<a href="https://colab.research.google.com/github/tbcreator71/Google-Colab-Notebooks/blob/GDCA_OCR_Notebooks/GDCA_Surya_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tbcreator71/Google-Colab-Notebooks/blob/GDCA_OCR_Notebooks/GDCA_Surya_GPU.ipynb)

# Notebook Version: v1.1.0-GDCA

**Updated:** 2026-11-15 | **Profile:** NVIDIA CUDA · Surya 2.5 Q8 · local VM inference · Drive checkpoints

Settings → Install → Public manifest → Models → Run → Benchmark → Export.

# OCR أرشيف العقارات العام — تجربة GPU ثم دفعات قابلة للاستئناف

**ملف محلي أُعدّ في 2026-11-15؛ لم يُشغّل على حساب Google أثناء إعداده.**

1. من **Runtime → Change runtime type** اختر NVIDIA GPU متاحاً بحسابك. يمكن تجربة A100 أو H100 إن ظهرا؛ النوع الفعلي والرصيد غير معلومين لهذا الدفتر.
2. شغّل الخلايا بالترتيب. الافتراضي **3 صفحات فقط**، دون تشغيل الأرشيف كله.
3. راجع النص والوقت، ثم عدّل الإعدادات لتكملة العينة أو دفعة كاملة.

يحتوي الدفتر على كود وروابط وبصمات الأرشيف الحكومي العام فقط: 1,522 ملفاً / 7,072 صفحة. لا يحتوي رواية القضية أو ملاحظاتك أو ملفات PDF. يجلب المستندات العامة إلى Google Colab عند التشغيل؛ هذه معالجة سحابية وليست محلية على اللابتوب.

النتائج غير مراجعة، ولا تثبت الملكية أو صحة محتوى الوثائق. درجة الثقة ليست نسبة دقة. لا تشارك الدفتر بعد إضافة مخرجات تحتوي بيانات تريد إبقاءها خاصة. حقوق النموذج: رخصة Datalab المرفقة بالمستودع الرسمي.


In [ ]:
#@title 1. Settings | الإعدادات
# الإعدادات: ابدأ بالعينة، ولا تفعل التشغيل الكامل قبل مراجعة القياس.
RUN_FULL = True  #@param {type:"boolean"}
MAX_PAGES = 7072  #@param {type:"integer"}
MAX_RUN_MINUTES = 720  #@param {type:"integer"}
PAGE_TIMEOUT_SECONDS = 900  #@param {type:"integer"}
DPI = 192  #@param [96, 128, 192, 300] {type:"raw"}
PERSIST_TO_DRIVE = True  #@param {type:"boolean"}
SAVE_PREVIEWS = True  #@param {type:"boolean"}
DRIVE_FOLDER = "GDCA-OCR-public-archive"  #@param {type:"string"}
assert isinstance(MAX_PAGES,int) and MAX_PAGES > 0
assert MAX_RUN_MINUTES > 0 and 60 <= PAGE_TIMEOUT_SECONDS <= 1800
assert 96 <= DPI <= 300


## 1 — اكتشاف GPU وتثبيت المحرك
يتوقف الدفتر إذا لم يجد NVIDIA؛ لا يعود إلى CPU بصمت. بناء CUDA الأول قد يستغرق عدة دقائق ويستهلك وقت الجلسة. لا Docker ولا خدمة OCR خارجية.

In [ ]:
#@title 2. Install CUDA runtime | تثبيت المحرك
import os, sys, json, subprocess, hashlib, time, shutil, socket, statistics
from pathlib import Path
from datetime import datetime, timezone
import urllib.request, urllib.parse
import threading, queue, signal

def run_step(label,cmd,timeout):
    print('▶ '+label,flush=True)
    started=time.monotonic()
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,start_new_session=True,
                             env={**os.environ,'PYTHONUNBUFFERED':'1','PIP_PROGRESS_BAR':'off'})
    lines=queue.Queue()
    def read_output():
        try:
            for line in process.stdout:lines.put(line.rstrip())
        finally:lines.put(None)
    reader=threading.Thread(target=read_output,daemon=True);reader.start()
    ended=False;heartbeat=time.monotonic()
    try:
        while not ended or process.poll() is None:
            elapsed=time.monotonic()-started
            if elapsed>timeout:raise TimeoutError(f'{label}: تجاوز المهلة ({timeout//60} دقيقة). راجع آخر رسائل الأمر قبل إعادة المحاولة.')
            try:
                line=lines.get(timeout=.5)
                if line is None:ended=True
                else:print(line,flush=True)
            except queue.Empty:pass
            if time.monotonic()-heartbeat>=15:
                print(f'⏳ {label} — مرّ {int(elapsed)} ثانية؛ العملية ما زالت تعمل (ليس تأكيداً لتقدّمها).',flush=True)
                heartbeat=time.monotonic()
        if process.returncode:raise RuntimeError(f'{label}: فشل الأمر برمز {process.returncode}؛ راجع السجل أعلاه.')
        print(f'✓ {label} — {time.monotonic()-started:.1f} ثانية',flush=True)
    finally:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=5)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
        reader.join(timeout=2)
        process.stdout.close()

WORK=Path('/content/gdca-ocr')
WORK.mkdir(parents=True,exist_ok=True)
GPU_INFO=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version,compute_cap','--format=csv,noheader'],text=True).strip()
print('GPU المتاح فعلياً:',GPU_INFO,flush=True)
if not GPU_INFO:raise RuntimeError('اختر GPU من إعدادات Runtime أولاً')
compute=GPU_INFO.splitlines()[0].split(',')[-1].strip().replace('.','')
if not compute.isdigit():raise RuntimeError('تعذر تحديد CUDA architecture')
if not shutil.which('nvcc'):raise Runtim12eError('CUDA toolkit غير متاح في هذه الجلسة؛ اختر جلسة GPU تدعم CUDA')
run_step('1/4 تثبيت Surya',[sys.executable,'-u','-m','pip','install','--disable-pip-version-check','--no-input','--timeout','60','--retries','2','surya-ocr==0.24.0'],600)
PACKAGES=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True,timeout=60)
LLAMA_COMMIT='dca216ad63c586f0db465324f785c5a418faef58'
source=WORK/'llama.cpp'
if not source.exists():run_step('2/4 تنزيل كود محرك CUDA',['git','clone','--progress','--depth','1','--branch','b12403','https://github.com/ggml-org/llama.cpp.git',str(source)],300)
else:print('2/4 كود المحرك موجود؛ التحقق من الإصدار',flush=True)
actual=subprocess.check_output(['git','-C',str(source),'rev-parse','HEAD'],text=True).strip()
assert actual==LLAMA_COMMIT,'Unexpected llama.cpp source revision'
build=source/('build-cuda-'+compute)
run_step('3/4 تهيئة بناء CUDA',['cmake','-S',str(source),'-B',str(build),'-DGGML_CUDA=ON','-DCMAKE_BUILD_TYPE=Release','-DBUILD_SHARED_LIBS=OFF','-DLLAMA_CURL=OFF','-DCMAKE_CUDA_ARCHITECTURES='+compute],300)
run_step('4/4 بناء محرك CUDA',['cmake','--build',str(build),'--config','Release','--target','llama-server','-j',str(min(4,os.cpu_count() or 2))],5400)
BINARY=build/'bin/llama-server'
assert BINARY.is_file()
print('محرك CUDA جاهز')


GPU المتاح فعلياً: Tesla T4, 15360 MiB, 580.82.07, 7.5
▶ 1/4 تثبيت Surya
✓ 1/4 تثبيت Surya — 1.4 ثانية
2/4 كود المحرك موجود؛ التحقق من الإصدار
▶ 3/4 تهيئة بناء CUDA
-- llama.cpp version: 1.0.0-dev
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native
-- CUDA Toolkit found
-- Using CMAKE_CUDA_ARCHITECTURES=75 CMAKE_CUDA_ARCHITECTURES_NATIVE=75-real
-- FlashAttention K-V type combinations: f16-f16;q4_0-q4_0;q8_0-q8_0;bf16-bf16;f8_e4m3-f8_e4m3
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 1.0.0
-- ggml commit:  dca216a
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
-- Configuring done (0.4s)
-- Generating done (0.7s)
-- Build files have been written to: /content/gdca-ocr/llama.c

## 2 — بيانات الأرشيف العامة ومحرك الصفحة
هذه الخلية تتضمن فهرساً تقنياً عاماً؛ لا ترفع مستودع القضية أو تربط GitHub الخاص به.

In [ ]:
#@title 3. Public archive manifest | الفهرس العام
DOCUMENTS = [{'source_id': 'GDCA-F-bd52dab60901e8f15f72', 'source_path': 'sources/gdca-recon/2026-11-14/files/bd52dab60901e8f15f72_242.pdf', 'source_sha256': '2fcb4837d8a30d4d0c325de606fbcbb440548441b46505820fa0c5cd39fe5753', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/242.pdf', 'bytes': 2076097}, {'source_id': 'GDCA-F-f2de3362e20d8837cc50', 'source_path': 'sources/gdca-recon/2026-11-14/files/f2de3362e20d8837cc50_665.pdf', 'source_sha256': '60814996ed140d4060686cc3ab2952114bb9ffa9418c8d42b5b5dcd5754d63dc', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/665.pdf', 'bytes': 2400942}, {'source_id': 'GDCA-F-f92a4b22a10eea59b585', 'source_path': 'sources/gdca-recon/2026-11-14/files/f92a4b22a10eea59b585_638.pdf', 'source_sha256': '44f4af2708baec086a55a9c84a88f313ecc1e4eec7e51497eaad8d10830e6dc8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/638.pdf', 'bytes': 1777510}, {'source_id': 'GDCA-F-8b65853a9982e592453a', 'source_path': 'sources/gdca-recon/2026-11-14/files/8b65853a9982e592453a_46.pdf', 'source_sha256': '575b871ad31a7e72b58b6438b47601598f9b5a79724fff46e743e3e7284a5483', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/46.pdf', 'bytes': 96286}, {'source_id': 'GDCA-F-da9491212680a094a0bb', 'source_path': 'sources/gdca-recon/2026-11-14/files/da9491212680a094a0bb_335.pdf', 'source_sha256': 'a6adc03a3ebc50a2fc761f5293fbdc51257c2be756bc726bd6bb50f1d51bc0cd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/335.pdf', 'bytes': 2232909}, {'source_id': 'GDCA-F-62733f6e8c01a80c8985', 'source_path': 'sources/gdca-recon/2026-11-14/files/62733f6e8c01a80c8985_1188.pdf', 'source_sha256': '6fb6969a928698d531faa11549d74af441a50b2da2a6d6c3c4311ba751bd47e4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1188.pdf', 'bytes': 1733024}, {'source_id': 'GDCA-F-750d307df8fc1c2a861b', 'source_path': 'sources/gdca-recon/2026-11-14/files/750d307df8fc1c2a861b_1059.pdf', 'source_sha256': '595ba5312665f80a2c39f52981db1ad7d6eebd5c6150d5a8dae364fa6c4d9c28', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1059.pdf', 'bytes': 2106532}, {'source_id': 'GDCA-F-e95692af8c2e2d36a51f', 'source_path': 'sources/gdca-recon/2026-11-14/files/e95692af8c2e2d36a51f_637.pdf', 'source_sha256': '57b79974e48eae87cf0b10c885282cee5198adcd74188465b29aed66fcbbd975', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/637.pdf', 'bytes': 2013382}, {'source_id': 'GDCA-F-2b8cb5803b310fac5487', 'source_path': 'sources/gdca-recon/2026-11-14/files/2b8cb5803b310fac5487_494.pdf', 'source_sha256': '0d2c87a5323a8f7a2c738fe96f175c7fd1559eb6bddcb40dbb8b67cfdfce6a97', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/494.pdf', 'bytes': 1829579}, {'source_id': 'GDCA-F-3439145fed34bbb5d6de', 'source_path': 'sources/gdca-recon/2026-11-14/files/3439145fed34bbb5d6de_1384.pdf', 'source_sha256': '16401ad8f81bf7bf78c1095960998dc2357deede059d9266f89308d8ecfda7f3', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1384.pdf', 'bytes': 2838683}, {'source_id': 'GDCA-F-8ea2e033d81e81994d54', 'source_path': 'sources/gdca-recon/2026-11-14/files/8ea2e033d81e81994d54_1057.pdf', 'source_sha256': '8e6801dbef28b3a179fbfa7051dbc35e94dd9d1a7f92f6766a9ab02dbfdbe14d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1057.pdf', 'bytes': 1926038}, {'source_id': 'GDCA-F-91fb5fb68bf36becbfd8', 'source_path': 'sources/gdca-recon/2026-11-14/files/91fb5fb68bf36becbfd8_199.pdf', 'source_sha256': '664e447cb05bdd5c7f4480376ab9e76aa42e7c217dcc6e1f22c428d50845cff2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/199.pdf', 'bytes': 2282267}, {'source_id': 'GDCA-F-e69339e4df7b76639cfa', 'source_path': 'sources/gdca-recon/2026-11-14/files/e69339e4df7b76639cfa_924.pdf', 'source_sha256': '3d1d4b82a2e138cfc02a09226a1ce63b6a32370cea47b4498d94ae3c65ff97e2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/924.pdf', 'bytes': 2123702}, {'source_id': 'GDCA-F-1ade0c5921c450f86a7c', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ade0c5921c450f86a7c_88.pdf', 'source_sha256': '740d0a79471e391d0bb2284213937fa2be8b1091bffd9b153822cc2c9029e8f3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/88.pdf', 'bytes': 1853402}, {'source_id': 'GDCA-F-12b5ec6af510833e03b8', 'source_path': 'sources/gdca-recon/2026-11-14/files/12b5ec6af510833e03b8_116.pdf', 'source_sha256': 'f6fb80052d511fcc791777fb6f87bb9ff6c5ec5b3965e77f723c0739fb501ad5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/116.pdf', 'bytes': 1988792}, {'source_id': 'GDCA-F-91e15ce78a85316f9b40', 'source_path': 'sources/gdca-recon/2026-11-14/files/91e15ce78a85316f9b40_619.pdf', 'source_sha256': '5e0cff86a8df8ce8de5a4ca6c45cc3a6af0956ed00aa307ef6c2de35f24d0f7a', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/619.pdf', 'bytes': 2078515}, {'source_id': 'GDCA-F-d51a5c98d3607e36854c', 'source_path': 'sources/gdca-recon/2026-11-14/files/d51a5c98d3607e36854c_187.pdf', 'source_sha256': '4ea815ced7dfea4314c92394911aa2f46fb964b9f215f516058e2662e49fc725', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/187.pdf', 'bytes': 2207310}, {'source_id': 'GDCA-F-703019bc6f6e8c0603e4', 'source_path': 'sources/gdca-recon/2026-11-14/files/703019bc6f6e8c0603e4_394.pdf', 'source_sha256': '4428cb6211a3ae0cf80563734e50ac8e0b6ecafccdd8f2a06a974e68479a005c', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/394.pdf', 'bytes': 2359603}, {'source_id': 'GDCA-F-b4adb4616b3a769f344a', 'source_path': 'sources/gdca-recon/2026-11-14/files/b4adb4616b3a769f344a_965.pdf', 'source_sha256': 'e08da12f5d36064188e3d2fffa68ddea0d88a24cbec439163b5ce26f829bf7b4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/965.pdf', 'bytes': 1875326}, {'source_id': 'GDCA-F-a411fd2178e250ec50b5', 'source_path': 'sources/gdca-recon/2026-11-14/files/a411fd2178e250ec50b5_377.pdf', 'source_sha256': 'cb5a65eb6515e99c33b835a261dc25d5b0639888dde4816d20e3ab7764f53f64', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/377.pdf', 'bytes': 2555290}, {'source_id': 'GDCA-F-3b4f5ddb207a3aad414b', 'source_path': 'sources/gdca-recon/2026-11-14/files/3b4f5ddb207a3aad414b_896.pdf', 'source_sha256': '57f92671ebd784a72e83a997134d56694040f5cad80b4ca42367bb77a4226b1f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/896.pdf', 'bytes': 2117399}, {'source_id': 'GDCA-F-2ded4d8b1f35e056a167', 'source_path': 'sources/gdca-recon/2026-11-14/files/2ded4d8b1f35e056a167_1260.pdf', 'source_sha256': 'efe06895ffb3d97a56ea7aeeee0541da1b665d878ed394627e2c1da1fbdb3df9', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1260.pdf', 'bytes': 1463094}, {'source_id': 'GDCA-F-5ee6bfeb285f0f2b15b4', 'source_path': 'sources/gdca-recon/2026-11-14/files/5ee6bfeb285f0f2b15b4_1002.pdf', 'source_sha256': '55ac0ad24f4977c961e7d76e614f89195bd7090fefcbcb6e71d77f4e2e5907d3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1002.pdf', 'bytes': 1816715}, {'source_id': 'GDCA-F-5259750821cb0d7f7df1', 'source_path': 'sources/gdca-recon/2026-11-14/files/5259750821cb0d7f7df1_1373.pdf', 'source_sha256': '193cac5c8343ff647d7621e2ed502df0a7afcf8c8b4f78a71c8471dcf1e8ca1f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1373.pdf', 'bytes': 2324736}, {'source_id': 'GDCA-F-d6bb309f255a1823533a', 'source_path': 'sources/gdca-recon/2026-11-14/files/d6bb309f255a1823533a_1173.pdf', 'source_sha256': '097f7294a50f1e61f563981123ddf00586efe03b3021f44ee63c8d5355af33ea', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1173.pdf', 'bytes': 2014093}, {'source_id': 'GDCA-F-afb2a247423243d82a7e', 'source_path': 'sources/gdca-recon/2026-11-14/files/afb2a247423243d82a7e_1466.pdf', 'source_sha256': 'dc6200aaf777f7d4ccdb318bc0fd7d0d19bd1c8fc576c2b86b74abd51864562e', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1466.pdf', 'bytes': 3212588}, {'source_id': 'GDCA-F-cbb08cafa3a1b09bd063', 'source_path': 'sources/gdca-recon/2026-11-14/files/cbb08cafa3a1b09bd063_709.pdf', 'source_sha256': 'f49d9b92288bb78ebde6a42dad184a2ec3f66acb119d4d90803fe2b4da792cbe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/709.pdf', 'bytes': 2512046}, {'source_id': 'GDCA-F-614fadf298d9085a4b0d', 'source_path': 'sources/gdca-recon/2026-11-14/files/614fadf298d9085a4b0d_935.pdf', 'source_sha256': 'b943ea1062dd1cd6af90335be36e8a36a661af2813f6ec3dc87026a6b2604fed', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/935.pdf', 'bytes': 1263241}, {'source_id': 'GDCA-F-7eedf1549a7edd2c8d2c', 'source_path': 'sources/gdca-recon/2026-11-14/files/7eedf1549a7edd2c8d2c_1483.pdf', 'source_sha256': '3fa93b3d2ebe2dce83f393fc70231964cf01649b0aeae562cfcb911924412f32', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1483.pdf', 'bytes': 2192307}, {'source_id': 'GDCA-F-8883d5feb210ff33a08c', 'source_path': 'sources/gdca-recon/2026-11-14/files/8883d5feb210ff33a08c_1268.pdf', 'source_sha256': 'f192332fc8315dfed8cb2626096ae7d1a82e5efb700b3291129f9d4fde70bb92', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1268.pdf', 'bytes': 2194604}, {'source_id': 'GDCA-F-4ab7a1452be3a3ef0da4', 'source_path': 'sources/gdca-recon/2026-11-14/files/4ab7a1452be3a3ef0da4_41.pdf', 'source_sha256': '87eda9573f2c6bd9e5556246f2506f2d4a290ba627f290b8389a158178fa22ba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/41.pdf', 'bytes': 97887}, {'source_id': 'GDCA-F-fbd56cfff27f95915619', 'source_path': 'sources/gdca-recon/2026-11-14/files/fbd56cfff27f95915619_34.pdf', 'source_sha256': '23bb70c7da48df4b612695493749e40a8c3a42ec7f5e8ba136b74239138497c2', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/34.pdf', 'bytes': 110518}, {'source_id': 'GDCA-F-f73e4b8e9a7f32b5e9b3', 'source_path': 'sources/gdca-recon/2026-11-14/files/f73e4b8e9a7f32b5e9b3_1080.pdf', 'source_sha256': 'f3ff2be99edfd185a4809399576c25b36ec26b15c5a6edcfe4ccf4e79ccf4ec5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1080.pdf', 'bytes': 1903343}, {'source_id': 'GDCA-F-b8bec33608e05397b6c3', 'source_path': 'sources/gdca-recon/2026-11-14/files/b8bec33608e05397b6c3_696.pdf', 'source_sha256': 'f6b8a5e990a5f0336e2eb5bf1988dd908dc31328788dc53ec7c0c70bca70ffec', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/696.pdf', 'bytes': 2236985}, {'source_id': 'GDCA-F-84f3672590dcc8eed9c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/84f3672590dcc8eed9c1_1033.pdf', 'source_sha256': '032c583eafdbcc25c7b5131d44d69f94e03b2007ea413cfa2761cedb118bb739', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1033.pdf', 'bytes': 1922510}, {'source_id': 'GDCA-F-1318272b9efcc55890f7', 'source_path': 'sources/gdca-recon/2026-11-14/files/1318272b9efcc55890f7_1010.pdf', 'source_sha256': '4ba6210c4e3c327d5f0047369656c585ac6cc030cbebde4865e7b126c713a322', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1010.pdf', 'bytes': 1987392}, {'source_id': 'GDCA-F-d16d1a6d264bb689b614', 'source_path': 'sources/gdca-recon/2026-11-14/files/d16d1a6d264bb689b614_947.pdf', 'source_sha256': '7b46fd22c199603053abaf0cd5322e4ea749c37ba331c2b4de0260810bd654c4', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/947.pdf', 'bytes': 2981556}, {'source_id': 'GDCA-F-134286b060ec1c414535', 'source_path': 'sources/gdca-recon/2026-11-14/files/134286b060ec1c414535_879.pdf', 'source_sha256': '321a4429cc2748df9ffff0dfdb23714219187eaeaff770fa7f2784ace1db82c6', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/879.pdf', 'bytes': 2339737}, {'source_id': 'GDCA-F-23e121844b163f6923d7', 'source_path': 'sources/gdca-recon/2026-11-14/files/23e121844b163f6923d7_755.pdf', 'source_sha256': '3f7a2ec46fe6d99a4c447b619a2ec10c16461887f6527f4228de2a6a015395ee', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/755.pdf', 'bytes': 2182286}, {'source_id': 'GDCA-F-9ae636184eb697df79f5', 'source_path': 'sources/gdca-recon/2026-11-14/files/9ae636184eb697df79f5_1022.pdf', 'source_sha256': '4d2c2d7e21e2c049f40db67e91d928ea214752da054f3bf14ffadc147ebaab1c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1022.pdf', 'bytes': 2251662}, {'source_id': 'GDCA-F-13c1d2503f8df4aacd2c', 'source_path': 'sources/gdca-recon/2026-11-14/files/13c1d2503f8df4aacd2c_1420.pdf', 'source_sha256': '781be095fbcf9158c4ce4617acede1485d1a4544e7d1bba4d961ed53834a1b68', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1420.pdf', 'bytes': 2731145}, {'source_id': 'GDCA-F-8fb2915b991f77d5ecdc', 'source_path': 'sources/gdca-recon/2026-11-14/files/8fb2915b991f77d5ecdc_1587.pdf', 'source_sha256': '632d02f33f09821fdfa2c6af039920408d43fa42928087d890f22cca68724149', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1587.pdf', 'bytes': 2036715}, {'source_id': 'GDCA-F-17bc096b8a12abf3de5c', 'source_path': 'sources/gdca-recon/2026-11-14/files/17bc096b8a12abf3de5c_19.pdf', 'source_sha256': 'd742ad54d3ac11080761880dc3ee1dfe2ea0a1fe2f234e8b82cc0ed3f14a3fc0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/19.pdf', 'bytes': 2106302}, {'source_id': 'GDCA-F-189c24780bdd7718f111', 'source_path': 'sources/gdca-recon/2026-11-14/files/189c24780bdd7718f111_1487.pdf', 'source_sha256': '3ec5ec6014bff89dda7d8bd37924e17bd58de0a8a39b07584fef45d5cc2b5c03', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1487.pdf', 'bytes': 2212530}, {'source_id': 'GDCA-F-08dc391926ac2555a9a7', 'source_path': 'sources/gdca-recon/2026-11-14/files/08dc391926ac2555a9a7_989.pdf', 'source_sha256': '0f2ef4bafad33c8bbded985dc66403cb60c965252636d0349d1d1ece277c18f2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/989.pdf', 'bytes': 2056128}, {'source_id': 'GDCA-F-26f0110228ca1d760568', 'source_path': 'sources/gdca-recon/2026-11-14/files/26f0110228ca1d760568_972.pdf', 'source_sha256': '0129393caacc592ad3c7e261251f41d19b66dbd7b62d388a042567b127d4f873', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/972.pdf', 'bytes': 2116626}, {'source_id': 'GDCA-F-6cd829f531290b407ba3', 'source_path': 'sources/gdca-recon/2026-11-14/files/6cd829f531290b407ba3_391.pdf', 'source_sha256': '191e58d461cef2578ff6d534c4524867ba77a0d513ceca8ee823a7fce4624f6e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/391.pdf', 'bytes': 2116908}, {'source_id': 'GDCA-F-8e453cd7cefd5882f615', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e453cd7cefd5882f615_679.pdf', 'source_sha256': 'af1ed3a4fbcfc670724ec81c5ba4ea50324d7a7bf6f5aa1e4a6583f10fcd0932', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/679.pdf', 'bytes': 2059193}, {'source_id': 'GDCA-F-f4afddb0c510931eeaf0', 'source_path': 'sources/gdca-recon/2026-11-14/files/f4afddb0c510931eeaf0_523.pdf', 'source_sha256': 'a33a4d0422e9dba62d271d4446ff645772d7a9db6df24f76072d2357bcc4414c', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/523.pdf', 'bytes': 2218431}, {'source_id': 'GDCA-F-2d0babf290189898f192', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d0babf290189898f192_325.pdf', 'source_sha256': '2c6236ee56147dae6af860187ac62bf6fe29f7819d32e6586ca760d7975dfb0f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/325.pdf', 'bytes': 1998537}, {'source_id': 'GDCA-F-143813910c056f92b811', 'source_path': 'sources/gdca-recon/2026-11-14/files/143813910c056f92b811_827.pdf', 'source_sha256': 'af48223ec7aa78e03bfe9ab5c24451e057fa6d74f7ada37dbb9dbafaca13d2d5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/827.pdf', 'bytes': 1672187}, {'source_id': 'GDCA-F-cf3e3d30c2c3dd7761cc', 'source_path': 'sources/gdca-recon/2026-11-14/files/cf3e3d30c2c3dd7761cc_1146.pdf', 'source_sha256': '30a44fafd8801297118e0a01777707ac69395e49c0da856cab118837e0a8db02', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1146.pdf', 'bytes': 2172009}, {'source_id': 'GDCA-F-8017b8c6aaf9a81dd16b', 'source_path': 'sources/gdca-recon/2026-11-14/files/8017b8c6aaf9a81dd16b_1578.pdf', 'source_sha256': 'f1282453e08805b424ec3934ec2dae1fb41f59490f9cc5150db997975a3ecaa2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1578.pdf', 'bytes': 2095993}, {'source_id': 'GDCA-F-e2c8a89c6ff5b7d7f50e', 'source_path': 'sources/gdca-recon/2026-11-14/files/e2c8a89c6ff5b7d7f50e_406.pdf', 'source_sha256': 'd3f2fd37027df528b3216311bb1207c940d559cb876c52ed3c575eee5be7161e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/406.pdf', 'bytes': 2546372}, {'source_id': 'GDCA-F-c17b41688a3c95a18f42', 'source_path': 'sources/gdca-recon/2026-11-14/files/c17b41688a3c95a18f42_1486.pdf', 'source_sha256': 'c5d1cf9ba3363a7fb9ddb3e6b5f3a8a10cc62fc52afdfb78f1472309a02b16e3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1486.pdf', 'bytes': 2200946}, {'source_id': 'GDCA-F-179c0712053a3ffa066e', 'source_path': 'sources/gdca-recon/2026-11-14/files/179c0712053a3ffa066e_708.pdf', 'source_sha256': 'c8c5401f81dd19d102ee6b3c77b1fd8ef39268c72fb96e7dc1f9f7bb05f7ebff', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/708.pdf', 'bytes': 1861356}, {'source_id': 'GDCA-F-9acd3eba2d87c3a088c8', 'source_path': 'sources/gdca-recon/2026-11-14/files/9acd3eba2d87c3a088c8_521.pdf', 'source_sha256': '506b6837be37d0d133c1165e2657bd008723d8e1ae7f25a3c2a11fbea6ef723c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/521.pdf', 'bytes': 2076131}, {'source_id': 'GDCA-F-6390d51ec17dd812af21', 'source_path': 'sources/gdca-recon/2026-11-14/files/6390d51ec17dd812af21_1374.pdf', 'source_sha256': '6e70023c557234cd7ed4fc1b92d3845a94b0125d72f06b28a030fe16184e6036', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1374.pdf', 'bytes': 2613495}, {'source_id': 'GDCA-F-9a150ac253b441f50ea5', 'source_path': 'sources/gdca-recon/2026-11-14/files/9a150ac253b441f50ea5_1419.pdf', 'source_sha256': '07914bf7b3f15c3c0c0bd2c6b74620629e72c9c90834d07fb5de08c67d34c0be', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1419.pdf', 'bytes': 2655136}, {'source_id': 'GDCA-F-8c7681e0db32514cf1e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c7681e0db32514cf1e5_411.pdf', 'source_sha256': '2531f9e866a9dfef070de7545ecfde18d9d9b0925a953444460ca7538b74058f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/411.pdf', 'bytes': 1543662}, {'source_id': 'GDCA-F-aaeca29d383da48aaa01', 'source_path': 'sources/gdca-recon/2026-11-14/files/aaeca29d383da48aaa01_1266.pdf', 'source_sha256': '59a09b86b1a308701a4aac3e5786e7eac2c610996a3e7b1a19f89636764bef3c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1266.pdf', 'bytes': 2181402}, {'source_id': 'GDCA-F-0baa2a6a912cd7792218', 'source_path': 'sources/gdca-recon/2026-11-14/files/0baa2a6a912cd7792218_1226.pdf', 'source_sha256': '7a48ccf0ca0236f7d6a18a0eef498e8ce917839aad62029365900d29a4359c5a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1226.pdf', 'bytes': 2051498}, {'source_id': 'GDCA-F-71f73533507d35e99970', 'source_path': 'sources/gdca-recon/2026-11-14/files/71f73533507d35e99970_507.pdf', 'source_sha256': 'a6cf2de0def57c7864bd7054451eb9ca1fd0e7d77b9cac0ea63d020194f4b85b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/507.pdf', 'bytes': 2028360}, {'source_id': 'GDCA-F-afaa7ab1c0e31a1134a5', 'source_path': 'sources/gdca-recon/2026-11-14/files/afaa7ab1c0e31a1134a5_248.pdf', 'source_sha256': 'dde3e6b0a4ac961e78b85fb20e9d491c764b51f5eb605b271ad2fa6ecb678196', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/248.pdf', 'bytes': 1624416}, {'source_id': 'GDCA-F-1ed92a931f5c49b662af', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ed92a931f5c49b662af_861.pdf', 'source_sha256': '0bd03f2a8f6480b3e5348387cfc6edbedc203caaca6548594006d194bcd26097', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/861.pdf', 'bytes': 2279870}, {'source_id': 'GDCA-F-cf02d68178872c242234', 'source_path': 'sources/gdca-recon/2026-11-14/files/cf02d68178872c242234_1292.pdf', 'source_sha256': '311567ee8cd7b0bed80998c44b265e5065117f2525e858907346fee2d913f0d6', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1292.pdf', 'bytes': 2523160}, {'source_id': 'GDCA-F-7de25f02f7ae37069886', 'source_path': 'sources/gdca-recon/2026-11-14/files/7de25f02f7ae37069886_1205.pdf', 'source_sha256': 'b86caa1989a1f7fdeca23d63b883c42f717b37d3f12f562698393ccac2fd4ca2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1205.pdf', 'bytes': 1721143}, {'source_id': 'GDCA-F-f3e72864b0a8aa7291a1', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3e72864b0a8aa7291a1_1039.pdf', 'source_sha256': '0e6dd5b21ac248060e35df173d1a6066152158dde950ff968e9d5a700e374424', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1039.pdf', 'bytes': 2238749}, {'source_id': 'GDCA-F-b82f98a0dc29b564678c', 'source_path': 'sources/gdca-recon/2026-11-14/files/b82f98a0dc29b564678c_35.pdf', 'source_sha256': 'ccd5c5ae9ec3d3eee068e77cd55dec1f8b32e44870512d0b0dc09632af727e6e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/35.pdf', 'bytes': 107296}, {'source_id': 'GDCA-F-5adf03f52f620e3b4ba7', 'source_path': 'sources/gdca-recon/2026-11-14/files/5adf03f52f620e3b4ba7_252.pdf', 'source_sha256': '08dabc936e55b1c876084b1795013367d6099468d37789d67d38ec7867f1e10f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/252.pdf', 'bytes': 2098527}, {'source_id': 'GDCA-F-8e6525f89857c95dddeb', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e6525f89857c95dddeb_234.pdf', 'source_sha256': '566e415f3ed4148ab485dfe80e21968ae3088d2dd3d1f648a30c295d48ae74bd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/234.pdf', 'bytes': 2338228}, {'source_id': 'GDCA-F-0e780117edeb0002a3b6', 'source_path': 'sources/gdca-recon/2026-11-14/files/0e780117edeb0002a3b6_532.pdf', 'source_sha256': '5c096b322a6f6ebb75ad84b2914297f40a0484d0203c269af06d9679ee845aaa', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/532.pdf', 'bytes': 1966938}, {'source_id': 'GDCA-F-80a5f552289ff259af7b', 'source_path': 'sources/gdca-recon/2026-11-14/files/80a5f552289ff259af7b_572.pdf', 'source_sha256': '683008f37aeae0001abfa86c65615242e48d3918ec294fd280eafc6de2977d9d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/572.pdf', 'bytes': 2359720}, {'source_id': 'GDCA-F-f7aaa45ec56d50c23140', 'source_path': 'sources/gdca-recon/2026-11-14/files/f7aaa45ec56d50c23140_1278.pdf', 'source_sha256': '27a3adedbbc87e69d445eadd87828e71ff80a74c566ae11f1183d81dc9af8796', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1278.pdf', 'bytes': 1850622}, {'source_id': 'GDCA-F-fd1bdbe481b9607704f8', 'source_path': 'sources/gdca-recon/2026-11-14/files/fd1bdbe481b9607704f8_913.pdf', 'source_sha256': '0bdc9e73172cb5054844f48c62d49af6f99c95c834cc7ce070b786b79bd0b599', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/913.pdf', 'bytes': 1820817}, {'source_id': 'GDCA-F-c14dccb847fcab6a8bec', 'source_path': 'sources/gdca-recon/2026-11-14/files/c14dccb847fcab6a8bec_1139.pdf', 'source_sha256': 'b98c9b2e2d882d45dd08f1f40c53035cda4998faa1dca00244ea6dcfb8b528f3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1139.pdf', 'bytes': 1737449}, {'source_id': 'GDCA-F-a7dc59e809cd95a58042', 'source_path': 'sources/gdca-recon/2026-11-14/files/a7dc59e809cd95a58042_1192.pdf', 'source_sha256': '380d5692c7aa2a8898ce528a169532272aca594de6762bf63e036db8542bba8c', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1192.pdf', 'bytes': 2324325}, {'source_id': 'GDCA-F-2bdc205e14d508364c0a', 'source_path': 'sources/gdca-recon/2026-11-14/files/2bdc205e14d508364c0a_1497.pdf', 'source_sha256': 'ca4829fd6beffbbf2b191acdbd35de4dde838a92a925192ea9c41203bd17a93c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1497.pdf', 'bytes': 1943729}, {'source_id': 'GDCA-F-0511ce32c89854c3aff4', 'source_path': 'sources/gdca-recon/2026-11-14/files/0511ce32c89854c3aff4_916.pdf', 'source_sha256': 'cd550920b97cf057667185d4d2682887dc54e8fa33ae371ce6d43bd225c742cd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/916.pdf', 'bytes': 1752304}, {'source_id': 'GDCA-F-49a4bc961d88b38e9335', 'source_path': 'sources/gdca-recon/2026-11-14/files/49a4bc961d88b38e9335_1568.pdf', 'source_sha256': '624baf902cb067a07b6532e3e25aea53a351267562313a7f1b3d1c09896475c3', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1568.pdf', 'bytes': 3424232}, {'source_id': 'GDCA-F-ea209443f5b5dd7b345c', 'source_path': 'sources/gdca-recon/2026-11-14/files/ea209443f5b5dd7b345c_146.pdf', 'source_sha256': '835ed5b80fef22a4b4033033c32fb2f02b3139ad8fff071446e8e560f90a960a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/146.pdf', 'bytes': 1932078}, {'source_id': 'GDCA-F-9589ba1cfd4d4777e8d2', 'source_path': 'sources/gdca-recon/2026-11-14/files/9589ba1cfd4d4777e8d2_1147.pdf', 'source_sha256': 'c46c9371e323ef07585b527dd484e85824c45e4bb1e62379aac1cef92f4d6287', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1147.pdf', 'bytes': 2056318}, {'source_id': 'GDCA-F-834592e5442d6095f386', 'source_path': 'sources/gdca-recon/2026-11-14/files/834592e5442d6095f386_1426.pdf', 'source_sha256': 'f6b95c0ede0328cbb4918181380a3d0980140400e71137f774087a8d0a1da26f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1426.pdf', 'bytes': 2194792}, {'source_id': 'GDCA-F-15228d83cec4adf00899', 'source_path': 'sources/gdca-recon/2026-11-14/files/15228d83cec4adf00899_233.pdf', 'source_sha256': '8b58b2b324c5385ac5afd497a7db81e019e63a29549f97b75a0589b4c3c7cb88', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/233.pdf', 'bytes': 2126856}, {'source_id': 'GDCA-F-b43da359c2714eb00ea0', 'source_path': 'sources/gdca-recon/2026-11-14/files/b43da359c2714eb00ea0_937.pdf', 'source_sha256': '848c645fd7dd801581da8aa4cd6853e974d02219adc3f730c9e0527f08e2eec8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/937.pdf', 'bytes': 2099707}, {'source_id': 'GDCA-F-83d0ef3ae882c2693087', 'source_path': 'sources/gdca-recon/2026-11-14/files/83d0ef3ae882c2693087_836.pdf', 'source_sha256': 'dca913e539ab573483cc49d0714e66e7cbad5534485a192925ec7c29d150d592', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/836.pdf', 'bytes': 1777958}, {'source_id': 'GDCA-F-625615f0f8dbbe77efb2', 'source_path': 'sources/gdca-recon/2026-11-14/files/625615f0f8dbbe77efb2_343.pdf', 'source_sha256': 'b4081380530bec888d5f06f0d1a2a6fb88021e415c5bb62243c073541c5347ba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/343.pdf', 'bytes': 1838276}, {'source_id': 'GDCA-F-26654506020c76ab624a', 'source_path': 'sources/gdca-recon/2026-11-14/files/26654506020c76ab624a_955.pdf', 'source_sha256': '59858af35d6ffc85488c0ffb8527da8fe842fc0c6bdaf8a7ed78f43ab1be2b2b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/955.pdf', 'bytes': 1853893}, {'source_id': 'GDCA-F-27af5038ece250e745bc', 'source_path': 'sources/gdca-recon/2026-11-14/files/27af5038ece250e745bc_568.pdf', 'source_sha256': '29aa9df12e3c03f9bd68724275efafe67103b5220fc52522bddffdf0bfaeec1f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/568.pdf', 'bytes': 1704006}, {'source_id': 'GDCA-F-d13e3be3f3e7a4400298', 'source_path': 'sources/gdca-recon/2026-11-14/files/d13e3be3f3e7a4400298_529.pdf', 'source_sha256': 'e5c8a7e407b8e0de45ca5c3e1a8018e41273a76d816bf5c9f1fb2a29cfae5986', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/529.pdf', 'bytes': 1332061}, {'source_id': 'GDCA-F-d15f54065a77ae59f252', 'source_path': 'sources/gdca-recon/2026-11-14/files/d15f54065a77ae59f252_734.pdf', 'source_sha256': '3d07d52fb3c214876f60087652c23b51d42de004b8e9fccc09b3e5264cea90e7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/734.pdf', 'bytes': 2005224}, {'source_id': 'GDCA-F-59c876d919ae0b6da6f7', 'source_path': 'sources/gdca-recon/2026-11-14/files/59c876d919ae0b6da6f7_173.pdf', 'source_sha256': 'b3ae1aa3540203ffb801bd608e6bea1b710f7e5225b4436ea7e146cfbfbf8a3d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/173.pdf', 'bytes': 1914882}, {'source_id': 'GDCA-F-e80e17c5419778d84756', 'source_path': 'sources/gdca-recon/2026-11-14/files/e80e17c5419778d84756_936.pdf', 'source_sha256': '12a4694ce8e5c99b9c401005b78915ddb446a5a346d6aa5673023af6233dd1f9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/936.pdf', 'bytes': 2021485}, {'source_id': 'GDCA-F-9ce35351498150c5b097', 'source_path': 'sources/gdca-recon/2026-11-14/files/9ce35351498150c5b097_1444.pdf', 'source_sha256': '3d5003a417c17a4ea3e45c844d1ab70ee12be7703a7de698c1434a1f053cc08a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1444.pdf', 'bytes': 2747002}, {'source_id': 'GDCA-F-6d7fb549b14f8c06a149', 'source_path': 'sources/gdca-recon/2026-11-14/files/6d7fb549b14f8c06a149_220.pdf', 'source_sha256': '82a7a161d8793c6491decbf7a167552af119b5078d234223f07676fa2a034d68', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/220.pdf', 'bytes': 1829619}, {'source_id': 'GDCA-F-94d03aa63b19a6b857f7', 'source_path': 'sources/gdca-recon/2026-11-14/files/94d03aa63b19a6b857f7_1382.pdf', 'source_sha256': 'ed3d9c1baf9d36a57bdc262621d27218ce395bbec2a1acf24208e3c8fa3204b4', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1382.pdf', 'bytes': 4164729}, {'source_id': 'GDCA-F-b65627ce5092e2fb9bfe', 'source_path': 'sources/gdca-recon/2026-11-14/files/b65627ce5092e2fb9bfe_634.pdf', 'source_sha256': 'ace899818d581f8f795c4d7669595d3a02681376be6f77d78c96f4c2e8d9d9da', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/634.pdf', 'bytes': 1699115}, {'source_id': 'GDCA-F-176690617d4b7bcd123e', 'source_path': 'sources/gdca-recon/2026-11-14/files/176690617d4b7bcd123e_68.pdf', 'source_sha256': '880fdcbfbdcc555bf126ccd2bff002640de301bfe95e0ca50db83379b7015763', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/68.pdf', 'bytes': 2113854}, {'source_id': 'GDCA-F-4dc9da600b2e408f057e', 'source_path': 'sources/gdca-recon/2026-11-14/files/4dc9da600b2e408f057e_273.pdf', 'source_sha256': 'e2a5e1b84189506d1f3d48fbff14ea65105bb84da2d2d07a79ad06e2e94922bb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/273.pdf', 'bytes': 2228705}, {'source_id': 'GDCA-F-ab3f61d5076f59acd6a2', 'source_path': 'sources/gdca-recon/2026-11-14/files/ab3f61d5076f59acd6a2_109.pdf', 'source_sha256': 'ab80e22ceb46b92a02a91957910376c6a7df3a616dd2f2eac863b67e95c8857d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/109.pdf', 'bytes': 1968413}, {'source_id': 'GDCA-F-6bd5bf60e1bae9316130', 'source_path': 'sources/gdca-recon/2026-11-14/files/6bd5bf60e1bae9316130_274.pdf', 'source_sha256': '6260ec870a4f7191316f8ed124abd6d8069f658dda86ca7210eb71ce439ec7ca', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/274.pdf', 'bytes': 2636097}, {'source_id': 'GDCA-F-1e8ca42895a41d9c391b', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e8ca42895a41d9c391b_1197.pdf', 'source_sha256': 'b1f346ab983be0426f09f4d0473a9c28c6281209044923e8b7f11e0da30412ac', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1197.pdf', 'bytes': 2300260}, {'source_id': 'GDCA-F-d29d42a70f0e1254cb70', 'source_path': 'sources/gdca-recon/2026-11-14/files/d29d42a70f0e1254cb70_919.pdf', 'source_sha256': 'bcb5a835171346fa41e7de54dd0fdd7de4089d7e66d4939a1df6a17a74aebcb0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/919.pdf', 'bytes': 1664762}, {'source_id': 'GDCA-F-8c6fef877160dc061f6c', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c6fef877160dc061f6c_309.pdf', 'source_sha256': '0fcccdfbcc213ae0ea39cba2056ac5b53de2090c7b5a8b39e617e9c39ef06533', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/309.pdf', 'bytes': 1926760}, {'source_id': 'GDCA-F-4a07211a1a6df13e5074', 'source_path': 'sources/gdca-recon/2026-11-14/files/4a07211a1a6df13e5074_326.pdf', 'source_sha256': 'c91923dcafc6c9b4f7c59a0ed0282132531e706b3f156668ce4b844305c6c30d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/326.pdf', 'bytes': 2131707}, {'source_id': 'GDCA-F-32d1190a14d582626518', 'source_path': 'sources/gdca-recon/2026-11-14/files/32d1190a14d582626518_653.pdf', 'source_sha256': 'ba2868e345997dcdfca1ab3464a647f5a379188a09376b27c11f454766369eba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/653.pdf', 'bytes': 1805489}, {'source_id': 'GDCA-F-7f0c33b617444f97a4f3', 'source_path': 'sources/gdca-recon/2026-11-14/files/7f0c33b617444f97a4f3_195.pdf', 'source_sha256': '2285f263a9c5f9c6c424a531fc3ace520fb229abea2cb15a2458575f2283390f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/195.pdf', 'bytes': 1967041}, {'source_id': 'GDCA-F-980abe7c800cf9463b78', 'source_path': 'sources/gdca-recon/2026-11-14/files/980abe7c800cf9463b78_497.pdf', 'source_sha256': '42084fa6945e58dbcfe5a9520ba4fd9b65cfae49aca163efbea95cf584c56e6f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/497.pdf', 'bytes': 2014353}, {'source_id': 'GDCA-F-042d0f474f6cdf22b452', 'source_path': 'sources/gdca-recon/2026-11-14/files/042d0f474f6cdf22b452_975.pdf', 'source_sha256': '20ea60b9d83da91b96ad77f263607b4bb013b72ff8278af28a17c9a5dd7a9752', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/975.pdf', 'bytes': 1790796}, {'source_id': 'GDCA-F-40a0387f1febef085a9b', 'source_path': 'sources/gdca-recon/2026-11-14/files/40a0387f1febef085a9b_909.pdf', 'source_sha256': '3415602dedf64d6682a0c05d2b1c655706a5d10c010182a46e8faf6b09d44be3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/909.pdf', 'bytes': 1739395}, {'source_id': 'GDCA-F-c4e21e4f96a7e5b0fff6', 'source_path': 'sources/gdca-recon/2026-11-14/files/c4e21e4f96a7e5b0fff6_70.pdf', 'source_sha256': 'ed03df35078501b5eb20652b088d93fc7b2ef340e5fa80963988b668df924f1c', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/70.pdf', 'bytes': 2506832}, {'source_id': 'GDCA-F-88279cb7d841251ef4bb', 'source_path': 'sources/gdca-recon/2026-11-14/files/88279cb7d841251ef4bb_680.pdf', 'source_sha256': '245071acc705c2eb1badf58fc4487b829058e5212f71966267488703dfb25101', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/680.pdf', 'bytes': 2338705}, {'source_id': 'GDCA-F-8e2b0d9ed3a3323559e8', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e2b0d9ed3a3323559e8_1032.pdf', 'source_sha256': 'b73a060c9aef46209c3eeab48661b3e0812f0cf6057ed90a2d0668bc44efb1b3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1032.pdf', 'bytes': 2037685}, {'source_id': 'GDCA-F-5801e5c93ef915b25fa6', 'source_path': 'sources/gdca-recon/2026-11-14/files/5801e5c93ef915b25fa6_373.pdf', 'source_sha256': '88f3c3a140c5a57787d6e7f14375693d52df058181a10976a9095443201f4f33', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/373.pdf', 'bytes': 1920808}, {'source_id': 'GDCA-F-bdb79817434af2c133cf', 'source_path': 'sources/gdca-recon/2026-11-14/files/bdb79817434af2c133cf_1274.pdf', 'source_sha256': '0660b7c5ffda76a269ccfde22ef51a35e156fb382d4d7fc36bf6a0dcbb8d2ad5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1274.pdf', 'bytes': 2023935}, {'source_id': 'GDCA-F-2de232ff881fc05a5e3d', 'source_path': 'sources/gdca-recon/2026-11-14/files/2de232ff881fc05a5e3d_1461.pdf', 'source_sha256': '8dcf410e6a2d819d1704edf89ebaecc53c74f855a370dddccb371c7e56dcf5d1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1461.pdf', 'bytes': 2298740}, {'source_id': 'GDCA-F-d9a4f32f49b34ca348aa', 'source_path': 'sources/gdca-recon/2026-11-14/files/d9a4f32f49b34ca348aa_1581.pdf', 'source_sha256': '9cad78496d6336004ba162dbd18f4693132495ac7e9c5ecc4ba1cbdf6acfb88f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1581.pdf', 'bytes': 2075273}, {'source_id': 'GDCA-F-7c7dd1a2d0327e505823', 'source_path': 'sources/gdca-recon/2026-11-14/files/7c7dd1a2d0327e505823_1165.pdf', 'source_sha256': 'c28171832e5c5b10c64f6780d91255714f70a2606e9cf02b34d3054a6c231fb8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1165.pdf', 'bytes': 1889550}, {'source_id': 'GDCA-F-2c5bd4523af57a37fbdd', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c5bd4523af57a37fbdd_29.pdf', 'source_sha256': '94036175efa9be223887197e5556b631851a1b5b8bb27393724e6c672055444d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/29.pdf', 'bytes': 91970}, {'source_id': 'GDCA-F-738fcea9736549d63b40', 'source_path': 'sources/gdca-recon/2026-11-14/files/738fcea9736549d63b40_830.pdf', 'source_sha256': '4c8174d6da6b4ba3e905de067aa8574602deab4274e46b88ad69bc6abb9d1a52', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/830.pdf', 'bytes': 1956034}, {'source_id': 'GDCA-F-7fd064d85470ace18ae2', 'source_path': 'sources/gdca-recon/2026-11-14/files/7fd064d85470ace18ae2_16.pdf', 'source_sha256': '501c25b719408e5cafd56f1f651cb7df89e4a8de3b49e9983ef997072a1d343e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/16.pdf', 'bytes': 2352481}, {'source_id': 'GDCA-F-2666499e4d6b5723b4a4', 'source_path': 'sources/gdca-recon/2026-11-14/files/2666499e4d6b5723b4a4_1574.pdf', 'source_sha256': '3d951fedbc30737aa7f407e1cdf8fa46823a78b6a74ec89a948ef68427f9a209', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1574.pdf', 'bytes': 2924890}, {'source_id': 'GDCA-F-50584a7d008503ad77aa', 'source_path': 'sources/gdca-recon/2026-11-14/files/50584a7d008503ad77aa_579.pdf', 'source_sha256': '24d6f1c9da4ec0184383d050253ab263a4218dbff45ec4c229a0e521d2c77333', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/579.pdf', 'bytes': 1611200}, {'source_id': 'GDCA-F-8eb629fdc8ec79bf8edd', 'source_path': 'sources/gdca-recon/2026-11-14/files/8eb629fdc8ec79bf8edd_933.pdf', 'source_sha256': '10a7b9440762c8c41cce8bd1bc115efbe21eea16aa4a60d4dd2edd04ed30a1dc', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/933.pdf', 'bytes': 2152897}, {'source_id': 'GDCA-F-0da8e209b65cf6910740', 'source_path': 'sources/gdca-recon/2026-11-14/files/0da8e209b65cf6910740_860.pdf', 'source_sha256': '9a8ac4ededbc4d627c04d9512d8f89adba7312e211e7b8d0e9049b209e6deaa1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/860.pdf', 'bytes': 2104956}, {'source_id': 'GDCA-F-351ce0581479674d5abf', 'source_path': 'sources/gdca-recon/2026-11-14/files/351ce0581479674d5abf_1253.pdf', 'source_sha256': '5cfe26bdb9c3e2be453b25504202133a63829b9974ed74d880b32c485d5564b3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1253.pdf', 'bytes': 1936523}, {'source_id': 'GDCA-F-90b924ff25afe95f7394', 'source_path': 'sources/gdca-recon/2026-11-14/files/90b924ff25afe95f7394_59.pdf', 'source_sha256': 'c96825082b3d61227b94329151621f43a9244065e86fb2acbe23229df56b7eca', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/59.pdf', 'bytes': 2033501}, {'source_id': 'GDCA-F-9f222ade0a0cd7c9bb48', 'source_path': 'sources/gdca-recon/2026-11-14/files/9f222ade0a0cd7c9bb48_662.pdf', 'source_sha256': '86ffad656ced2f35a0c73ec1a52bc46ba16dcd5c6d9c45883867ac137909092a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/662.pdf', 'bytes': 1770310}, {'source_id': 'GDCA-F-8b91e58d44551f9e8e7d', 'source_path': 'sources/gdca-recon/2026-11-14/files/8b91e58d44551f9e8e7d_884.pdf', 'source_sha256': '0990bb0a5cd4f30f3a0c79b0890c57f28589bdbc9fb0a430ea8dafb231cb86e9', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/884.pdf', 'bytes': 2417676}, {'source_id': 'GDCA-F-58df6efa9149348d1f41', 'source_path': 'sources/gdca-recon/2026-11-14/files/58df6efa9149348d1f41_1000.pdf', 'source_sha256': 'f23251d7157623d91c9fa1bfd2de3f42cee8a8d7a415e7c8b3c2ff1160697057', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1000.pdf', 'bytes': 2187543}, {'source_id': 'GDCA-F-cd48bd6436ed765384fa', 'source_path': 'sources/gdca-recon/2026-11-14/files/cd48bd6436ed765384fa_1454.pdf', 'source_sha256': '9202444f09b7861a40b1c93ffd35a5ccd08c1cb1ce79ef48edd5d245ab4e4259', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1454.pdf', 'bytes': 2581283}, {'source_id': 'GDCA-F-8d1879fd5b9829951e46', 'source_path': 'sources/gdca-recon/2026-11-14/files/8d1879fd5b9829951e46_838.pdf', 'source_sha256': 'f51a3477813d8836f16f1b7e2e1d52017c8d106be2b8deb0cd3356b41849c5c6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/838.pdf', 'bytes': 2004264}, {'source_id': 'GDCA-F-22c702692adbf9485025', 'source_path': 'sources/gdca-recon/2026-11-14/files/22c702692adbf9485025_1484.pdf', 'source_sha256': '4caca280581ca2f6d4cf2b01f784236eddfeb561e88e8068c47d48eec1128694', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1484.pdf', 'bytes': 2262154}, {'source_id': 'GDCA-F-1026b4da05bf88b4eb45', 'source_path': 'sources/gdca-recon/2026-11-14/files/1026b4da05bf88b4eb45_943.pdf', 'source_sha256': 'c4e4f7e5b4acde1b8eb35c0d143f005a0f80166acc795341295dfe37e3af18a3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/943.pdf', 'bytes': 1874453}, {'source_id': 'GDCA-F-37c0f11e2b8e6f3462f7', 'source_path': 'sources/gdca-recon/2026-11-14/files/37c0f11e2b8e6f3462f7_1005.pdf', 'source_sha256': 'c40fead0f0ebf7d7ffb7cb30facf4dd6ee4fd8b395bfef873db32963c80b769b', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1005.pdf', 'bytes': 2301096}, {'source_id': 'GDCA-F-d8a8b8a879421f8c3adf', 'source_path': 'sources/gdca-recon/2026-11-14/files/d8a8b8a879421f8c3adf_241.pdf', 'source_sha256': '569526059577769258c39a66fdd71aa0754b1400d814994b66f9a895645c9d07', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/241.pdf', 'bytes': 1836551}, {'source_id': 'GDCA-F-db6d2f9599041c9a8b99', 'source_path': 'sources/gdca-recon/2026-11-14/files/db6d2f9599041c9a8b99_815.pdf', 'source_sha256': 'e5214ddc40cadc5d93be1557cb1ecbc119f54ee73f3db600c55c8cc44e778203', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/815.pdf', 'bytes': 2093271}, {'source_id': 'GDCA-F-ea6186b3093b7417289a', 'source_path': 'sources/gdca-recon/2026-11-14/files/ea6186b3093b7417289a_601.pdf', 'source_sha256': 'd4e5aedcabef3a571088a75a578e6a95ba16b495f6d02596ac806ac787603d69', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/601.pdf', 'bytes': 1766688}, {'source_id': 'GDCA-F-0c90a307402adc02c0e9', 'source_path': 'sources/gdca-recon/2026-11-14/files/0c90a307402adc02c0e9_692.pdf', 'source_sha256': 'adeabdb80cb1fd6a3c64b5804fd5c030a386c659c924e41e650f46b1b47920dc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/692.pdf', 'bytes': 2044453}, {'source_id': 'GDCA-F-ad0b74b5d633ec70bedb', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad0b74b5d633ec70bedb_209.pdf', 'source_sha256': '7c3569f9d06c45d0072955e13d5f304b6352026afeaeeaef73041d43fb7b762a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/209.pdf', 'bytes': 1980066}, {'source_id': 'GDCA-F-fa9604922dc4e902978c', 'source_path': 'sources/gdca-recon/2026-11-14/files/fa9604922dc4e902978c_712.pdf', 'source_sha256': '0ab23904ef023eceb4b72392d9da4908d702ac696dfd04cf5fb086a23bb69aa7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/712.pdf', 'bytes': 1775618}, {'source_id': 'GDCA-F-c42473f7db479c43ef1e', 'source_path': 'sources/gdca-recon/2026-11-14/files/c42473f7db479c43ef1e_137.pdf', 'source_sha256': 'f07e6f315633fa64863c4a0aae43ffdd86f54a884d3ea0f0d066acf970775e57', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/137.pdf', 'bytes': 2066712}, {'source_id': 'GDCA-F-d693c934031353e26ca7', 'source_path': 'sources/gdca-recon/2026-11-14/files/d693c934031353e26ca7_362.pdf', 'source_sha256': '5d68af98340ee54a773b8999965b020cea52e857e1c9e688d16b45863a358064', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/362.pdf', 'bytes': 1934702}, {'source_id': 'GDCA-F-e4c733d22db940fd7ceb', 'source_path': 'sources/gdca-recon/2026-11-14/files/e4c733d22db940fd7ceb_1272.pdf', 'source_sha256': 'ce953da99008f2f9129481d22938810dd6f9b8f8fde0f3f971730bb557223bcf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1272.pdf', 'bytes': 1879666}, {'source_id': 'GDCA-F-ad743da67d4553bbf3b4', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad743da67d4553bbf3b4_873.pdf', 'source_sha256': '199c57874502d06372d00f4b15cabbe6c83285103a52aad6e1265f7a238b25ac', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/873.pdf', 'bytes': 2253503}, {'source_id': 'GDCA-F-01f095e3e014acc3554f', 'source_path': 'sources/gdca-recon/2026-11-14/files/01f095e3e014acc3554f_635.pdf', 'source_sha256': '84cbad766a7e1099a848b9510bbae5437b8e7d46943461875a5334e1d242a1cd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/635.pdf', 'bytes': 2006548}, {'source_id': 'GDCA-F-d389fa1c5cfc8006f837', 'source_path': 'sources/gdca-recon/2026-11-14/files/d389fa1c5cfc8006f837_1304.pdf', 'source_sha256': '61ccb08b8fd0450fa481a5af511b8f5358b3b526bdb68cf7302782c808ae3bab', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1304.pdf', 'bytes': 2550775}, {'source_id': 'GDCA-F-84220d313ebe1d3b4a21', 'source_path': 'sources/gdca-recon/2026-11-14/files/84220d313ebe1d3b4a21_1550.pdf', 'source_sha256': '968739f9295c6c4cf696b6f7d5bbf56ce4281ca10371eaebfa39d1308fc17ab7', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1550.pdf', 'bytes': 3591764}, {'source_id': 'GDCA-F-07a3e9f5c7359d445384', 'source_path': 'sources/gdca-recon/2026-11-14/files/07a3e9f5c7359d445384_138.pdf', 'source_sha256': '674ea84e4f2c8f5a2cf128b22216b6aa3567b271797adc5386917e610ec8ccbf', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/138.pdf', 'bytes': 2444431}, {'source_id': 'GDCA-F-ac09926439000f8b5820', 'source_path': 'sources/gdca-recon/2026-11-14/files/ac09926439000f8b5820_1449.pdf', 'source_sha256': 'b4d1f71c98373b8000f835f94ade2c3ebc8af8fc6db42f1c0745e84230b79403', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1449.pdf', 'bytes': 2657279}, {'source_id': 'GDCA-F-577ebbebb2417cc749db', 'source_path': 'sources/gdca-recon/2026-11-14/files/577ebbebb2417cc749db_1219.pdf', 'source_sha256': '4fbf75c59dfd288692e05f19985426e7385f004e3f1227aa450943dd1bc46bf7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1219.pdf', 'bytes': 2277470}, {'source_id': 'GDCA-F-7c8c0254487e8ba1c0f9', 'source_path': 'sources/gdca-recon/2026-11-14/files/7c8c0254487e8ba1c0f9_651.pdf', 'source_sha256': '0da8ee152862f77650a106ba3355230833e68c4a15615ecc89856509f37c3bbf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/651.pdf', 'bytes': 1811557}, {'source_id': 'GDCA-F-565e6dfc2b12f4344820', 'source_path': 'sources/gdca-recon/2026-11-14/files/565e6dfc2b12f4344820_1277.pdf', 'source_sha256': '830d9afae904f5fd1ac322a9ad5fbcfb9b8c197317173ce17517475f40024a01', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1277.pdf', 'bytes': 901538}, {'source_id': 'GDCA-F-09d42e9cd3574108e2c2', 'source_path': 'sources/gdca-recon/2026-11-14/files/09d42e9cd3574108e2c2_800.pdf', 'source_sha256': 'df1b5cf57795750ba6ad95cc05337bf8a8ef7d3afa779907a3fe76844c4f4a50', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/800.pdf', 'bytes': 1200963}, {'source_id': 'GDCA-F-5d7552e0d4bac7eee325', 'source_path': 'sources/gdca-recon/2026-11-14/files/5d7552e0d4bac7eee325_1024.pdf', 'source_sha256': 'c5d6c3bde44a33e4908cd5cb3c7acfc6daacdfa621067652858f4ca1c39ffefc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1024.pdf', 'bytes': 2377059}, {'source_id': 'GDCA-F-4a16e2d1ce29e8e6ba48', 'source_path': 'sources/gdca-recon/2026-11-14/files/4a16e2d1ce29e8e6ba48_282.pdf', 'source_sha256': '6ecf1aa6a876c22ed813b66197674bdaa2c01bc2b0ab419761fb59c9fb1759a9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/282.pdf', 'bytes': 1835250}, {'source_id': 'GDCA-F-8f81e59fb5f687ecc353', 'source_path': 'sources/gdca-recon/2026-11-14/files/8f81e59fb5f687ecc353_950.pdf', 'source_sha256': '40db03a5e3b73b6184543ef35c46496cad158acc8ace3604a03fd779eb051a0b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/950.pdf', 'bytes': 2223710}, {'source_id': 'GDCA-F-881b5ba9e687e3572313', 'source_path': 'sources/gdca-recon/2026-11-14/files/881b5ba9e687e3572313_1142.pdf', 'source_sha256': '1700814b19933ee300348e4189020d4ad3b1e03f1178f52e6dc7b415bf97fb7a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1142.pdf', 'bytes': 1930467}, {'source_id': 'GDCA-F-676b81e7a047713a3b1c', 'source_path': 'sources/gdca-recon/2026-11-14/files/676b81e7a047713a3b1c_327.pdf', 'source_sha256': 'b8885193521504ff6dcc643600a5c81022c98acf835a45424a5ca413db775ae1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/327.pdf', 'bytes': 2145313}, {'source_id': 'GDCA-F-0786bcc4c110c0dc45bf', 'source_path': 'sources/gdca-recon/2026-11-14/files/0786bcc4c110c0dc45bf_540.pdf', 'source_sha256': '68eeff41216502077b7e2e4f0b0facc02b8d066551db4c50db93e096e0c2d808', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/540.pdf', 'bytes': 1998139}, {'source_id': 'GDCA-F-3a5c48bed8f5767fa53b', 'source_path': 'sources/gdca-recon/2026-11-14/files/3a5c48bed8f5767fa53b_1423.pdf', 'source_sha256': 'c2daa779906963f3009b9a82329aa7a598fec0eec18cb0953b2fbea522f04964', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1423.pdf', 'bytes': 2295927}, {'source_id': 'GDCA-F-6ec843bcabf961f9c4d8', 'source_path': 'sources/gdca-recon/2026-11-14/files/6ec843bcabf961f9c4d8_478.pdf', 'source_sha256': 'e7e5294aa48c2c38a0d48cb968a0ee1896e1ff7cf3512cd744499b79dd3e2171', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/478.pdf', 'bytes': 1186656}, {'source_id': 'GDCA-F-26d34d380a5240bfbf75', 'source_path': 'sources/gdca-recon/2026-11-14/files/26d34d380a5240bfbf75_32.pdf', 'source_sha256': '3fb96722bc1420231afe093a5655a22dbbc095f2d7d43b176e74dc982eff72ba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/32.pdf', 'bytes': 97330}, {'source_id': 'GDCA-F-f1a0e0d86331d6ddbad0', 'source_path': 'sources/gdca-recon/2026-11-14/files/f1a0e0d86331d6ddbad0_1319.pdf', 'source_sha256': 'ef62f0d9605742ca44ea7b5ca3112a9b4ec5aae731d2f91f6bee37efad53a850', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1319.pdf', 'bytes': 2279490}, {'source_id': 'GDCA-F-f31503f599fb97a4ab31', 'source_path': 'sources/gdca-recon/2026-11-14/files/f31503f599fb97a4ab31_991.pdf', 'source_sha256': 'fe697ecbfcff2088033bafe4ab80dda8d75f727c23bcf069faacbf5373e076a0', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/991.pdf', 'bytes': 3042302}, {'source_id': 'GDCA-F-5c03b561cb362c991cc7', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c03b561cb362c991cc7_544.pdf', 'source_sha256': '8cd241f8296492a1f556f382db300cfde2530609db8850643a55c40e660376ba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/544.pdf', 'bytes': 1749156}, {'source_id': 'GDCA-F-303517a73ecb939275c9', 'source_path': 'sources/gdca-recon/2026-11-14/files/303517a73ecb939275c9_857.pdf', 'source_sha256': '21dfc754ddb9de662d9f508879583dcbfbe56d63a2b7aaa82a20742b1a7b179e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/857.pdf', 'bytes': 1888071}, {'source_id': 'GDCA-F-ca9dea6b57849daf4a70', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca9dea6b57849daf4a70_1580.pdf', 'source_sha256': 'c01845cc4ade24ec2cfe28cbb85c183e018473c9f2d9d3c7be428845b0346d1b', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1580.pdf', 'bytes': 2395797}, {'source_id': 'GDCA-F-deb97946a6ccc944517c', 'source_path': 'sources/gdca-recon/2026-11-14/files/deb97946a6ccc944517c_964.pdf', 'source_sha256': 'a518acf1d3ac50b93ed3058941d46f48bd998f2a77887844b0140d5391cf8065', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/964.pdf', 'bytes': 1665271}, {'source_id': 'GDCA-F-acac5765f0d1413be32e', 'source_path': 'sources/gdca-recon/2026-11-14/files/acac5765f0d1413be32e_954.pdf', 'source_sha256': '0bffeb63203299bd121cc2a18110a14f7e9ba152a69eb7738f5503c0037d6606', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/954.pdf', 'bytes': 2067458}, {'source_id': 'GDCA-F-b7a5dd8ade1a677650ac', 'source_path': 'sources/gdca-recon/2026-11-14/files/b7a5dd8ade1a677650ac_1153.pdf', 'source_sha256': 'afbbf836af9fe9f4685d03c579f54736a7466f7298884c89454a9072e79dfb8f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1153.pdf', 'bytes': 2033713}, {'source_id': 'GDCA-F-1646ec8553103c66d18f', 'source_path': 'sources/gdca-recon/2026-11-14/files/1646ec8553103c66d18f_990.pdf', 'source_sha256': 'f952e6bfe82a717dbbcf9b35cd883f466d5a25687d0020c29bed97ee36bdd26d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/990.pdf', 'bytes': 1908654}, {'source_id': 'GDCA-F-0f2cbc9d853fd3dac8f6', 'source_path': 'sources/gdca-recon/2026-11-14/files/0f2cbc9d853fd3dac8f6_247.pdf', 'source_sha256': '2755f80ab36760753e0587a228e287c20b6fbadc30898ddd2fb0c36a9d04eb7a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/247.pdf', 'bytes': 2222532}, {'source_id': 'GDCA-F-0f15112cb224bcbb2550', 'source_path': 'sources/gdca-recon/2026-11-14/files/0f15112cb224bcbb2550_280.pdf', 'source_sha256': 'a8ea7826db15fc8227ee84c2e484f494b1822cc48fbec815c7e8759764103ada', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/280.pdf', 'bytes': 2038320}, {'source_id': 'GDCA-F-642e5dc1369b76babd45', 'source_path': 'sources/gdca-recon/2026-11-14/files/642e5dc1369b76babd45_1116.pdf', 'source_sha256': '5a78a606095202f17606d9481d2be659ae38149f8172ea18519beb6885f3206b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1116.pdf', 'bytes': 1767595}, {'source_id': 'GDCA-F-58342d293bea4bfdca4f', 'source_path': 'sources/gdca-recon/2026-11-14/files/58342d293bea4bfdca4f_447.pdf', 'source_sha256': 'aa6f299b7d552dfedac3096098bcc0f598830632fded30cf29edecc683e1c275', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/447.pdf', 'bytes': 1921392}, {'source_id': 'GDCA-F-1ee6275d4ad8873609ae', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ee6275d4ad8873609ae_672.pdf', 'source_sha256': 'cf52b5ac4af0dec130d0a3b8347ec0c0fc62fb7fcfd860805b512a0088f8d4d7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/672.pdf', 'bytes': 1814041}, {'source_id': 'GDCA-F-4d8077c6348587649fc9', 'source_path': 'sources/gdca-recon/2026-11-14/files/4d8077c6348587649fc9_569.pdf', 'source_sha256': '33a2e6ae59051bb49d2205b5a426a53aadf3a521c1a278345a54fb3c4662aad2', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/569.pdf', 'bytes': 1356741}, {'source_id': 'GDCA-F-7be93c69f05a7087e036', 'source_path': 'sources/gdca-recon/2026-11-14/files/7be93c69f05a7087e036_832.pdf', 'source_sha256': '97749310fc8cdbb8a5178eed4f3d7f093ac4a6f1904da12d6a7b4f23c2de4276', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/832.pdf', 'bytes': 1076847}, {'source_id': 'GDCA-F-12efd51aba0dae18ee63', 'source_path': 'sources/gdca-recon/2026-11-14/files/12efd51aba0dae18ee63_244.pdf', 'source_sha256': '23b240fca6950343731e9d6738aeb853340cc021ba2ba7ad7a7cc79c48c3a2ce', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/244.pdf', 'bytes': 2240932}, {'source_id': 'GDCA-F-e1db1aa65d76dbce466b', 'source_path': 'sources/gdca-recon/2026-11-14/files/e1db1aa65d76dbce466b_74.pdf', 'source_sha256': '191d51659c3a5695f395b43219c80968fe1422932e8e27febccf29beb1848db2', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/74.pdf', 'bytes': 2153086}, {'source_id': 'GDCA-F-930a5383f2400d7fe004', 'source_path': 'sources/gdca-recon/2026-11-14/files/930a5383f2400d7fe004_472.pdf', 'source_sha256': 'e372f338d21c0c8b95e5f11442b594c434cef1eff061c23a2184be40b29c1afa', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/472.pdf', 'bytes': 1998671}, {'source_id': 'GDCA-F-e2c62a21560c2842e180', 'source_path': 'sources/gdca-recon/2026-11-14/files/e2c62a21560c2842e180_667.pdf', 'source_sha256': '95ba09e54d51242428fedb1f34685c9390b66d41013313e2cf7f12995f8bf139', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/667.pdf', 'bytes': 1816106}, {'source_id': 'GDCA-F-c94b59ac617eeb664189', 'source_path': 'sources/gdca-recon/2026-11-14/files/c94b59ac617eeb664189_1370.pdf', 'source_sha256': '03a2f6f4e5efcb7d7612e17b5166713a1c346a539c69a81eedf20a5b90440dab', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1370.pdf', 'bytes': 3017057}, {'source_id': 'GDCA-F-56d50154899e69495b31', 'source_path': 'sources/gdca-recon/2026-11-14/files/56d50154899e69495b31_995.pdf', 'source_sha256': 'd206c8e9ad705baff4dc0575bba1c4fdd81d3ba9d8d75e807e032d1273c948c8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/995.pdf', 'bytes': 2255419}, {'source_id': 'GDCA-F-335a995ed734a7e6c494', 'source_path': 'sources/gdca-recon/2026-11-14/files/335a995ed734a7e6c494_49.pdf', 'source_sha256': 'add23bef97c78c11f0a73841ef4f56efc4a71bbf5494c5524443ed922dab4216', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/49.pdf', 'bytes': 1984696}, {'source_id': 'GDCA-F-ecae8cbcd13b0f89acc5', 'source_path': 'sources/gdca-recon/2026-11-14/files/ecae8cbcd13b0f89acc5_143.pdf', 'source_sha256': '0ba2a86702703d7cef69b8ad23dc07f351c2fb7c294bf0f1c502801a3c06c87a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/143.pdf', 'bytes': 2054809}, {'source_id': 'GDCA-F-5ca7a5cae008eff013d4', 'source_path': 'sources/gdca-recon/2026-11-14/files/5ca7a5cae008eff013d4_238.pdf', 'source_sha256': 'c1c0c77cb8e92e77132b3ba4c84b59ed56cd7fe59a06261db6b5c168babd652a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/238.pdf', 'bytes': 2252108}, {'source_id': 'GDCA-F-be753dcd56a7114196a0', 'source_path': 'sources/gdca-recon/2026-11-14/files/be753dcd56a7114196a0_136.pdf', 'source_sha256': '13bea4783308a481f0f2c9f379722e7b5e27630b8481df8c2098c47b41f4e4b0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/136.pdf', 'bytes': 1797425}, {'source_id': 'GDCA-F-57810e70883bee2acdcd', 'source_path': 'sources/gdca-recon/2026-11-14/files/57810e70883bee2acdcd_770.pdf', 'source_sha256': 'ddc85132240c49527114036a2389b5857e6f362c1f806bb16d3dafee2dabf242', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/770.pdf', 'bytes': 1871489}, {'source_id': 'GDCA-F-cb074ea1f93f0f635683', 'source_path': 'sources/gdca-recon/2026-11-14/files/cb074ea1f93f0f635683_1105.pdf', 'source_sha256': '8c0d74c2b12d400c10ac916ebc72e5d090e5e06933b0388b466166df7b399a6f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1105.pdf', 'bytes': 2220048}, {'source_id': 'GDCA-F-0eaef4651d912a1b85e8', 'source_path': 'sources/gdca-recon/2026-11-14/files/0eaef4651d912a1b85e8_889.pdf', 'source_sha256': '1f0a2a12fce800ae44c1559b0acfb1b64cb28bcbad4ee3dbd14141ddfed7a4dc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/889.pdf', 'bytes': 1989565}, {'source_id': 'GDCA-F-4e27e92aac588455f833', 'source_path': 'sources/gdca-recon/2026-11-14/files/4e27e92aac588455f833_1052.pdf', 'source_sha256': 'f85b8e9a38f92ca7f7364094ee4c431515368633842c36eb8163e29399633978', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1052.pdf', 'bytes': 2338914}, {'source_id': 'GDCA-F-7f7a9862533377d065c4', 'source_path': 'sources/gdca-recon/2026-11-14/files/7f7a9862533377d065c4_348.pdf', 'source_sha256': '342ba42490df86062f64a5e62502411decc6dfe4795e42b4586e375946ae784b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/348.pdf', 'bytes': 1977743}, {'source_id': 'GDCA-F-ca229a5a4cbfdea16fe8', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca229a5a4cbfdea16fe8_1179.pdf', 'source_sha256': '795eb043affb2bec07022147b73dce12d1d32534ac1af4d3e5d3cf8cbf8f694f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1179.pdf', 'bytes': 1879196}, {'source_id': 'GDCA-F-6f5628b626eaf1c3a735', 'source_path': 'sources/gdca-recon/2026-11-14/files/6f5628b626eaf1c3a735_197.pdf', 'source_sha256': '5acb8e9c35dfcf8a190935eaa75ebeef0a21ee72b235825ccfa17cf8fe2ce8d2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/197.pdf', 'bytes': 2091894}, {'source_id': 'GDCA-F-1f1fb80ad5131d1894fa', 'source_path': 'sources/gdca-recon/2026-11-14/files/1f1fb80ad5131d1894fa_504.pdf', 'source_sha256': 'cc4edc902fcfce347b0dbbf7a88df016707d9febbae30db629a5954499c2700b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/504.pdf', 'bytes': 2396650}, {'source_id': 'GDCA-F-2e8562edcb087c6a6d0d', 'source_path': 'sources/gdca-recon/2026-11-14/files/2e8562edcb087c6a6d0d_285.pdf', 'source_sha256': '31d58d033cf8edc2385ace4a1a530a23b635f4669792e93b20c52da92fea47d4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/285.pdf', 'bytes': 2280812}, {'source_id': 'GDCA-F-a0b34ca202afe6a39755', 'source_path': 'sources/gdca-recon/2026-11-14/files/a0b34ca202afe6a39755_1152.pdf', 'source_sha256': 'b514973f1c9a776c2167a8e750830a0e2338334137f2feecdf5d478fe623ccdc', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1152.pdf', 'bytes': 2227834}, {'source_id': 'GDCA-F-4ff607dcb91f9192be93', 'source_path': 'sources/gdca-recon/2026-11-14/files/4ff607dcb91f9192be93_697.pdf', 'source_sha256': '0d71cc2944664ec57d364a7eac3b3a0520043b3acbd0765c97cd6536a872f077', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/697.pdf', 'bytes': 1848773}, {'source_id': 'GDCA-F-14bfa5753cdc8c40e6f8', 'source_path': 'sources/gdca-recon/2026-11-14/files/14bfa5753cdc8c40e6f8_717.pdf', 'source_sha256': 'e2888d33be4b87fe30eb9ca1830195fa067f5d1f15c05cc8e1f515e518af892d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/717.pdf', 'bytes': 2021741}, {'source_id': 'GDCA-F-3829545208bfb6f620f4', 'source_path': 'sources/gdca-recon/2026-11-14/files/3829545208bfb6f620f4_419.pdf', 'source_sha256': '63e236c7245e59a9d0be1998189de3e934a6b921e8e019d44ed591cd13d1f3d6', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/419.pdf', 'bytes': 2558573}, {'source_id': 'GDCA-F-b609cd66993580c209d9', 'source_path': 'sources/gdca-recon/2026-11-14/files/b609cd66993580c209d9_573.pdf', 'source_sha256': '9912be898ff456db72e68b9e467f366297da34e8b3a0613c307aabafffc982ee', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/573.pdf', 'bytes': 1801279}, {'source_id': 'GDCA-F-fb2541a18d44f94148d6', 'source_path': 'sources/gdca-recon/2026-11-14/files/fb2541a18d44f94148d6_1310.pdf', 'source_sha256': '55212f5821c3b56811a547116a8f6703ab149e16a8196222162cd8be67bc08ec', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1310.pdf', 'bytes': 2184417}, {'source_id': 'GDCA-F-c15b1611f5fd211608e7', 'source_path': 'sources/gdca-recon/2026-11-14/files/c15b1611f5fd211608e7_314.pdf', 'source_sha256': '2001934e8783f7d337e8692d5e8d64585ec9b431347f2ebd1009d7ce9399ed94', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/314.pdf', 'bytes': 1626722}, {'source_id': 'GDCA-F-ca9276199ae69e068275', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca9276199ae69e068275_1459.pdf', 'source_sha256': '1613d83f6792d5d04ef0f2c907eeafd8d17f72771aa9e61c47731aad09992c5f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1459.pdf', 'bytes': 2291737}, {'source_id': 'GDCA-F-965cf2f773ccbde026a1', 'source_path': 'sources/gdca-recon/2026-11-14/files/965cf2f773ccbde026a1_467.pdf', 'source_sha256': '9e18c7f879cf6b33177afa12d31223c9e09ed913f1c53d9a4d82c4f12016b3f9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/467.pdf', 'bytes': 2214478}, {'source_id': 'GDCA-F-dcf8f2107a2105353a4c', 'source_path': 'sources/gdca-recon/2026-11-14/files/dcf8f2107a2105353a4c_1216.pdf', 'source_sha256': '71a679045b9af191ae540d4d6339819d3e39946e6ee15cda1c9e71628b53dbb0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1216.pdf', 'bytes': 2156501}, {'source_id': 'GDCA-F-52b7c23685a42ce95ec5', 'source_path': 'sources/gdca-recon/2026-11-14/files/52b7c23685a42ce95ec5_72.pdf', 'source_sha256': 'd9529d9b2a926e3f437aae712fa619aa4fa5b8f00184f8f665a55993a0ad49b5', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/72.pdf', 'bytes': 2264889}, {'source_id': 'GDCA-F-338f1439db766cf44e91', 'source_path': 'sources/gdca-recon/2026-11-14/files/338f1439db766cf44e91_514.pdf', 'source_sha256': '931243f1351f0eed03b7f7fd4362f96a7ff08d8978aaadba48b20016c3858244', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/514.pdf', 'bytes': 1776920}, {'source_id': 'GDCA-F-a488ebc81c218f8962aa', 'source_path': 'sources/gdca-recon/2026-11-14/files/a488ebc81c218f8962aa_243.pdf', 'source_sha256': '31cd4bded0cbd209b7e4acdf2f131278773f3c1327acd28f3317397ee96eb3aa', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/243.pdf', 'bytes': 2140536}, {'source_id': 'GDCA-F-49d74dadba88edd4d353', 'source_path': 'sources/gdca-recon/2026-11-14/files/49d74dadba88edd4d353_1552.pdf', 'source_sha256': '35390a57c7ff488521c4247d04ad47cb0cf3267eb460680b2e7c8d05e8359cad', 'document_pages': 11, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1552.pdf', 'bytes': 4510283}, {'source_id': 'GDCA-F-4d06a7c4f212c2f817cb', 'source_path': 'sources/gdca-recon/2026-11-14/files/4d06a7c4f212c2f817cb_646.pdf', 'source_sha256': '5ea38c589e6050850bf671a816e36a3cc93d205e2e6ae9be60cd92f5cab1ea3e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/646.pdf', 'bytes': 2007265}, {'source_id': 'GDCA-F-c82a155dec4cde4a5e04', 'source_path': 'sources/gdca-recon/2026-11-14/files/c82a155dec4cde4a5e04_237.pdf', 'source_sha256': 'ad672f1af782ea1fbcb966e57e5013a05ca32c414f6cfbba3332079b7c2d64ba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/237.pdf', 'bytes': 2080049}, {'source_id': 'GDCA-F-ad94c7c851d929f4b313', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad94c7c851d929f4b313_2.pdf', 'source_sha256': '78aee97f802e5aaca0e2cbd194974c44dd15b3c58d9e7fc317aa7dc3f4162dfe', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/2.pdf', 'bytes': 3029757}, {'source_id': 'GDCA-F-1e54ae484fc81391bb56', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e54ae484fc81391bb56_1453.pdf', 'source_sha256': '84d61fc488ea120e49d6627107a4aee7470ba2a1d4413b2929edf2a626b46648', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1453.pdf', 'bytes': 2429959}, {'source_id': 'GDCA-F-a5dda89f05ceb30b63d5', 'source_path': 'sources/gdca-recon/2026-11-14/files/a5dda89f05ceb30b63d5_530.pdf', 'source_sha256': '92928e68f84593ba26743c2980a37d9d4cb19725ac033f18452c328e1aaf85cd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/530.pdf', 'bytes': 1910686}, {'source_id': 'GDCA-F-3730e46fd4e2004f3ab3', 'source_path': 'sources/gdca-recon/2026-11-14/files/3730e46fd4e2004f3ab3_486.pdf', 'source_sha256': 'e4dd3ab3dd7a2e5a53f80095902a4a79af0ef65ec9bb52140ca49c5c2fdb42c9', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/486.pdf', 'bytes': 1412408}, {'source_id': 'GDCA-F-6754f6d5e96f916fd409', 'source_path': 'sources/gdca-recon/2026-11-14/files/6754f6d5e96f916fd409_994.pdf', 'source_sha256': 'd1ec7fa226c77e97b78781ce0eb013f908ca0ba67ac8a17b0d2dfed1039b54db', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/994.pdf', 'bytes': 919232}, {'source_id': 'GDCA-F-ed22c555c741c5574288', 'source_path': 'sources/gdca-recon/2026-11-14/files/ed22c555c741c5574288_1488.pdf', 'source_sha256': 'b70e51387d588f3396defc75cb693cfb71efdc2bab38d6a7d5ce2830ef12d5d4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1488.pdf', 'bytes': 2611924}, {'source_id': 'GDCA-F-ff64a1a01283124dde66', 'source_path': 'sources/gdca-recon/2026-11-14/files/ff64a1a01283124dde66_1103.pdf', 'source_sha256': '6b5b56a6be36b61e5b68f6f8e29f4f5f450d7b7c653e607a7c2f019a9dc41d84', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1103.pdf', 'bytes': 2368370}, {'source_id': 'GDCA-F-55268bc4b20c8ff9ed27', 'source_path': 'sources/gdca-recon/2026-11-14/files/55268bc4b20c8ff9ed27_210.pdf', 'source_sha256': '0dc3ee6a91fc80c2af09fcf6ef896be01c66e3b7755a38f43a3d1312d6435c93', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/210.pdf', 'bytes': 2049917}, {'source_id': 'GDCA-F-efe93ac7c2c93a005166', 'source_path': 'sources/gdca-recon/2026-11-14/files/efe93ac7c2c93a005166_1194.pdf', 'source_sha256': '64608b2312227d08eaf92e69c6591de85eff0c5abed5c83a957b053f9cfe903d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1194.pdf', 'bytes': 2200902}, {'source_id': 'GDCA-F-5dcc472cd5263809b375', 'source_path': 'sources/gdca-recon/2026-11-14/files/5dcc472cd5263809b375_780.pdf', 'source_sha256': 'b53eec5c2d314e1f219b34fbe182e3e719139604c41accb4cf5bcdf3a76df6b4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/780.pdf', 'bytes': 2022980}, {'source_id': 'GDCA-F-93fa8c7ded9f589a18ea', 'source_path': 'sources/gdca-recon/2026-11-14/files/93fa8c7ded9f589a18ea_1109.pdf', 'source_sha256': 'cab76dc53e76ae6a39a404886881a61921afaf3a36b9775c4ec65fd309bfc2c4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1109.pdf', 'bytes': 2066338}, {'source_id': 'GDCA-F-f0245783d799665c9b57', 'source_path': 'sources/gdca-recon/2026-11-14/files/f0245783d799665c9b57_1140.pdf', 'source_sha256': '289fa8056498bb85f437635a1a586c727c5437e59dd33931688a53dceea63e09', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1140.pdf', 'bytes': 1778181}, {'source_id': 'GDCA-F-2de37474695a3d3a32cb', 'source_path': 'sources/gdca-recon/2026-11-14/files/2de37474695a3d3a32cb_1434.pdf', 'source_sha256': '87eb65fe24503285f3f016afe69c333a9847574175c4bd8a784e373cf3554b48', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1434.pdf', 'bytes': 2578603}, {'source_id': 'GDCA-F-eee48672d77f67de7642', 'source_path': 'sources/gdca-recon/2026-11-14/files/eee48672d77f67de7642_968.pdf', 'source_sha256': '41b2dc67fced0d74dca5b14b351d331e335462df904826c57e82a62f814dd47e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/968.pdf', 'bytes': 2270282}, {'source_id': 'GDCA-F-bffd5446a6d8d427b997', 'source_path': 'sources/gdca-recon/2026-11-14/files/bffd5446a6d8d427b997_465.pdf', 'source_sha256': '11025b9d62397a5d83a7e323f09d0b318b0823704ca0eee0410c962cb0f70995', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/465.pdf', 'bytes': 1796253}, {'source_id': 'GDCA-F-2ac875df554b136b4997', 'source_path': 'sources/gdca-recon/2026-11-14/files/2ac875df554b136b4997_736.pdf', 'source_sha256': 'ee1a13f1d3d555327589ac4821c250cd6f810f17ebcc0054de3d2148e05d3c29', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/736.pdf', 'bytes': 1311755}, {'source_id': 'GDCA-F-e6b1034eb24ca1940e61', 'source_path': 'sources/gdca-recon/2026-11-14/files/e6b1034eb24ca1940e61_629.pdf', 'source_sha256': '042c5417b5f84ac5aa9894139fd9884c221e0ea64a2ce001f3d381fa4c818732', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/629.pdf', 'bytes': 1988425}, {'source_id': 'GDCA-F-be8dc35a004b626a2cff', 'source_path': 'sources/gdca-recon/2026-11-14/files/be8dc35a004b626a2cff_702.pdf', 'source_sha256': 'a03f61339be453c9f0068e5f5ff933b66c8df6da2f3c2ef2968c579be1a0504a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/702.pdf', 'bytes': 1959987}, {'source_id': 'GDCA-F-13d4607e381d9dbcb1bf', 'source_path': 'sources/gdca-recon/2026-11-14/files/13d4607e381d9dbcb1bf_1135.pdf', 'source_sha256': '992dc62a901818c0d9dd9a0c5e080f573edf67161965d34bb71ba336d2668c93', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1135.pdf', 'bytes': 2024487}, {'source_id': 'GDCA-F-364e361faf47fe4803ef', 'source_path': 'sources/gdca-recon/2026-11-14/files/364e361faf47fe4803ef_333.pdf', 'source_sha256': '5066a7f45896429fc955dda7ddda5a738da6dad3ae9389ba76881e1b7bc446bc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/333.pdf', 'bytes': 1837681}, {'source_id': 'GDCA-F-73abef3d89fe31968014', 'source_path': 'sources/gdca-recon/2026-11-14/files/73abef3d89fe31968014_657.pdf', 'source_sha256': 'a8dec5f56c596ea081ebc378099ae4104d5a5323047446aa3ffa02a343ea992e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/657.pdf', 'bytes': 1959437}, {'source_id': 'GDCA-F-2fe2cd048d6330b0f819', 'source_path': 'sources/gdca-recon/2026-11-14/files/2fe2cd048d6330b0f819_1112.pdf', 'source_sha256': 'f67c6b8d4b85c9ca563e15192356a088fbcf6ed4f7ba61cca4864602e92ffc33', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1112.pdf', 'bytes': 2017356}, {'source_id': 'GDCA-F-2414b8afd4260cdea50c', 'source_path': 'sources/gdca-recon/2026-11-14/files/2414b8afd4260cdea50c_925.pdf', 'source_sha256': '272d04177d13897a1e3e0f0b5fd6eec53c8db20749f534ff425b01e64e5ae0fd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/925.pdf', 'bytes': 1766059}, {'source_id': 'GDCA-F-7e20dab606f54e5d3dab', 'source_path': 'sources/gdca-recon/2026-11-14/files/7e20dab606f54e5d3dab_899.pdf', 'source_sha256': '390c60820354b230335cadbda140b65c98ceb4f25680191ebaba423695c96ed3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/899.pdf', 'bytes': 2106678}, {'source_id': 'GDCA-F-b5c9d43dfce9f70c4fc7', 'source_path': 'sources/gdca-recon/2026-11-14/files/b5c9d43dfce9f70c4fc7_1019.pdf', 'source_sha256': '4498d7c68f926779f8ef3562a81e51474d1a380a40f9f710196fc4f02ff1cb0f', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1019.pdf', 'bytes': 2440415}, {'source_id': 'GDCA-F-d12c43e801ac6b841120', 'source_path': 'sources/gdca-recon/2026-11-14/files/d12c43e801ac6b841120_1001.pdf', 'source_sha256': '56be7ffa1b37394226b1ef1617bad18029d172e7c3e55bddce43ef6d00573e27', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1001.pdf', 'bytes': 1868449}, {'source_id': 'GDCA-F-362c525a9ecae65e2c9b', 'source_path': 'sources/gdca-recon/2026-11-14/files/362c525a9ecae65e2c9b_500.pdf', 'source_sha256': '6832394aad7c01ea1370808748a2920186b879c43b62e7b43ce375595e220546', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/500.pdf', 'bytes': 2398034}, {'source_id': 'GDCA-F-9d711b77a492d18467d2', 'source_path': 'sources/gdca-recon/2026-11-14/files/9d711b77a492d18467d2_737.pdf', 'source_sha256': '8dbefd8c2c87af09f2991008e3e0438b2bc688db8d94870a59c491695416d4d1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/737.pdf', 'bytes': 2085545}, {'source_id': 'GDCA-F-735143a9300b6960690b', 'source_path': 'sources/gdca-recon/2026-11-14/files/735143a9300b6960690b_1170.pdf', 'source_sha256': '46623fa81bf104374d0ab76b941bec92e7869bab01a2e56ee276d0f1c757ceda', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1170.pdf', 'bytes': 2009976}, {'source_id': 'GDCA-F-e4e689adf823c009b75e', 'source_path': 'sources/gdca-recon/2026-11-14/files/e4e689adf823c009b75e_683.pdf', 'source_sha256': '3234f6b2dbe33c9e1817753a02a8081b08b318ba5fc06f67e0c0039e4b47a773', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/683.pdf', 'bytes': 1772066}, {'source_id': 'GDCA-F-5ec9f85388499daa0f52', 'source_path': 'sources/gdca-recon/2026-11-14/files/5ec9f85388499daa0f52_622.pdf', 'source_sha256': '8e3813ba8cf1e7d297bd6fd5d9af9a04dba81846fc61fb3dde2014574c5df952', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/622.pdf', 'bytes': 1802139}, {'source_id': 'GDCA-F-557412642bf4b31ffd8b', 'source_path': 'sources/gdca-recon/2026-11-14/files/557412642bf4b31ffd8b_760.pdf', 'source_sha256': '0c7a531b80ad3adae1e24a7d3ee8e9d0e6c70119c7325f7f724120e5d817675c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/760.pdf', 'bytes': 1930159}, {'source_id': 'GDCA-F-8c73d7307949140318dc', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c73d7307949140318dc_1564.pdf', 'source_sha256': 'd7cba830539247b779c510488e7c6516d366acedf455114c730dbc5909b50320', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1564.pdf', 'bytes': 3490275}, {'source_id': 'GDCA-F-52b2ef07a9ede072d1f0', 'source_path': 'sources/gdca-recon/2026-11-14/files/52b2ef07a9ede072d1f0_227.pdf', 'source_sha256': 'a4b4a9b74eb97b03c7c341c99a32cc45e76ca49466f79388026c9e1afd7bf589', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/227.pdf', 'bytes': 1950593}, {'source_id': 'GDCA-F-e6f8654292f2b2b8d3cc', 'source_path': 'sources/gdca-recon/2026-11-14/files/e6f8654292f2b2b8d3cc_1577.pdf', 'source_sha256': '8f70906e5e3f3f79181029cad2240816f3e047536a5b5495162dbd295cd11e01', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1577.pdf', 'bytes': 3283546}, {'source_id': 'GDCA-F-440ed4d73da54ff8792f', 'source_path': 'sources/gdca-recon/2026-11-14/files/440ed4d73da54ff8792f_1490.pdf', 'source_sha256': '213eaeb34ff9a73bbfceafee853951d45d844093d01e605784de75ee973285ca', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1490.pdf', 'bytes': 1983460}, {'source_id': 'GDCA-F-0a88b18e2ae5a240dcb7', 'source_path': 'sources/gdca-recon/2026-11-14/files/0a88b18e2ae5a240dcb7_525.pdf', 'source_sha256': '3282c886dcb92023abdc198320bbeb2f661394c65dd6ff9bacea8cfb6443ed21', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/525.pdf', 'bytes': 2399771}, {'source_id': 'GDCA-F-a31de8e8c8c8b28c0ed6', 'source_path': 'sources/gdca-recon/2026-11-14/files/a31de8e8c8c8b28c0ed6_359.pdf', 'source_sha256': '8cce7d046676a6bf48770d928f870cd347dd6bbe4131129afe3eeb9d061832cd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/359.pdf', 'bytes': 1936048}, {'source_id': 'GDCA-F-a5ae3ae2552e0d93236a', 'source_path': 'sources/gdca-recon/2026-11-14/files/a5ae3ae2552e0d93236a_820.pdf', 'source_sha256': '62d7c1ca3b56f8913021fc6c6b5fd8ce68f08edee25e6a8568645a0afdc91250', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/820.pdf', 'bytes': 3223431}, {'source_id': 'GDCA-F-df3d93cafffd7e0c15a4', 'source_path': 'sources/gdca-recon/2026-11-14/files/df3d93cafffd7e0c15a4_1107.pdf', 'source_sha256': 'aff02eec2267d29b226e51b9cb69c18b8e9e47814f7e728f332e67bc4110a6b3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1107.pdf', 'bytes': 1957747}, {'source_id': 'GDCA-F-6d37a8c9345a3cc069af', 'source_path': 'sources/gdca-recon/2026-11-14/files/6d37a8c9345a3cc069af_1141.pdf', 'source_sha256': '101c993d6d4835b1da674ae00879799da3d05d541afe469347417d0ba333d699', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1141.pdf', 'bytes': 1370774}, {'source_id': 'GDCA-F-21f65532430919597235', 'source_path': 'sources/gdca-recon/2026-11-14/files/21f65532430919597235_1096.pdf', 'source_sha256': 'd3c50641b21b0e3b4bb4dd9f867fb9229334b4dd7e1b8a4da52f301f81e5fa9e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1096.pdf', 'bytes': 2200671}, {'source_id': 'GDCA-F-16fe03a2a2a10af3cc45', 'source_path': 'sources/gdca-recon/2026-11-14/files/16fe03a2a2a10af3cc45_1430.pdf', 'source_sha256': '7ca30643ce3df6ccd6efa35cf2122803cf13d5fd6f88903f2f62007eeecfe1b1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1430.pdf', 'bytes': 2270693}, {'source_id': 'GDCA-F-ccfb7279dbd49fadbe80', 'source_path': 'sources/gdca-recon/2026-11-14/files/ccfb7279dbd49fadbe80_644.pdf', 'source_sha256': 'a24fe5a1245514465e9ae109086fbee905b1094de5d9d0e35c28008688ed6706', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/644.pdf', 'bytes': 2181705}, {'source_id': 'GDCA-F-72f6c20eb210dc90eb09', 'source_path': 'sources/gdca-recon/2026-11-14/files/72f6c20eb210dc90eb09_374.pdf', 'source_sha256': '23b411ee4748e44d5602f5a1af55afda99b16698782c2ec0389faa5fc0e60f1e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/374.pdf', 'bytes': 2252849}, {'source_id': 'GDCA-F-62367b8952263dd42e2a', 'source_path': 'sources/gdca-recon/2026-11-14/files/62367b8952263dd42e2a_779.pdf', 'source_sha256': '8d6f6cdc5d082b5e7eae6c4f44ae0b7980b16b31246647d2aa21a78a1f1227a4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/779.pdf', 'bytes': 2126332}, {'source_id': 'GDCA-F-daefd722039a14cfe196', 'source_path': 'sources/gdca-recon/2026-11-14/files/daefd722039a14cfe196_172.pdf', 'source_sha256': 'ac3259a806b64fc77ebc6b546f113c361c259c8a60cd3c16f99a48a69caca57f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/172.pdf', 'bytes': 1946417}, {'source_id': 'GDCA-F-9d655612801e8d213746', 'source_path': 'sources/gdca-recon/2026-11-14/files/9d655612801e8d213746_407.pdf', 'source_sha256': 'f2d62ebf0209b58d97e915efd0b6be2df0022ce371a5fca55da210518d99ac8b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/407.pdf', 'bytes': 2102949}, {'source_id': 'GDCA-F-e95ff3cc9e5866d37427', 'source_path': 'sources/gdca-recon/2026-11-14/files/e95ff3cc9e5866d37427_952.pdf', 'source_sha256': '7c5d9b3ede70ac733abd87744b66084fb77bc75237ccecc1355294ad41368979', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/952.pdf', 'bytes': 2185590}, {'source_id': 'GDCA-F-8f5fef4be8982eb96918', 'source_path': 'sources/gdca-recon/2026-11-14/files/8f5fef4be8982eb96918_267.pdf', 'source_sha256': '0f0061fbcd846a0f37930a8f1a0912d57a57994350390591053d71c83ac29093', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/267.pdf', 'bytes': 1839308}, {'source_id': 'GDCA-F-ecfd4b5733b283aaaf96', 'source_path': 'sources/gdca-recon/2026-11-14/files/ecfd4b5733b283aaaf96_58.pdf', 'source_sha256': 'ff6ef97c9a0ad84ba748ededa42a1fddad28d5456303ddb33a8bd5deddc5cefa', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/58.pdf', 'bytes': 2328842}, {'source_id': 'GDCA-F-74bc23f4eaeaf7b8da2d', 'source_path': 'sources/gdca-recon/2026-11-14/files/74bc23f4eaeaf7b8da2d_767.pdf', 'source_sha256': '2defb732cf167e6b40328099c50417cf235fde5f2e4d7f4a287466c1bb070b4c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/767.pdf', 'bytes': 2105920}, {'source_id': 'GDCA-F-4dc669eb7f2c7d76fd52', 'source_path': 'sources/gdca-recon/2026-11-14/files/4dc669eb7f2c7d76fd52_1462.pdf', 'source_sha256': 'b8188b09b447e30e95b43e6084b4630ff16069603dcceb6b8a5b71a89466a92d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1462.pdf', 'bytes': 2335545}, {'source_id': 'GDCA-F-0a38520bd98e5d7d7341', 'source_path': 'sources/gdca-recon/2026-11-14/files/0a38520bd98e5d7d7341_758.pdf', 'source_sha256': 'a62c17612eb496c02854a79d75d719a3183ce6c5543446e635d077cbc348084b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/758.pdf', 'bytes': 1850586}, {'source_id': 'GDCA-F-3884eff3654e3a26544e', 'source_path': 'sources/gdca-recon/2026-11-14/files/3884eff3654e3a26544e_1167.pdf', 'source_sha256': 'c2ba30a3c1f00295fa0f8a1b9a6efb0e62347f1bb54d64d07fc3f9db42e4520d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1167.pdf', 'bytes': 1897272}, {'source_id': 'GDCA-F-60c669b93966bafca934', 'source_path': 'sources/gdca-recon/2026-11-14/files/60c669b93966bafca934_1565.pdf', 'source_sha256': 'adae516b439d3ebbd8363eb574ae7da9e42add341e41fab10b92e654a612cccf', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1565.pdf', 'bytes': 3535761}, {'source_id': 'GDCA-F-b063c276399c945729a7', 'source_path': 'sources/gdca-recon/2026-11-14/files/b063c276399c945729a7_741.pdf', 'source_sha256': '3ac022f1e27e4a4f1fcb8f37d3d1938c36614276864c42f5f28dba992c01691e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/741.pdf', 'bytes': 1712145}, {'source_id': 'GDCA-F-b7a6f4fb213a469684b8', 'source_path': 'sources/gdca-recon/2026-11-14/files/b7a6f4fb213a469684b8_1396.pdf', 'source_sha256': '0ffae9f909a55bc4151c5b0f6fef7cb5147a45e79e7a8a8e652ba78cd47d583f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1396.pdf', 'bytes': 2324592}, {'source_id': 'GDCA-F-0684916af77e6b1d5069', 'source_path': 'sources/gdca-recon/2026-11-14/files/0684916af77e6b1d5069_695.pdf', 'source_sha256': '3c99337f227d8d69abc32a04a36a2f3305f8d6fc77c2442e9735de720058ac28', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/695.pdf', 'bytes': 2467865}, {'source_id': 'GDCA-F-5c1987fa323cea0ceec6', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c1987fa323cea0ceec6_283.pdf', 'source_sha256': '892c9603f37b1834e19f88767e61e85671830bea14f011d718293c36d8358f17', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/283.pdf', 'bytes': 1922888}, {'source_id': 'GDCA-F-c8106502c32cce68a2f7', 'source_path': 'sources/gdca-recon/2026-11-14/files/c8106502c32cce68a2f7_740.pdf', 'source_sha256': '3634db34b6834fcc7c366ddd1fe8a71eb3bedfb7885d13bab5c0f22a7bac8ab9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/740.pdf', 'bytes': 1827042}, {'source_id': 'GDCA-F-b3e6ddcdeda70cf7659a', 'source_path': 'sources/gdca-recon/2026-11-14/files/b3e6ddcdeda70cf7659a_1163.pdf', 'source_sha256': 'ac8ff22f685ef808820fe9d1bad4648db9556ff52fd825e02555e5a2d0108a50', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1163.pdf', 'bytes': 1940509}, {'source_id': 'GDCA-F-34370db73cacdbe41c9f', 'source_path': 'sources/gdca-recon/2026-11-14/files/34370db73cacdbe41c9f_1224.pdf', 'source_sha256': 'b1041b446dae91274c330223ba345e5c7702d1bd96aa21a828dee9b0c4e8ca5a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1224.pdf', 'bytes': 1916567}, {'source_id': 'GDCA-F-b3b07a482fded3e69df4', 'source_path': 'sources/gdca-recon/2026-11-14/files/b3b07a482fded3e69df4_3.pdf', 'source_sha256': 'f40adc83d8478370d063f71074eeefc6e639472bb6240484f1fac78eca09183b', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/3.pdf', 'bytes': 2668366}, {'source_id': 'GDCA-F-3aa5aa33faf4405cd6f4', 'source_path': 'sources/gdca-recon/2026-11-14/files/3aa5aa33faf4405cd6f4_1101.pdf', 'source_sha256': '1ee0195978b682ea197d51c558bbd7ba5117738109c0be3eca41929e581ba2d5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1101.pdf', 'bytes': 1696916}, {'source_id': 'GDCA-F-bbea7a833640e674c018', 'source_path': 'sources/gdca-recon/2026-11-14/files/bbea7a833640e674c018_1193.pdf', 'source_sha256': '6c7b576a329840ddcd838e57cd0434fefd217639cd9b4d3365116da2d57f13bc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1193.pdf', 'bytes': 1889766}, {'source_id': 'GDCA-F-27b59cf8c2dceed649cb', 'source_path': 'sources/gdca-recon/2026-11-14/files/27b59cf8c2dceed649cb_161.pdf', 'source_sha256': '6cb9ebc65b073d5c67309a70cb421d057e3870a22d1e420f181216241742dcab', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/161.pdf', 'bytes': 2066166}, {'source_id': 'GDCA-F-ae7df0be9bf420ad7b4e', 'source_path': 'sources/gdca-recon/2026-11-14/files/ae7df0be9bf420ad7b4e_1290.pdf', 'source_sha256': 'cff0464cb5a9d128f886198c04518a83e080a3f02fcbb4574fb65a9d404492a0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1290.pdf', 'bytes': 1935159}, {'source_id': 'GDCA-F-cc240c7792fc8a453915', 'source_path': 'sources/gdca-recon/2026-11-14/files/cc240c7792fc8a453915_598.pdf', 'source_sha256': 'f1a368522be7c008ebd85df9e4e8221463f189264d7cfffc1bb6ca39440c19b5', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/598.pdf', 'bytes': 2016446}, {'source_id': 'GDCA-F-157a73ff524cff32abc7', 'source_path': 'sources/gdca-recon/2026-11-14/files/157a73ff524cff32abc7_1388.pdf', 'source_sha256': 'cce24d3125000ae6b46ad51806f55e4723f1ae6b4b6b9180025217179cfd0f3a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1388.pdf', 'bytes': 2086865}, {'source_id': 'GDCA-F-fb60b287e4e9a4e02168', 'source_path': 'sources/gdca-recon/2026-11-14/files/fb60b287e4e9a4e02168_257.pdf', 'source_sha256': 'ce9fbdb5bacfa3dfcc3dfbfb8d2c450669b68dd31c70d554971c5d42d9643107', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/257.pdf', 'bytes': 1998775}, {'source_id': 'GDCA-F-78dd21551eaf7500e52f', 'source_path': 'sources/gdca-recon/2026-11-14/files/78dd21551eaf7500e52f_1438.pdf', 'source_sha256': '2ae5bf3850c8c54ac7d5916237fd13eb10533cec1a09e042cab46b36dfee61df', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1438.pdf', 'bytes': 2365504}, {'source_id': 'GDCA-F-933d515f7fb1e6905b2b', 'source_path': 'sources/gdca-recon/2026-11-14/files/933d515f7fb1e6905b2b_869.pdf', 'source_sha256': 'f70056f4a8c7c2ab7588be59d9b036853f30a70c7a287cc70fafe22241cc5464', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/869.pdf', 'bytes': 1878837}, {'source_id': 'GDCA-F-9a1d71576678e750607f', 'source_path': 'sources/gdca-recon/2026-11-14/files/9a1d71576678e750607f_1418.pdf', 'source_sha256': 'be9a08859cb18229af51ce272202799fb9610f0b1453bdcf755373ace11fe5b3', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1418.pdf', 'bytes': 4043023}, {'source_id': 'GDCA-F-f6470c66c11957583402', 'source_path': 'sources/gdca-recon/2026-11-14/files/f6470c66c11957583402_400.pdf', 'source_sha256': '57e9c7ce5fef0e0b2942cc2ba6cac51ae52df1d58780c9e6173257c939caa03e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/400.pdf', 'bytes': 2327304}, {'source_id': 'GDCA-F-2f159ef240c0ff30eca7', 'source_path': 'sources/gdca-recon/2026-11-14/files/2f159ef240c0ff30eca7_923.pdf', 'source_sha256': '3fd78c974e76ce41e28b103127e13d51ab1668a12f04b9fc8fa361c5928b3b2b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/923.pdf', 'bytes': 2169079}, {'source_id': 'GDCA-F-37151b22f236226ca98e', 'source_path': 'sources/gdca-recon/2026-11-14/files/37151b22f236226ca98e_430.pdf', 'source_sha256': '47ed9d7fa549b2903755e792740e732f2be3407d06b506217fd87d3d231e2e05', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/430.pdf', 'bytes': 2242751}, {'source_id': 'GDCA-F-5adb886a13bbc5e7a0b2', 'source_path': 'sources/gdca-recon/2026-11-14/files/5adb886a13bbc5e7a0b2_907.pdf', 'source_sha256': '23f9c281917739020ebb351b8f2ef03a4dcab617c7f9142589153b3ef55b392c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/907.pdf', 'bytes': 2011489}, {'source_id': 'GDCA-F-3bb4fa5274765fa8f494', 'source_path': 'sources/gdca-recon/2026-11-14/files/3bb4fa5274765fa8f494_1077.pdf', 'source_sha256': '736275bbc46d16430686edd972b8905e60896b3b42561e8547a23a4e9c51f1a5', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1077.pdf', 'bytes': 2290458}, {'source_id': 'GDCA-F-396eb0391226c6697843', 'source_path': 'sources/gdca-recon/2026-11-14/files/396eb0391226c6697843_1262.pdf', 'source_sha256': 'eb5c53b4cc5c4c1243891a80c2ebc632ae22f5b444098637a9bf870776840d1d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1262.pdf', 'bytes': 2026293}, {'source_id': 'GDCA-F-b36f169c9caebc32bc32', 'source_path': 'sources/gdca-recon/2026-11-14/files/b36f169c9caebc32bc32_410.pdf', 'source_sha256': 'ce3afe6e5b786da9ee7de20db66233dcab9aa22cf5f2190fd5f6fa10429805fe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/410.pdf', 'bytes': 2236315}, {'source_id': 'GDCA-F-a32c4dff5822d6532eb3', 'source_path': 'sources/gdca-recon/2026-11-14/files/a32c4dff5822d6532eb3_560.pdf', 'source_sha256': 'c344e36fa1c7654b5ee138698a1cc5aefe2e7fc64093c73b707f59d60a095a06', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/560.pdf', 'bytes': 1877448}, {'source_id': 'GDCA-F-bf1e0b0e0870d76b5618', 'source_path': 'sources/gdca-recon/2026-11-14/files/bf1e0b0e0870d76b5618_224.pdf', 'source_sha256': '737cc5aa74cb20a4e8ff21cd09fbf4b177d2646defedea2aa37d58e7a769981d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/224.pdf', 'bytes': 1908955}, {'source_id': 'GDCA-F-f2f559d183cd4a607879', 'source_path': 'sources/gdca-recon/2026-11-14/files/f2f559d183cd4a607879_122.pdf', 'source_sha256': '5fe784b3f306b5d8ffe4f283e1ab27c8aa385fd9326e7bbfe6424e9b275240a4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/122.pdf', 'bytes': 2006497}, {'source_id': 'GDCA-F-b60b44023c4b9464b7f3', 'source_path': 'sources/gdca-recon/2026-11-14/files/b60b44023c4b9464b7f3_1018.pdf', 'source_sha256': 'd97d1b1cd49cd3c968a9703a81ad13f97cfbda62c4f281fdc58fe658414e618c', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1018.pdf', 'bytes': 2319881}, {'source_id': 'GDCA-F-51fb8680b5b0e9f70474', 'source_path': 'sources/gdca-recon/2026-11-14/files/51fb8680b5b0e9f70474_978.pdf', 'source_sha256': '9bd94ee5047517a83a6807315e9a15fe8fe671b4503558aaf87fa844beb7d3f3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/978.pdf', 'bytes': 2231673}, {'source_id': 'GDCA-F-d23a9cfb148ae1c78802', 'source_path': 'sources/gdca-recon/2026-11-14/files/d23a9cfb148ae1c78802_304.pdf', 'source_sha256': '87c7d8ea11cabc4fbb3dfa2481237f01d657c06beb8aa11097d565d81310e174', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/304.pdf', 'bytes': 2162981}, {'source_id': 'GDCA-F-762bde25d9754ffea0fa', 'source_path': 'sources/gdca-recon/2026-11-14/files/762bde25d9754ffea0fa_839.pdf', 'source_sha256': '670eee3d31e3c093db29b28fdaf1f4a806231e68b40f694d1437998c15c33dfa', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/839.pdf', 'bytes': 1956525}, {'source_id': 'GDCA-F-e9ba0337f77e6de2780e', 'source_path': 'sources/gdca-recon/2026-11-14/files/e9ba0337f77e6de2780e_492.pdf', 'source_sha256': 'ef77cfb4fc1d0bb429e44acf7982cc324050f4786a4241aa3019910c0f6ffdca', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/492.pdf', 'bytes': 1802349}, {'source_id': 'GDCA-F-794402c865725cf58254', 'source_path': 'sources/gdca-recon/2026-11-14/files/794402c865725cf58254_483.pdf', 'source_sha256': 'b40e42d4e6f090659982b43c1920e9d3e0a9cdc45cdf0b0e81a42d0b04a69966', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/483.pdf', 'bytes': 1662478}, {'source_id': 'GDCA-F-021f59961b280ab67c95', 'source_path': 'sources/gdca-recon/2026-11-14/files/021f59961b280ab67c95_111.pdf', 'source_sha256': '69d20e265965eb37d740ea7dc6ee55d5727aad94e2f83440d2d77933dbdad6d4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/111.pdf', 'bytes': 2327518}, {'source_id': 'GDCA-F-e75ceb8d99f49ba853dd', 'source_path': 'sources/gdca-recon/2026-11-14/files/e75ceb8d99f49ba853dd_321.pdf', 'source_sha256': '0c43affa67eb24af2e649a16f3a0ff4590acf3d60dd82a5fa2525bee209f888a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/321.pdf', 'bytes': 1988465}, {'source_id': 'GDCA-F-21dca950c7f77f2a98fa', 'source_path': 'sources/gdca-recon/2026-11-14/files/21dca950c7f77f2a98fa_642.pdf', 'source_sha256': '63abb8e5b021c33021530b08caedab6a7f1ff5d2a018bcae435dda4a659ca2dc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/642.pdf', 'bytes': 1816765}, {'source_id': 'GDCA-F-146563c9eba0d9b73b7f', 'source_path': 'sources/gdca-recon/2026-11-14/files/146563c9eba0d9b73b7f_750.pdf', 'source_sha256': '576ebe7f2fc43d22d8d024e6720165be2800ce85e85464b20282ee4edb8f4ef9', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/750.pdf', 'bytes': 2501128}, {'source_id': 'GDCA-F-22d883bde2a34c12587d', 'source_path': 'sources/gdca-recon/2026-11-14/files/22d883bde2a34c12587d_855.pdf', 'source_sha256': 'bffa8173e2886bb67eae7a415bde8d8854204a9829ef67060d18910a7c21d27d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/855.pdf', 'bytes': 2602929}, {'source_id': 'GDCA-F-e68720886599a71e7cc8', 'source_path': 'sources/gdca-recon/2026-11-14/files/e68720886599a71e7cc8_1013.pdf', 'source_sha256': '7006d4b008dcfdf09842c4ca810bf98014f0a9af4a4840eb443234739b216185', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1013.pdf', 'bytes': 2129712}, {'source_id': 'GDCA-F-710874046e3ce6af6de8', 'source_path': 'sources/gdca-recon/2026-11-14/files/710874046e3ce6af6de8_1130.pdf', 'source_sha256': '3350df3bd29aee2700167ef44057b6e695fa5588327774502568efadbc87b5eb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1130.pdf', 'bytes': 1865484}, {'source_id': 'GDCA-F-4a1dc111c778c93002ed', 'source_path': 'sources/gdca-recon/2026-11-14/files/4a1dc111c778c93002ed_1156.pdf', 'source_sha256': '9c6199e561ee4896b3983f5e753d63c1030917fc521e7000c62b6be5c2b5325d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1156.pdf', 'bytes': 1927613}, {'source_id': 'GDCA-F-93ca6ad8c517bc1d1f21', 'source_path': 'sources/gdca-recon/2026-11-14/files/93ca6ad8c517bc1d1f21_1020.pdf', 'source_sha256': '3c373778adfcfc4559015cb18bf7bd4d6e43caba51c85c9a94244bad47e8b548', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1020.pdf', 'bytes': 2307898}, {'source_id': 'GDCA-F-c4ab0cf1f1adbeff0735', 'source_path': 'sources/gdca-recon/2026-11-14/files/c4ab0cf1f1adbeff0735_1203.pdf', 'source_sha256': '3f7abbea131712de7930824428241d753daea8810ba7851b8d44fdbe57bacb07', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1203.pdf', 'bytes': 2028768}, {'source_id': 'GDCA-F-02c42d633df8974d9376', 'source_path': 'sources/gdca-recon/2026-11-14/files/02c42d633df8974d9376_33.pdf', 'source_sha256': 'de252ce24c45e06f0ac75f580f88ec33c9826d845f0175368d728e169a4c71c3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/33.pdf', 'bytes': 98168}, {'source_id': 'GDCA-F-4af17b64d1e91919ac40', 'source_path': 'sources/gdca-recon/2026-11-14/files/4af17b64d1e91919ac40_217.pdf', 'source_sha256': '2de7b8d615d6ed9148f41ed883db2932baf4a64b1d4ab79e26c1366d683479e0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/217.pdf', 'bytes': 1846028}, {'source_id': 'GDCA-F-601c067e480ad3acded1', 'source_path': 'sources/gdca-recon/2026-11-14/files/601c067e480ad3acded1_1481.pdf', 'source_sha256': 'c29a2392d256272a15461250e519a521ab35cd488dd2cee2786a0ac24ea97a70', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1481.pdf', 'bytes': 2625335}, {'source_id': 'GDCA-F-29ffd0e37b4533c6fcdc', 'source_path': 'sources/gdca-recon/2026-11-14/files/29ffd0e37b4533c6fcdc_317.pdf', 'source_sha256': 'c7eb332bef019cb0967031a0d46048bcdfae3b38423e95d6208f5b62b2bb3cbe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/317.pdf', 'bytes': 2077092}, {'source_id': 'GDCA-F-1cf07a514a41fa0e3f18', 'source_path': 'sources/gdca-recon/2026-11-14/files/1cf07a514a41fa0e3f18_1380.pdf', 'source_sha256': 'd3cab86837bec985c7122c490c745349730121bbb4caf25b3b7663de94fb27e1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1380.pdf', 'bytes': 2137472}, {'source_id': 'GDCA-F-aa09ebf7337764aba5aa', 'source_path': 'sources/gdca-recon/2026-11-14/files/aa09ebf7337764aba5aa_370.pdf', 'source_sha256': '4876754ddb505fd037891a32efb36964a03920186f12e7a60fdbfe51aa7cbfbd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/370.pdf', 'bytes': 2011392}, {'source_id': 'GDCA-F-4ef9f649628ad00b4923', 'source_path': 'sources/gdca-recon/2026-11-14/files/4ef9f649628ad00b4923_587.pdf', 'source_sha256': '18df5a03a60bfde4c63f0097920509138d84a74c97e99af0c1e1d1e045b2b0bc', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/587.pdf', 'bytes': 2372948}, {'source_id': 'GDCA-F-94706ba47dc336c15d59', 'source_path': 'sources/gdca-recon/2026-11-14/files/94706ba47dc336c15d59_1067.pdf', 'source_sha256': '3c375e166b2c29fc35a585790c9156bef39857bb407418cf3d8e34222596c1f2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1067.pdf', 'bytes': 2078792}, {'source_id': 'GDCA-F-622d217989281ca74002', 'source_path': 'sources/gdca-recon/2026-11-14/files/622d217989281ca74002_962.pdf', 'source_sha256': 'dfdd1b4198b24485c0ec1351c0436567dcf469ad8615c8a1c6827c795aa42fd7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/962.pdf', 'bytes': 1795394}, {'source_id': 'GDCA-F-f3031731ed58f986f672', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3031731ed58f986f672_425.pdf', 'source_sha256': '6a2120a049b25bc744bc1e82f644c91feced6cd2dbf47bae1c8dc08b1912da89', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/425.pdf', 'bytes': 7736932}, {'source_id': 'GDCA-F-c1cb72312dfa1c7e2f60', 'source_path': 'sources/gdca-recon/2026-11-14/files/c1cb72312dfa1c7e2f60_277.pdf', 'source_sha256': '7316725232a5f870f54247a4e73deee9cb5c4e774b927c5cf82a772f6ac57087', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/277.pdf', 'bytes': 1819753}, {'source_id': 'GDCA-F-622130f400278f5ee68f', 'source_path': 'sources/gdca-recon/2026-11-14/files/622130f400278f5ee68f_1255.pdf', 'source_sha256': '1407bb760792378973ede44f9c26919bd232a62c5d817653e1d1c341d791b4d1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1255.pdf', 'bytes': 1867376}, {'source_id': 'GDCA-F-18a0a7bf05514bc893fb', 'source_path': 'sources/gdca-recon/2026-11-14/files/18a0a7bf05514bc893fb_202.pdf', 'source_sha256': 'f7ab287e857aa06a4f42178de767726bd1db9c4d1b9ecbd12219afae121390a7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/202.pdf', 'bytes': 1890270}, {'source_id': 'GDCA-F-f039631363897834b083', 'source_path': 'sources/gdca-recon/2026-11-14/files/f039631363897834b083_456.pdf', 'source_sha256': 'd8dc0355e8ad03e89543e5cbfa79835ffb2356ae1642dfe4b625413d8cf0c349', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/456.pdf', 'bytes': 1227596}, {'source_id': 'GDCA-F-fca0657f5ce56ad1f92c', 'source_path': 'sources/gdca-recon/2026-11-14/files/fca0657f5ce56ad1f92c_1460.pdf', 'source_sha256': '72eb9ba692afe8928953476daeb429d7a746b23c4670b70e7098d213c27501b2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1460.pdf', 'bytes': 2828115}, {'source_id': 'GDCA-F-3a356f57573566967fff', 'source_path': 'sources/gdca-recon/2026-11-14/files/3a356f57573566967fff_308.pdf', 'source_sha256': '341cdb2a66b9238dbc19bf389e479bfd57e075519cd24def893c88a1256df051', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/308.pdf', 'bytes': 1869728}, {'source_id': 'GDCA-F-b05d0a42746d5ba30a00', 'source_path': 'sources/gdca-recon/2026-11-14/files/b05d0a42746d5ba30a00_1083.pdf', 'source_sha256': '635c9e0552653b8b830515de7303ef05b463378d90c281b7510516557b21fcbe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1083.pdf', 'bytes': 2222214}, {'source_id': 'GDCA-F-71bbd1a9ae364e3412c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/71bbd1a9ae364e3412c1_1064.pdf', 'source_sha256': '7a5d8d2855e8988f678c7be49f84ed8232995d05e126170e334db05a82bd198e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1064.pdf', 'bytes': 1790647}, {'source_id': 'GDCA-F-79d3851f7e508c12db7f', 'source_path': 'sources/gdca-recon/2026-11-14/files/79d3851f7e508c12db7f_306.pdf', 'source_sha256': '7be0770ac494911999de6d08156d75a8c54522daa76b1423adac706e1cc4ad93', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/306.pdf', 'bytes': 1823542}, {'source_id': 'GDCA-F-26a4655df5e43b7bc442', 'source_path': 'sources/gdca-recon/2026-11-14/files/26a4655df5e43b7bc442_214.pdf', 'source_sha256': '82b4b30a20da3646aae3f66a7a1b0d5f9c0855415c335d7922e16d53f15cee8f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/214.pdf', 'bytes': 1589744}, {'source_id': 'GDCA-F-c70c310802e2d0c00eea', 'source_path': 'sources/gdca-recon/2026-11-14/files/c70c310802e2d0c00eea_535.pdf', 'source_sha256': 'e8fe91497847743cc349057e2c1ec012782cf63b1efd6429fadbeab8c918c183', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/535.pdf', 'bytes': 1778131}, {'source_id': 'GDCA-F-95f2fdf68d78575d1023', 'source_path': 'sources/gdca-recon/2026-11-14/files/95f2fdf68d78575d1023_1479.pdf', 'source_sha256': '553b8da140aaa8076a134aaca6fa21ab545806959b7b027df54fcbfed3373ea7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1479.pdf', 'bytes': 2320619}, {'source_id': 'GDCA-F-0a69408fbf4ee69cbaed', 'source_path': 'sources/gdca-recon/2026-11-14/files/0a69408fbf4ee69cbaed_256.pdf', 'source_sha256': '6ceb6f74d6c6cd29adf41f3401c331bd951b97fcdf8d2377cac045c953919c6f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/256.pdf', 'bytes': 2113433}, {'source_id': 'GDCA-F-8661fc1460e516d7593b', 'source_path': 'sources/gdca-recon/2026-11-14/files/8661fc1460e516d7593b_1050.pdf', 'source_sha256': 'edd7ee7b3118b08b9a0b535cc01b89e3a336e5d9434287842d07ce510392b687', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1050.pdf', 'bytes': 1808096}, {'source_id': 'GDCA-F-1f6e849de949c0b4119c', 'source_path': 'sources/gdca-recon/2026-11-14/files/1f6e849de949c0b4119c_563.pdf', 'source_sha256': 'f383184d357ac095f11e6347169b07a18aaaf17c393f8d7213802f3cc7e9a605', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/563.pdf', 'bytes': 2426341}, {'source_id': 'GDCA-F-0633526d3261e64e2546', 'source_path': 'sources/gdca-recon/2026-11-14/files/0633526d3261e64e2546_969.pdf', 'source_sha256': 'f4f88436b8cf21debcd9ea8b05e8550ba8efb50d23c38c6c51088948ffad99b6', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/969.pdf', 'bytes': 2438751}, {'source_id': 'GDCA-F-8552af562968f6f9e756', 'source_path': 'sources/gdca-recon/2026-11-14/files/8552af562968f6f9e756_254.pdf', 'source_sha256': '6b419cc020761fafd73280f486880dcd886e83dac51fb0c0193c451cea0f9f11', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/254.pdf', 'bytes': 1801988}, {'source_id': 'GDCA-F-24dbfb8a49fbea8180f5', 'source_path': 'sources/gdca-recon/2026-11-14/files/24dbfb8a49fbea8180f5_959.pdf', 'source_sha256': 'f1b0d7954bc7513aa4cb3eee25a9961a05a822d715b83d334b846e5cc39778d1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/959.pdf', 'bytes': 1681120}, {'source_id': 'GDCA-F-26e3411944d9ae1ff1f9', 'source_path': 'sources/gdca-recon/2026-11-14/files/26e3411944d9ae1ff1f9_196.pdf', 'source_sha256': '9c24d05a882cbab88065fc65db26707c3e2b000ac94cb7d354f82ec9819d8398', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/196.pdf', 'bytes': 1919340}, {'source_id': 'GDCA-F-b67152474e7c37115913', 'source_path': 'sources/gdca-recon/2026-11-14/files/b67152474e7c37115913_1042.pdf', 'source_sha256': '43dbef6c5ff95b5f6a22aca2694a33d5a6a6a21d0cd005a86c1f4f3317b0a807', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1042.pdf', 'bytes': 2112893}, {'source_id': 'GDCA-F-47886c5e555a9fa86afc', 'source_path': 'sources/gdca-recon/2026-11-14/files/47886c5e555a9fa86afc_1157.pdf', 'source_sha256': '21c08024f8747a3c42e68b04799451fd9454cc2aa3ba0e190c1031752d2375b3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1157.pdf', 'bytes': 1967311}, {'source_id': 'GDCA-F-74d9313b61674c820a45', 'source_path': 'sources/gdca-recon/2026-11-14/files/74d9313b61674c820a45_1401.pdf', 'source_sha256': '8709ed669289fac3870048cfba433d0b06fda90952f91d2d45779ebf455f12dd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1401.pdf', 'bytes': 1980748}, {'source_id': 'GDCA-F-5289a93f9b432ba65d80', 'source_path': 'sources/gdca-recon/2026-11-14/files/5289a93f9b432ba65d80_1398.pdf', 'source_sha256': '34b7279876994c6bba077460172fc241de95a5029ffc4bb83c48d345a3ee4408', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1398.pdf', 'bytes': 2227827}, {'source_id': 'GDCA-F-98f9802af6ecbb62ea6a', 'source_path': 'sources/gdca-recon/2026-11-14/files/98f9802af6ecbb62ea6a_204.pdf', 'source_sha256': '990010a4dd03dde95404291b19fcc715b63f0887e0c88321a9fec3b252b0ad3c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/204.pdf', 'bytes': 1852375}, {'source_id': 'GDCA-F-6a02506549e627a6179b', 'source_path': 'sources/gdca-recon/2026-11-14/files/6a02506549e627a6179b_1110.pdf', 'source_sha256': 'bee7691da2482c810a01944890fb702ca7f5cd22ea9414b2738cce6c9a6acd74', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1110.pdf', 'bytes': 1839672}, {'source_id': 'GDCA-F-d67987dfe97499080d3f', 'source_path': 'sources/gdca-recon/2026-11-14/files/d67987dfe97499080d3f_1008.pdf', 'source_sha256': '843cb13d3e07e0236be3964e1d8d4da71137996af7d4e50976fe25edfea92956', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1008.pdf', 'bytes': 2331907}, {'source_id': 'GDCA-F-537c5a3272d97baeb659', 'source_path': 'sources/gdca-recon/2026-11-14/files/537c5a3272d97baeb659_27.pdf', 'source_sha256': 'e7739b8714aa869c2b61f415eb3dffbb7c0e949d2e4e9a6caf89d1163472fec2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/27.pdf', 'bytes': 93835}, {'source_id': 'GDCA-F-86135861c2c0d51742f4', 'source_path': 'sources/gdca-recon/2026-11-14/files/86135861c2c0d51742f4_1493.pdf', 'source_sha256': 'd856747a1dc954302d8fdb9b4bc38b86b8e2716e8e3b17cf7712c956443ded57', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1493.pdf', 'bytes': 2215906}, {'source_id': 'GDCA-F-d3c5952768080eff9ee6', 'source_path': 'sources/gdca-recon/2026-11-14/files/d3c5952768080eff9ee6_1015.pdf', 'source_sha256': 'd71adc4ba225f46f09b0ed9cf764319fbffd527eb79ee0d23268edf7d5af2f4c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1015.pdf', 'bytes': 2035919}, {'source_id': 'GDCA-F-bae668acac1070db5cd5', 'source_path': 'sources/gdca-recon/2026-11-14/files/bae668acac1070db5cd5_1315.pdf', 'source_sha256': '26b52e4646099843e4d50a2532356dc5924d6195afa7c90c8717584531a35f65', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1315.pdf', 'bytes': 1242452}, {'source_id': 'GDCA-F-d349e5c424b06a82d487', 'source_path': 'sources/gdca-recon/2026-11-14/files/d349e5c424b06a82d487_230.pdf', 'source_sha256': 'd3619285f1b19c5501368f2b72ae1cf47a0d869180aafe53c6aa7d1ad7964a04', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/230.pdf', 'bytes': 1890331}, {'source_id': 'GDCA-F-f9eaf74a3ebfbc626a53', 'source_path': 'sources/gdca-recon/2026-11-14/files/f9eaf74a3ebfbc626a53_809.pdf', 'source_sha256': 'f29dd63f580c940a05a8a8deb45fae0d798b68dc980d3470e06c3b6e8f6d28b3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/809.pdf', 'bytes': 1796809}, {'source_id': 'GDCA-F-702dfc9e7a95793cf320', 'source_path': 'sources/gdca-recon/2026-11-14/files/702dfc9e7a95793cf320_60.pdf', 'source_sha256': '107655bc6d3a4d4fe8c730149f79565e3420a2cb2d77238f062712b16926b9e1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/60.pdf', 'bytes': 1845206}, {'source_id': 'GDCA-F-07bcee63504684b0e212', 'source_path': 'sources/gdca-recon/2026-11-14/files/07bcee63504684b0e212_357.pdf', 'source_sha256': 'fd3a918254c24f4e0552b4acd78bb0d0cfe81c2911598952671414b7653448f1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/357.pdf', 'bytes': 1878387}, {'source_id': 'GDCA-F-8b384015da3337eb10fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/8b384015da3337eb10fd_1298.pdf', 'source_sha256': '1e3894f0c9e787b6cdfe7eca5980d7dbe0a71a3f29e941685b515b2f41df8d28', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1298.pdf', 'bytes': 1873035}, {'source_id': 'GDCA-F-2ceb3cafe9f5b9bc88f8', 'source_path': 'sources/gdca-recon/2026-11-14/files/2ceb3cafe9f5b9bc88f8_360.pdf', 'source_sha256': '324990c18bee953b6875c99c6cfe26ef6db0d0435c630136652fd27da8492d3f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/360.pdf', 'bytes': 1876262}, {'source_id': 'GDCA-F-5a1bde168f22b2ae6def', 'source_path': 'sources/gdca-recon/2026-11-14/files/5a1bde168f22b2ae6def_365.pdf', 'source_sha256': 'f69b4bf3d33681550e3150006818bb9909f0797fba57c2dfbf63bdd7bb2c080a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/365.pdf', 'bytes': 2206368}, {'source_id': 'GDCA-F-4aa1981fb79e21d1d4b4', 'source_path': 'sources/gdca-recon/2026-11-14/files/4aa1981fb79e21d1d4b4_1436.pdf', 'source_sha256': '1c05b02a475d9a4a18e09c201e5c0ed6c04a05d4e214d248b29d4a2e3304a156', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1436.pdf', 'bytes': 2348062}, {'source_id': 'GDCA-F-25b6fc9ea2e68959cbec', 'source_path': 'sources/gdca-recon/2026-11-14/files/25b6fc9ea2e68959cbec_170.pdf', 'source_sha256': 'fc0fea5b1ebfe25a36317fc2897ea8729c1322134e9648b7fba60a82b017e43f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/170.pdf', 'bytes': 1868088}, {'source_id': 'GDCA-F-91f496222326d6a51ab7', 'source_path': 'sources/gdca-recon/2026-11-14/files/91f496222326d6a51ab7_616.pdf', 'source_sha256': 'f293a49be49523aad76e3fe0ee54410d0c53b42a520f1b19c3c1370265a517f2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/616.pdf', 'bytes': 1896135}, {'source_id': 'GDCA-F-a6e86a1748de37f6bb12', 'source_path': 'sources/gdca-recon/2026-11-14/files/a6e86a1748de37f6bb12_1279.pdf', 'source_sha256': '5d5e8b2659209f777d48126fa85e6fad759eab45567fbd968f8bacf06223b727', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1279.pdf', 'bytes': 1923725}, {'source_id': 'GDCA-F-cac4dbb21ef1c095835b', 'source_path': 'sources/gdca-recon/2026-11-14/files/cac4dbb21ef1c095835b_1183.pdf', 'source_sha256': '0ef2671edefa92e39c0db108b381195ae36e040282017cfa05e9666dd1242549', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1183.pdf', 'bytes': 1946375}, {'source_id': 'GDCA-F-7db7d8f6fac7c39b4f87', 'source_path': 'sources/gdca-recon/2026-11-14/files/7db7d8f6fac7c39b4f87_1155.pdf', 'source_sha256': 'ee4c6e7cb204612eee9d67e793ab50e58019e43de3c328a4cbbe96d5807df6b1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1155.pdf', 'bytes': 2072346}, {'source_id': 'GDCA-F-deee2846c3d9c3675600', 'source_path': 'sources/gdca-recon/2026-11-14/files/deee2846c3d9c3675600_1371.pdf', 'source_sha256': '275d3d16b44b1127a9f5bc6e7f4b92d8bb97286840be54894339d648feab1950', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1371.pdf', 'bytes': 1261158}, {'source_id': 'GDCA-F-46f5bc2e34eaa2aa5ca1', 'source_path': 'sources/gdca-recon/2026-11-14/files/46f5bc2e34eaa2aa5ca1_344.pdf', 'source_sha256': '7444ebd88d2d0b3986e647119b3b5c01f3c769f5286a822d2b0ce5da9be08dc7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/344.pdf', 'bytes': 2053465}, {'source_id': 'GDCA-F-7b089da0ce5d2940c281', 'source_path': 'sources/gdca-recon/2026-11-14/files/7b089da0ce5d2940c281_12.pdf', 'source_sha256': 'db2f2c200aee853781a1357c0b5e7c301cd4727c435aff6013e583d8b731ac83', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/12.pdf', 'bytes': 2850486}, {'source_id': 'GDCA-F-e4b67bb6fae9b1e4f47f', 'source_path': 'sources/gdca-recon/2026-11-14/files/e4b67bb6fae9b1e4f47f_1440.pdf', 'source_sha256': '98af48b41a86afc348369899abf79e916bfc3e475ee85dd6fb9fb3886c0dbd3d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1440.pdf', 'bytes': 2297049}, {'source_id': 'GDCA-F-d708569bec66b4c35324', 'source_path': 'sources/gdca-recon/2026-11-14/files/d708569bec66b4c35324_1062.pdf', 'source_sha256': '30103d726a3cad440631644faea72302aa5c9859378a6b63059d3b0b6a0e15ec', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1062.pdf', 'bytes': 1228198}, {'source_id': 'GDCA-F-7104059709a803ce8110', 'source_path': 'sources/gdca-recon/2026-11-14/files/7104059709a803ce8110_805.pdf', 'source_sha256': 'fa145501f8dd078741031df589acac1c1c4b60d43ea131ea412d3e192d5187c7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/805.pdf', 'bytes': 2282338}, {'source_id': 'GDCA-F-7ce7d25215cdad9918b1', 'source_path': 'sources/gdca-recon/2026-11-14/files/7ce7d25215cdad9918b1_1489.pdf', 'source_sha256': '366dd510eca98d603ca57632e092449f3b9b96d2c43f2cfe31c96daad86de17a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1489.pdf', 'bytes': 2558945}, {'source_id': 'GDCA-F-fbf48fb8da6f32e2abd7', 'source_path': 'sources/gdca-recon/2026-11-14/files/fbf48fb8da6f32e2abd7_501.pdf', 'source_sha256': 'd446a677da4c92534d7370f88b3fbf984cb9d17154c9ee817581eb7e175e0c47', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/501.pdf', 'bytes': 1779734}, {'source_id': 'GDCA-F-f7e02cf4c78e5752f442', 'source_path': 'sources/gdca-recon/2026-11-14/files/f7e02cf4c78e5752f442_609.pdf', 'source_sha256': '6713539ca8a803ee3bd36f6559f72fcb1d8b1e9ff8c2ee7833c82ca824e7ca10', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/609.pdf', 'bytes': 2303665}, {'source_id': 'GDCA-F-0502819f6a1c37552f34', 'source_path': 'sources/gdca-recon/2026-11-14/files/0502819f6a1c37552f34_607.pdf', 'source_sha256': '89501c297a4d407d4c89bbe117a13cbc341c4b47f17bcc3af5f1f755b0ece3d7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/607.pdf', 'bytes': 2120207}, {'source_id': 'GDCA-F-b071823fa07c1abc9a50', 'source_path': 'sources/gdca-recon/2026-11-14/files/b071823fa07c1abc9a50_883.pdf', 'source_sha256': '29302f253f614972be56254d52ebc586f55f105f455b0e0c642fd683dd20f3be', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/883.pdf', 'bytes': 2300491}, {'source_id': 'GDCA-F-a344dc7bedadbbe23982', 'source_path': 'sources/gdca-recon/2026-11-14/files/a344dc7bedadbbe23982_1122.pdf', 'source_sha256': '169a84c8a0ae7cffcb4dadde67347c973553e274362e9a9720c88edd6c633a07', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1122.pdf', 'bytes': 1975813}, {'source_id': 'GDCA-F-3e168a0aecc09a10f98f', 'source_path': 'sources/gdca-recon/2026-11-14/files/3e168a0aecc09a10f98f_401.pdf', 'source_sha256': '542f5862b8cc8fa68299011a212e2646aaad097879d6b1a7aa7d24e96decb296', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/401.pdf', 'bytes': 2282056}, {'source_id': 'GDCA-F-bff101f574079c4f312d', 'source_path': 'sources/gdca-recon/2026-11-14/files/bff101f574079c4f312d_582.pdf', 'source_sha256': '647f5aa8148c1b00efcffb41c6b64df940e5be87211263579420f558e39df5b9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/582.pdf', 'bytes': 2232833}, {'source_id': 'GDCA-F-d7502f9e51d530386cf1', 'source_path': 'sources/gdca-recon/2026-11-14/files/d7502f9e51d530386cf1_85.pdf', 'source_sha256': '73a434e61b76c3e0a14658b6941273b4445e9ca431de9a85f7cb504662aaa7f8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/85.pdf', 'bytes': 2077609}, {'source_id': 'GDCA-F-d151bd3313826f9b97dc', 'source_path': 'sources/gdca-recon/2026-11-14/files/d151bd3313826f9b97dc_371.pdf', 'source_sha256': 'ab7fcc5f9d5fb264919278da3046d9be7cb7725f948a8c8ef88ede04d13d12be', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/371.pdf', 'bytes': 2044958}, {'source_id': 'GDCA-F-8e375fc0e3f1966bea72', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e375fc0e3f1966bea72_332.pdf', 'source_sha256': '639a0ad3a66e77fbb3123388b80c2b53ed4398012e2b3d3e6c8300a05db29a8e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/332.pdf', 'bytes': 1945440}, {'source_id': 'GDCA-F-b53ee43c677d3fe2f135', 'source_path': 'sources/gdca-recon/2026-11-14/files/b53ee43c677d3fe2f135_1084.pdf', 'source_sha256': '8f00c6fefc81e70c6a56e07edeb8763575a8eeceefdef111ef223e5cea71916f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1084.pdf', 'bytes': 1995455}, {'source_id': 'GDCA-F-bd1c4afa8fd51d00f3b2', 'source_path': 'sources/gdca-recon/2026-11-14/files/bd1c4afa8fd51d00f3b2_53.pdf', 'source_sha256': 'd50c80111da96abc28f43209e3b0c42b2db3671d65138385d34ced577ec283e2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/53.pdf', 'bytes': 2030894}, {'source_id': 'GDCA-F-6e1e4b656b975589dafc', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e1e4b656b975589dafc_1588.pdf', 'source_sha256': 'eb35b07b9979e25e0e9707fd0fbaa61cb2c1d08ea565b7216da0365dbe428f6d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1588.pdf', 'bytes': 1485850}, {'source_id': 'GDCA-F-3c8c87fe1dcfad681b2f', 'source_path': 'sources/gdca-recon/2026-11-14/files/3c8c87fe1dcfad681b2f_1003.pdf', 'source_sha256': '7f2c654f3e87a04874b2f08d0a3d40a40c6c8421902c7a1558be10baa9672014', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1003.pdf', 'bytes': 1975704}, {'source_id': 'GDCA-F-3560b782c91013eb8525', 'source_path': 'sources/gdca-recon/2026-11-14/files/3560b782c91013eb8525_835.pdf', 'source_sha256': '66296b1569172a3f5855599173f2adaa92c7423dec8224819eb26427c86ba47a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/835.pdf', 'bytes': 1901131}, {'source_id': 'GDCA-F-2d53942bef687d421c8f', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d53942bef687d421c8f_1047.pdf', 'source_sha256': '425f7f94a32ff86f2271a62a8927b37fd5c168ea693d0842e9a0c697e6581435', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1047.pdf', 'bytes': 2031730}, {'source_id': 'GDCA-F-1dbfdb06062c1e2f5cfb', 'source_path': 'sources/gdca-recon/2026-11-14/files/1dbfdb06062c1e2f5cfb_347.pdf', 'source_sha256': '9f63ed50f0a162ba1bcdb59c89f29cf6290bb60b5f0c81bfccf954c0c4bf453e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/347.pdf', 'bytes': 2293161}, {'source_id': 'GDCA-F-7175de1c8d628bde2efa', 'source_path': 'sources/gdca-recon/2026-11-14/files/7175de1c8d628bde2efa_356.pdf', 'source_sha256': '36ed8e457c7d362fb359327301d6187a2a1afa6ac7f4e738bbc26a48b4e3d5e4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/356.pdf', 'bytes': 2009034}, {'source_id': 'GDCA-F-a31a28231de59e06f7b3', 'source_path': 'sources/gdca-recon/2026-11-14/files/a31a28231de59e06f7b3_733.pdf', 'source_sha256': 'e9e99d7917a60fd9e8e2116e3e8429ff64333b6a6ada88d536a8a47d608704cb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/733.pdf', 'bytes': 1974859}, {'source_id': 'GDCA-F-fe97615cfe34230a7cfc', 'source_path': 'sources/gdca-recon/2026-11-14/files/fe97615cfe34230a7cfc_1547.pdf', 'source_sha256': 'bdd046adb012b776da9affa2c44e7e1ff630098c7b9e79c15feec04582d8ae3a', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1547.pdf', 'bytes': 3623183}, {'source_id': 'GDCA-F-554660cc942ab0ee117f', 'source_path': 'sources/gdca-recon/2026-11-14/files/554660cc942ab0ee117f_153.pdf', 'source_sha256': 'dddbd1763d9f56da9e09259f69e201a76465dbb76b169c30315f3a6e4b5bd665', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/153.pdf', 'bytes': 1801286}, {'source_id': 'GDCA-F-38a4965969bd6e1f3836', 'source_path': 'sources/gdca-recon/2026-11-14/files/38a4965969bd6e1f3836_562.pdf', 'source_sha256': 'd406646018b3539d2d20ae89d9918f54a34bd5144732574fdbc1637aa0fe3b8e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/562.pdf', 'bytes': 1862762}, {'source_id': 'GDCA-F-d0f42ac33c8c22c5f606', 'source_path': 'sources/gdca-recon/2026-11-14/files/d0f42ac33c8c22c5f606_1421.pdf', 'source_sha256': '7f34c6a4bbc013a608b22b0f873fe4631f48797e3a95b9277461b32df4f8f971', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1421.pdf', 'bytes': 1589858}, {'source_id': 'GDCA-F-c3e9c8b884b77e23ff79', 'source_path': 'sources/gdca-recon/2026-11-14/files/c3e9c8b884b77e23ff79_977.pdf', 'source_sha256': '439490f2f253eba31c5387b9e45a3c2854f9a2f893b35e07c85d8d03381f3eb8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/977.pdf', 'bytes': 2147894}, {'source_id': 'GDCA-F-95a07640b11348cadea3', 'source_path': 'sources/gdca-recon/2026-11-14/files/95a07640b11348cadea3_284.pdf', 'source_sha256': '72bb721ff65971e3199c9f7617e1e34c3f3eddf120a0e90dfbd820247a553d71', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/284.pdf', 'bytes': 2030240}, {'source_id': 'GDCA-F-c9959c864a01cb4c3f4f', 'source_path': 'sources/gdca-recon/2026-11-14/files/c9959c864a01cb4c3f4f_167.pdf', 'source_sha256': '3877f90f79a5118ecfead9f9e4cc29b6278bc6bafb6d9f1cabda61ff282f7dba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/167.pdf', 'bytes': 1978465}, {'source_id': 'GDCA-F-3f967efa3932a5772ac9', 'source_path': 'sources/gdca-recon/2026-11-14/files/3f967efa3932a5772ac9_618.pdf', 'source_sha256': '9ae032d4b07f08de2a0df329fd105a41794d5961d23e84076d142822071fb745', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/618.pdf', 'bytes': 2129411}, {'source_id': 'GDCA-F-0f4a16190a1391850928', 'source_path': 'sources/gdca-recon/2026-11-14/files/0f4a16190a1391850928_62.pdf', 'source_sha256': 'dc8d5c797f559d556ef7e5186d07796e9636a8d15df07b1e630e2030adbf832a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/62.pdf', 'bytes': 2176181}, {'source_id': 'GDCA-F-f993520c2c2cdbefbec8', 'source_path': 'sources/gdca-recon/2026-11-14/files/f993520c2c2cdbefbec8_613.pdf', 'source_sha256': '4cf914cd5a08e9c26f0253799fcdc2221951b966486a0ade44627363757fea53', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/613.pdf', 'bytes': 1746112}, {'source_id': 'GDCA-F-2aa3495d763b82c8f704', 'source_path': 'sources/gdca-recon/2026-11-14/files/2aa3495d763b82c8f704_1180.pdf', 'source_sha256': '4fa9b955a3997da1b51b8ee8d8a7aae8bc7af23591bef0da521f3434ddabd4a0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1180.pdf', 'bytes': 1859716}, {'source_id': 'GDCA-F-94521f90bbd5f472d646', 'source_path': 'sources/gdca-recon/2026-11-14/files/94521f90bbd5f472d646_427.pdf', 'source_sha256': 'c5a2d9afce9302cba329a75917e1f23cd6ac7687fc67ece8b18b15b28e434964', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/427.pdf', 'bytes': 2365310}, {'source_id': 'GDCA-F-dfb77bbefbcae2e588da', 'source_path': 'sources/gdca-recon/2026-11-14/files/dfb77bbefbcae2e588da_895.pdf', 'source_sha256': '09c021bf2c43dced80c1ed79321fe351710131a350d57b4e89a1e36eeb2d4bcb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/895.pdf', 'bytes': 2014465}, {'source_id': 'GDCA-F-47c2bf00a773450a80b4', 'source_path': 'sources/gdca-recon/2026-11-14/files/47c2bf00a773450a80b4_1095.pdf', 'source_sha256': '571689c01b4becf5e85631a5aa45b9f26e8ef26bbcde39137d42c7eccd281200', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1095.pdf', 'bytes': 1855539}, {'source_id': 'GDCA-F-745ca1da706a89399f41', 'source_path': 'sources/gdca-recon/2026-11-14/files/745ca1da706a89399f41_1437.pdf', 'source_sha256': 'c6dc784daffc26c1d8e354f65477e3a5744d747de040f3a00cdfab3d6b9786d4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1437.pdf', 'bytes': 2273589}, {'source_id': 'GDCA-F-ad89f1a0c9a76579e3a2', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad89f1a0c9a76579e3a2_95.pdf', 'source_sha256': 'b612d7934e7613a471f79c08b1b4be1006844584980df6477b87fbc86c109efa', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/95.pdf', 'bytes': 1867946}, {'source_id': 'GDCA-F-8a08ea16cbe21db974ab', 'source_path': 'sources/gdca-recon/2026-11-14/files/8a08ea16cbe21db974ab_1136.pdf', 'source_sha256': '72e10e456aec0a03ed3d3c4a083f0a6130e56bb956f7ef357be53c65469a6bfe', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1136.pdf', 'bytes': 1886426}, {'source_id': 'GDCA-F-d601fc40ecd1512c872e', 'source_path': 'sources/gdca-recon/2026-11-14/files/d601fc40ecd1512c872e_268.pdf', 'source_sha256': '2bd1d0137e07bafccf57304ac2cbe9e8d471ed58bc413b1888c2cfc5b9ca80f4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/268.pdf', 'bytes': 2068190}, {'source_id': 'GDCA-F-4ac3fdfc09036cb34c98', 'source_path': 'sources/gdca-recon/2026-11-14/files/4ac3fdfc09036cb34c98_518.pdf', 'source_sha256': 'a216698b01da462116b278f5b9ebb0daf7265e05fbfba936d4731e3984203b88', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/518.pdf', 'bytes': 1954595}, {'source_id': 'GDCA-F-2b33e1117e0b2cf60515', 'source_path': 'sources/gdca-recon/2026-11-14/files/2b33e1117e0b2cf60515_553.pdf', 'source_sha256': '56e8115e5c23a3ac242f7885b096d281b5cec03b6a6fee9908b5889dc1510b16', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/553.pdf', 'bytes': 1767446}, {'source_id': 'GDCA-F-332a1e29dec187331f4b', 'source_path': 'sources/gdca-recon/2026-11-14/files/332a1e29dec187331f4b_788.pdf', 'source_sha256': 'd4877a45f216dab661bf7fb23599edda81269c3a4dac1d2040cd9fde046c7643', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/788.pdf', 'bytes': 1923109}, {'source_id': 'GDCA-F-66a8bf62aedb023d328e', 'source_path': 'sources/gdca-recon/2026-11-14/files/66a8bf62aedb023d328e_11.pdf', 'source_sha256': 'cd9e5f92ef85979331cc08891cd002e21c279af1d16067c53f344f364c8d7549', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/11.pdf', 'bytes': 2425795}, {'source_id': 'GDCA-F-eb883221f210294972e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb883221f210294972e3_1221.pdf', 'source_sha256': '1c8bc80b4b34dce48927698e29e442f13b1cb9f97d62f75dccdcb6a520587ad9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1221.pdf', 'bytes': 1948402}, {'source_id': 'GDCA-F-7c92d1669134f29b53cd', 'source_path': 'sources/gdca-recon/2026-11-14/files/7c92d1669134f29b53cd_951.pdf', 'source_sha256': '520f865084fc2673e2e7bd815b724987aab7167848fa38791093e08f79df5c77', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/951.pdf', 'bytes': 1588157}, {'source_id': 'GDCA-F-a9a49a4c5f04f9be2126', 'source_path': 'sources/gdca-recon/2026-11-14/files/a9a49a4c5f04f9be2126_1235.pdf', 'source_sha256': '765d9ccfd59e0b6915b5f1cbbf009b03e163c73e0fd1ee6646c6cc5e0e36f240', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1235.pdf', 'bytes': 1779321}, {'source_id': 'GDCA-F-0867bcc1406a9d19aabd', 'source_path': 'sources/gdca-recon/2026-11-14/files/0867bcc1406a9d19aabd_626.pdf', 'source_sha256': 'a7b77bc2c49587ee9b4834ae9ed0459334ee9a0d41a10f975c52374bc43333d7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/626.pdf', 'bytes': 2335406}, {'source_id': 'GDCA-F-3b766865db3caeac8304', 'source_path': 'sources/gdca-recon/2026-11-14/files/3b766865db3caeac8304_328.pdf', 'source_sha256': '7740f4c309e069eb492005f320d6532892e2ef9cbbc1314308cba755161a7c44', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/328.pdf', 'bytes': 2288456}, {'source_id': 'GDCA-F-fa605a5365c37e11807f', 'source_path': 'sources/gdca-recon/2026-11-14/files/fa605a5365c37e11807f_1390.pdf', 'source_sha256': 'fcc30ab35e7a65e8c3a31919e62b55190515254d231f1502204271021e099f4c', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1390.pdf', 'bytes': 3025112}, {'source_id': 'GDCA-F-4bc047ac059872787ceb', 'source_path': 'sources/gdca-recon/2026-11-14/files/4bc047ac059872787ceb_1259.pdf', 'source_sha256': 'cce7ac1fd963ad9ccfdd81e3409ecf1d1e2693ec9359fa91627dbc266db87673', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1259.pdf', 'bytes': 2342120}, {'source_id': 'GDCA-F-128ffc94932167ca4755', 'source_path': 'sources/gdca-recon/2026-11-14/files/128ffc94932167ca4755_409.pdf', 'source_sha256': '5e469b6b87f6671321d491f41d78edc2b829f1cb5a294c91785be2c408acfc93', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/409.pdf', 'bytes': 2158597}, {'source_id': 'GDCA-F-d5cf4fe1a9ad184cf234', 'source_path': 'sources/gdca-recon/2026-11-14/files/d5cf4fe1a9ad184cf234_1450.pdf', 'source_sha256': '23bbaa7b30a1c324b6cb9aaef1838d5b3444a461112823e4b3e5d68047f949f1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1450.pdf', 'bytes': 2353949}, {'source_id': 'GDCA-F-ce7324c9317c66c24bb0', 'source_path': 'sources/gdca-recon/2026-11-14/files/ce7324c9317c66c24bb0_439.pdf', 'source_sha256': 'b6a88a7e7c81b5bca12098e336a8acd7e25409b5f2d93f91e787a3bbed4bfad2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/439.pdf', 'bytes': 2075935}, {'source_id': 'GDCA-F-8f109a7558ab23aa51b0', 'source_path': 'sources/gdca-recon/2026-11-14/files/8f109a7558ab23aa51b0_350.pdf', 'source_sha256': '34176f79d24bed67d8a4ca627617dc602e5f62ca6a77c1be5d7d8bd88b86e3a0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/350.pdf', 'bytes': 1882671}, {'source_id': 'GDCA-F-0e7bc16938188a8acf71', 'source_path': 'sources/gdca-recon/2026-11-14/files/0e7bc16938188a8acf71_536.pdf', 'source_sha256': '4a7d314c0e0ca551f53e3ffd75ddca9f0d2efcb3a1ac9b0bd6a03bd173c53ea0', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/536.pdf', 'bytes': 1998036}, {'source_id': 'GDCA-F-0eedd8ea0a7c80e167f0', 'source_path': 'sources/gdca-recon/2026-11-14/files/0eedd8ea0a7c80e167f0_812.pdf', 'source_sha256': '34ada844396b67e12c6a1ca7b303aea4e819613c4f80f2d222fa3f1a64652dcd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/812.pdf', 'bytes': 1582759}, {'source_id': 'GDCA-F-10a4b1b8aec1a2b02fb4', 'source_path': 'sources/gdca-recon/2026-11-14/files/10a4b1b8aec1a2b02fb4_1161.pdf', 'source_sha256': '0680edab72f713ebb28a91cb9696188bc6ebb160952cd38c9ca21104ca8df09d', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1161.pdf', 'bytes': 1140345}, {'source_id': 'GDCA-F-bb9504705e2e1a22b429', 'source_path': 'sources/gdca-recon/2026-11-14/files/bb9504705e2e1a22b429_1200.pdf', 'source_sha256': 'cea8a9f3ec0392e84d3d1ff70aa8345a877674085b2aab1fc33338093bd30fa2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1200.pdf', 'bytes': 2083895}, {'source_id': 'GDCA-F-f951fd236a4cb171cee9', 'source_path': 'sources/gdca-recon/2026-11-14/files/f951fd236a4cb171cee9_1395.pdf', 'source_sha256': 'a797d2ea2fdbd4de453eb653465537e6459939484d5ffcf9748042e06b9c9214', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1395.pdf', 'bytes': 1995852}, {'source_id': 'GDCA-F-0e10824d1f86141a87cd', 'source_path': 'sources/gdca-recon/2026-11-14/files/0e10824d1f86141a87cd_1257.pdf', 'source_sha256': '9c344f1d0aff2a99eea63cb12afe1368c895b0a50f7dcc81406a161deb02ee1f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1257.pdf', 'bytes': 1769434}, {'source_id': 'GDCA-F-ea5fee86eae09e1c80a1', 'source_path': 'sources/gdca-recon/2026-11-14/files/ea5fee86eae09e1c80a1_1425.pdf', 'source_sha256': 'ca9e22e24c6b7266c595db49fd84415c452cfdc912afafda408cc49823ffb56f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1425.pdf', 'bytes': 2183396}, {'source_id': 'GDCA-F-a07804950b8a5c3ed09d', 'source_path': 'sources/gdca-recon/2026-11-14/files/a07804950b8a5c3ed09d_495.pdf', 'source_sha256': '2b0857e2f329ad3cfdc6a5522780649766d0aaa731fc748c00f32f0ef2bdc0d2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/495.pdf', 'bytes': 1779875}, {'source_id': 'GDCA-F-9296e454d1a5ac12d229', 'source_path': 'sources/gdca-recon/2026-11-14/files/9296e454d1a5ac12d229_606.pdf', 'source_sha256': 'f8a1bb2d637f8d0963c6ea5dd1e3641954b6e1ae5e422a84b1c42fc05386701d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/606.pdf', 'bytes': 2350800}, {'source_id': 'GDCA-F-0bd6fd7ee65a1547af98', 'source_path': 'sources/gdca-recon/2026-11-14/files/0bd6fd7ee65a1547af98_546.pdf', 'source_sha256': '0847d16118eb86fe2f2e025fe3f11c0d938a6e433e5365543acfad48fa9a7566', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/546.pdf', 'bytes': 1907324}, {'source_id': 'GDCA-F-6f99ca42965c21688d2f', 'source_path': 'sources/gdca-recon/2026-11-14/files/6f99ca42965c21688d2f_331.pdf', 'source_sha256': '8757c8c654fc897aca411dd785f3a0c129c437720750f0e9a85fae10c864f9f9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/331.pdf', 'bytes': 2343668}, {'source_id': 'GDCA-F-335faa029765413d9bc4', 'source_path': 'sources/gdca-recon/2026-11-14/files/335faa029765413d9bc4_269.pdf', 'source_sha256': '9e7c7dadac211aa8d698a9a1aaaa11b31a821007085d8823b1012e4eff18dabe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/269.pdf', 'bytes': 2393270}, {'source_id': 'GDCA-F-1042acdfd15f23a8be3c', 'source_path': 'sources/gdca-recon/2026-11-14/files/1042acdfd15f23a8be3c_643.pdf', 'source_sha256': 'f90ed36a28db122b6185249fa90858273fb6132e694556398ef230cb8e13ade7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/643.pdf', 'bytes': 1745258}, {'source_id': 'GDCA-F-b26e1acf65572f17a290', 'source_path': 'sources/gdca-recon/2026-11-14/files/b26e1acf65572f17a290_1078.pdf', 'source_sha256': '80f9dd55f1ffcb7465d132572aa1ba1b8879861820caf506f4e67802238313b2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1078.pdf', 'bytes': 2009167}, {'source_id': 'GDCA-F-81af7f55a269b72b553a', 'source_path': 'sources/gdca-recon/2026-11-14/files/81af7f55a269b72b553a_516.pdf', 'source_sha256': 'fa1c2a0ab14ddb0c9880d1082a18e2b5a6d1e1657978bf8e1dcb3e82e01d55fb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/516.pdf', 'bytes': 1888672}, {'source_id': 'GDCA-F-c6cfc7cc8104decec02c', 'source_path': 'sources/gdca-recon/2026-11-14/files/c6cfc7cc8104decec02c_491.pdf', 'source_sha256': '6697ba6ba2975d7d1bc6a87b12cf24bd9aa34db3f1eca90139927e3b8229ab88', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/491.pdf', 'bytes': 1621167}, {'source_id': 'GDCA-F-a96ee7772e1fb2cb8c60', 'source_path': 'sources/gdca-recon/2026-11-14/files/a96ee7772e1fb2cb8c60_1068.pdf', 'source_sha256': '31cfd8f0667bd57da53989d24478207eca8ec37963519b6b5dce862732388f9b', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1068.pdf', 'bytes': 2792010}, {'source_id': 'GDCA-F-1023332b86c0cb29283a', 'source_path': 'sources/gdca-recon/2026-11-14/files/1023332b86c0cb29283a_912.pdf', 'source_sha256': 'e8949e5d0426def4f4f6d719ffc03a78f06caaac8ac271dada31be04f4de2e08', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/912.pdf', 'bytes': 2018799}, {'source_id': 'GDCA-F-b7a4b7cea31a3ff900bf', 'source_path': 'sources/gdca-recon/2026-11-14/files/b7a4b7cea31a3ff900bf_828.pdf', 'source_sha256': 'e444f1eed67e478abf4f5fbc6364e685d5c40e51f94a93eb8393c2ad9ef860c8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/828.pdf', 'bytes': 1671183}, {'source_id': 'GDCA-F-8313bf568f4784b8151b', 'source_path': 'sources/gdca-recon/2026-11-14/files/8313bf568f4784b8151b_297.pdf', 'source_sha256': '85fb47fcd8a09a8e09a9bd752aacdc1a6d8d7250a685c798a790368c4ac71eaf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/297.pdf', 'bytes': 1605050}, {'source_id': 'GDCA-F-5fef0702d43cb72f5569', 'source_path': 'sources/gdca-recon/2026-11-14/files/5fef0702d43cb72f5569_932.pdf', 'source_sha256': '92611246fcfc5cc0228ff17d308bb55049e9eaca2785ef4054b2eff7e98fe29c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/932.pdf', 'bytes': 2103002}, {'source_id': 'GDCA-F-71d1a700031ae1b90b0e', 'source_path': 'sources/gdca-recon/2026-11-14/files/71d1a700031ae1b90b0e_751.pdf', 'source_sha256': '43b87303087350ff1c5a2e404750123bd08a646f77b982823314df4553c560aa', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/751.pdf', 'bytes': 2252796}, {'source_id': 'GDCA-F-e674750569264c687bfe', 'source_path': 'sources/gdca-recon/2026-11-14/files/e674750569264c687bfe_1496.pdf', 'source_sha256': 'aa8d86bfbcf47bc1dcef9e46283687f6e3ec779cef6b4b9ccc746a39555a55de', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1496.pdf', 'bytes': 1458402}, {'source_id': 'GDCA-F-41ea881191b697198e9a', 'source_path': 'sources/gdca-recon/2026-11-14/files/41ea881191b697198e9a_453.pdf', 'source_sha256': '450f0cb30a14a164a0cf199a32114437772b5f1af776188959812eaff454414f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/453.pdf', 'bytes': 2385666}, {'source_id': 'GDCA-F-aa594d985b6d3f7dd65f', 'source_path': 'sources/gdca-recon/2026-11-14/files/aa594d985b6d3f7dd65f_464.pdf', 'source_sha256': 'c05f7ee3730c5047ebbc3dfe96f2df67c57f68e013622a2f2618a15b5a71f4d2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/464.pdf', 'bytes': 1788233}, {'source_id': 'GDCA-F-63ff632eff18af67a2af', 'source_path': 'sources/gdca-recon/2026-11-14/files/63ff632eff18af67a2af_479.pdf', 'source_sha256': 'ef319b73725645ca0987cef0ddfc141be708328f395ba00b6608709db61dc7e4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/479.pdf', 'bytes': 1695042}, {'source_id': 'GDCA-F-f501d239396ea521e0ea', 'source_path': 'sources/gdca-recon/2026-11-14/files/f501d239396ea521e0ea_576.pdf', 'source_sha256': 'effd57ec421e3c826669809294decffe7b7ef7b497a4dd8e51883f4813957af4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/576.pdf', 'bytes': 1825814}, {'source_id': 'GDCA-F-b6c258d11ba208868401', 'source_path': 'sources/gdca-recon/2026-11-14/files/b6c258d11ba208868401_910.pdf', 'source_sha256': 'eac57d437283125da002d27e3d0283255478e3b50d199701d74921146561a4d5', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/910.pdf', 'bytes': 2141482}, {'source_id': 'GDCA-F-105253a804bcc0f54f30', 'source_path': 'sources/gdca-recon/2026-11-14/files/105253a804bcc0f54f30_561.pdf', 'source_sha256': '43c267bfb7fac574858be6e196234094a9a2b0d087ece96183e33e176e5db8c2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/561.pdf', 'bytes': 2033443}, {'source_id': 'GDCA-F-af6ea6531c007fa85579', 'source_path': 'sources/gdca-recon/2026-11-14/files/af6ea6531c007fa85579_777.pdf', 'source_sha256': '5053a8ea105b3c7a280450fd6a757de74c148ce4e77c5ae552092973ef945a8c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/777.pdf', 'bytes': 1879412}, {'source_id': 'GDCA-F-d800eff9b30ef939e99b', 'source_path': 'sources/gdca-recon/2026-11-14/files/d800eff9b30ef939e99b_508.pdf', 'source_sha256': '82532053aa192f8118e448caf450bcd7b7e64de701f3b41e15925c3c91cebe23', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/508.pdf', 'bytes': 1847821}, {'source_id': 'GDCA-F-00f60c3a41df63cfe8fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/00f60c3a41df63cfe8fd_781.pdf', 'source_sha256': '2209ec3d9a7886e9a92839220df41f868ec49f649512d26ede25cd203d3bc3c1', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/781.pdf', 'bytes': 2467080}, {'source_id': 'GDCA-F-f621c62a6c16d564306b', 'source_path': 'sources/gdca-recon/2026-11-14/files/f621c62a6c16d564306b_1549.pdf', 'source_sha256': '44e900c8c46c531b4326a09e03a4740740e514bec668e68dfc490fd81b322169', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1549.pdf', 'bytes': 3917577}, {'source_id': 'GDCA-F-39bde580c7edb16a9d42', 'source_path': 'sources/gdca-recon/2026-11-14/files/39bde580c7edb16a9d42_1184.pdf', 'source_sha256': 'c471bc8d92aad5f3e6a6c229f42b83c2c14f5377d01bed543703e265da2c5928', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1184.pdf', 'bytes': 2162270}, {'source_id': 'GDCA-F-1927b9ffc5797c83c427', 'source_path': 'sources/gdca-recon/2026-11-14/files/1927b9ffc5797c83c427_920.pdf', 'source_sha256': 'd42b76e762d3ceda06788e95d0242c1245c0bf84e2fcb995173920108ad205e8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/920.pdf', 'bytes': 1944892}, {'source_id': 'GDCA-F-5c266f626ec95ba8e4a8', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c266f626ec95ba8e4a8_1412.pdf', 'source_sha256': '2aef694842a750884d1910f89477ed56db5fa4d497381b2676ed7f9f868ab1b2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1412.pdf', 'bytes': 2437601}, {'source_id': 'GDCA-F-265075210e9e4c1556b0', 'source_path': 'sources/gdca-recon/2026-11-14/files/265075210e9e4c1556b0_1191.pdf', 'source_sha256': '14d7c91af13ee3120d7df9c978fde1cf3c6604bce3c031e1e048e6df943093d3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1191.pdf', 'bytes': 1532968}, {'source_id': 'GDCA-F-c164613ea726772734ae', 'source_path': 'sources/gdca-recon/2026-11-14/files/c164613ea726772734ae_351.pdf', 'source_sha256': 'f64b13ee626a1b49f705adfc135711fd14dbf8978a733579c29c919823edc924', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/351.pdf', 'bytes': 2059998}, {'source_id': 'GDCA-F-5037aeacf3e52889478d', 'source_path': 'sources/gdca-recon/2026-11-14/files/5037aeacf3e52889478d_17.pdf', 'source_sha256': '2783e0c5e8ae1971b3a3a57a368a0c4f8724f44c2147d7254beb522cba1441f9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/17.pdf', 'bytes': 2161203}, {'source_id': 'GDCA-F-4db004fadfa63f4d66f9', 'source_path': 'sources/gdca-recon/2026-11-14/files/4db004fadfa63f4d66f9_97.pdf', 'source_sha256': '42b35f9dbce4b9c6da69b064b04f716a20a41728e60a8597e06c2eb60df19567', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/97.pdf', 'bytes': 1858391}, {'source_id': 'GDCA-F-8256a1a7fb6f1800aab7', 'source_path': 'sources/gdca-recon/2026-11-14/files/8256a1a7fb6f1800aab7_1102.pdf', 'source_sha256': '7e2fb452ccc73f4e09273679c1d536349e289006d85b395601c11753571a0d82', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1102.pdf', 'bytes': 2057288}, {'source_id': 'GDCA-F-53d38fe911b3aa3886cf', 'source_path': 'sources/gdca-recon/2026-11-14/files/53d38fe911b3aa3886cf_129.pdf', 'source_sha256': '7e719707adf5ad44ef393ab365325dab4d5e9c2327c82af14a4a026efa3e03f2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/129.pdf', 'bytes': 2006313}, {'source_id': 'GDCA-F-cdd5dab9c87f2ea9ce82', 'source_path': 'sources/gdca-recon/2026-11-14/files/cdd5dab9c87f2ea9ce82_382.pdf', 'source_sha256': '708316b9061ed339a8b66af85f72edf53e8325da4661bae3620ceb5a02d664cf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/382.pdf', 'bytes': 2001946}, {'source_id': 'GDCA-F-fcaf526f8bb1fce4680b', 'source_path': 'sources/gdca-recon/2026-11-14/files/fcaf526f8bb1fce4680b_449.pdf', 'source_sha256': 'a9b1e8ee7b564919c079cb1923abc0ca3c4da002f397d44dd867eabf0d7cfc68', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/449.pdf', 'bytes': 2228773}, {'source_id': 'GDCA-F-e28b4425403d4504edb1', 'source_path': 'sources/gdca-recon/2026-11-14/files/e28b4425403d4504edb1_498.pdf', 'source_sha256': '05238d047fddbb8f1fe1be23bdee2057c3513c6935966b3636a455d8b51fda64', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/498.pdf', 'bytes': 1753772}, {'source_id': 'GDCA-F-4a549c8ad4851aab71d5', 'source_path': 'sources/gdca-recon/2026-11-14/files/4a549c8ad4851aab71d5_510.pdf', 'source_sha256': '4e70bf53a934d056e805fb07f3120a90bd32312163d88d4036a128f92d38a2b9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/510.pdf', 'bytes': 2042284}, {'source_id': 'GDCA-F-539bdf5a89ac957e0f97', 'source_path': 'sources/gdca-recon/2026-11-14/files/539bdf5a89ac957e0f97_904.pdf', 'source_sha256': '7f09ca8e7e006ee2a144262a9de6253d5eac27e90654e67d2ac906c9d38879da', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/904.pdf', 'bytes': 2259350}, {'source_id': 'GDCA-F-45a8d84b534261d92560', 'source_path': 'sources/gdca-recon/2026-11-14/files/45a8d84b534261d92560_1296.pdf', 'source_sha256': 'f5ba759d0836a811c22e8b15bd30be1f521abaf3d6fd41670f2d5e649726e34e', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1296.pdf', 'bytes': 2922912}, {'source_id': 'GDCA-F-5511b91148f6e213c8ac', 'source_path': 'sources/gdca-recon/2026-11-14/files/5511b91148f6e213c8ac_376.pdf', 'source_sha256': '369a5803e069c4577d2377f43f4c5710b6f39dd6e290cc1a49bc0b6e02f973a9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/376.pdf', 'bytes': 2213216}, {'source_id': 'GDCA-F-92de1128fbff9445d609', 'source_path': 'sources/gdca-recon/2026-11-14/files/92de1128fbff9445d609_813.pdf', 'source_sha256': '279b72ea7c3a61dc72a3e8da538020ff7412990f71dc186c1a0cf7e316d4a1ff', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/813.pdf', 'bytes': 1810309}, {'source_id': 'GDCA-F-7bf3e4a683e37d96b6ea', 'source_path': 'sources/gdca-recon/2026-11-14/files/7bf3e4a683e37d96b6ea_1443.pdf', 'source_sha256': '749883d340032c791a7ccb46224f19b2b483114397ff0f03d6015b7815313fda', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1443.pdf', 'bytes': 2662721}, {'source_id': 'GDCA-F-d618133f901992779637', 'source_path': 'sources/gdca-recon/2026-11-14/files/d618133f901992779637_610.pdf', 'source_sha256': '8999c4f8046a8f1d755b83ea809deec70d6779656c21594a4f4db6c190cae979', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/610.pdf', 'bytes': 1799940}, {'source_id': 'GDCA-F-abc2822f4a8b5569a925', 'source_path': 'sources/gdca-recon/2026-11-14/files/abc2822f4a8b5569a925_1044.pdf', 'source_sha256': '01ff1c4e4145f2fdcf530f018c85a6b213555a74ef9c832c9ec28707ade6c578', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1044.pdf', 'bytes': 2299016}, {'source_id': 'GDCA-F-c0bc9c3a8482e8264061', 'source_path': 'sources/gdca-recon/2026-11-14/files/c0bc9c3a8482e8264061_275.pdf', 'source_sha256': '710cd8882af56ebf5374c42d553eeb231798cec92618c52d14c33eb3a05e652f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/275.pdf', 'bytes': 1895632}, {'source_id': 'GDCA-F-a05e8db42d576a72ee42', 'source_path': 'sources/gdca-recon/2026-11-14/files/a05e8db42d576a72ee42_155.pdf', 'source_sha256': 'aafa877cfdd9048bf0db1325ffc52aa64ada2eea942493415e771f6c77f04c02', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/155.pdf', 'bytes': 2034826}, {'source_id': 'GDCA-F-95362d97bd6a309fc8fe', 'source_path': 'sources/gdca-recon/2026-11-14/files/95362d97bd6a309fc8fe_998.pdf', 'source_sha256': '4a15b11ac5fd249dc88bd21290afbf2c872eb64c56e3623c7dcf7f62ac03b298', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/998.pdf', 'bytes': 1724643}, {'source_id': 'GDCA-F-c905c2d05a12e292436e', 'source_path': 'sources/gdca-recon/2026-11-14/files/c905c2d05a12e292436e_79.pdf', 'source_sha256': 'f9e2e49778fd19274e4f41d7dd75bf833b1e585f395f323dca26334e3f600ebd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/79.pdf', 'bytes': 1927222}, {'source_id': 'GDCA-F-134eb2e24a7f6bfa97bb', 'source_path': 'sources/gdca-recon/2026-11-14/files/134eb2e24a7f6bfa97bb_1480.pdf', 'source_sha256': '5e5cfa1a30f15d55b4b64d896bc4c9015d0c3b53ed0c3dc338291e66138190f0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1480.pdf', 'bytes': 2372948}, {'source_id': 'GDCA-F-1366a552a9c1d00d290a', 'source_path': 'sources/gdca-recon/2026-11-14/files/1366a552a9c1d00d290a_302.pdf', 'source_sha256': '1ba4381b7fee39450e73ca34a80ee4d961c90fa1372a2066808a17f72879af2b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/302.pdf', 'bytes': 1808345}, {'source_id': 'GDCA-F-21f013fb092490ebed6a', 'source_path': 'sources/gdca-recon/2026-11-14/files/21f013fb092490ebed6a_39.pdf', 'source_sha256': 'd113eff7e5a2c5887fd85ba5d112f28685bbf1669a337f4dda1ffc81ce7a72b7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/39.pdf', 'bytes': 74449}, {'source_id': 'GDCA-F-1ff20f9fe232d9249bfc', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ff20f9fe232d9249bfc_1557.pdf', 'source_sha256': 'b47481d5a54bd1497e473037dadb03666103c0d6498f3aa1cbde3120a2bdd601', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1557.pdf', 'bytes': 2469528}, {'source_id': 'GDCA-F-8cfcc163417c93261765', 'source_path': 'sources/gdca-recon/2026-11-14/files/8cfcc163417c93261765_1121.pdf', 'source_sha256': '20d35c5c53841a441cc093c3111405638b38334b39f6f2f105372905a79617b3', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1121.pdf', 'bytes': 1241426}, {'source_id': 'GDCA-F-f1046ab473136b840789', 'source_path': 'sources/gdca-recon/2026-11-14/files/f1046ab473136b840789_381.pdf', 'source_sha256': '147c433799e2d59f0e4b634a4638d0e3efd29358600bcbf0aa865ebf48ba668f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/381.pdf', 'bytes': 2164394}, {'source_id': 'GDCA-F-74e2fac2449aac4c89b9', 'source_path': 'sources/gdca-recon/2026-11-14/files/74e2fac2449aac4c89b9_113.pdf', 'source_sha256': 'c7a701f8727b681498d4875ca7b517a825dea07958603894f97d40f173652f43', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/113.pdf', 'bytes': 2042333}, {'source_id': 'GDCA-F-9eb7bd3048e365d999cb', 'source_path': 'sources/gdca-recon/2026-11-14/files/9eb7bd3048e365d999cb_1472.pdf', 'source_sha256': '7b3ae9880334fc97c0a35f9115badf32b8fa75cba0c2afedae35c68deb773e50', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1472.pdf', 'bytes': 2704470}, {'source_id': 'GDCA-F-8e811a2951c11dc6fddb', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e811a2951c11dc6fddb_470.pdf', 'source_sha256': '413440d3215f7c36f6c33a36cf75531eb49cdc03d325c449259cf0574500ec0a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/470.pdf', 'bytes': 1760539}, {'source_id': 'GDCA-F-f9394133fb5f2ed02665', 'source_path': 'sources/gdca-recon/2026-11-14/files/f9394133fb5f2ed02665_944.pdf', 'source_sha256': '85b521262f83ff044b902b0642d514583280c104101dcebd0d4f48ae1de005b7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/944.pdf', 'bytes': 1951039}, {'source_id': 'GDCA-F-2b101d9484893285df6b', 'source_path': 'sources/gdca-recon/2026-11-14/files/2b101d9484893285df6b_43.pdf', 'source_sha256': 'c4d4b8bad9dd923b073ca220737acd6663c4136576b14de3fc6acc8a507c3f72', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/43.pdf', 'bytes': 105190}, {'source_id': 'GDCA-F-6380b4b7f75eb4f8bae8', 'source_path': 'sources/gdca-recon/2026-11-14/files/6380b4b7f75eb4f8bae8_721.pdf', 'source_sha256': '87ec4f180aa058431184b717758254a5cf7170e6392b9ddf9e520e36b2f4dca5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/721.pdf', 'bytes': 1737875}, {'source_id': 'GDCA-F-41cbf7681e142904ed0d', 'source_path': 'sources/gdca-recon/2026-11-14/files/41cbf7681e142904ed0d_1403.pdf', 'source_sha256': 'c8cf63cb919dbf5b4b343e4cb4810326bd15cdc248b61482401bad3c1839bcfc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1403.pdf', 'bytes': 1990974}, {'source_id': 'GDCA-F-89bfeab3e5d41a0a2604', 'source_path': 'sources/gdca-recon/2026-11-14/files/89bfeab3e5d41a0a2604_1408.pdf', 'source_sha256': '03a269b7048200ba3d96e8f3b4784f115d091a0be48d1ab0a2309a792acbe873', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1408.pdf', 'bytes': 2139403}, {'source_id': 'GDCA-F-b99a206f6b6ad74c106d', 'source_path': 'sources/gdca-recon/2026-11-14/files/b99a206f6b6ad74c106d_1431.pdf', 'source_sha256': '4d442020136a9e0affd5db1b0e81c3306e41877b175b2828970f8be962c05c5a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1431.pdf', 'bytes': 2328976}, {'source_id': 'GDCA-F-0d97a79db7a5045dc4d7', 'source_path': 'sources/gdca-recon/2026-11-14/files/0d97a79db7a5045dc4d7_157.pdf', 'source_sha256': 'cc3d678c8627c4bab5ba8a6b8187d16e6d8f337b622b1273620e82d35d340940', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/157.pdf', 'bytes': 1885944}, {'source_id': 'GDCA-F-82268e712f96bb53819e', 'source_path': 'sources/gdca-recon/2026-11-14/files/82268e712f96bb53819e_743.pdf', 'source_sha256': '4cdf33b9880d0074954673187a8e5f9769127f0ee09006a40777040575b4531a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/743.pdf', 'bytes': 2043316}, {'source_id': 'GDCA-F-70dc9f13a3e0814c30ec', 'source_path': 'sources/gdca-recon/2026-11-14/files/70dc9f13a3e0814c30ec_703.pdf', 'source_sha256': '1d088b04c843cebf89d3f924b4ff34c789aa2cd4158909945a9af6a52114634b', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/703.pdf', 'bytes': 2278839}, {'source_id': 'GDCA-F-e7a6e7579e2c33643ea9', 'source_path': 'sources/gdca-recon/2026-11-14/files/e7a6e7579e2c33643ea9_670.pdf', 'source_sha256': '7c2cd3fe985f86c9a97d357974248f3420baff9cdce3f6c1420ed6914a882ddf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/670.pdf', 'bytes': 1878216}, {'source_id': 'GDCA-F-d348582b32577ad3296f', 'source_path': 'sources/gdca-recon/2026-11-14/files/d348582b32577ad3296f_1111.pdf', 'source_sha256': '37dbde2bd72245202cb552f4a186c8f2de1478d420df14d5a496286377eb807c', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1111.pdf', 'bytes': 1153757}, {'source_id': 'GDCA-F-c86e3e35fe49d213b6d8', 'source_path': 'sources/gdca-recon/2026-11-14/files/c86e3e35fe49d213b6d8_1391.pdf', 'source_sha256': '19992b79adacc5c7ca7a28a54f36c4cc400f11ee50f1ebb3ca770ff2303f9656', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1391.pdf', 'bytes': 1330488}, {'source_id': 'GDCA-F-95b87335778ef594144b', 'source_path': 'sources/gdca-recon/2026-11-14/files/95b87335778ef594144b_258.pdf', 'source_sha256': '6f43b3514d362067eea828537ef75165f4f91b9858b674bcb5b291b5c3620892', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/258.pdf', 'bytes': 1841680}, {'source_id': 'GDCA-F-be8b238302fe9f2baec0', 'source_path': 'sources/gdca-recon/2026-11-14/files/be8b238302fe9f2baec0_1201.pdf', 'source_sha256': '2d37702c9da884111671580e1b4c71bccbc4b50aeaf27fa9894bb70e38514a57', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1201.pdf', 'bytes': 998623}, {'source_id': 'GDCA-F-e62c074193fa80c5d7f5', 'source_path': 'sources/gdca-recon/2026-11-14/files/e62c074193fa80c5d7f5_174.pdf', 'source_sha256': 'a93dc319d72744a791780291ab1e6cb9f1e1b33d4d23f7ed6dcd721beb398db9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/174.pdf', 'bytes': 1937274}, {'source_id': 'GDCA-F-4b22f8e32c6240ddadd4', 'source_path': 'sources/gdca-recon/2026-11-14/files/4b22f8e32c6240ddadd4_57.pdf', 'source_sha256': 'b57a805329c34c65bd75223fd816f17c1e719001f27ffe4098bec215bfe737e6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/57.pdf', 'bytes': 2198040}, {'source_id': 'GDCA-F-0025c5685088c3b9ac7e', 'source_path': 'sources/gdca-recon/2026-11-14/files/0025c5685088c3b9ac7e_1134.pdf', 'source_sha256': '103439c2ee4043b22b6480a73ebbae63f022b57c881725ce997f09ba08ed20ca', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1134.pdf', 'bytes': 1861247}, {'source_id': 'GDCA-F-f58e11aebdd2fa22e49a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f58e11aebdd2fa22e49a_688.pdf', 'source_sha256': 'e265ede26f9e981d10a4baf8f4d8f1107042040d517e4abd9c984562824b482a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/688.pdf', 'bytes': 2043587}, {'source_id': 'GDCA-F-ad36057302995fa24269', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad36057302995fa24269_1322.pdf', 'source_sha256': 'da8e422a16901fe0cc75d65b8f0b51a684b7ce34cf3081c8ca8b24e8f13d4d2a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1322.pdf', 'bytes': 1577112}, {'source_id': 'GDCA-F-5477564eafb390ba5c78', 'source_path': 'sources/gdca-recon/2026-11-14/files/5477564eafb390ba5c78_1237.pdf', 'source_sha256': '3da7344a0fe2493febd5798951212b8c41e8d34220242079e02e6a1914a11036', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1237.pdf', 'bytes': 1737891}, {'source_id': 'GDCA-F-d6ea54c671bf0e82a36b', 'source_path': 'sources/gdca-recon/2026-11-14/files/d6ea54c671bf0e82a36b_856.pdf', 'source_sha256': '57c087cb57bf384f7eb60e5dd5144c3b2a458513f3da2b5e45ad6a0989361689', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/856.pdf', 'bytes': 1686379}, {'source_id': 'GDCA-F-c0f4119812a3dabc506e', 'source_path': 'sources/gdca-recon/2026-11-14/files/c0f4119812a3dabc506e_1478.pdf', 'source_sha256': 'ac0d68d6dcfb2860754e3cbc007020137d7b1d2229af10b7fe2be5468eebb520', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1478.pdf', 'bytes': 2637078}, {'source_id': 'GDCA-F-ba343136cc7389e58297', 'source_path': 'sources/gdca-recon/2026-11-14/files/ba343136cc7389e58297_551.pdf', 'source_sha256': 'ad15854f5b0994a20f0222f0ce756cb26f0655323fbefe68fbb7bf1331d063ce', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/551.pdf', 'bytes': 1862991}, {'source_id': 'GDCA-F-44234f00e9639c6427cc', 'source_path': 'sources/gdca-recon/2026-11-14/files/44234f00e9639c6427cc_825.pdf', 'source_sha256': '469f7ce2f0e1ee46e8d0f914e3da1dc1783a9bf87c112bfb152cb5e52be6b244', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/825.pdf', 'bytes': 2480137}, {'source_id': 'GDCA-F-fe8f128da0c80d90fa13', 'source_path': 'sources/gdca-recon/2026-11-14/files/fe8f128da0c80d90fa13_126.pdf', 'source_sha256': '17192c6332b4931e064c65351fd12ed7158934e5134c9d44c3f1a7ad3cfd8dbc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/126.pdf', 'bytes': 2292399}, {'source_id': 'GDCA-F-f76583d698a466e08045', 'source_path': 'sources/gdca-recon/2026-11-14/files/f76583d698a466e08045_1046.pdf', 'source_sha256': 'd358fb7d99d37500161fe5fc0c1558219e76571c7f5d979f6ac479adc29be7b4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1046.pdf', 'bytes': 1574771}, {'source_id': 'GDCA-F-486f672e48bcd78a8a35', 'source_path': 'sources/gdca-recon/2026-11-14/files/486f672e48bcd78a8a35_707.pdf', 'source_sha256': '6fc5340dc9decfcc26078268da21614dd114d95e13f1822467956acb1c98513a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/707.pdf', 'bytes': 1958644}, {'source_id': 'GDCA-F-42769db6ec4ba7ef31c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/42769db6ec4ba7ef31c1_548.pdf', 'source_sha256': '39926ce7b104aeda491ec17fa16c33617445d16749e779401b45aa768888f5a6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/548.pdf', 'bytes': 1795827}, {'source_id': 'GDCA-F-5e3e5fc1dcc2859afe21', 'source_path': 'sources/gdca-recon/2026-11-14/files/5e3e5fc1dcc2859afe21_854.pdf', 'source_sha256': '5abe7c59eb75cc7e591b4b08a4d6c95e5653b041c46b210ce8d7ece32897b602', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/854.pdf', 'bytes': 1963462}, {'source_id': 'GDCA-F-2cb9f70473deac9f3178', 'source_path': 'sources/gdca-recon/2026-11-14/files/2cb9f70473deac9f3178_713.pdf', 'source_sha256': 'a6fccd7f4f752c4ba6db96166622b1082dad8fbd0da9f3d90ed92466b5e344a2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/713.pdf', 'bytes': 1643473}, {'source_id': 'GDCA-F-70fbd58c01285a6353db', 'source_path': 'sources/gdca-recon/2026-11-14/files/70fbd58c01285a6353db_1316.pdf', 'source_sha256': '560d07d195cfbd23bedf29cea633b9bfec0aae18930c8b252b681877ed646589', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1316.pdf', 'bytes': 1700747}, {'source_id': 'GDCA-F-540a4503254d11590ad7', 'source_path': 'sources/gdca-recon/2026-11-14/files/540a4503254d11590ad7_83.pdf', 'source_sha256': 'bc74f310c5b624e053ad6b23622bc2cb138c6a0f925b9e883eb501dab0fd85df', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/83.pdf', 'bytes': 1961832}, {'source_id': 'GDCA-F-5bf109c2d9df550226c6', 'source_path': 'sources/gdca-recon/2026-11-14/files/5bf109c2d9df550226c6_1264.pdf', 'source_sha256': '3e907c765a3023d76568ea9041ab7366c1efa59ac6dc26edf13e9d2cbdec0900', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1264.pdf', 'bytes': 2274693}, {'source_id': 'GDCA-F-869e2b8334e7f2c7e95e', 'source_path': 'sources/gdca-recon/2026-11-14/files/869e2b8334e7f2c7e95e_1563.pdf', 'source_sha256': '639b908d929f5a65fd087f28caa5eb1fffa1f94c1b10c9e2972f2645190c6dcd', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1563.pdf', 'bytes': 3482199}, {'source_id': 'GDCA-F-e5d2dc7ee7ee0f148492', 'source_path': 'sources/gdca-recon/2026-11-14/files/e5d2dc7ee7ee0f148492_1017.pdf', 'source_sha256': '0eda10156f16f58d3b9bc4e54dd1ec2e782923b8714f4d5be88b079336993542', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1017.pdf', 'bytes': 2230722}, {'source_id': 'GDCA-F-394b9df7f2344e9a6b7e', 'source_path': 'sources/gdca-recon/2026-11-14/files/394b9df7f2344e9a6b7e_810.pdf', 'source_sha256': '44f7df34fb344875d6f159726403f51e9fcf8c10f6b07649cc3e39f3472475b5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/810.pdf', 'bytes': 1924441}, {'source_id': 'GDCA-F-437870845faff0aae31e', 'source_path': 'sources/gdca-recon/2026-11-14/files/437870845faff0aae31e_1321.pdf', 'source_sha256': '4828816ed074b5a06cce9d6652681122b849ef9a0cf5eedd8493f6edf19b5751', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1321.pdf', 'bytes': 1661324}, {'source_id': 'GDCA-F-3158572b43b9dc228da7', 'source_path': 'sources/gdca-recon/2026-11-14/files/3158572b43b9dc228da7_668.pdf', 'source_sha256': 'f83a298a75c69d5d29d0cbf17174bdb4a66737a62097346723f674408faa6ea7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/668.pdf', 'bytes': 2146602}, {'source_id': 'GDCA-F-9d7e5b042b7a80730e29', 'source_path': 'sources/gdca-recon/2026-11-14/files/9d7e5b042b7a80730e29_542.pdf', 'source_sha256': '3d5483ed0d4e7bd1a28d249e49c7aa5f4e3bce5e68d9251591be5a8c30b6ee87', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/542.pdf', 'bytes': 1940793}, {'source_id': 'GDCA-F-0a592e1021cec4cf1a39', 'source_path': 'sources/gdca-recon/2026-11-14/files/0a592e1021cec4cf1a39_1127.pdf', 'source_sha256': 'ba18be0bbff1ad93b2d1e74da53da3d063ef71d81965469c5af9b267452c7249', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1127.pdf', 'bytes': 2069089}, {'source_id': 'GDCA-F-188153462b5afe3eae40', 'source_path': 'sources/gdca-recon/2026-11-14/files/188153462b5afe3eae40_358.pdf', 'source_sha256': '5859ea32ce51009698be0bbe92889bba458ef946536712539a15368e20fac20b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/358.pdf', 'bytes': 1924715}, {'source_id': 'GDCA-F-80b5d823d6fb7efae4f1', 'source_path': 'sources/gdca-recon/2026-11-14/files/80b5d823d6fb7efae4f1_775.pdf', 'source_sha256': '2a126422626245ea6f336778ccd329e7c4b14e8e63d14c71b70d7a8086bcbf7d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/775.pdf', 'bytes': 2357353}, {'source_id': 'GDCA-F-4acc6a2ddbc43268e0c0', 'source_path': 'sources/gdca-recon/2026-11-14/files/4acc6a2ddbc43268e0c0_1402.pdf', 'source_sha256': '0479327cb0034f8337fdf0c55734889e2753ef3be04d0a7009edc62601752d49', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1402.pdf', 'bytes': 2864503}, {'source_id': 'GDCA-F-b57f09bd601269891cd8', 'source_path': 'sources/gdca-recon/2026-11-14/files/b57f09bd601269891cd8_389.pdf', 'source_sha256': '4a9dd5674054d8779f4b43d00b19e1536b5bdb11394b8f345c93934a2d700081', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/389.pdf', 'bytes': 1930006}, {'source_id': 'GDCA-F-33ca351e16891a9c26fa', 'source_path': 'sources/gdca-recon/2026-11-14/files/33ca351e16891a9c26fa_1558.pdf', 'source_sha256': 'e6dfaaabe711cc6b12a698d7d5a1d28ff6f84752889056459594ab997ff729a3', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1558.pdf', 'bytes': 3343840}, {'source_id': 'GDCA-F-9cc92f8d911f21a42cd4', 'source_path': 'sources/gdca-recon/2026-11-14/files/9cc92f8d911f21a42cd4_1166.pdf', 'source_sha256': 'c85a2a27b2ea4f695ed50b745988ed86225051602b08376637052cb0eb0c1ce7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1166.pdf', 'bytes': 2176703}, {'source_id': 'GDCA-F-8617d91b0ce6913486af', 'source_path': 'sources/gdca-recon/2026-11-14/files/8617d91b0ce6913486af_471.pdf', 'source_sha256': 'e4a1ee772f48a70c3c0a6dc8940630cb7eb0f5f26491759c243b315e27062f1a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/471.pdf', 'bytes': 1696864}, {'source_id': 'GDCA-F-f42c8f01ab22a6d6a6e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/f42c8f01ab22a6d6a6e3_1447.pdf', 'source_sha256': 'c30d2998bb4af9a67796285051be1ddba5c51284102600d46d961033fad1abe2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1447.pdf', 'bytes': 2353183}, {'source_id': 'GDCA-F-7afec9e8faeb58cf48c7', 'source_path': 'sources/gdca-recon/2026-11-14/files/7afec9e8faeb58cf48c7_476.pdf', 'source_sha256': 'f6cc796d556f7a0a2b72e11140170d20281d4e240dfbd2f179877a681d113a15', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/476.pdf', 'bytes': 1337095}, {'source_id': 'GDCA-F-bee187ea704a186b9c89', 'source_path': 'sources/gdca-recon/2026-11-14/files/bee187ea704a186b9c89_1576.pdf', 'source_sha256': '5a9a505b368291f85d01157c9c3939e00418510610fb98f7c9c9629e28260d5e', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1576.pdf', 'bytes': 2243695}, {'source_id': 'GDCA-F-982ad2efd87b221f0a9f', 'source_path': 'sources/gdca-recon/2026-11-14/files/982ad2efd87b221f0a9f_557.pdf', 'source_sha256': '7208f6e02b9599e792880bff773ab0849c4a2af65a25e70f7b0863e4399ccee6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/557.pdf', 'bytes': 1798903}, {'source_id': 'GDCA-F-22694fd7c1d8c6202219', 'source_path': 'sources/gdca-recon/2026-11-14/files/22694fd7c1d8c6202219_1433.pdf', 'source_sha256': '102f450cc3ee235c75ec14ec2f44112b5dbfa106777b829d5f39973901b12686', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1433.pdf', 'bytes': 2308907}, {'source_id': 'GDCA-F-456b0616c953627d10b7', 'source_path': 'sources/gdca-recon/2026-11-14/files/456b0616c953627d10b7_961.pdf', 'source_sha256': '102810949eb4a8e1e154a570de6b5742d8eb968aa02cc29af8928ceef501653f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/961.pdf', 'bytes': 2090204}, {'source_id': 'GDCA-F-d2be5dc981f4fbde9c9f', 'source_path': 'sources/gdca-recon/2026-11-14/files/d2be5dc981f4fbde9c9f_76.pdf', 'source_sha256': '78eb603ac136abb0141c15b0d68e9525f86ac8cc5c1217168c182fe13ef0498a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/76.pdf', 'bytes': 1937520}, {'source_id': 'GDCA-F-f55ae1cc666d841fda50', 'source_path': 'sources/gdca-recon/2026-11-14/files/f55ae1cc666d841fda50_941.pdf', 'source_sha256': 'c4d7d15ce64e734490ade229fbbfa58ed4ddaaf5cb4f509d8985a8b9987ce0e2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/941.pdf', 'bytes': 2062722}, {'source_id': 'GDCA-F-1cc94aecff6ae6e876a1', 'source_path': 'sources/gdca-recon/2026-11-14/files/1cc94aecff6ae6e876a1_271.pdf', 'source_sha256': '40d0fff01c9fb10c42101f95890e4874c32efc42ceb5e70def3cc438849bf9f0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/271.pdf', 'bytes': 1902515}, {'source_id': 'GDCA-F-ce487e75c152efd636db', 'source_path': 'sources/gdca-recon/2026-11-14/files/ce487e75c152efd636db_593.pdf', 'source_sha256': 'a05d44b2532176afaa15c5fc7529e0bce8268fdc59a6876f1fdfb251241d4c49', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/593.pdf', 'bytes': 1652904}, {'source_id': 'GDCA-F-8d3a62a6309e9ea23aad', 'source_path': 'sources/gdca-recon/2026-11-14/files/8d3a62a6309e9ea23aad_1143.pdf', 'source_sha256': 'b9e9fff7c75d9eb5c84e9e88310a81fa6cdc95b400e4a50196956db6dd9f36de', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1143.pdf', 'bytes': 1770677}, {'source_id': 'GDCA-F-f3c88234f0fb1537e48a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3c88234f0fb1537e48a_773.pdf', 'source_sha256': '4bf9d446c387db5cd1e4e81717dd46df7fe3aeaad723f3eeaec065fab6577cc6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/773.pdf', 'bytes': 2172900}, {'source_id': 'GDCA-F-489ad35c75ee231fa3b5', 'source_path': 'sources/gdca-recon/2026-11-14/files/489ad35c75ee231fa3b5_134.pdf', 'source_sha256': 'c92456e4b6d73a424326a3fc0e67dd28a04c3c2745208ae715c0a3ffd924cbcd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/134.pdf', 'bytes': 1873952}, {'source_id': 'GDCA-F-eb86466e6053d090e842', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb86466e6053d090e842_1254.pdf', 'source_sha256': 'f37f10a57022b84aabc0dfe5503cbe01672ad68722e1478d57b914a055e064b8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1254.pdf', 'bytes': 1848101}, {'source_id': 'GDCA-F-dcc3ed1c76fba5febacb', 'source_path': 'sources/gdca-recon/2026-11-14/files/dcc3ed1c76fba5febacb_1245.pdf', 'source_sha256': 'da09d5e0e428b0f9f5326c6f628227a3ef2a59843ba5704b7468c3c4463219bf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1245.pdf', 'bytes': 1636463}, {'source_id': 'GDCA-F-2491ccba03da1b3e66e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/2491ccba03da1b3e66e5_892.pdf', 'source_sha256': '9331243d9ebf82414c6c09021ea7cbed0f9c27d542b51f9b1a4b15d17425adf7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/892.pdf', 'bytes': 2122163}, {'source_id': 'GDCA-F-1300edf852a3b891560b', 'source_path': 'sources/gdca-recon/2026-11-14/files/1300edf852a3b891560b_1270.pdf', 'source_sha256': '71d407a640379a8644b9566b81d302ea4d5d7b21b48f65f52331d822b298c1b0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1270.pdf', 'bytes': 1940914}, {'source_id': 'GDCA-F-65948da54fbca42fcf0e', 'source_path': 'sources/gdca-recon/2026-11-14/files/65948da54fbca42fcf0e_1468.pdf', 'source_sha256': '6d4c476bce8f721db5b45e8ad39b33d886b7938a8c6adc8d7a2690f925e76486', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1468.pdf', 'bytes': 2339829}, {'source_id': 'GDCA-F-ebfb3e60cc4e40c9a953', 'source_path': 'sources/gdca-recon/2026-11-14/files/ebfb3e60cc4e40c9a953_1422.pdf', 'source_sha256': '8dcab55b787974aa09585fc6b8edb4fd33f333f7954be8cf4b8236b7976c619a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1422.pdf', 'bytes': 2981162}, {'source_id': 'GDCA-F-909bb9805469d7713707', 'source_path': 'sources/gdca-recon/2026-11-14/files/909bb9805469d7713707_1457.pdf', 'source_sha256': '211e74d53bb72a5f8387ab304b31ab1e5b328dc9b14b89be50118e2826e5739c', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1457.pdf', 'bytes': 3780202}, {'source_id': 'GDCA-F-375924af9188c958c800', 'source_path': 'sources/gdca-recon/2026-11-14/files/375924af9188c958c800_513.pdf', 'source_sha256': 'b9e6d30a4bee2acbdb7dad0934f1bee161c5ee28b20b3bf0554aa0e2d38bc457', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/513.pdf', 'bytes': 1855145}, {'source_id': 'GDCA-F-f3eea12caa7821ffcb57', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3eea12caa7821ffcb57_1582.pdf', 'source_sha256': '25b62dedad5b81f775f80fdfbe53d12a46a4f6750874a6fea45d14302a3624fa', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1582.pdf', 'bytes': 3074660}, {'source_id': 'GDCA-F-50df9b2ba4a65e81be44', 'source_path': 'sources/gdca-recon/2026-11-14/files/50df9b2ba4a65e81be44_1464.pdf', 'source_sha256': 'c385903a3e130d3ab0871ca8e716046fbf5f3f62a194e2b5c8f101f3e5b3b3e2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1464.pdf', 'bytes': 2270501}, {'source_id': 'GDCA-F-17085201cf71b60b8cf6', 'source_path': 'sources/gdca-recon/2026-11-14/files/17085201cf71b60b8cf6_881.pdf', 'source_sha256': 'bac337faa59de393912311d08c753ce2d0b3f39ef4b7441c90bca010eeaf689e', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/881.pdf', 'bytes': 1165443}, {'source_id': 'GDCA-F-1b9ed187ed113b428855', 'source_path': 'sources/gdca-recon/2026-11-14/files/1b9ed187ed113b428855_473.pdf', 'source_sha256': 'ec4e87b2430494a4ade8dfe02629844947d45b105e1cfaaa4f177f81c3bb4b0e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/473.pdf', 'bytes': 2322741}, {'source_id': 'GDCA-F-bb0f58d85945e15469fb', 'source_path': 'sources/gdca-recon/2026-11-14/files/bb0f58d85945e15469fb_1119.pdf', 'source_sha256': 'c262024eaed49d64b25b184e3b5f0e05a1e5f5b82c6ac65c8b7370864d8f8f04', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1119.pdf', 'bytes': 1836859}, {'source_id': 'GDCA-F-8c750e25a73a4263fae2', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c750e25a73a4263fae2_1123.pdf', 'source_sha256': '825b590b793ffb6f7320f090dacc303ab3a46db46df67a99f6512e563c367061', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1123.pdf', 'bytes': 2043824}, {'source_id': 'GDCA-F-ae4d40fe37a7c417838d', 'source_path': 'sources/gdca-recon/2026-11-14/files/ae4d40fe37a7c417838d_336.pdf', 'source_sha256': '716a8e964facf9e71d11d7fe2d012ae00c12980dd636583c8d5c3023e1b5ca69', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/336.pdf', 'bytes': 1803281}, {'source_id': 'GDCA-F-29c168c195bda9155163', 'source_path': 'sources/gdca-recon/2026-11-14/files/29c168c195bda9155163_698.pdf', 'source_sha256': 'a6a21bca2319c8ce6a78b4368632bb6cba2fb14397b248784499b7fd1a156d54', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/698.pdf', 'bytes': 1977683}, {'source_id': 'GDCA-F-d85c0d5919eeabc697bb', 'source_path': 'sources/gdca-recon/2026-11-14/files/d85c0d5919eeabc697bb_541.pdf', 'source_sha256': '4bdaac7cd224015ae3bdec0991a58ca1add7c6382f54dd960d0f2146fc560013', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/541.pdf', 'bytes': 1987318}, {'source_id': 'GDCA-F-cce676676730381d25ab', 'source_path': 'sources/gdca-recon/2026-11-14/files/cce676676730381d25ab_440.pdf', 'source_sha256': '64fa768cbd4a91279eff545b24b598e4c548cd29817c9085373fe5416ee4fc03', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/440.pdf', 'bytes': 2336707}, {'source_id': 'GDCA-F-0aa9b9d709b78d423104', 'source_path': 'sources/gdca-recon/2026-11-14/files/0aa9b9d709b78d423104_1295.pdf', 'source_sha256': '36a22bb1c0767398e512264e437f3ba8485997bde303fac31ee723d56df64f48', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1295.pdf', 'bytes': 2267139}, {'source_id': 'GDCA-F-c7da0b626885e38ebbd1', 'source_path': 'sources/gdca-recon/2026-11-14/files/c7da0b626885e38ebbd1_226.pdf', 'source_sha256': '407e393a43d7d82758ed90e76637fb720937856b94399910615dc324e4276bf3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/226.pdf', 'bytes': 2286099}, {'source_id': 'GDCA-F-889971dd7a4f70d38de4', 'source_path': 'sources/gdca-recon/2026-11-14/files/889971dd7a4f70d38de4_710.pdf', 'source_sha256': 'df1bdd1b92c44d4c1ef33f82e3aaa5fc8826bb4ccc0f136800897b6eaf1b49cd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/710.pdf', 'bytes': 2293891}, {'source_id': 'GDCA-F-a15a7f276b560468d2c3', 'source_path': 'sources/gdca-recon/2026-11-14/files/a15a7f276b560468d2c3_346.pdf', 'source_sha256': 'aaf25dd23155e75fa9cb73f93043d7a4dbff9176af67a19e7a1626824e880e04', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/346.pdf', 'bytes': 2214475}, {'source_id': 'GDCA-F-5538ac085cc3291a44a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/5538ac085cc3291a44a9_1476.pdf', 'source_sha256': 'b583412be706dcd52fc5a553b595ea7e3eaaaa95f81d4ba907d26adc859ca715', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1476.pdf', 'bytes': 2644736}, {'source_id': 'GDCA-F-fe5397d10e3504f2f274', 'source_path': 'sources/gdca-recon/2026-11-14/files/fe5397d10e3504f2f274_50.pdf', 'source_sha256': '3c9f6a042fe65715ec2f0f97360adbdcac31f930617458714b99fee36d2c348a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/50.pdf', 'bytes': 1852532}, {'source_id': 'GDCA-F-fe76e542b49d753c27e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/fe76e542b49d753c27e5_640.pdf', 'source_sha256': '53dea9f6ba3b9041b4734d081b48598cff0a6827c8b2ae140e088accce73832f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/640.pdf', 'bytes': 1823726}, {'source_id': 'GDCA-F-478378b42b09e23216f0', 'source_path': 'sources/gdca-recon/2026-11-14/files/478378b42b09e23216f0_86.pdf', 'source_sha256': '6bd0940136c22c24e870f94c094496a38195738b4dc86d9bb27899d68557641b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/86.pdf', 'bytes': 1933969}, {'source_id': 'GDCA-F-97525a38d7198dfdd123', 'source_path': 'sources/gdca-recon/2026-11-14/files/97525a38d7198dfdd123_1150.pdf', 'source_sha256': '78df7b3d49a5363775269ff551a87aeb0e323b67507089807a121d3a96e29f6f', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1150.pdf', 'bytes': 2455191}, {'source_id': 'GDCA-F-6c25658ad65aa3ee459f', 'source_path': 'sources/gdca-recon/2026-11-14/files/6c25658ad65aa3ee459f_522.pdf', 'source_sha256': '702a502f41c70acb34c59eda5352ab49b3cdf4ab66cabf9041815eb3cb1ac08b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/522.pdf', 'bytes': 2153058}, {'source_id': 'GDCA-F-350ad3b1cdf417e9a169', 'source_path': 'sources/gdca-recon/2026-11-14/files/350ad3b1cdf417e9a169_1311.pdf', 'source_sha256': '218d1b3e4f2b99556831fa0a3d964bd01e23af46081ad299625a1db17e8c9fee', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1311.pdf', 'bytes': 2107420}, {'source_id': 'GDCA-F-28cdab592f49c0897763', 'source_path': 'sources/gdca-recon/2026-11-14/files/28cdab592f49c0897763_585.pdf', 'source_sha256': '9947a765c42661240289b71650199621be52eac8f33f2d82edff93758d39bc42', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/585.pdf', 'bytes': 2114222}, {'source_id': 'GDCA-F-bd81be6235f29ea5d24f', 'source_path': 'sources/gdca-recon/2026-11-14/files/bd81be6235f29ea5d24f_674.pdf', 'source_sha256': 'e821609056711133e045c4d9fbba0fe50ee0ee52e9e42dc4a0297dc4d7233bf6', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/674.pdf', 'bytes': 2740747}, {'source_id': 'GDCA-F-ad25a2f15f157f7ce72e', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad25a2f15f157f7ce72e_604.pdf', 'source_sha256': '0bfdf52be91df7ff6fdb4bce07939053d0efe7ee2fa8b1583f5178d1252a7845', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/604.pdf', 'bytes': 2114401}, {'source_id': 'GDCA-F-e6816468f30ecde2b139', 'source_path': 'sources/gdca-recon/2026-11-14/files/e6816468f30ecde2b139_732.pdf', 'source_sha256': '20b7c42e1b81518a81f877c9a7518e38315a86acfade5a68a8298b57b3e921ef', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/732.pdf', 'bytes': 1787052}, {'source_id': 'GDCA-F-2ec4dbb9c3f9e5da51d6', 'source_path': 'sources/gdca-recon/2026-11-14/files/2ec4dbb9c3f9e5da51d6_728.pdf', 'source_sha256': '17ec1aae548efc584a9c29d8cd4eea36796dd09033f5ea4434c3b55b3b5e5e85', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/728.pdf', 'bytes': 1755753}, {'source_id': 'GDCA-F-df48363cda0201bd08d4', 'source_path': 'sources/gdca-recon/2026-11-14/files/df48363cda0201bd08d4_487.pdf', 'source_sha256': '388c6907148713d3902d53322bf37ab2da6e34798cc1e588059cdb97fdf7b230', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/487.pdf', 'bytes': 2229287}, {'source_id': 'GDCA-F-2c3c1331c3a73b799dae', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c3c1331c3a73b799dae_822.pdf', 'source_sha256': 'b1a293366c96b8d8a2f893d911c81903c361cea44a6779af2291ce8c31d26a2e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/822.pdf', 'bytes': 1666301}, {'source_id': 'GDCA-F-70d457cb4e936e743eff', 'source_path': 'sources/gdca-recon/2026-11-14/files/70d457cb4e936e743eff_949.pdf', 'source_sha256': 'b0f6ba0af84d71d7c879ec7a1e1a499ff2e939dec70d8e57f4485da9ae9fd7a5', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/949.pdf', 'bytes': 2205955}, {'source_id': 'GDCA-F-e4c560dfc496c3b48840', 'source_path': 'sources/gdca-recon/2026-11-14/files/e4c560dfc496c3b48840_669.pdf', 'source_sha256': 'b792b0455f51fe7d190a45b54e73d8445b876556bb0d08a65086d41302b4aec3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/669.pdf', 'bytes': 1900318}, {'source_id': 'GDCA-F-2b82c2a950b1266ba12e', 'source_path': 'sources/gdca-recon/2026-11-14/files/2b82c2a950b1266ba12e_1416.pdf', 'source_sha256': '26e97cc15b0af2340290336571a88a8f490ebf3dc431b339e2d7095c071bc723', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1416.pdf', 'bytes': 2045832}, {'source_id': 'GDCA-F-ba8691ec2eb503f8769a', 'source_path': 'sources/gdca-recon/2026-11-14/files/ba8691ec2eb503f8769a_301.pdf', 'source_sha256': '8284ace74205e1b0128ab5199e775953d6301b4fd9a346c08725b699f397edfe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/301.pdf', 'bytes': 1790933}, {'source_id': 'GDCA-F-57743237ffa8027c0292', 'source_path': 'sources/gdca-recon/2026-11-14/files/57743237ffa8027c0292_318.pdf', 'source_sha256': '0573f8933148944ab5b4daac281496a124841d11f89bab0e4bebf1e98ee0eebe', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/318.pdf', 'bytes': 2312006}, {'source_id': 'GDCA-F-343e92766c8e3cb91aae', 'source_path': 'sources/gdca-recon/2026-11-14/files/343e92766c8e3cb91aae_1251.pdf', 'source_sha256': '85fb5ee0767a2af080a064cc1026b036a7cbc24285c7d39a895344540081172a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1251.pdf', 'bytes': 1974721}, {'source_id': 'GDCA-F-f12212bc389a73df3fa7', 'source_path': 'sources/gdca-recon/2026-11-14/files/f12212bc389a73df3fa7_729.pdf', 'source_sha256': 'bac46da822859c6c55535aace65c77c3806cf82f2868d79b72107f5d5404b9be', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/729.pdf', 'bytes': 1785403}, {'source_id': 'GDCA-F-3fe1b92211f4aee894d4', 'source_path': 'sources/gdca-recon/2026-11-14/files/3fe1b92211f4aee894d4_205.pdf', 'source_sha256': '75ae0608120ebd368b65348dd773cc9f7259c14c5ee9ee09945d287fa89823fa', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/205.pdf', 'bytes': 1901151}, {'source_id': 'GDCA-F-72e9818d4887e1e61b1f', 'source_path': 'sources/gdca-recon/2026-11-14/files/72e9818d4887e1e61b1f_714.pdf', 'source_sha256': '5add59bee0919f7bd12070d763bd4a3b20918274938b46b6eebefc347776cd10', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/714.pdf', 'bytes': 2025074}, {'source_id': 'GDCA-F-016abef3080a4b1e6c1b', 'source_path': 'sources/gdca-recon/2026-11-14/files/016abef3080a4b1e6c1b_1214.pdf', 'source_sha256': 'bf10ccb9c19f56e09193b4530458ed5f172c620bcff4922a458c736f5e20b878', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1214.pdf', 'bytes': 2010343}, {'source_id': 'GDCA-F-2fb1b9a2068a8bc14995', 'source_path': 'sources/gdca-recon/2026-11-14/files/2fb1b9a2068a8bc14995_450.pdf', 'source_sha256': '344866b548643c031478f4411af170e0027430223a478f66640c011f9886a4bf', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/450.pdf', 'bytes': 2101828}, {'source_id': 'GDCA-F-803ccf8f3fa239ea7d2a', 'source_path': 'sources/gdca-recon/2026-11-14/files/803ccf8f3fa239ea7d2a_493.pdf', 'source_sha256': 'e751223cca384c4928bace116946ff1b52d25c51cbef90857d14d2595e9a711c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/493.pdf', 'bytes': 1820461}, {'source_id': 'GDCA-F-dd347b046f5502214990', 'source_path': 'sources/gdca-recon/2026-11-14/files/dd347b046f5502214990_1394.pdf', 'source_sha256': '24f3b2331c867cebe9616a9adce96e9df1e220762475c3308af6c21b2bd39b34', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1394.pdf', 'bytes': 1744043}, {'source_id': 'GDCA-F-bd44dd34817b2eed92dc', 'source_path': 'sources/gdca-recon/2026-11-14/files/bd44dd34817b2eed92dc_94.pdf', 'source_sha256': '34998e42690e90c066f1a349a5356bc432f5aede40da8e669043723b601cf42b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/94.pdf', 'bytes': 1923767}, {'source_id': 'GDCA-F-501aa88a6e2cfc6cf9f0', 'source_path': 'sources/gdca-recon/2026-11-14/files/501aa88a6e2cfc6cf9f0_1090.pdf', 'source_sha256': '3727221e8f691ee9e7f416ce7811b4bd01344c2755c718baccb5848249083a9e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1090.pdf', 'bytes': 2082761}, {'source_id': 'GDCA-F-3c888202afbf0db1fb73', 'source_path': 'sources/gdca-recon/2026-11-14/files/3c888202afbf0db1fb73_859.pdf', 'source_sha256': '3539f2293513bd154478c52dc3762920fa5969873bb8d18f8555d5208ef3b904', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/859.pdf', 'bytes': 2554692}, {'source_id': 'GDCA-F-3154d1db3ff9ed1e800c', 'source_path': 'sources/gdca-recon/2026-11-14/files/3154d1db3ff9ed1e800c_176.pdf', 'source_sha256': '38070b89f8de3d4be9f4af0cc23dec0758da0a91c51e0866e05d463f484b6e44', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/176.pdf', 'bytes': 2089507}, {'source_id': 'GDCA-F-9c34ff46a4cedba2635f', 'source_path': 'sources/gdca-recon/2026-11-14/files/9c34ff46a4cedba2635f_276.pdf', 'source_sha256': '87a9ac8b3a748b096e195e9618580db5b866ac6bf16a2c62b2d653ede74b9274', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/276.pdf', 'bytes': 1028880}, {'source_id': 'GDCA-F-936e7b25500c0361e7f4', 'source_path': 'sources/gdca-recon/2026-11-14/files/936e7b25500c0361e7f4_631.pdf', 'source_sha256': '8477f13908926e8ffb21da26f5349ebaf4ed5570bcc71a401d2d5f69d405220b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/631.pdf', 'bytes': 2026651}, {'source_id': 'GDCA-F-5a8a4e2a91ad18109b48', 'source_path': 'sources/gdca-recon/2026-11-14/files/5a8a4e2a91ad18109b48_421.pdf', 'source_sha256': '6e0ed41d852b73babf243170b41bf0d57fc32c29b2e64ada76c2bf1c3f479104', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/421.pdf', 'bytes': 1366066}, {'source_id': 'GDCA-F-b347bc9615bd8d508b39', 'source_path': 'sources/gdca-recon/2026-11-14/files/b347bc9615bd8d508b39_1492.pdf', 'source_sha256': 'd78d13e579cf79dea0486478f078ec4f9a74be31bf6fd959fe3ed44b7d9bc2df', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1492.pdf', 'bytes': 1996628}, {'source_id': 'GDCA-F-52216c3f5f733d7dfb68', 'source_path': 'sources/gdca-recon/2026-11-14/files/52216c3f5f733d7dfb68_22.pdf', 'source_sha256': '6fa912d3afef7531724e5ab15eb3d1b24769c1e4059191ddb7cde6b6c26ab557', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/22.pdf', 'bytes': 119149}, {'source_id': 'GDCA-F-5ebfbd184ca9eebc3f2e', 'source_path': 'sources/gdca-recon/2026-11-14/files/5ebfbd184ca9eebc3f2e_144.pdf', 'source_sha256': '71efa434a91fac91f8c75d8b973365833efaa39e8eb9aa5b691adc22536cd21e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/144.pdf', 'bytes': 1920416}, {'source_id': 'GDCA-F-67d425ea38c58f137244', 'source_path': 'sources/gdca-recon/2026-11-14/files/67d425ea38c58f137244_139.pdf', 'source_sha256': 'e0ea73f9eeb3fddb3f807325420cd7e6006b4b6a86f30335375697623130bb1e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/139.pdf', 'bytes': 1957390}, {'source_id': 'GDCA-F-e51e84f413e494df8c4c', 'source_path': 'sources/gdca-recon/2026-11-14/files/e51e84f413e494df8c4c_310.pdf', 'source_sha256': '65791f6404fb3f2cf5df4fc58b337eddf7678e91063a3ffac5dafca49ab84e01', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/310.pdf', 'bytes': 2352535}, {'source_id': 'GDCA-F-2613c14acc500f6531ae', 'source_path': 'sources/gdca-recon/2026-11-14/files/2613c14acc500f6531ae_1569.pdf', 'source_sha256': '97bfc20c2c5099d3bde5de377fae40b344446c32941c4435813331a65e08e847', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1569.pdf', 'bytes': 3904506}, {'source_id': 'GDCA-F-7146379794be8b1af90e', 'source_path': 'sources/gdca-recon/2026-11-14/files/7146379794be8b1af90e_392.pdf', 'source_sha256': '71b45bb64cfb7470bc61289f874a8ae20cb74253cce862b398da44917d73ac80', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/392.pdf', 'bytes': 1970168}, {'source_id': 'GDCA-F-443a3c704058b52e99a0', 'source_path': 'sources/gdca-recon/2026-11-14/files/443a3c704058b52e99a0_1456.pdf', 'source_sha256': '9899fadb2c55aa169be6531cf77e90ed9571ccdce657b220a2f31e40f7c15db6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1456.pdf', 'bytes': 2185029}, {'source_id': 'GDCA-F-47f0a9bfc2ec70fe2341', 'source_path': 'sources/gdca-recon/2026-11-14/files/47f0a9bfc2ec70fe2341_281.pdf', 'source_sha256': 'aae25909536a44e24a1d706db7a75f1adfdee12c2e38152bcbcba1dfed99b79a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/281.pdf', 'bytes': 2290155}, {'source_id': 'GDCA-F-76d3daf34ff1118753b1', 'source_path': 'sources/gdca-recon/2026-11-14/files/76d3daf34ff1118753b1_31.pdf', 'source_sha256': '27c3f2644efdc3b8f8371d477e559cd3f866540d7a9d4d6db4d4edd1a3445bb8', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/31.pdf', 'bytes': 78761}, {'source_id': 'GDCA-F-85d5809cdf5e4d402765', 'source_path': 'sources/gdca-recon/2026-11-14/files/85d5809cdf5e4d402765_385.pdf', 'source_sha256': 'f44f80cb17f7483ca3cdafbfe029f6de59072df3a0a94c71c8474e8120124cde', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/385.pdf', 'bytes': 2033520}, {'source_id': 'GDCA-F-c5bca5176eaed95775ee', 'source_path': 'sources/gdca-recon/2026-11-14/files/c5bca5176eaed95775ee_1225.pdf', 'source_sha256': '3b8f0094331d6a75169c9b65912e842f3082cc92a9e27751d4beb49245a2c135', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1225.pdf', 'bytes': 2097012}, {'source_id': 'GDCA-F-0d2a1d9123a595ec85a3', 'source_path': 'sources/gdca-recon/2026-11-14/files/0d2a1d9123a595ec85a3_686.pdf', 'source_sha256': '369e5529d0675b4c03e51ea918d7f2b1a3dba2895726dc062277bf5fc1a66d92', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/686.pdf', 'bytes': 1783577}, {'source_id': 'GDCA-F-fd79d05d785347391e6f', 'source_path': 'sources/gdca-recon/2026-11-14/files/fd79d05d785347391e6f_875.pdf', 'source_sha256': 'e4e5b9243d706f32dd808dc26953c349fa1bbabe1cdfe79dc6ace15803f345e4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/875.pdf', 'bytes': 1917348}, {'source_id': 'GDCA-F-62f089a5c18c4e390de7', 'source_path': 'sources/gdca-recon/2026-11-14/files/62f089a5c18c4e390de7_420.pdf', 'source_sha256': 'b4f4da555f0e415dcf8975b08db1afbe980bc58fc56427e719ad7db1894cab61', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/420.pdf', 'bytes': 2266049}, {'source_id': 'GDCA-F-2f882313b708628fc080', 'source_path': 'sources/gdca-recon/2026-11-14/files/2f882313b708628fc080_1553.pdf', 'source_sha256': '72a79a960505e51e0d561144fd42cef6b37cc59e8b9fc5f6c95e096ebc2224b4', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1553.pdf', 'bytes': 3244319}, {'source_id': 'GDCA-F-697b29469b2446a3acd0', 'source_path': 'sources/gdca-recon/2026-11-14/files/697b29469b2446a3acd0_1126.pdf', 'source_sha256': 'ca492ab0708f9dab3ea1cafaa889e21164a26ebc90a66d59010bf22f5ad382bf', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1126.pdf', 'bytes': 2210126}, {'source_id': 'GDCA-F-67d32050c0dd4b75d3e7', 'source_path': 'sources/gdca-recon/2026-11-14/files/67d32050c0dd4b75d3e7_434.pdf', 'source_sha256': 'b6e8ca5b2cf6dd6ba4cccb3116d0b2a617c29e1079b02625079516df6d390d5e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/434.pdf', 'bytes': 2202655}, {'source_id': 'GDCA-F-f982cb92eebb3c10448a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f982cb92eebb3c10448a_123.pdf', 'source_sha256': '3737cb48b060845a4afd3bb65921d4e72e6346a60028f333945a83ffe96f899d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/123.pdf', 'bytes': 2068557}, {'source_id': 'GDCA-F-337df06046b90f9708ec', 'source_path': 'sources/gdca-recon/2026-11-14/files/337df06046b90f9708ec_874.pdf', 'source_sha256': '4587cec0991288f76121505a487af411dddf9da75ad4b9ea094c431e861b91ec', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/874.pdf', 'bytes': 1549019}, {'source_id': 'GDCA-F-0aed1746cc3087a23464', 'source_path': 'sources/gdca-recon/2026-11-14/files/0aed1746cc3087a23464_1570.pdf', 'source_sha256': '8a5dd932f4e75f93d704b93bd00670f3fc3b07ca04be354855e5627c9c19da8f', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1570.pdf', 'bytes': 3270418}, {'source_id': 'GDCA-F-e70409f7b8e7e7f355c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/e70409f7b8e7e7f355c1_1240.pdf', 'source_sha256': '8b53d1bc9141d5c19a55addf4d1a8364f2702c27213115274ba1bf0ca3fd36d9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1240.pdf', 'bytes': 1764289}, {'source_id': 'GDCA-F-8b2d884e252ee9b926e6', 'source_path': 'sources/gdca-recon/2026-11-14/files/8b2d884e252ee9b926e6_559.pdf', 'source_sha256': '880c1f3764da3495f2a009eb3fef147c0e9d3654971ab6e27f9ba4d3fac504c4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/559.pdf', 'bytes': 2046177}, {'source_id': 'GDCA-F-92fd9c66395c93f29e83', 'source_path': 'sources/gdca-recon/2026-11-14/files/92fd9c66395c93f29e83_1036.pdf', 'source_sha256': 'ae65fd0eaa0082f17a4384e3fa42a6d31b98f3fac5d9af2c61be735989ed2068', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1036.pdf', 'bytes': 1738957}, {'source_id': 'GDCA-F-365cf358666ef1abfe8e', 'source_path': 'sources/gdca-recon/2026-11-14/files/365cf358666ef1abfe8e_402.pdf', 'source_sha256': '8b67ec89a8e9acbd54e4ecdaf9d107964701cefe1275e09146388f30ef72aa2c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/402.pdf', 'bytes': 2145970}, {'source_id': 'GDCA-F-7e980a46539cd0117f8e', 'source_path': 'sources/gdca-recon/2026-11-14/files/7e980a46539cd0117f8e_90.pdf', 'source_sha256': '9001530e50089f8e5838d2c15e491c863d1921aa3061c14044d492e7f5d3272d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/90.pdf', 'bytes': 1773207}, {'source_id': 'GDCA-F-08f85ebd3badd26b7a73', 'source_path': 'sources/gdca-recon/2026-11-14/files/08f85ebd3badd26b7a73_201.pdf', 'source_sha256': 'd20197de41259d5598b31f0dcd91b2c6ab762f26f9f30b751871d923e9dc242e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/201.pdf', 'bytes': 1987807}, {'source_id': 'GDCA-F-a7fa34d62cd67bd26295', 'source_path': 'sources/gdca-recon/2026-11-14/files/a7fa34d62cd67bd26295_454.pdf', 'source_sha256': '28d989528f7a60943d9562f13c9b24cf64f3979181847d65771f42208ec63cb3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/454.pdf', 'bytes': 1902947}, {'source_id': 'GDCA-F-86f70605b3b56c3f7de3', 'source_path': 'sources/gdca-recon/2026-11-14/files/86f70605b3b56c3f7de3_435.pdf', 'source_sha256': '3fadd658847b7326d1bbaac2bdf08aacbe996696f60f83864fec4fb28b499935', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/435.pdf', 'bytes': 2345160}, {'source_id': 'GDCA-F-eb4174777aaa2bf8b593', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb4174777aaa2bf8b593_1575.pdf', 'source_sha256': '3baf29d452f150a995c7e436e1cdb6e5ae5ca36e1d145fdada1538d870f6e752', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1575.pdf', 'bytes': 3965765}, {'source_id': 'GDCA-F-79a01efef669def573e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/79a01efef669def573e3_1051.pdf', 'source_sha256': '6dd3b70ca00db6561f619bf8303ed0cadb89f98577348a9794db3b1db198d011', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1051.pdf', 'bytes': 1877465}, {'source_id': 'GDCA-F-47c50ced79d6d1b2419f', 'source_path': 'sources/gdca-recon/2026-11-14/files/47c50ced79d6d1b2419f_1231.pdf', 'source_sha256': '8897b814f078b1e3e79773c51db14f32c87af6ac8eca32c0b3eefc865be2971e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1231.pdf', 'bytes': 2093917}, {'source_id': 'GDCA-F-3abd39ba5eb80c922d0f', 'source_path': 'sources/gdca-recon/2026-11-14/files/3abd39ba5eb80c922d0f_1037.pdf', 'source_sha256': 'be418c65cda9d4b23bb8db0473d0d3fcdd4a9d1422af510d1928da19acd633d7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1037.pdf', 'bytes': 1939811}, {'source_id': 'GDCA-F-2e0a663896cc8d8adcec', 'source_path': 'sources/gdca-recon/2026-11-14/files/2e0a663896cc8d8adcec_1287.pdf', 'source_sha256': 'eb304e2ff97de417272a6713b3fd2f2a90c30b46b0f070af010c866df6a83a4d', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1287.pdf', 'bytes': 991954}, {'source_id': 'GDCA-F-39c193006a4e06f0334f', 'source_path': 'sources/gdca-recon/2026-11-14/files/39c193006a4e06f0334f_876.pdf', 'source_sha256': 'c17156256285fecb35c2b7f95e98e3cb64a36fb49ea9eb1704e207fb7125c1c1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/876.pdf', 'bytes': 2048812}, {'source_id': 'GDCA-F-c41a4b44f238fe451764', 'source_path': 'sources/gdca-recon/2026-11-14/files/c41a4b44f238fe451764_1196.pdf', 'source_sha256': 'a7f4a84793694689ba5b927ea52eb9f9055097c64df8e8dc15c237019f6a396e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1196.pdf', 'bytes': 1848574}, {'source_id': 'GDCA-F-53f71074f57aa72dc3ec', 'source_path': 'sources/gdca-recon/2026-11-14/files/53f71074f57aa72dc3ec_390.pdf', 'source_sha256': '5dc4ff516f792ce947f6bd91dbb390a44985c89fc71e4cb7eeb48f42e8e4f7b7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/390.pdf', 'bytes': 2108966}, {'source_id': 'GDCA-F-e1f57cb4c500b71e22d7', 'source_path': 'sources/gdca-recon/2026-11-14/files/e1f57cb4c500b71e22d7_480.pdf', 'source_sha256': '1eea8ad1515000f1e5393b8ac280bac46a0f7e6fbba7f0cc97419a7b7469ee4c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/480.pdf', 'bytes': 1822204}, {'source_id': 'GDCA-F-2d33e0b16141ac766279', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d33e0b16141ac766279_457.pdf', 'source_sha256': 'b6acc79a1aa40b3882cb8ac25f7a4bc71d1032c0f54bca0796c81e5a2586a946', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/457.pdf', 'bytes': 1978756}, {'source_id': 'GDCA-F-d92feb6f85c82dc94776', 'source_path': 'sources/gdca-recon/2026-11-14/files/d92feb6f85c82dc94776_814.pdf', 'source_sha256': '86a1893a030025a54cd2b78253ec6e3f818f7980c1f61ee66167bbc341f9c494', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/814.pdf', 'bytes': 2013669}, {'source_id': 'GDCA-F-c9fa7134b28fd5798d4b', 'source_path': 'sources/gdca-recon/2026-11-14/files/c9fa7134b28fd5798d4b_9.pdf', 'source_sha256': '0c359806c63025e9e02d7863af8804372d622de55315a664b9a9cf34d561768e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/9.pdf', 'bytes': 2459228}, {'source_id': 'GDCA-F-4654c429287fa3e71147', 'source_path': 'sources/gdca-recon/2026-11-14/files/4654c429287fa3e71147_1307.pdf', 'source_sha256': 'a07722a5f3af2cdbe852f855f496b999888a837e98af3ed42fa6eca4f2fa9ebf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1307.pdf', 'bytes': 1973177}, {'source_id': 'GDCA-F-58bd968571f2064858cd', 'source_path': 'sources/gdca-recon/2026-11-14/files/58bd968571f2064858cd_386.pdf', 'source_sha256': 'caeffe79009b4ab1f10b4433b081d7a96a6156011f97eeeabed5ab5003a0509b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/386.pdf', 'bytes': 2239153}, {'source_id': 'GDCA-F-9326b6581f4f939543a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/9326b6581f4f939543a9_565.pdf', 'source_sha256': '30b8d1e993ddeadf7852b4d5665d179c8e41a0d3626b839f3bdb79ded46707a0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/565.pdf', 'bytes': 2364081}, {'source_id': 'GDCA-F-f67311086a7e73daa330', 'source_path': 'sources/gdca-recon/2026-11-14/files/f67311086a7e73daa330_272.pdf', 'source_sha256': '15dcd8d31b04eba6c884c6614f27dc1219b1bb9b74f534d9fc8569b57eab332b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/272.pdf', 'bytes': 1976106}, {'source_id': 'GDCA-F-ac789c6202a4949a0675', 'source_path': 'sources/gdca-recon/2026-11-14/files/ac789c6202a4949a0675_206.pdf', 'source_sha256': 'db2eaca592df523b94da6d2ae912758768ea4622908d9871d0ff0e6bca195e96', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/206.pdf', 'bytes': 1814319}, {'source_id': 'GDCA-F-86dd22836023a19f4fa5', 'source_path': 'sources/gdca-recon/2026-11-14/files/86dd22836023a19f4fa5_30.pdf', 'source_sha256': 'ad8e5842f2eaf31fb80eed91d14357944711bf7b0333fcce73ea251595d75c42', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/30.pdf', 'bytes': 91032}, {'source_id': 'GDCA-F-b0f02762da0020261cc9', 'source_path': 'sources/gdca-recon/2026-11-14/files/b0f02762da0020261cc9_588.pdf', 'source_sha256': '831d7323817819631b9bb06ae0cfd85b6a871930e6615d743dcdc071c309978b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/588.pdf', 'bytes': 1582823}, {'source_id': 'GDCA-F-fd8eb66b17ebe64dc774', 'source_path': 'sources/gdca-recon/2026-11-14/files/fd8eb66b17ebe64dc774_804.pdf', 'source_sha256': '1e0c754b530299c1b8888c56e2581c4b00a323a79c874657fbf0f96b63971455', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/804.pdf', 'bytes': 1817108}, {'source_id': 'GDCA-F-6c0bb92b2c9a6b2bac13', 'source_path': 'sources/gdca-recon/2026-11-14/files/6c0bb92b2c9a6b2bac13_624.pdf', 'source_sha256': 'd2d986285f42875a75c6d6d300825404cf76e46288679ba91b8895474846f62a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/624.pdf', 'bytes': 1755373}, {'source_id': 'GDCA-F-31977f898d14af1258e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/31977f898d14af1258e5_938.pdf', 'source_sha256': 'bf88c87b8a0740faff8dce39ba4fd222605e919566ca8dc7af16019d64e1acee', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/938.pdf', 'bytes': 1966424}, {'source_id': 'GDCA-F-99a8bcdff84d8ec15be0', 'source_path': 'sources/gdca-recon/2026-11-14/files/99a8bcdff84d8ec15be0_418.pdf', 'source_sha256': '7b213df39b50b8e8dfe9c5565caacb32b6a45a3362fd4bba9f43852ad417064f', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/418.pdf', 'bytes': 2553435}, {'source_id': 'GDCA-F-c3b7b11020d24813f9c7', 'source_path': 'sources/gdca-recon/2026-11-14/files/c3b7b11020d24813f9c7_18.pdf', 'source_sha256': 'f65c90b2aa02212945f935e622bfd69f33ee407196cb85b4ed7ac1571ef966ae', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/18.pdf', 'bytes': 2216368}, {'source_id': 'GDCA-F-57b89423c8fdce94c5b1', 'source_path': 'sources/gdca-recon/2026-11-14/files/57b89423c8fdce94c5b1_112.pdf', 'source_sha256': '77bb4b7fefa99b815e7ac00be410e149237686ad7a26061bb54a24e1b900391d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/112.pdf', 'bytes': 196758}, {'source_id': 'GDCA-F-f1da07d0a82ce389dbfd', 'source_path': 'sources/gdca-recon/2026-11-14/files/f1da07d0a82ce389dbfd_1548.pdf', 'source_sha256': '9535ed3889b69c01319bb14ea20a7b561cc16970d059dce572a90d2e4680cec9', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1548.pdf', 'bytes': 3932811}, {'source_id': 'GDCA-F-a325c74a5346a6620c5a', 'source_path': 'sources/gdca-recon/2026-11-14/files/a325c74a5346a6620c5a_1566.pdf', 'source_sha256': '4afcf31a55b0528e1b00a68c9de90fa979e775376e146666d459424ab51e28b4', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1566.pdf', 'bytes': 3195035}, {'source_id': 'GDCA-F-a0d4c2f9326fa7ba623c', 'source_path': 'sources/gdca-recon/2026-11-14/files/a0d4c2f9326fa7ba623c_1273.pdf', 'source_sha256': 'ebfb36c295f8ad46909f7d79afef152608d7c7fc212b9229a3bd70c45b4eefa7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1273.pdf', 'bytes': 2296011}, {'source_id': 'GDCA-F-5402c8154727eb17d659', 'source_path': 'sources/gdca-recon/2026-11-14/files/5402c8154727eb17d659_194.pdf', 'source_sha256': '037155dc924983d0682cb0777debfbf4cc5b84f9d4db73cefba049a5c7fde985', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/194.pdf', 'bytes': 2141664}, {'source_id': 'GDCA-F-9760d6950d23a3cbe68b', 'source_path': 'sources/gdca-recon/2026-11-14/files/9760d6950d23a3cbe68b_502.pdf', 'source_sha256': '81154a00b47949b0288c2cb8b0b1b38f95c6db8d1f3245a183a0c238a1d21e5b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/502.pdf', 'bytes': 1989713}, {'source_id': 'GDCA-F-4306506250f591e0ba1f', 'source_path': 'sources/gdca-recon/2026-11-14/files/4306506250f591e0ba1f_191.pdf', 'source_sha256': '1af2eed5ce56996925a13b22d832a7dd1e1caf43568c1668bb46e8cde51b2061', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/191.pdf', 'bytes': 2403068}, {'source_id': 'GDCA-F-f97723a7ef81755f5052', 'source_path': 'sources/gdca-recon/2026-11-14/files/f97723a7ef81755f5052_715.pdf', 'source_sha256': '9014bef5bc912eff56d7d24d71e8cc4cf5015f54e35e80a9788d85b2ea2ffcef', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/715.pdf', 'bytes': 1717518}, {'source_id': 'GDCA-F-99d5231e5c779e64632d', 'source_path': 'sources/gdca-recon/2026-11-14/files/99d5231e5c779e64632d_324.pdf', 'source_sha256': '0cdbd2347ab77de7e333a3a80210270581ee81d158e085fe114f6dc4616ade22', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/324.pdf', 'bytes': 1922555}, {'source_id': 'GDCA-F-8a4dada9a48ee21ed16c', 'source_path': 'sources/gdca-recon/2026-11-14/files/8a4dada9a48ee21ed16c_477.pdf', 'source_sha256': '50fa07a6c2dd6b4453baf69a15f6442d4fd5ea8e071aaba39a3e02d1b0438371', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/477.pdf', 'bytes': 2612576}, {'source_id': 'GDCA-F-e54d8afa98d045bf4864', 'source_path': 'sources/gdca-recon/2026-11-14/files/e54d8afa98d045bf4864_1467.pdf', 'source_sha256': '3b8952871a5f89cf0726fd21d07a49bf672735e7e1d12c231a56b0a5e02d8d85', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1467.pdf', 'bytes': 2368797}, {'source_id': 'GDCA-F-b346c0794492b8310021', 'source_path': 'sources/gdca-recon/2026-11-14/files/b346c0794492b8310021_581.pdf', 'source_sha256': '93c2549b4e8b247753e12d4f583c9de4b388d2080ccd654897fcbd7ee84da30d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/581.pdf', 'bytes': 1967295}, {'source_id': 'GDCA-F-1ef0c912da979f1da6b2', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ef0c912da979f1da6b2_208.pdf', 'source_sha256': '0e2856b6335125cff9cba3eb8523bec34e07a8fbb4655ca7890425c2732b0467', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/208.pdf', 'bytes': 1908114}, {'source_id': 'GDCA-F-cac08d6d17464aea6e8b', 'source_path': 'sources/gdca-recon/2026-11-14/files/cac08d6d17464aea6e8b_1385.pdf', 'source_sha256': 'bb2c4a92a7c83f11ed7ae87539f23c9491bb0801840cce96bc94404bed083410', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1385.pdf', 'bytes': 2226960}, {'source_id': 'GDCA-F-c099ec3177859279987c', 'source_path': 'sources/gdca-recon/2026-11-14/files/c099ec3177859279987c_298.pdf', 'source_sha256': '649563d76db5d4dbb606e35bd0981bc9a01915cb5ce9a28f313b49b5b5e6afae', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/298.pdf', 'bytes': 1898034}, {'source_id': 'GDCA-F-da0b3200b475d9db41de', 'source_path': 'sources/gdca-recon/2026-11-14/files/da0b3200b475d9db41de_659.pdf', 'source_sha256': 'a49fea740eb6be637cbf7ed65759798d652c52c932c82ac65dcec0a3913c0a84', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/659.pdf', 'bytes': 1930141}, {'source_id': 'GDCA-F-576a98ffb812dd2b0446', 'source_path': 'sources/gdca-recon/2026-11-14/files/576a98ffb812dd2b0446_160.pdf', 'source_sha256': '1f35d4b87769919fdc22d06763c38a8fd41d46b20cbcaa74360f6703383bb16b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/160.pdf', 'bytes': 2049764}, {'source_id': 'GDCA-F-a37f99486e61c01aeadb', 'source_path': 'sources/gdca-recon/2026-11-14/files/a37f99486e61c01aeadb_413.pdf', 'source_sha256': 'e7a2fb0d8f96cfa0669060de74de37c1ca7fe55ab1d06ae251fa5b20744901cf', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/413.pdf', 'bytes': 2332146}, {'source_id': 'GDCA-F-d3f507b89d6f67202d70', 'source_path': 'sources/gdca-recon/2026-11-14/files/d3f507b89d6f67202d70_985.pdf', 'source_sha256': '1c521211e10583a3af9ed7d01a318c80620a611cba99f7f91433f8bc508e28f4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/985.pdf', 'bytes': 1776763}, {'source_id': 'GDCA-F-ef5d2bb3a563eb95dd83', 'source_path': 'sources/gdca-recon/2026-11-14/files/ef5d2bb3a563eb95dd83_104.pdf', 'source_sha256': 'f6a12dad6291ff1ebefdce3443fbaab456a457f71c1db1bf66778427cc0371d7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/104.pdf', 'bytes': 2924928}, {'source_id': 'GDCA-F-42da34738193e6545ffd', 'source_path': 'sources/gdca-recon/2026-11-14/files/42da34738193e6545ffd_887.pdf', 'source_sha256': '88c39c1bf7fc8a1c8fc9019751720dce0b31a7268f0390028799e8df2ac86c30', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/887.pdf', 'bytes': 2140794}, {'source_id': 'GDCA-F-b9ddff7c186d687f677a', 'source_path': 'sources/gdca-recon/2026-11-14/files/b9ddff7c186d687f677a_806.pdf', 'source_sha256': 'bfc3108be69d9e70f0dfede372c81d51b23f47977b594d6e24633ebfc064ff71', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/806.pdf', 'bytes': 1857886}, {'source_id': 'GDCA-F-ca6cde3633b2d0c74eee', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca6cde3633b2d0c74eee_125.pdf', 'source_sha256': 'e882eff2541696c535923563948a21700d083946c6fa0d4d37e3fa8678fc2692', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/125.pdf', 'bytes': 2591152}, {'source_id': 'GDCA-F-07cbfc8939b4e0d5f0e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/07cbfc8939b4e0d5f0e3_671.pdf', 'source_sha256': '9e9cf669d6a4339be7b9336f2a867d9c6381abca9e4b65d4631b5e866f1243cd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/671.pdf', 'bytes': 1850182}, {'source_id': 'GDCA-F-714b39a82f6af66e9d20', 'source_path': 'sources/gdca-recon/2026-11-14/files/714b39a82f6af66e9d20_766.pdf', 'source_sha256': 'e64f4b883afc3cfc607278993510355ff7555213f1fbe3e0e18d84d399c61b76', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/766.pdf', 'bytes': 2109685}, {'source_id': 'GDCA-F-75951ec4a66a742ce9d4', 'source_path': 'sources/gdca-recon/2026-11-14/files/75951ec4a66a742ce9d4_1151.pdf', 'source_sha256': '01be0541947e4079bd3c65b251ce8a4b17b4f19583a4ee2233167cd51e793787', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1151.pdf', 'bytes': 852797}, {'source_id': 'GDCA-F-5d0173d950502832e455', 'source_path': 'sources/gdca-recon/2026-11-14/files/5d0173d950502832e455_1470.pdf', 'source_sha256': 'f5996eb491002cee70ba65138812e92283393091bae9085b5d2b66092abeb966', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1470.pdf', 'bytes': 2682345}, {'source_id': 'GDCA-F-1e4f6006911971a45384', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e4f6006911971a45384_699.pdf', 'source_sha256': 'ea8b71340fb6fc71eeace10e05aba7a1b928892b05e2d4cace2cd14105aa8f91', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/699.pdf', 'bytes': 1872755}, {'source_id': 'GDCA-F-ab6eb5fc9d1949b54f5b', 'source_path': 'sources/gdca-recon/2026-11-14/files/ab6eb5fc9d1949b54f5b_444.pdf', 'source_sha256': '46d4967fc714708c495b6d409aae79ee7b1a7b54fb0bb14df24c7c92ac7ffa85', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/444.pdf', 'bytes': 2161255}, {'source_id': 'GDCA-F-7d7bf17c14da0b944bf6', 'source_path': 'sources/gdca-recon/2026-11-14/files/7d7bf17c14da0b944bf6_474.pdf', 'source_sha256': '572cff8cf6bffc1ad32bc57a559aa7834f4408b95bbd6347473c234500816ee8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/474.pdf', 'bytes': 1784401}, {'source_id': 'GDCA-F-27914a62b7dafd675aee', 'source_path': 'sources/gdca-recon/2026-11-14/files/27914a62b7dafd675aee_1399.pdf', 'source_sha256': '6bafe7e78e33efe6b784263c70c0d094a1a7b4af8bbef408d24631703e1a27f0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1399.pdf', 'bytes': 2266897}, {'source_id': 'GDCA-F-d899a7bee9f4a9875790', 'source_path': 'sources/gdca-recon/2026-11-14/files/d899a7bee9f4a9875790_36.pdf', 'source_sha256': '911a14277b2e4af6a2b092d95138e025c4623ea68d9c877bb015f09a2bde1921', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/36.pdf', 'bytes': 111216}, {'source_id': 'GDCA-F-1e8984724377761f57a5', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e8984724377761f57a5_158.pdf', 'source_sha256': '82f56d486b3863dd890afc64d8770325be06c1600c1c9fc6a77423793fe4a312', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/158.pdf', 'bytes': 1893272}, {'source_id': 'GDCA-F-14527295ad87a75914f6', 'source_path': 'sources/gdca-recon/2026-11-14/files/14527295ad87a75914f6_127.pdf', 'source_sha256': '75b2b3871bef99fbd6dc7d9bd73065357b204a800799c95fb1d01060322c2038', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/127.pdf', 'bytes': 2066176}, {'source_id': 'GDCA-F-d79bfb3caebd20880dd4', 'source_path': 'sources/gdca-recon/2026-11-14/files/d79bfb3caebd20880dd4_589.pdf', 'source_sha256': '4d68746b46ea8c03bbc50ebc20682ddaa32c437e03b30fea8c2404621ea954a4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/589.pdf', 'bytes': 2144820}, {'source_id': 'GDCA-F-2d6ed2c9c8795a9cffeb', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d6ed2c9c8795a9cffeb_749.pdf', 'source_sha256': 'be6718a93c26a484930a949020b734ec538ca520581dc46be93f64cad7752231', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/749.pdf', 'bytes': 1848374}, {'source_id': 'GDCA-F-33d6c0c542ec8386ff59', 'source_path': 'sources/gdca-recon/2026-11-14/files/33d6c0c542ec8386ff59_739.pdf', 'source_sha256': '56e720a6e7525e4b5a6148e70015f97a21d5aeec71415e8f878a535d0b6dcdbf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/739.pdf', 'bytes': 1861203}, {'source_id': 'GDCA-F-e728417275884ae95ebe', 'source_path': 'sources/gdca-recon/2026-11-14/files/e728417275884ae95ebe_432.pdf', 'source_sha256': '97f47399b676eb880813d4a934fe201212e980103584a1664ca7958c6c70b0b0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/432.pdf', 'bytes': 2138768}, {'source_id': 'GDCA-F-3d41b21c80ee4ae88cc3', 'source_path': 'sources/gdca-recon/2026-11-14/files/3d41b21c80ee4ae88cc3_1498.pdf', 'source_sha256': 'a59d598176becc7745e1dca7d4ac76a34ccdc8c892dba9b3c13174678a668ffb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1498.pdf', 'bytes': 1928354}, {'source_id': 'GDCA-F-78ae9186a2acdea8baf1', 'source_path': 'sources/gdca-recon/2026-11-14/files/78ae9186a2acdea8baf1_1113.pdf', 'source_sha256': '60cb24ebb7c64bbb439784795bfd2ff874bc9551cca05fb82d3c75898d151a70', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1113.pdf', 'bytes': 2430115}, {'source_id': 'GDCA-F-e8ba4188d47573ff7aea', 'source_path': 'sources/gdca-recon/2026-11-14/files/e8ba4188d47573ff7aea_132.pdf', 'source_sha256': 'b069dffd0dc506d0192d09c19261b22b7b422864d6c12fbe6a1214fd4cdd087f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/132.pdf', 'bytes': 2094348}, {'source_id': 'GDCA-F-ad2b17dd188de5499ed1', 'source_path': 'sources/gdca-recon/2026-11-14/files/ad2b17dd188de5499ed1_1554.pdf', 'source_sha256': '1d9ae6d3d66a34e8463dbe857f4b88983cf2a058a9bb8044c768d7fd69719061', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1554.pdf', 'bytes': 2806907}, {'source_id': 'GDCA-F-1b40ad46e953f8b83a1a', 'source_path': 'sources/gdca-recon/2026-11-14/files/1b40ad46e953f8b83a1a_1376.pdf', 'source_sha256': 'd27ec4a9c3923f22b5002bfccbffc9a3e3bd7afb8784d993e5d853b50cd752e6', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1376.pdf', 'bytes': 3589945}, {'source_id': 'GDCA-F-701c74ba36f203af1e53', 'source_path': 'sources/gdca-recon/2026-11-14/files/701c74ba36f203af1e53_240.pdf', 'source_sha256': 'f948e4dd3816d6be853a1b82b9261b648591991ecc704739405344cf42214890', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/240.pdf', 'bytes': 1946622}, {'source_id': 'GDCA-F-f6401157574ecd14070a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f6401157574ecd14070a_801.pdf', 'source_sha256': 'db615fbfc0c8e1157c8cacf56055e4dc1af51c060508b4b06be74858ba75b901', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/801.pdf', 'bytes': 2251995}, {'source_id': 'GDCA-F-77ea0bdca21456455524', 'source_path': 'sources/gdca-recon/2026-11-14/files/77ea0bdca21456455524_1261.pdf', 'source_sha256': '67b73b75fd8fee9de056437d4d0dd14e5915a37c65803cedbb45210b267d515b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1261.pdf', 'bytes': 2110436}, {'source_id': 'GDCA-F-d831d0533c36bb0ba98e', 'source_path': 'sources/gdca-recon/2026-11-14/files/d831d0533c36bb0ba98e_1125.pdf', 'source_sha256': '112d3fdce1d3acf764681aa21dbcbf5f7939c29073dc73acbf3e9e62f154f11e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1125.pdf', 'bytes': 2087480}, {'source_id': 'GDCA-F-c74b7944ba92f946b992', 'source_path': 'sources/gdca-recon/2026-11-14/files/c74b7944ba92f946b992_73.pdf', 'source_sha256': '49a86ce6401b130c3332e5547360f8fd34e88eae200d885401abc3777ab466e1', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/73.pdf', 'bytes': 2372770}, {'source_id': 'GDCA-F-d3c3eb5b03e6f65afa5f', 'source_path': 'sources/gdca-recon/2026-11-14/files/d3c3eb5b03e6f65afa5f_142.pdf', 'source_sha256': '6a55eb61a130cd43ac0feb6173e8f4d9885cdbb10b741cf4a5f46ba0c1c6b77b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/142.pdf', 'bytes': 2046555}, {'source_id': 'GDCA-F-4bd8a5f9dca09ff757b0', 'source_path': 'sources/gdca-recon/2026-11-14/files/4bd8a5f9dca09ff757b0_1204.pdf', 'source_sha256': 'dafaa94c04036156b7336e7bef9152bb52d543ea99b52b3ffe1f517bb8975dc3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1204.pdf', 'bytes': 1999881}, {'source_id': 'GDCA-F-8e0d1e73aee625173aa8', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e0d1e73aee625173aa8_159.pdf', 'source_sha256': 'f713439af0aea830bc61caa22b2ca1a0eba6f97c2e1b4cfe92175af7efcf5d9d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/159.pdf', 'bytes': 1875794}, {'source_id': 'GDCA-F-aeee45f5158a09339731', 'source_path': 'sources/gdca-recon/2026-11-14/files/aeee45f5158a09339731_1168.pdf', 'source_sha256': '4544f8e9af04b53df208a7e63b983308c8444d7af841618762e345b40c97ed72', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1168.pdf', 'bytes': 2106980}, {'source_id': 'GDCA-F-0f012b6aa057ce53bc3c', 'source_path': 'sources/gdca-recon/2026-11-14/files/0f012b6aa057ce53bc3c_787.pdf', 'source_sha256': '96a08c460d6aecc90cdae8e029e07bf6fa1f1561c1bea1922a6f6159e70b5260', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/787.pdf', 'bytes': 1895853}, {'source_id': 'GDCA-F-c2ee1dd70981133a1130', 'source_path': 'sources/gdca-recon/2026-11-14/files/c2ee1dd70981133a1130_496.pdf', 'source_sha256': '663abc17f0a2930f3c4695ac28b2e6846e5933933ebcfe31bd459dab5c469a91', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/496.pdf', 'bytes': 1510210}, {'source_id': 'GDCA-F-eb24d0b7a10b1cfca975', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb24d0b7a10b1cfca975_178.pdf', 'source_sha256': '0ad7969df96e4866cbe4b0cda5d6d7ee7dd6a46f314094372526a608715f2a22', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/178.pdf', 'bytes': 1763234}, {'source_id': 'GDCA-F-b37f8dfb7ca4e79583c4', 'source_path': 'sources/gdca-recon/2026-11-14/files/b37f8dfb7ca4e79583c4_816.pdf', 'source_sha256': '658199b4d6fff3eb64f1459249193b021956e82f58ecb59b4ed43bcc7caa1179', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/816.pdf', 'bytes': 1850650}, {'source_id': 'GDCA-F-413abf94afe1fe658b4a', 'source_path': 'sources/gdca-recon/2026-11-14/files/413abf94afe1fe658b4a_219.pdf', 'source_sha256': 'ea9c87de8ebf1a4fa93b2cdcc87ac501ce73a3c508703f48b890217c70fb799d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/219.pdf', 'bytes': 2186253}, {'source_id': 'GDCA-F-a905c3160c5af8003dc5', 'source_path': 'sources/gdca-recon/2026-11-14/files/a905c3160c5af8003dc5_1120.pdf', 'source_sha256': '75e9a8db9be6c10ce8fb3b0c59d46723eb7e0c19925246f75bd42214eb0a65a6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1120.pdf', 'bytes': 1911196}, {'source_id': 'GDCA-F-3b34ab35eb3e91ec9b00', 'source_path': 'sources/gdca-recon/2026-11-14/files/3b34ab35eb3e91ec9b00_315.pdf', 'source_sha256': '0159bfe4131e3914648bf156a6d0b9bc4d0049ede38bb5765bfc142463379278', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/315.pdf', 'bytes': 2338141}, {'source_id': 'GDCA-F-5b97f708e6c5ca8db342', 'source_path': 'sources/gdca-recon/2026-11-14/files/5b97f708e6c5ca8db342_388.pdf', 'source_sha256': 'a4171bc670a922e68e330c479b728275a2439cc25801509218d4388460320032', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/388.pdf', 'bytes': 2245327}, {'source_id': 'GDCA-F-d8cc534ab3cbc8b8a2ca', 'source_path': 'sources/gdca-recon/2026-11-14/files/d8cc534ab3cbc8b8a2ca_8.pdf', 'source_sha256': 'b981aaee9bdc5ed66be11f0be69c1b85be7ae9a2af4546ec567f0a96a14b2e73', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/8.pdf', 'bytes': 2079545}, {'source_id': 'GDCA-F-7111cc443597903e9744', 'source_path': 'sources/gdca-recon/2026-11-14/files/7111cc443597903e9744_292.pdf', 'source_sha256': '197a9f00822a91b8dae99c5e5fa8e22e6514a85992e41a889b42e3f194554106', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/292.pdf', 'bytes': 1892811}, {'source_id': 'GDCA-F-96303d2bd8682b0bdd43', 'source_path': 'sources/gdca-recon/2026-11-14/files/96303d2bd8682b0bdd43_722.pdf', 'source_sha256': '717c71ff688d830202b4acc4f2f1103e2f0773da419c497df1c30812841e241f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/722.pdf', 'bytes': 2231432}, {'source_id': 'GDCA-F-a1d15f80482d7a62407f', 'source_path': 'sources/gdca-recon/2026-11-14/files/a1d15f80482d7a62407f_981.pdf', 'source_sha256': '0502ae942d89b6b52ce77524bb024fb3c6bf372694a9a09653ce1316bbfea6ce', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/981.pdf', 'bytes': 2210385}, {'source_id': 'GDCA-F-ffab13c965b1a3f2a90a', 'source_path': 'sources/gdca-recon/2026-11-14/files/ffab13c965b1a3f2a90a_1023.pdf', 'source_sha256': 'cfe2ef7f802144f41bda9dbbbbe83d9808ff25cf7ec144ac6f2d7b61247f85c5', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1023.pdf', 'bytes': 2151608}, {'source_id': 'GDCA-F-d8b9cfccc34833183827', 'source_path': 'sources/gdca-recon/2026-11-14/files/d8b9cfccc34833183827_80.pdf', 'source_sha256': '84992284438fc05aea341e75f6fec4d9bf7551e909100c11cf179a3e5fb0a8d2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/80.pdf', 'bytes': 2116828}, {'source_id': 'GDCA-F-bea7801f395fa9108325', 'source_path': 'sources/gdca-recon/2026-11-14/files/bea7801f395fa9108325_263.pdf', 'source_sha256': '200f8a95f956a5731d8ccc8d84ba6c6ee4392b6949f90013b20568881af1cbea', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/263.pdf', 'bytes': 2227398}, {'source_id': 'GDCA-F-4e2c4a3115c004002a4f', 'source_path': 'sources/gdca-recon/2026-11-14/files/4e2c4a3115c004002a4f_1473.pdf', 'source_sha256': 'a61b22176f9ed2cde9a98a196b80061d11ba69d652ab4badae4d5345cb29026a', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1473.pdf', 'bytes': 3479056}, {'source_id': 'GDCA-F-36d709bed964ae08c1d0', 'source_path': 'sources/gdca-recon/2026-11-14/files/36d709bed964ae08c1d0_54.pdf', 'source_sha256': 'a87b99e63bde981df2a5385be4fcfcb3c57b2534355c5c8760702ae8d72fe41b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/54.pdf', 'bytes': 1933610}, {'source_id': 'GDCA-F-80b7f37a1b2d64f786c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/80b7f37a1b2d64f786c1_612.pdf', 'source_sha256': 'ba5a36e277d515acc413e1a6ed9dc4e2accbbc0045e8dea26ef3b3611039708e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/612.pdf', 'bytes': 1975777}, {'source_id': 'GDCA-F-2ac8cf00b7d0f75e5732', 'source_path': 'sources/gdca-recon/2026-11-14/files/2ac8cf00b7d0f75e5732_119.pdf', 'source_sha256': '04ca7d00c26ac179352a38d789a7ee23b9c9fc0b2f1c1e9e28957a6f0eef07bd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/119.pdf', 'bytes': 2170534}, {'source_id': 'GDCA-F-71c1b954ffef2d07f724', 'source_path': 'sources/gdca-recon/2026-11-14/files/71c1b954ffef2d07f724_1070.pdf', 'source_sha256': 'b387ecfb0cb40f3a6b051812d70fcdc62464096a90def208131405bf18c297c8', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1070.pdf', 'bytes': 2729222}, {'source_id': 'GDCA-F-d3eef8a772322391f2c3', 'source_path': 'sources/gdca-recon/2026-11-14/files/d3eef8a772322391f2c3_1247.pdf', 'source_sha256': '935117079c18b8f437e99176d44cd54809e77895f76e7fca53adb4ffbef8917b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1247.pdf', 'bytes': 1779400}, {'source_id': 'GDCA-F-f4fad40a2cd444ecf3a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/f4fad40a2cd444ecf3a9_339.pdf', 'source_sha256': '9a9efed80ffc408baacb39e109fa311c9c613271c72cf9b5681e171e6f38de70', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/339.pdf', 'bytes': 1963921}, {'source_id': 'GDCA-F-5c780857ab9068e15d88', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c780857ab9068e15d88_934.pdf', 'source_sha256': '17c2a2bc3e6c129f6f2d2de4a19818555c5ab608411929797bca6a26df58b78f', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/934.pdf', 'bytes': 2573539}, {'source_id': 'GDCA-F-3b179cc28fe260d5a525', 'source_path': 'sources/gdca-recon/2026-11-14/files/3b179cc28fe260d5a525_997.pdf', 'source_sha256': 'd42eadd4a6b387ff81edb1f3677c702a65c4d0e100b2e2b909420d577f0c9f11', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/997.pdf', 'bytes': 2363425}, {'source_id': 'GDCA-F-7ed79367b221eab8a6ba', 'source_path': 'sources/gdca-recon/2026-11-14/files/7ed79367b221eab8a6ba_570.pdf', 'source_sha256': 'db4797cf907e617700387554f81b93b0eb5724bc76f2d331c2250831e3879c04', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/570.pdf', 'bytes': 1775099}, {'source_id': 'GDCA-F-a80c86b63d0c2be413f1', 'source_path': 'sources/gdca-recon/2026-11-14/files/a80c86b63d0c2be413f1_355.pdf', 'source_sha256': '3f9b9e92fe74fa9180734502e4f8d954d29b69dd9b4f7cf4e835ff3509bbcec7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/355.pdf', 'bytes': 2047861}, {'source_id': 'GDCA-F-66d00ec12d1016aeefb8', 'source_path': 'sources/gdca-recon/2026-11-14/files/66d00ec12d1016aeefb8_1413.pdf', 'source_sha256': '4bbd72ccf2516eacd20d211a1829b8f58d975404d6aa28551160e5573c3e7b57', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1413.pdf', 'bytes': 2298508}, {'source_id': 'GDCA-F-d39c00ec60a921fa761c', 'source_path': 'sources/gdca-recon/2026-11-14/files/d39c00ec60a921fa761c_963.pdf', 'source_sha256': '83b54df84e7f1d06ba39bb2d76c66736da6e3c6ab88abbc2790b7d537ca9b730', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/963.pdf', 'bytes': 1913742}, {'source_id': 'GDCA-F-e9b2e1cbfc14ce4146a0', 'source_path': 'sources/gdca-recon/2026-11-14/files/e9b2e1cbfc14ce4146a0_1054.pdf', 'source_sha256': '70f95a3f062a68b85d5a25e7ca709a91d8ce67d00e4a64474085b43c2ed7386a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1054.pdf', 'bytes': 2092695}, {'source_id': 'GDCA-F-af9e76b2b59bc15739bc', 'source_path': 'sources/gdca-recon/2026-11-14/files/af9e76b2b59bc15739bc_1006.pdf', 'source_sha256': '4d0c0ea4fce74eb746209d7a028548ff445cdb3cb6ea5d89708a9168094a956a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1006.pdf', 'bytes': 1708750}, {'source_id': 'GDCA-F-4ce36db1790893d47b59', 'source_path': 'sources/gdca-recon/2026-11-14/files/4ce36db1790893d47b59_212.pdf', 'source_sha256': '1080cfb4b7aa0bcee9093d526be1bf4c8944daa76698e3cb9fd824c27c3dbcdb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/212.pdf', 'bytes': 2333320}, {'source_id': 'GDCA-F-13281e9c0bf61f5ff28c', 'source_path': 'sources/gdca-recon/2026-11-14/files/13281e9c0bf61f5ff28c_971.pdf', 'source_sha256': 'ca38eb7cab7effa6e8e40d83300bcd956da8088d146970eb30f89344e07158ab', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/971.pdf', 'bytes': 2307658}, {'source_id': 'GDCA-F-35cb099210d82dc3bf13', 'source_path': 'sources/gdca-recon/2026-11-14/files/35cb099210d82dc3bf13_384.pdf', 'source_sha256': '158557c6ee299c962feade99d7ec9e4e24631cea2679c5c1581d3b9f78d19dc1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/384.pdf', 'bytes': 1919993}, {'source_id': 'GDCA-F-8c92b23b2463e0c18e08', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c92b23b2463e0c18e08_1190.pdf', 'source_sha256': '199a01e02c1a980a6a273191c7ff1bc03ca232191647c8f396310fa2ff93b5ee', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1190.pdf', 'bytes': 1778109}, {'source_id': 'GDCA-F-1de28a5e472b65356b3f', 'source_path': 'sources/gdca-recon/2026-11-14/files/1de28a5e472b65356b3f_363.pdf', 'source_sha256': 'bcf335d41295d58f166c1948935c33cc21a434cdbd93d748f9d45b1e39250c51', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/363.pdf', 'bytes': 1907871}, {'source_id': 'GDCA-F-9fb32360fdc199da00dd', 'source_path': 'sources/gdca-recon/2026-11-14/files/9fb32360fdc199da00dd_1303.pdf', 'source_sha256': 'cc312ee20c561a07c6ae02d098f8a58f6c8376d1f061326523410cee5e1023bc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1303.pdf', 'bytes': 2062968}, {'source_id': 'GDCA-F-be8d1cb3919d2b721422', 'source_path': 'sources/gdca-recon/2026-11-14/files/be8d1cb3919d2b721422_537.pdf', 'source_sha256': '6119a02e84ec750e10b99fe7628a281d2ec888c9ee8dcddbe9b415aea5064df1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/537.pdf', 'bytes': 1722312}, {'source_id': 'GDCA-F-11ce62e559986b70abeb', 'source_path': 'sources/gdca-recon/2026-11-14/files/11ce62e559986b70abeb_397.pdf', 'source_sha256': '8dd23cd7fbd245f697af25b32ed3e30d3f0a9ef263f19515711b4cc634a426e0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/397.pdf', 'bytes': 1841560}, {'source_id': 'GDCA-F-e216f0ed516ebd89611b', 'source_path': 'sources/gdca-recon/2026-11-14/files/e216f0ed516ebd89611b_1211.pdf', 'source_sha256': '5e11574a8394903fdd57f3b6c3397cad876a9df21eb0c202c09381e2447a1954', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1211.pdf', 'bytes': 957746}, {'source_id': 'GDCA-F-ac10ee376fb19181a8b7', 'source_path': 'sources/gdca-recon/2026-11-14/files/ac10ee376fb19181a8b7_782.pdf', 'source_sha256': 'd85c7168a090c8194500471097e9a34f7be0c19231617e675b5dcdd6f8f1bbab', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/782.pdf', 'bytes': 1717212}, {'source_id': 'GDCA-F-53852e68de884d98e01c', 'source_path': 'sources/gdca-recon/2026-11-14/files/53852e68de884d98e01c_690.pdf', 'source_sha256': 'e383bf31e23e0056b27ff6e1d9200d9830f05ed47546ca6ca5bb9f12869ef5c6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/690.pdf', 'bytes': 1956931}, {'source_id': 'GDCA-F-52c83a78fde57a49bc29', 'source_path': 'sources/gdca-recon/2026-11-14/files/52c83a78fde57a49bc29_1169.pdf', 'source_sha256': '1ee1bc4cd042fb1b33dd18afcb6e30b90080ac447de03866111731bcf6e976e1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1169.pdf', 'bytes': 1827281}, {'source_id': 'GDCA-F-3a7c5aed4a19570ab858', 'source_path': 'sources/gdca-recon/2026-11-14/files/3a7c5aed4a19570ab858_313.pdf', 'source_sha256': '4ae56b70e287cbf3ee3cd44e2062767aa09625caaa8cff9569220c7f242e2354', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/313.pdf', 'bytes': 1909579}, {'source_id': 'GDCA-F-f58238947a4e48d61e4a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f58238947a4e48d61e4a_225.pdf', 'source_sha256': '32158ef2f6b33ecf731819104af888231b0319266f46b94d368942dc17c02626', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/225.pdf', 'bytes': 1890546}, {'source_id': 'GDCA-F-5c967522121de0cfdbfb', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c967522121de0cfdbfb_270.pdf', 'source_sha256': 'd420c2f014a4fcc287e59e5c8c0825c20ecf1bb7155117412d157e9cc145aab2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/270.pdf', 'bytes': 1866331}, {'source_id': 'GDCA-F-26a7883d1913ff702d88', 'source_path': 'sources/gdca-recon/2026-11-14/files/26a7883d1913ff702d88_1097.pdf', 'source_sha256': 'd8d3ef9513565403b22dc65e565342fb1913337ff37d0586fb99f7743447e2d5', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1097.pdf', 'bytes': 2475775}, {'source_id': 'GDCA-F-f45da4401b428f2f9fc2', 'source_path': 'sources/gdca-recon/2026-11-14/files/f45da4401b428f2f9fc2_649.pdf', 'source_sha256': '38f5e5c6dfd9fa3a71c4b55c6d4722c93441527d678f71247a894329fcd5dc2f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/649.pdf', 'bytes': 2229041}, {'source_id': 'GDCA-F-da72408d50f6cc911f33', 'source_path': 'sources/gdca-recon/2026-11-14/files/da72408d50f6cc911f33_1159.pdf', 'source_sha256': 'e7aa53b36955a1166f4bc009ccc036f19df659ace796ab24f70fcbf9720fca68', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1159.pdf', 'bytes': 1977067}, {'source_id': 'GDCA-F-a202cf0eea73883dc615', 'source_path': 'sources/gdca-recon/2026-11-14/files/a202cf0eea73883dc615_1269.pdf', 'source_sha256': 'f7b057424affe0c6aab10be97b3441c3e38222d84f680796b89ae1a678112107', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1269.pdf', 'bytes': 1871998}, {'source_id': 'GDCA-F-e13288c9cb185bf8cf48', 'source_path': 'sources/gdca-recon/2026-11-14/files/e13288c9cb185bf8cf48_1248.pdf', 'source_sha256': 'd0a77c65d43953a000db9e8d20d1d36aee367e64ac70820210be48fa39feff1e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1248.pdf', 'bytes': 1844739}, {'source_id': 'GDCA-F-28e15bef25c5acf8b620', 'source_path': 'sources/gdca-recon/2026-11-14/files/28e15bef25c5acf8b620_742.pdf', 'source_sha256': 'e6411f2d128f76ae2dd674281137f354422d0c0163c26ecb523655d01a7c1a88', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/742.pdf', 'bytes': 1441984}, {'source_id': 'GDCA-F-765a34b370eb335351be', 'source_path': 'sources/gdca-recon/2026-11-14/files/765a34b370eb335351be_1435.pdf', 'source_sha256': 'f19d5a660ceb2ac98ca65f4a032d70e4d243e923258970753ab4bc920d52c574', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1435.pdf', 'bytes': 2263468}, {'source_id': 'GDCA-F-1131dbd6b55a47695c33', 'source_path': 'sources/gdca-recon/2026-11-14/files/1131dbd6b55a47695c33_422.pdf', 'source_sha256': '2102beb0fa1b28d4969f557e8ffbf549989979fec7490a97c6da8a318af9e52b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/422.pdf', 'bytes': 7200476}, {'source_id': 'GDCA-F-ca69d14a3cff5975cec9', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca69d14a3cff5975cec9_231.pdf', 'source_sha256': '20bdf373a37e09c0bec3408df207e59cf5e0aab95deececb116bd2926c2b3eab', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/231.pdf', 'bytes': 1806361}, {'source_id': 'GDCA-F-6fc4b7fc168087c0c88f', 'source_path': 'sources/gdca-recon/2026-11-14/files/6fc4b7fc168087c0c88f_1034.pdf', 'source_sha256': '8a33c2a5c357febc9dde6de7c82536ab2e515c2ffd83301a6df5ed7447fd0a61', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1034.pdf', 'bytes': 1184348}, {'source_id': 'GDCA-F-21ee1e4d22766b6789a2', 'source_path': 'sources/gdca-recon/2026-11-14/files/21ee1e4d22766b6789a2_259.pdf', 'source_sha256': 'e30d3670e3298ed0715bbe738b575206b94d13de94c66c8445fc230a5f9523d0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/259.pdf', 'bytes': 1895679}, {'source_id': 'GDCA-F-ed7167d0f5c1f0df0c58', 'source_path': 'sources/gdca-recon/2026-11-14/files/ed7167d0f5c1f0df0c58_719.pdf', 'source_sha256': '5075f9db9014d295bc2a276280404a37acab422789f587151f440ddcfb605bd8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/719.pdf', 'bytes': 2066305}, {'source_id': 'GDCA-F-e08afd44c5f1e74df514', 'source_path': 'sources/gdca-recon/2026-11-14/files/e08afd44c5f1e74df514_655.pdf', 'source_sha256': '7cf256effe730d9b5e121b83ce7f39884fba076cc386ea2f0af81f22b34ca11d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/655.pdf', 'bytes': 1833775}, {'source_id': 'GDCA-F-4fd415923bf88d7a5a3e', 'source_path': 'sources/gdca-recon/2026-11-14/files/4fd415923bf88d7a5a3e_658.pdf', 'source_sha256': '054da30b781b055783ed03ba3b42d8386dfc8df61b05026df0a7c2db1cadc13b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/658.pdf', 'bytes': 2028710}, {'source_id': 'GDCA-F-5f97be56d648d64bc898', 'source_path': 'sources/gdca-recon/2026-11-14/files/5f97be56d648d64bc898_120.pdf', 'source_sha256': 'e1cb548149dc05793c354a1de86217f8f3726f402e630a2a329ae0e20d28b8fa', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/120.pdf', 'bytes': 2158032}, {'source_id': 'GDCA-F-280bedaeb716a24bc0a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/280bedaeb716a24bc0a9_24.pdf', 'source_sha256': 'b42f5778eaf60a66d115e1dc1a056d025014ee30ba915828c939e4cd0c139f43', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/24.pdf', 'bytes': 110337}, {'source_id': 'GDCA-F-999cdbb41d53a31805db', 'source_path': 'sources/gdca-recon/2026-11-14/files/999cdbb41d53a31805db_603.pdf', 'source_sha256': 'ebf8b59d8fd05cc2f76f139e6125b1857684d558ee7aed7e947b8432555534d4', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/603.pdf', 'bytes': 1340308}, {'source_id': 'GDCA-F-f2830bedd6b58314bc77', 'source_path': 'sources/gdca-recon/2026-11-14/files/f2830bedd6b58314bc77_1325.pdf', 'source_sha256': 'ec582b2d243a6b858a3addc5129f684ee9a91698d9001aefd50d32b1e6442a50', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1325.pdf', 'bytes': 1876250}, {'source_id': 'GDCA-F-7de593b7606f3c246885', 'source_path': 'sources/gdca-recon/2026-11-14/files/7de593b7606f3c246885_1117.pdf', 'source_sha256': '95cb2c86a6dfc898287ce212a944466c33a8d09ce31db941d161fb0dc4ebd883', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1117.pdf', 'bytes': 1815286}, {'source_id': 'GDCA-F-78452fb8a1ffc194e203', 'source_path': 'sources/gdca-recon/2026-11-14/files/78452fb8a1ffc194e203_617.pdf', 'source_sha256': '52f9477604ec08845d40e2b2b12969d3b4bd5461638af10b547e76403f361843', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/617.pdf', 'bytes': 1738549}, {'source_id': 'GDCA-F-8a4c740087ada008711f', 'source_path': 'sources/gdca-recon/2026-11-14/files/8a4c740087ada008711f_1271.pdf', 'source_sha256': '44434dfd25087c2ae842dc7e1e22e40dabd7f25330272c3b364d7ff9f7bdf41a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1271.pdf', 'bytes': 1928195}, {'source_id': 'GDCA-F-710ebe4449cf3da74e7d', 'source_path': 'sources/gdca-recon/2026-11-14/files/710ebe4449cf3da74e7d_21.pdf', 'source_sha256': '1dee0a9d692333ae370a12e10fa5acd56a1ce15223828db91b883cc2402aff96', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/21.pdf', 'bytes': 107807}, {'source_id': 'GDCA-F-eadf896321c11ea20043', 'source_path': 'sources/gdca-recon/2026-11-14/files/eadf896321c11ea20043_452.pdf', 'source_sha256': 'bf60e8134be03d129bf0a2810829148fee11ae04aa6db15a5dab613fc150d267', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/452.pdf', 'bytes': 2404685}, {'source_id': 'GDCA-F-fe121826d8478418af17', 'source_path': 'sources/gdca-recon/2026-11-14/files/fe121826d8478418af17_597.pdf', 'source_sha256': 'f89d82d5c32d3039cb5f0bf438a57f03c4cc27d3f4b1054a7fdf8974aa02900e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/597.pdf', 'bytes': 1881605}, {'source_id': 'GDCA-F-641b96d7feb5cca666c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/641b96d7feb5cca666c1_1372.pdf', 'source_sha256': 'aa94b5ea4879feb33a1357e029c04c05285f60a070dc833286228a2b5a6faf4f', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1372.pdf', 'bytes': 851149}, {'source_id': 'GDCA-F-948350d2a0ca37352a0d', 'source_path': 'sources/gdca-recon/2026-11-14/files/948350d2a0ca37352a0d_768.pdf', 'source_sha256': '32603554bb7deffb035cbb47c012f2856480f6f4d7da7794c65fd2e0e8307cac', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/768.pdf', 'bytes': 1965815}, {'source_id': 'GDCA-F-cdadc5a3f5edd0a99ab5', 'source_path': 'sources/gdca-recon/2026-11-14/files/cdadc5a3f5edd0a99ab5_1267.pdf', 'source_sha256': 'c74c33061eb01d35b5f1d441b904ce96fcb5a38ce358f42380d89d226855481c', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1267.pdf', 'bytes': 1178193}, {'source_id': 'GDCA-F-e41d232cc51da1659fcc', 'source_path': 'sources/gdca-recon/2026-11-14/files/e41d232cc51da1659fcc_1185.pdf', 'source_sha256': '6151ce105e664c51c92bc5b09276c3999f6f5c8b9b7b09bfb80cc8a850d9eb3f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1185.pdf', 'bytes': 1792759}, {'source_id': 'GDCA-F-047cbbfd8dccfba2973d', 'source_path': 'sources/gdca-recon/2026-11-14/files/047cbbfd8dccfba2973d_615.pdf', 'source_sha256': 'f8b503effe687e18bb453f5296abc5b9f61c144b621aa40dd7fecedf7aa5c1e9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/615.pdf', 'bytes': 1589500}, {'source_id': 'GDCA-F-e59716745ab70d85f2ce', 'source_path': 'sources/gdca-recon/2026-11-14/files/e59716745ab70d85f2ce_1556.pdf', 'source_sha256': '5eb1fe8b84b85a42ef12052f2c570fd557655b3d1b60f5ae3d45a73acd36e8be', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1556.pdf', 'bytes': 2801054}, {'source_id': 'GDCA-F-8249769c13a68110c1e0', 'source_path': 'sources/gdca-recon/2026-11-14/files/8249769c13a68110c1e0_66.pdf', 'source_sha256': 'cff75327677e7e5fe1fbec4389920150da562733e00a5208e853986b67e2c93c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/66.pdf', 'bytes': 2158536}, {'source_id': 'GDCA-F-b42b1b6f3bbb2da93c09', 'source_path': 'sources/gdca-recon/2026-11-14/files/b42b1b6f3bbb2da93c09_1252.pdf', 'source_sha256': 'd9a2fb103c653cbdd57ab15a96241c26ee9587049909cfaed468dc8533888171', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1252.pdf', 'bytes': 1866353}, {'source_id': 'GDCA-F-734146f08d9701d33fdf', 'source_path': 'sources/gdca-recon/2026-11-14/files/734146f08d9701d33fdf_287.pdf', 'source_sha256': '6862dfdf1bc7110a2bd9b3cdcf67ded71927fb3f15404f039ce75c09e4df7293', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/287.pdf', 'bytes': 1946525}, {'source_id': 'GDCA-F-2e0b3e893dd4e8c640c9', 'source_path': 'sources/gdca-recon/2026-11-14/files/2e0b3e893dd4e8c640c9_482.pdf', 'source_sha256': 'd2ac52ca6a5c3d38dae02077f879ba08d5dfca85cfe506f3e032f1cce065e6af', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/482.pdf', 'bytes': 1813052}, {'source_id': 'GDCA-F-1f9bf59b0831fcebf013', 'source_path': 'sources/gdca-recon/2026-11-14/files/1f9bf59b0831fcebf013_293.pdf', 'source_sha256': '1b8499860ed31f8a0171a36889d8fd193cd87e5b67a24a2a0d3f5f2bbf7ce6dc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/293.pdf', 'bytes': 1848017}, {'source_id': 'GDCA-F-c84f9d6ff7394f5e05be', 'source_path': 'sources/gdca-recon/2026-11-14/files/c84f9d6ff7394f5e05be_223.pdf', 'source_sha256': '83b7812822e5bbcf6cb6d075f3152ef75578665da40692aa88b4cdd4e7e2f6d9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/223.pdf', 'bytes': 1906231}, {'source_id': 'GDCA-F-424980694d056a4f20ea', 'source_path': 'sources/gdca-recon/2026-11-14/files/424980694d056a4f20ea_1207.pdf', 'source_sha256': '71e582ff503f09ca909e62f881987f8d316af6766635f013dd90e4a88b6f908b', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1207.pdf', 'bytes': 1464906}, {'source_id': 'GDCA-F-86f575de41e9bab94d30', 'source_path': 'sources/gdca-recon/2026-11-14/files/86f575de41e9bab94d30_246.pdf', 'source_sha256': '7b12b9266e192157c7f531fe97829ea5d6c975f2caacac63fe9b364b61653116', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/246.pdf', 'bytes': 1846175}, {'source_id': 'GDCA-F-21cfdbc11e861da1c781', 'source_path': 'sources/gdca-recon/2026-11-14/files/21cfdbc11e861da1c781_291.pdf', 'source_sha256': 'acec78b2cb3a921bc2e0296821072000139241392d464cfd23c95518aacfe062', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/291.pdf', 'bytes': 1950130}, {'source_id': 'GDCA-F-dcfa73cf9854cc0f7baa', 'source_path': 'sources/gdca-recon/2026-11-14/files/dcfa73cf9854cc0f7baa_1265.pdf', 'source_sha256': '81f524cb8b700aafdc133df66393b99630d75d753949a00e8d421431759ec9cc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1265.pdf', 'bytes': 1870994}, {'source_id': 'GDCA-F-7c83c8b252eb314019e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/7c83c8b252eb314019e5_630.pdf', 'source_sha256': '4180bf3164c13ac31bafe2fb18f91f6afa5e939f1ed01da33a438b5aecd2ca02', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/630.pdf', 'bytes': 1912908}, {'source_id': 'GDCA-F-262d08c4bb569876414b', 'source_path': 'sources/gdca-recon/2026-11-14/files/262d08c4bb569876414b_316.pdf', 'source_sha256': 'a64e281e8b91d949a4930f916b90721b530a6ab526a0ae3d5ebca4439493ee6a', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/316.pdf', 'bytes': 4578018}, {'source_id': 'GDCA-F-ccb3da41296f2ca58a85', 'source_path': 'sources/gdca-recon/2026-11-14/files/ccb3da41296f2ca58a85_771.pdf', 'source_sha256': '86cd0d1a6a7e281b202d81a901eb7f3154d9f1c35d908cc3a89103b460d373c0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/771.pdf', 'bytes': 1807703}, {'source_id': 'GDCA-F-02d5a5e67bece295d490', 'source_path': 'sources/gdca-recon/2026-11-14/files/02d5a5e67bece295d490_946.pdf', 'source_sha256': '2ecd238d42b47499b7d17e513e6ed005deb4f9b85da0f64daf6935b2169a61bc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/946.pdf', 'bytes': 2154326}, {'source_id': 'GDCA-F-44f93262d7443a4175c4', 'source_path': 'sources/gdca-recon/2026-11-14/files/44f93262d7443a4175c4_1071.pdf', 'source_sha256': '03d1cd3ca233145df0ea3c86ab8b628ae1f3843fa90a571f7e9c7ece886ba157', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1071.pdf', 'bytes': 1921731}, {'source_id': 'GDCA-F-35d758c1f68bf81f6088', 'source_path': 'sources/gdca-recon/2026-11-14/files/35d758c1f68bf81f6088_1389.pdf', 'source_sha256': '02bc73c4dd55b505040a4ca37cdc187dbb0a563cc3b473d225863c30fefc1000', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1389.pdf', 'bytes': 1839592}, {'source_id': 'GDCA-F-6141a31df35035fc286d', 'source_path': 'sources/gdca-recon/2026-11-14/files/6141a31df35035fc286d_1004.pdf', 'source_sha256': '84a96952310699e416996c3c36cf2419d4496f6ff91aca835b81a129cbdafc42', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1004.pdf', 'bytes': 1521496}, {'source_id': 'GDCA-F-9f8f6ffba7cc660f5539', 'source_path': 'sources/gdca-recon/2026-11-14/files/9f8f6ffba7cc660f5539_448.pdf', 'source_sha256': '4b09223151e0bfdc53b89d08ae72c80d190bb6bce6fb01c276ecc8e67375d2fc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/448.pdf', 'bytes': 1824287}, {'source_id': 'GDCA-F-ac4843cdc87879233ace', 'source_path': 'sources/gdca-recon/2026-11-14/files/ac4843cdc87879233ace_490.pdf', 'source_sha256': '4b8bb7e3d547209e494112cbc0ba4cb2ee0932e62de520652c196d984a7dac59', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/490.pdf', 'bytes': 2120363}, {'source_id': 'GDCA-F-ce028d37256ea9875fc2', 'source_path': 'sources/gdca-recon/2026-11-14/files/ce028d37256ea9875fc2_730.pdf', 'source_sha256': 'b7e29de7bcd1930dbf70168de2486ca0ab8bca911c9f01ff308b222c027a2657', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/730.pdf', 'bytes': 2135345}, {'source_id': 'GDCA-F-8d0076390a7c000dfeed', 'source_path': 'sources/gdca-recon/2026-11-14/files/8d0076390a7c000dfeed_864.pdf', 'source_sha256': '295c8dc6e77138c96227ba7162a9083822acb77d89e0bd211cbc12a8de589aaa', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/864.pdf', 'bytes': 2926747}, {'source_id': 'GDCA-F-083807e37b85013fd1a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/083807e37b85013fd1a9_369.pdf', 'source_sha256': '902720de5a860f7f6616d5e0b62b38e836186fd897aea1a3045252aee106ec00', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/369.pdf', 'bytes': 2078137}, {'source_id': 'GDCA-F-ddcd5eaf9d9d1c74ccf0', 'source_path': 'sources/gdca-recon/2026-11-14/files/ddcd5eaf9d9d1c74ccf0_545.pdf', 'source_sha256': 'b4166d76db39de45eac757504970a1fcc8f3acee331d8f96f6b1b7667355249a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/545.pdf', 'bytes': 1668139}, {'source_id': 'GDCA-F-6f08c7fccbea9bb9ec76', 'source_path': 'sources/gdca-recon/2026-11-14/files/6f08c7fccbea9bb9ec76_716.pdf', 'source_sha256': '501d4456e908db73bc4b2ae3d92a11c47990e2d574f100ca137ed19eaa144501', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/716.pdf', 'bytes': 1782336}, {'source_id': 'GDCA-F-66be224cfad1b8604ee0', 'source_path': 'sources/gdca-recon/2026-11-14/files/66be224cfad1b8604ee0_1031.pdf', 'source_sha256': 'b43a9ae839a1f090ad7062a16a34f7bd0367cbab790ffde5545476eff0c1c460', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1031.pdf', 'bytes': 2035334}, {'source_id': 'GDCA-F-172ff2f50bf7cbc3e3f1', 'source_path': 'sources/gdca-recon/2026-11-14/files/172ff2f50bf7cbc3e3f1_803.pdf', 'source_sha256': '209dee1035d8e59db4dd39ea6f0c34de5a97df3952604f7b470810f1102db290', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/803.pdf', 'bytes': 2067438}, {'source_id': 'GDCA-F-0181ac0fa3f4415304cd', 'source_path': 'sources/gdca-recon/2026-11-14/files/0181ac0fa3f4415304cd_577.pdf', 'source_sha256': 'ddca8c26bfefa08f6d17d3f1c5da2f71fac43fb2b074f5091cb76769713f12a0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/577.pdf', 'bytes': 1872083}, {'source_id': 'GDCA-F-a845f9ec0082704f65df', 'source_path': 'sources/gdca-recon/2026-11-14/files/a845f9ec0082704f65df_162.pdf', 'source_sha256': 'e331a77c1970ea25d75aca9b52fbb3412d6431335ddba2a6f50cef22c6dbbf64', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/162.pdf', 'bytes': 2214938}, {'source_id': 'GDCA-F-0102546859b318a3fae6', 'source_path': 'sources/gdca-recon/2026-11-14/files/0102546859b318a3fae6_905.pdf', 'source_sha256': '255b446834f684caa29801914201a21d1af50a1d16a3c6bf341198975f064306', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/905.pdf', 'bytes': 1986288}, {'source_id': 'GDCA-F-403a87f719c7315be022', 'source_path': 'sources/gdca-recon/2026-11-14/files/403a87f719c7315be022_442.pdf', 'source_sha256': '2cafe89c0bfa1af4e98c107755e944fa329a4147a8a141e23e492186515d1199', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/442.pdf', 'bytes': 2538533}, {'source_id': 'GDCA-F-4e821f0ef9b23cc66981', 'source_path': 'sources/gdca-recon/2026-11-14/files/4e821f0ef9b23cc66981_956.pdf', 'source_sha256': '383afd6a8048ed4a5694b51a5a6367fd3b511bc621549d4fcc5ada01e807d984', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/956.pdf', 'bytes': 2155962}, {'source_id': 'GDCA-F-6ad4330dcd431e917377', 'source_path': 'sources/gdca-recon/2026-11-14/files/6ad4330dcd431e917377_1586.pdf', 'source_sha256': '6d0a4d35dee66f06be52b8e65b75188703698989461767caa5f10c0ba29f5713', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1586.pdf', 'bytes': 1966652}, {'source_id': 'GDCA-F-b40fba9e25b661defb45', 'source_path': 'sources/gdca-recon/2026-11-14/files/b40fba9e25b661defb45_1572.pdf', 'source_sha256': '2c79e2adec2ce8c741c4b1b29022560ba1f9b75ac685b7df735edd8f9f8b1638', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1572.pdf', 'bytes': 3308838}, {'source_id': 'GDCA-F-982a55acaa4e8a5b34c3', 'source_path': 'sources/gdca-recon/2026-11-14/files/982a55acaa4e8a5b34c3_1171.pdf', 'source_sha256': 'e629db1287e1aa66fb6250ca16afcccd5ee4ae09c5ce4121ca91670ba984a23c', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1171.pdf', 'bytes': 1103525}, {'source_id': 'GDCA-F-5102856e7a07e89e5868', 'source_path': 'sources/gdca-recon/2026-11-14/files/5102856e7a07e89e5868_1011.pdf', 'source_sha256': 'f325917d2e4d75c8d153f8fcc58a660671d043afd3c44a0b7cb2491447a1f3d4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1011.pdf', 'bytes': 2299206}, {'source_id': 'GDCA-F-9228e5359d7b771dcf8f', 'source_path': 'sources/gdca-recon/2026-11-14/files/9228e5359d7b771dcf8f_1066.pdf', 'source_sha256': 'de5b61b1716f7de9efa00986335f59f0c47c9a2d07d5a19c905f43e65d096a38', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1066.pdf', 'bytes': 2167138}, {'source_id': 'GDCA-F-b4f2b15ed13c524b2ca5', 'source_path': 'sources/gdca-recon/2026-11-14/files/b4f2b15ed13c524b2ca5_261.pdf', 'source_sha256': 'f9021952984c6abb024652053393bf54fba7549771f8532654a9def9537b17c5', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/261.pdf', 'bytes': 1598320}, {'source_id': 'GDCA-F-91e5f73f25ec476ebada', 'source_path': 'sources/gdca-recon/2026-11-14/files/91e5f73f25ec476ebada_307.pdf', 'source_sha256': '6be4d79b90bf85cd901e17800c96b164bf6e4ce337722d1993266bc9be5da90f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/307.pdf', 'bytes': 2253874}, {'source_id': 'GDCA-F-1cb89e6d826d145e125b', 'source_path': 'sources/gdca-recon/2026-11-14/files/1cb89e6d826d145e125b_648.pdf', 'source_sha256': '18b8be0bc3f29053203a8bd7332816cc627294e9901bc67e35fbd63bc0b15cf8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/648.pdf', 'bytes': 1834053}, {'source_id': 'GDCA-F-ecf79ef0fb0e12a73d71', 'source_path': 'sources/gdca-recon/2026-11-14/files/ecf79ef0fb0e12a73d71_1555.pdf', 'source_sha256': 'fd44824711e1346ceace01687fb4e3cfae50dc67f3cf2f7dd8e09b74e7f63829', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1555.pdf', 'bytes': 3557028}, {'source_id': 'GDCA-F-dd18b3d22e4977075ae4', 'source_path': 'sources/gdca-recon/2026-11-14/files/dd18b3d22e4977075ae4_45.pdf', 'source_sha256': '2af251b403eb8dfe1094ec213c02ef6a52af706d5625f9498293d148d5981592', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/45.pdf', 'bytes': 106490}, {'source_id': 'GDCA-F-03d1cfa6db36663ce0be', 'source_path': 'sources/gdca-recon/2026-11-14/files/03d1cfa6db36663ce0be_337.pdf', 'source_sha256': '49cd8e6d5842f7ffb858b0ccbb54e208284ea3d69ef0ff490dcec057126cb6f3', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/337.pdf', 'bytes': 1407703}, {'source_id': 'GDCA-F-1063a093f01cc664f475', 'source_path': 'sources/gdca-recon/2026-11-14/files/1063a093f01cc664f475_461.pdf', 'source_sha256': 'a162eba5dbe30582101bc4cc896f2bda2bc07c568f901a73d13ae80c1bee03ce', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/461.pdf', 'bytes': 1750309}, {'source_id': 'GDCA-F-b3f18a03422e78c21dc4', 'source_path': 'sources/gdca-recon/2026-11-14/files/b3f18a03422e78c21dc4_645.pdf', 'source_sha256': '898121d982b392828510f61ce2fa855b6d0898769f044af3befd0b9116297644', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/645.pdf', 'bytes': 1773273}, {'source_id': 'GDCA-F-337e7b2300f92b307c2c', 'source_path': 'sources/gdca-recon/2026-11-14/files/337e7b2300f92b307c2c_1030.pdf', 'source_sha256': '136a18b9626c4aef219b5a40413686bdc0bcffb7d434ac8a2fa715cf48ca15f3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1030.pdf', 'bytes': 2058978}, {'source_id': 'GDCA-F-db5326523071a77cd4e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/db5326523071a77cd4e1_1495.pdf', 'source_sha256': '885109ab1553102343f78917b6cdd04d0313ae2a2176cbba97af60d4a15b75ee', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1495.pdf', 'bytes': 2362186}, {'source_id': 'GDCA-F-f945a75aaae981d5904e', 'source_path': 'sources/gdca-recon/2026-11-14/files/f945a75aaae981d5904e_150.pdf', 'source_sha256': '243708a0250b16bec8da7ab68a08e4a9fd5fe96914c5f96172fa5ee1006adc60', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/150.pdf', 'bytes': 1923230}, {'source_id': 'GDCA-F-24dc64c55aade1b307a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/24dc64c55aade1b307a9_772.pdf', 'source_sha256': '91b37e94e3bbc566bf54fb1b0a952470a4e2c163cc76d818b95164e34694cef8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/772.pdf', 'bytes': 1850400}, {'source_id': 'GDCA-F-2dc0a75a6a1c085a0e5a', 'source_path': 'sources/gdca-recon/2026-11-14/files/2dc0a75a6a1c085a0e5a_776.pdf', 'source_sha256': '30e7ae9b293c45c1d710e7b966ba1fd80a647a8aa2034a515642881125ef26e6', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/776.pdf', 'bytes': 1099736}, {'source_id': 'GDCA-F-dd2890b636122af5849c', 'source_path': 'sources/gdca-recon/2026-11-14/files/dd2890b636122af5849c_687.pdf', 'source_sha256': '19bbf227d34d3e06cf37f73d8a77ae47409b6ce5593942f66743f2b5670877f5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/687.pdf', 'bytes': 1861777}, {'source_id': 'GDCA-F-2c2ea08dea065aa1e957', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c2ea08dea065aa1e957_620.pdf', 'source_sha256': 'd9c368c9a0884c670c5910709a4d9305cfa4945b084fec8dfcc106fd2027291b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/620.pdf', 'bytes': 1851104}, {'source_id': 'GDCA-F-c0dd3e50b04e5c133fcc', 'source_path': 'sources/gdca-recon/2026-11-14/files/c0dd3e50b04e5c133fcc_890.pdf', 'source_sha256': 'f3f723cf79f0c4c50fcb244fe8d2f2a0f5900aed6e32a8a6ed4e0b3acc9d85a4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/890.pdf', 'bytes': 1908025}, {'source_id': 'GDCA-F-5212719a88714ab8a5b6', 'source_path': 'sources/gdca-recon/2026-11-14/files/5212719a88714ab8a5b6_789.pdf', 'source_sha256': 'ca47b7e283f5d033429c2f65a496d10dc6fc43324981e933bab82ccc9576dd43', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/789.pdf', 'bytes': 2087833}, {'source_id': 'GDCA-F-2d7cdb725e3749dc7703', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d7cdb725e3749dc7703_940.pdf', 'source_sha256': 'c2768db48cb8e8456f516d14137943259357fc53c56f56a92d6212b1a6471e10', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/940.pdf', 'bytes': 2315921}, {'source_id': 'GDCA-F-d0aa047d13f2dd832d29', 'source_path': 'sources/gdca-recon/2026-11-14/files/d0aa047d13f2dd832d29_1029.pdf', 'source_sha256': 'cbbe136a2b8cc04371cee4cb6b93dd0649776e883e8f5f368ba29e69c418d357', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1029.pdf', 'bytes': 1932188}, {'source_id': 'GDCA-F-8638f5691ef9a00ef02d', 'source_path': 'sources/gdca-recon/2026-11-14/files/8638f5691ef9a00ef02d_554.pdf', 'source_sha256': '101e70d5fca070f8dd50532b0c5eb0a462abad235c108a5b74f5957fce256069', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/554.pdf', 'bytes': 1324715}, {'source_id': 'GDCA-F-de68509afd0c4826b227', 'source_path': 'sources/gdca-recon/2026-11-14/files/de68509afd0c4826b227_590.pdf', 'source_sha256': 'bbb3c8966e8c560606354e75264819e3a449bea42dac8ad48c873d722821c465', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/590.pdf', 'bytes': 2097205}, {'source_id': 'GDCA-F-c59cbf28d61a2d0be06f', 'source_path': 'sources/gdca-recon/2026-11-14/files/c59cbf28d61a2d0be06f_322.pdf', 'source_sha256': '4aa16724f950167c23721ddb6620c3fb06d0065a935d5540507d9f000a6db951', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/322.pdf', 'bytes': 1967937}, {'source_id': 'GDCA-F-4877304b0680817594bf', 'source_path': 'sources/gdca-recon/2026-11-14/files/4877304b0680817594bf_1369.pdf', 'source_sha256': '0876b7a61f9f47fbbce1dee29b1dd7202a6fe5db3e68240d1d7d7f0577394ba2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1369.pdf', 'bytes': 1539796}, {'source_id': 'GDCA-F-7da7857845a70690efc0', 'source_path': 'sources/gdca-recon/2026-11-14/files/7da7857845a70690efc0_970.pdf', 'source_sha256': 'df4aaea4d7c64967ad8a78df48f2914d11a7cc9ef5e79b826beef917300d9715', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/970.pdf', 'bytes': 2227232}, {'source_id': 'GDCA-F-17d7b73ce6e4c554e5b9', 'source_path': 'sources/gdca-recon/2026-11-14/files/17d7b73ce6e4c554e5b9_1256.pdf', 'source_sha256': '13655a7eeed6e5791570a245d73712f0b32cf4c9d133136b446e332ee3f7d1e9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1256.pdf', 'bytes': 1877256}, {'source_id': 'GDCA-F-5d41b24e798d98febeb1', 'source_path': 'sources/gdca-recon/2026-11-14/files/5d41b24e798d98febeb1_1474.pdf', 'source_sha256': '0142bb4dc7bd7abc7a6febc944373839cd7705bc45f218ff6ac9b61466f6e2ff', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1474.pdf', 'bytes': 2317471}, {'source_id': 'GDCA-F-d1a79f7e548ad806e20e', 'source_path': 'sources/gdca-recon/2026-11-14/files/d1a79f7e548ad806e20e_558.pdf', 'source_sha256': '07c8e2087a160aa87404312fb306df9662ec3bd41f30011a7e79801c94da1f65', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/558.pdf', 'bytes': 1828580}, {'source_id': 'GDCA-F-d33e2e67cb35d54c2a89', 'source_path': 'sources/gdca-recon/2026-11-14/files/d33e2e67cb35d54c2a89_650.pdf', 'source_sha256': '49ed70c53ec5348b5cfe0a266c8535dae266c77d00cec3078b57e4b610d4c1d2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/650.pdf', 'bytes': 1935034}, {'source_id': 'GDCA-F-7fc347de2eb53c77f637', 'source_path': 'sources/gdca-recon/2026-11-14/files/7fc347de2eb53c77f637_1187.pdf', 'source_sha256': '3e71141b80a6055de8bd39607dfea766eb9574a6c09dc3e9e9b270d35cdef1e9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1187.pdf', 'bytes': 2023229}, {'source_id': 'GDCA-F-124995016a2083d06520', 'source_path': 'sources/gdca-recon/2026-11-14/files/124995016a2083d06520_1148.pdf', 'source_sha256': 'ecc40df1d0eac55eda7c8fb545deb0e96ff9ed482f4bda37bae9d84437506937', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1148.pdf', 'bytes': 1706473}, {'source_id': 'GDCA-F-fe91b2b75ab01f2a7597', 'source_path': 'sources/gdca-recon/2026-11-14/files/fe91b2b75ab01f2a7597_1387.pdf', 'source_sha256': 'a0ca96a24b280bfc98a2f06dcf13eb56218a9b60ba6f61323d2f4cf4f8b520fa', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1387.pdf', 'bytes': 1776681}, {'source_id': 'GDCA-F-3354fcd1ee4ca2314552', 'source_path': 'sources/gdca-recon/2026-11-14/files/3354fcd1ee4ca2314552_77.pdf', 'source_sha256': '496326225afd3365eef2076ad3c640a270d2e6134be0dd37957f0d293116de3f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/77.pdf', 'bytes': 2148155}, {'source_id': 'GDCA-F-130c415e13a1411c6b5c', 'source_path': 'sources/gdca-recon/2026-11-14/files/130c415e13a1411c6b5c_303.pdf', 'source_sha256': '208faa4630809098fcd2b1a5a3fbdd40ed4003a910ba4eadb38f4cff916e91a8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/303.pdf', 'bytes': 1735246}, {'source_id': 'GDCA-F-2be94d62e8b0fbfad8be', 'source_path': 'sources/gdca-recon/2026-11-14/files/2be94d62e8b0fbfad8be_720.pdf', 'source_sha256': '24e33920de87e735db62e91d5e625ebcce3f41549b0b512e60bc27816b663543', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/720.pdf', 'bytes': 2237319}, {'source_id': 'GDCA-F-1a8d5de224652c7e418a', 'source_path': 'sources/gdca-recon/2026-11-14/files/1a8d5de224652c7e418a_255.pdf', 'source_sha256': 'f1d081d96e57d347e85a4b521ce9c193d0c2fb3df9e43ea5fa88fe7d96f0b688', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/255.pdf', 'bytes': 2310094}, {'source_id': 'GDCA-F-80ecc1b5da8b287aad2b', 'source_path': 'sources/gdca-recon/2026-11-14/files/80ecc1b5da8b287aad2b_221.pdf', 'source_sha256': '5bf208b7cefdb065abe57801f8f5af1e60e00e5da3d5f3a4ff38868ca15741ee', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/221.pdf', 'bytes': 2054173}, {'source_id': 'GDCA-F-603dcb7f8f7e8e348e6d', 'source_path': 'sources/gdca-recon/2026-11-14/files/603dcb7f8f7e8e348e6d_906.pdf', 'source_sha256': '6551f75e02c9b62571ae4611fb00b163f1f1af478e4bb1aaacbafce17769977d', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/906.pdf', 'bytes': 2823426}, {'source_id': 'GDCA-F-655566a7f7e99e1f4f45', 'source_path': 'sources/gdca-recon/2026-11-14/files/655566a7f7e99e1f4f45_182.pdf', 'source_sha256': 'a12672fad2fb163954b4a09d34d15712679c7242859a2777454ef72b1554dec5', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/182.pdf', 'bytes': 2413436}, {'source_id': 'GDCA-F-1e1d4ca39da47effa4c7', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e1d4ca39da47effa4c7_625.pdf', 'source_sha256': 'abe442385d64697d7ef11f06839172d6b59f5e9a303e54d03f042dff0c5b87ae', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/625.pdf', 'bytes': 1805102}, {'source_id': 'GDCA-F-1a0ef585cd8320f8ad8f', 'source_path': 'sources/gdca-recon/2026-11-14/files/1a0ef585cd8320f8ad8f_1160.pdf', 'source_sha256': '0e31c958cd84585367f8115089d665d7bc9ce9a61dbe74817a51e58b921f8f81', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1160.pdf', 'bytes': 1963835}, {'source_id': 'GDCA-F-796e19a31b4cfd1b1b64', 'source_path': 'sources/gdca-recon/2026-11-14/files/796e19a31b4cfd1b1b64_102.pdf', 'source_sha256': '16a8587dcd7de80ec6d3bbbaaf1a70b81bd368efc26e350cd858278d7ec80243', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/102.pdf', 'bytes': 2183489}, {'source_id': 'GDCA-F-94e9f33b0dbd2bdc1871', 'source_path': 'sources/gdca-recon/2026-11-14/files/94e9f33b0dbd2bdc1871_1285.pdf', 'source_sha256': '23c3bd8af2dbb95061ea8ac948feda374ec9e9b4d4329e92bfb0f8c489233b10', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1285.pdf', 'bytes': 2840664}, {'source_id': 'GDCA-F-88d6299e68ffa6d85813', 'source_path': 'sources/gdca-recon/2026-11-14/files/88d6299e68ffa6d85813_124.pdf', 'source_sha256': '2473c498b8af216e2ae47f1ccdef80a5b6979419d9564dfce0aa7c7f70f4a388', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/124.pdf', 'bytes': 2123521}, {'source_id': 'GDCA-F-77563ba1381fbfdd1470', 'source_path': 'sources/gdca-recon/2026-11-14/files/77563ba1381fbfdd1470_1244.pdf', 'source_sha256': '3ab73b4beb6a9fbece3248ae3aed60878bd76b4be0a13e27f5eec87bc35d7f81', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1244.pdf', 'bytes': 1827546}, {'source_id': 'GDCA-F-7bbd4ca2ddd3322a7ac2', 'source_path': 'sources/gdca-recon/2026-11-14/files/7bbd4ca2ddd3322a7ac2_1405.pdf', 'source_sha256': '24ab94bb4852f6427c94a018d8fa7a4d376c3c813d65ba346f62ef64e90f0ecc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1405.pdf', 'bytes': 1722957}, {'source_id': 'GDCA-F-f00cd3bae6af101ad624', 'source_path': 'sources/gdca-recon/2026-11-14/files/f00cd3bae6af101ad624_40.pdf', 'source_sha256': 'ca68e908c6ad7ce89f1289d68e71210273ef18764896d444462250095ea5a808', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/40.pdf', 'bytes': 125941}, {'source_id': 'GDCA-F-47bfc47af41dacaaf776', 'source_path': 'sources/gdca-recon/2026-11-14/files/47bfc47af41dacaaf776_1491.pdf', 'source_sha256': 'c1c694cc854c4a6d378b40b102af3e8ed15d57d8effc0ff47b70f62ea689ea1d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1491.pdf', 'bytes': 2148023}, {'source_id': 'GDCA-F-a0704e8b97bf61b2828c', 'source_path': 'sources/gdca-recon/2026-11-14/files/a0704e8b97bf61b2828c_181.pdf', 'source_sha256': '0d1f8b8e1159a67ab81087cbe665a4c6fd842c9ab42b1d4f3183bd0abb4f8c8c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/181.pdf', 'bytes': 2061218}, {'source_id': 'GDCA-F-155f36ae63ac83a182f2', 'source_path': 'sources/gdca-recon/2026-11-14/files/155f36ae63ac83a182f2_403.pdf', 'source_sha256': '6a3d0c7804806807fd49a57a348c6653607c3d386e1e67e1ed21bf9eb38431c4', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/403.pdf', 'bytes': 1363883}, {'source_id': 'GDCA-F-057485022452bd0b9c91', 'source_path': 'sources/gdca-recon/2026-11-14/files/057485022452bd0b9c91_1291.pdf', 'source_sha256': 'cab66f08468a1e179054576593618e3209674fea98270a0816ae5f5fd2ab0076', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1291.pdf', 'bytes': 2497288}, {'source_id': 'GDCA-F-64dcc1a77d9472c15570', 'source_path': 'sources/gdca-recon/2026-11-14/files/64dcc1a77d9472c15570_627.pdf', 'source_sha256': '85479f34d3d0fc823162b1c41527ce428bb732ae6e728641b243e6e536d767b7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/627.pdf', 'bytes': 2333116}, {'source_id': 'GDCA-F-dd4758db63a88a887aea', 'source_path': 'sources/gdca-recon/2026-11-14/files/dd4758db63a88a887aea_600.pdf', 'source_sha256': '11bb740789bdd658ec73deb228880380ebaf0f0cae9c7d81e2249188c7220279', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/600.pdf', 'bytes': 2154998}, {'source_id': 'GDCA-F-ee4466491e5582c1d716', 'source_path': 'sources/gdca-recon/2026-11-14/files/ee4466491e5582c1d716_236.pdf', 'source_sha256': '25996c0b11689353e45f3e862a6d5930f9c243c79e62d06bbfefe70e5ea71106', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/236.pdf', 'bytes': 1921561}, {'source_id': 'GDCA-F-6340947fc82b349ad1d0', 'source_path': 'sources/gdca-recon/2026-11-14/files/6340947fc82b349ad1d0_633.pdf', 'source_sha256': '1b294ba7237f29b1065d784e4c7f34463ed6ccf0f17ef1b8ab8eb1b5d7d2906b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/633.pdf', 'bytes': 1782434}, {'source_id': 'GDCA-F-3ba01d7063fbb63335b2', 'source_path': 'sources/gdca-recon/2026-11-14/files/3ba01d7063fbb63335b2_117.pdf', 'source_sha256': '78975a8c2898abb3dbf9a88dad538b67fd73d4e9facf24e6494a8894aa63a433', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/117.pdf', 'bytes': 1902184}, {'source_id': 'GDCA-F-848d9145cd91e63ebbb0', 'source_path': 'sources/gdca-recon/2026-11-14/files/848d9145cd91e63ebbb0_549.pdf', 'source_sha256': '70e4ebc90a05b83ada4d4637c13f1c4db166a09d9967820f295c99e68ca28baf', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/549.pdf', 'bytes': 1988259}, {'source_id': 'GDCA-F-53661ad47d16d734fdf6', 'source_path': 'sources/gdca-recon/2026-11-14/files/53661ad47d16d734fdf6_151.pdf', 'source_sha256': '6670859dafaa5c31984bb5f09a116cce54ba03ef5b99724f448b6e0bc22a636b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/151.pdf', 'bytes': 2019409}, {'source_id': 'GDCA-F-bc974c98a88a733873d5', 'source_path': 'sources/gdca-recon/2026-11-14/files/bc974c98a88a733873d5_517.pdf', 'source_sha256': 'd85d33d6a075c4ae0c19309b445cdad8939a6900ef49a235ed8a5878df9982e5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/517.pdf', 'bytes': 1878018}, {'source_id': 'GDCA-F-4fc8e13e437b6496baa1', 'source_path': 'sources/gdca-recon/2026-11-14/files/4fc8e13e437b6496baa1_1439.pdf', 'source_sha256': 'b4b8f9c1738e7432ffc5fceeeacf828cfe18281587fdb52ef9db83c647e79cee', 'document_pages': 1, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1439.pdf', 'bytes': 619948}, {'source_id': 'GDCA-F-d711410e5a7c6e9f28aa', 'source_path': 'sources/gdca-recon/2026-11-14/files/d711410e5a7c6e9f28aa_1115.pdf', 'source_sha256': '331099f36c2990f683d874eb43193af2a03cf90eaa94f276ac601dbe5e5cb9ec', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1115.pdf', 'bytes': 2123888}, {'source_id': 'GDCA-F-5125501509666314ee23', 'source_path': 'sources/gdca-recon/2026-11-14/files/5125501509666314ee23_663.pdf', 'source_sha256': '8f9cf94285ab4e17cb326539e26830e94fd1c3a33b9fb750e55950ed2220f18f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/663.pdf', 'bytes': 1980933}, {'source_id': 'GDCA-F-863dcaa392e165d57192', 'source_path': 'sources/gdca-recon/2026-11-14/files/863dcaa392e165d57192_169.pdf', 'source_sha256': '4ced1c2de7863c7e636febd806bf5536f55118daf7f9f1c33bdd00772f96e72d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/169.pdf', 'bytes': 1957367}, {'source_id': 'GDCA-F-e1ec1b1eb734694688ae', 'source_path': 'sources/gdca-recon/2026-11-14/files/e1ec1b1eb734694688ae_1246.pdf', 'source_sha256': '6ee9597d13f1f8d468b6bcc4fac054caa4c440d44566eb2c648f59b75c47d00e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1246.pdf', 'bytes': 1976489}, {'source_id': 'GDCA-F-915a86d7411aaa3de1e8', 'source_path': 'sources/gdca-recon/2026-11-14/files/915a86d7411aaa3de1e8_747.pdf', 'source_sha256': 'b54d29b406325706a65fbede98a05e87214f31d04cf8d0ea636f63c3326fe580', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/747.pdf', 'bytes': 1797202}, {'source_id': 'GDCA-F-d14f82c1bf63c99a951c', 'source_path': 'sources/gdca-recon/2026-11-14/files/d14f82c1bf63c99a951c_821.pdf', 'source_sha256': '59c6480624e8315357c40e3b579613e893a479bbfe947d1538ac3d802f0fbf76', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/821.pdf', 'bytes': 1779737}, {'source_id': 'GDCA-F-72f346945b55311fb425', 'source_path': 'sources/gdca-recon/2026-11-14/files/72f346945b55311fb425_200.pdf', 'source_sha256': 'a4c87abbf0b489bc258160084cdd91b763d9c335b8d5ab2bcd9b2d2e08fb47ee', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/200.pdf', 'bytes': 2166671}, {'source_id': 'GDCA-F-06190968869eb893983e', 'source_path': 'sources/gdca-recon/2026-11-14/files/06190968869eb893983e_75.pdf', 'source_sha256': 'ba94cc3f586f838bcdf91b64ab5242d4b5a156bf4a11724e6bb162062ae925be', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/75.pdf', 'bytes': 2252776}, {'source_id': 'GDCA-F-2ea53cd42075d55e6407', 'source_path': 'sources/gdca-recon/2026-11-14/files/2ea53cd42075d55e6407_1446.pdf', 'source_sha256': 'a8ec5eed716d32dc7a7856870467ee41602aaedfabc7fe145efbb2a6d3078556', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1446.pdf', 'bytes': 2339986}, {'source_id': 'GDCA-F-d3f9cc2a0082a8c65a79', 'source_path': 'sources/gdca-recon/2026-11-14/files/d3f9cc2a0082a8c65a79_878.pdf', 'source_sha256': '74b21e4f6fe2d5893305a02006f64f9dac0b71b899affdaeec72d157841c332f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/878.pdf', 'bytes': 1920256}, {'source_id': 'GDCA-F-2e9e21588d1b9fd5c431', 'source_path': 'sources/gdca-recon/2026-11-14/files/2e9e21588d1b9fd5c431_1218.pdf', 'source_sha256': '97f9dd515c7854fea00844e3169d251ec76d86f3d4168bd0656b321e74a28efc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1218.pdf', 'bytes': 2200664}, {'source_id': 'GDCA-F-796bcab0441086f5b17e', 'source_path': 'sources/gdca-recon/2026-11-14/files/796bcab0441086f5b17e_249.pdf', 'source_sha256': 'c1196ff65dc9deee3489540a669c6c655ba2642d76571b74c060dea622eed3f8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/249.pdf', 'bytes': 1847439}, {'source_id': 'GDCA-F-4d0b898c888418c8b3c3', 'source_path': 'sources/gdca-recon/2026-11-14/files/4d0b898c888418c8b3c3_398.pdf', 'source_sha256': '8f2235dedf07c208b01610a8668bd095a350b1376585eec7654ac5747ab43a74', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/398.pdf', 'bytes': 1912342}, {'source_id': 'GDCA-F-582b527d5116d17d25cc', 'source_path': 'sources/gdca-recon/2026-11-14/files/582b527d5116d17d25cc_1063.pdf', 'source_sha256': '3252c11a29e5da77ddc85aeabcb7c9594c163d5d3aa6d87462fd9a7b8b26a70d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1063.pdf', 'bytes': 2444930}, {'source_id': 'GDCA-F-f66a8b105e1c7e0928eb', 'source_path': 'sources/gdca-recon/2026-11-14/files/f66a8b105e1c7e0928eb_1469.pdf', 'source_sha256': 'ee7f72020762c25d8589e7c3c5fdcfa6904a872dd26c4afca77e8456e7264634', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1469.pdf', 'bytes': 2372638}, {'source_id': 'GDCA-F-f693a8d31e2ff32c76d8', 'source_path': 'sources/gdca-recon/2026-11-14/files/f693a8d31e2ff32c76d8_1305.pdf', 'source_sha256': 'bf824f2448cd96b059aa9ef47e7e94d94aee2d4d148407ec30d5591a18c8df77', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1305.pdf', 'bytes': 1312091}, {'source_id': 'GDCA-F-377faaa3a9d0581f8299', 'source_path': 'sources/gdca-recon/2026-11-14/files/377faaa3a9d0581f8299_1198.pdf', 'source_sha256': '7e4d5c16c0c928f9acf168e0546f7e15af36d74123e7584ea5733134031faeff', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1198.pdf', 'bytes': 2012229}, {'source_id': 'GDCA-F-03bf1d70782a667da6e4', 'source_path': 'sources/gdca-recon/2026-11-14/files/03bf1d70782a667da6e4_96.pdf', 'source_sha256': 'e251b41f551c6e393cd729dc35d76847195dad68ce66ed41ded4a99a2d81ca28', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/96.pdf', 'bytes': 1773744}, {'source_id': 'GDCA-F-bdbb2ac2c931aff2eec9', 'source_path': 'sources/gdca-recon/2026-11-14/files/bdbb2ac2c931aff2eec9_1243.pdf', 'source_sha256': 'c2292de7803feca715cb4f56144d36dbabc7b69d17339a2cfd83961b300262dc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1243.pdf', 'bytes': 1881526}, {'source_id': 'GDCA-F-e1e58aea00efbe08cc5b', 'source_path': 'sources/gdca-recon/2026-11-14/files/e1e58aea00efbe08cc5b_918.pdf', 'source_sha256': 'b324b02d4ea62234d31c1e82880869aa7d4d6e679dfaa2ee0c06d1023ad1d7f9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/918.pdf', 'bytes': 1734563}, {'source_id': 'GDCA-F-a174350c282bad082896', 'source_path': 'sources/gdca-recon/2026-11-14/files/a174350c282bad082896_527.pdf', 'source_sha256': '29675631452a7088f9bcb684b73fce12cd68c7b1250aa99a24f1e04ba49b1fcb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/527.pdf', 'bytes': 1863094}, {'source_id': 'GDCA-F-3fcd2805c405e9e242fb', 'source_path': 'sources/gdca-recon/2026-11-14/files/3fcd2805c405e9e242fb_1404.pdf', 'source_sha256': '9bb0f1d0a25a3b1114f3c73be55822397beaef77f8a6f6ace997e48e70270ec3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1404.pdf', 'bytes': 2441239}, {'source_id': 'GDCA-F-cb5b9e4ab466869e6ee3', 'source_path': 'sources/gdca-recon/2026-11-14/files/cb5b9e4ab466869e6ee3_926.pdf', 'source_sha256': '9a7bd178af4119a16fa1b9f1d08b5bab61b8ce027ef2707372f5799f9f37dbc3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/926.pdf', 'bytes': 2051818}, {'source_id': 'GDCA-F-592c54d8d0d97acb6929', 'source_path': 'sources/gdca-recon/2026-11-14/files/592c54d8d0d97acb6929_785.pdf', 'source_sha256': 'f5666482d8fc396fff8b0724a6bc2cb0272a93a77f5a2cf2512d9cd5d8d86aa1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/785.pdf', 'bytes': 1842034}, {'source_id': 'GDCA-F-808a58631d9940796ead', 'source_path': 'sources/gdca-recon/2026-11-14/files/808a58631d9940796ead_110.pdf', 'source_sha256': '555dceed5efee03afafa74dd4558e6d1e64e4067ea50f2eba42b7c8001c6cb47', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/110.pdf', 'bytes': 2256531}, {'source_id': 'GDCA-F-3d6f00473198ec2181ff', 'source_path': 'sources/gdca-recon/2026-11-14/files/3d6f00473198ec2181ff_886.pdf', 'source_sha256': '5a909661d00add985d5449de045618daf11442ee13c562efcb1bc76db9c883fa', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/886.pdf', 'bytes': 1457630}, {'source_id': 'GDCA-F-34cf8748bff8eac641e9', 'source_path': 'sources/gdca-recon/2026-11-14/files/34cf8748bff8eac641e9_526.pdf', 'source_sha256': '45436c29862984b3b6c4d88afbe27a966eed75d872bb0235339c6bea71d6a701', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/526.pdf', 'bytes': 2034422}, {'source_id': 'GDCA-F-37d991bf2314fb5d5dac', 'source_path': 'sources/gdca-recon/2026-11-14/files/37d991bf2314fb5d5dac_424.pdf', 'source_sha256': 'debdb294c72b4657512fe37c04f1053dd7fc35c9710e0d7abaa1df710fc28e07', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/424.pdf', 'bytes': 6812014}, {'source_id': 'GDCA-F-81397232a09f187d0499', 'source_path': 'sources/gdca-recon/2026-11-14/files/81397232a09f187d0499_368.pdf', 'source_sha256': 'ce409da034ac7b74e3351b1256bb36b2d5af36689712789ec7097872724cd475', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/368.pdf', 'bytes': 2022718}, {'source_id': 'GDCA-F-b71de3ec21b56589b46a', 'source_path': 'sources/gdca-recon/2026-11-14/files/b71de3ec21b56589b46a_481.pdf', 'source_sha256': '6aa2201d0d6802c699c7dafb9bde48e81b982e206feddcda08a2a2116148904c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/481.pdf', 'bytes': 1801608}, {'source_id': 'GDCA-F-d2891e6a1aeca985c5ca', 'source_path': 'sources/gdca-recon/2026-11-14/files/d2891e6a1aeca985c5ca_1181.pdf', 'source_sha256': '0feac957af6bbbf382cc5c718761f79b1a84284323276a5a75beba123f8fbdc3', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1181.pdf', 'bytes': 1249809}, {'source_id': 'GDCA-F-6b987515f2aedb3b90b9', 'source_path': 'sources/gdca-recon/2026-11-14/files/6b987515f2aedb3b90b9_1242.pdf', 'source_sha256': '49976ae5231238e877ad19206f697da60427b33945321b02948615360c905c93', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1242.pdf', 'bytes': 1657737}, {'source_id': 'GDCA-F-b614910a6a167463d999', 'source_path': 'sources/gdca-recon/2026-11-14/files/b614910a6a167463d999_42.pdf', 'source_sha256': '4d73ed81857b634c2d9df4a57657e8cfc526ac41353336ed3c6ca9313f0f764c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/42.pdf', 'bytes': 86567}, {'source_id': 'GDCA-F-01755feab33db4248504', 'source_path': 'sources/gdca-recon/2026-11-14/files/01755feab33db4248504_1060.pdf', 'source_sha256': '1f623d537f3188474e115956ebb64165e0f4e1f81cc9cab61c3cf9a4d2a07010', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1060.pdf', 'bytes': 2340926}, {'source_id': 'GDCA-F-7ee54b44230b1022f495', 'source_path': 'sources/gdca-recon/2026-11-14/files/7ee54b44230b1022f495_858.pdf', 'source_sha256': '1c8147735b53e05d46b75300dc886d941e06a408675d67d371463cd25d6fe203', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/858.pdf', 'bytes': 2249564}, {'source_id': 'GDCA-F-eddce350380fa7fafce9', 'source_path': 'sources/gdca-recon/2026-11-14/files/eddce350380fa7fafce9_1573.pdf', 'source_sha256': '339a5ecde0f0a993fe08edbb579a91a5d9a403f1c1a41f9f5c342936b114d992', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1573.pdf', 'bytes': 3014156}, {'source_id': 'GDCA-F-7565f0d571e2424fbf30', 'source_path': 'sources/gdca-recon/2026-11-14/files/7565f0d571e2424fbf30_939.pdf', 'source_sha256': 'cb4586475030292797ad812aaa463b40417561443bd1a414a85c74f82064ba63', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/939.pdf', 'bytes': 1993379}, {'source_id': 'GDCA-F-99d6b6f06425ce340547', 'source_path': 'sources/gdca-recon/2026-11-14/files/99d6b6f06425ce340547_1208.pdf', 'source_sha256': '4162f2f418f415efb53287732a6f474f9e99ae77e9d664932238eeead0fa203d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1208.pdf', 'bytes': 1625264}, {'source_id': 'GDCA-F-51f8f6d532398d9c79e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/51f8f6d532398d9c79e1_893.pdf', 'source_sha256': '274fe2c5b164d72821f53b7a0be0721740fb0a0592891ddb1793eae71f8c07e0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/893.pdf', 'bytes': 2048650}, {'source_id': 'GDCA-F-c5220ab8bde985a8b229', 'source_path': 'sources/gdca-recon/2026-11-14/files/c5220ab8bde985a8b229_1009.pdf', 'source_sha256': '66f823c94837ad436171dbdfa173a601d9b1b2dd5f782a337d4d40f73f453343', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1009.pdf', 'bytes': 2139197}, {'source_id': 'GDCA-F-ace8e0bcfc148e558ce4', 'source_path': 'sources/gdca-recon/2026-11-14/files/ace8e0bcfc148e558ce4_677.pdf', 'source_sha256': '13fbc2c3ad899b941bdbe0fbb51f44b0148c3364bbe7618c074cbd36aa8328f5', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/677.pdf', 'bytes': 1986810}, {'source_id': 'GDCA-F-bfc325f3fa891cd11e88', 'source_path': 'sources/gdca-recon/2026-11-14/files/bfc325f3fa891cd11e88_1410.pdf', 'source_sha256': 'ba452132038f0182766f87550b0bdc1dc5339b6a64a22c135c80cfdff48e0f03', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1410.pdf', 'bytes': 2052643}, {'source_id': 'GDCA-F-eb5c38ed5af6fe38dd33', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb5c38ed5af6fe38dd33_908.pdf', 'source_sha256': '817a26eb64f0b2662d0e27adf63c890b40571df6995080b886502569df48326b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/908.pdf', 'bytes': 1629694}, {'source_id': 'GDCA-F-07e4e924f1112170bf59', 'source_path': 'sources/gdca-recon/2026-11-14/files/07e4e924f1112170bf59_594.pdf', 'source_sha256': 'e4440af7b3ee39bca71b573a3802b0b5f5aa0676214f406c15ba9f0d91575698', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/594.pdf', 'bytes': 1966763}, {'source_id': 'GDCA-F-29f12e3f9a5b14f916a5', 'source_path': 'sources/gdca-recon/2026-11-14/files/29f12e3f9a5b14f916a5_20.pdf', 'source_sha256': 'ef5824e8c584e71de9986dda3eee4067ba331c5be45f47d97ddace028a31f71a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/20.pdf', 'bytes': 89334}, {'source_id': 'GDCA-F-168f6156c895ec273b0a', 'source_path': 'sources/gdca-recon/2026-11-14/files/168f6156c895ec273b0a_1284.pdf', 'source_sha256': '2285f0e8de1e5e2c752eaba3621146fdd5df8831ce6f0b830fd3188f27d083cc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1284.pdf', 'bytes': 2206329}, {'source_id': 'GDCA-F-8c0959d216ec53186f93', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c0959d216ec53186f93_1085.pdf', 'source_sha256': 'b0f32f3155e6bd6950aae77f0d8ee51e73f66d41510f817d4afaf7ccee301a85', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1085.pdf', 'bytes': 1905356}, {'source_id': 'GDCA-F-a0ebd509330d59e6eea2', 'source_path': 'sources/gdca-recon/2026-11-14/files/a0ebd509330d59e6eea2_1081.pdf', 'source_sha256': 'ecead21f6ed4ac4ebe402f04ad00c728111bb875cf4996f24656629db079ded1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1081.pdf', 'bytes': 2013573}, {'source_id': 'GDCA-F-345f63f382aa727c51fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/345f63f382aa727c51fd_868.pdf', 'source_sha256': 'f8580391a0dc5657c03de120c30705ec5b48fee5e7afdfa30ab289bcddcf888f', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/868.pdf', 'bytes': 3035883}, {'source_id': 'GDCA-F-9cbe49a332dc4775dba2', 'source_path': 'sources/gdca-recon/2026-11-14/files/9cbe49a332dc4775dba2_628.pdf', 'source_sha256': 'b27d01a05dd0645dc550b94fe4926cb7aca994851a4abc30d075d8ba7006e0da', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/628.pdf', 'bytes': 2320163}, {'source_id': 'GDCA-F-57084f5d6cf43a092b12', 'source_path': 'sources/gdca-recon/2026-11-14/files/57084f5d6cf43a092b12_135.pdf', 'source_sha256': 'd15b618bd005ef8e293c30ed3050290e1da40b505142cebcd13b55c98d9ae331', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/135.pdf', 'bytes': 2068942}, {'source_id': 'GDCA-F-1ee74a863eb2720ca40e', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ee74a863eb2720ca40e_152.pdf', 'source_sha256': '13ba0f69aed7f326da035a45a5a61a646616c919c35bc451eb3ef579a58d3eac', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/152.pdf', 'bytes': 1948247}, {'source_id': 'GDCA-F-bf0bb0a8f7e750876557', 'source_path': 'sources/gdca-recon/2026-11-14/files/bf0bb0a8f7e750876557_681.pdf', 'source_sha256': 'd7d7af4c08386e148c4ad0b88c11bd982407c642202d74f7ec9c48b011fe1f1e', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/681.pdf', 'bytes': 2421831}, {'source_id': 'GDCA-F-99d4a84bf6536ca00365', 'source_path': 'sources/gdca-recon/2026-11-14/files/99d4a84bf6536ca00365_1.pdf', 'source_sha256': 'e3182e94abe2dfcff859c1ed10a683295152c6ab11b871d40320f2ecca461501', 'document_pages': 9, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1.pdf', 'bytes': 3091750}, {'source_id': 'GDCA-F-f72e57fcef244622625d', 'source_path': 'sources/gdca-recon/2026-11-14/files/f72e57fcef244622625d_897.pdf', 'source_sha256': 'ffcba1fb48948dc1db86ac38619dbc2e075d4420044566b39d2cb96037ebd8dc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/897.pdf', 'bytes': 2093433}, {'source_id': 'GDCA-F-72b41f882c6e2ad7c28b', 'source_path': 'sources/gdca-recon/2026-11-14/files/72b41f882c6e2ad7c28b_753.pdf', 'source_sha256': '3d947e5b7ca9ed5e949c61f135c2780526b0c4b7d9b3462ebe4b6c3a29980e5b', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/753.pdf', 'bytes': 3040675}, {'source_id': 'GDCA-F-580b131aae89e2948a27', 'source_path': 'sources/gdca-recon/2026-11-14/files/580b131aae89e2948a27_870.pdf', 'source_sha256': 'c2668a328b4c94b99f7c359001ec778f8e320af7cf7046f7fd3aa18b33c8d366', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/870.pdf', 'bytes': 2112850}, {'source_id': 'GDCA-F-eefaf69245869a02013e', 'source_path': 'sources/gdca-recon/2026-11-14/files/eefaf69245869a02013e_819.pdf', 'source_sha256': '43484afeb48c1351b1f2621a8e165ace3ad59d598be5c259a2df8c7f7ee32018', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/819.pdf', 'bytes': 2508018}, {'source_id': 'GDCA-F-3aa7205078cdd7044209', 'source_path': 'sources/gdca-recon/2026-11-14/files/3aa7205078cdd7044209_867.pdf', 'source_sha256': '93ee7e1c06233781c480fb89a0e1f887f428db3a63c5c02a802a7084a0174db1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/867.pdf', 'bytes': 2091047}, {'source_id': 'GDCA-F-f57161efcbac3e0fd337', 'source_path': 'sources/gdca-recon/2026-11-14/files/f57161efcbac3e0fd337_894.pdf', 'source_sha256': 'b58814cfcd3170994d80ad5e9a84b63a6c7ad2559e989a598a3e30ebbd065b72', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/894.pdf', 'bytes': 1969935}, {'source_id': 'GDCA-F-99d5f8aa5861250490d1', 'source_path': 'sources/gdca-recon/2026-11-14/files/99d5f8aa5861250490d1_228.pdf', 'source_sha256': 'f753d71568a51274920f275403dc4d4043e130d640bf8b6ecc2f1101e25f542a', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/228.pdf', 'bytes': 1637401}, {'source_id': 'GDCA-F-34fe9437b34009db9371', 'source_path': 'sources/gdca-recon/2026-11-14/files/34fe9437b34009db9371_65.pdf', 'source_sha256': 'ac5820ffd87361f2ded439a5bedcb58c92627ba244d9c2fb00488c3a199bbf45', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/65.pdf', 'bytes': 1892378}, {'source_id': 'GDCA-F-a20c26fb8f746c4ccabc', 'source_path': 'sources/gdca-recon/2026-11-14/files/a20c26fb8f746c4ccabc_1154.pdf', 'source_sha256': '6f82c4c8fd568519efa553ff1668d6b7f7b44f8935ffefc748d2423a8267330d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1154.pdf', 'bytes': 2002724}, {'source_id': 'GDCA-F-5463a06b3c6af20caf0a', 'source_path': 'sources/gdca-recon/2026-11-14/files/5463a06b3c6af20caf0a_1424.pdf', 'source_sha256': '702d08720b6138d16cf9b1b28d85555a1826fc9b17f8aeb0f606f020963b707d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1424.pdf', 'bytes': 2196293}, {'source_id': 'GDCA-F-a9f8a9873f125e799c68', 'source_path': 'sources/gdca-recon/2026-11-14/files/a9f8a9873f125e799c68_818.pdf', 'source_sha256': '7967967c0db33c9f24f09c3b2bebfe728294ee2edb079d238bd7833ee0a4cbd4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/818.pdf', 'bytes': 1667249}, {'source_id': 'GDCA-F-ac906ea0794bfdc49142', 'source_path': 'sources/gdca-recon/2026-11-14/files/ac906ea0794bfdc49142_7.pdf', 'source_sha256': '7fc2fdc828baa1add659ea439ea91360e72c96b32f463c333e8ccfb6e6682bcc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/7.pdf', 'bytes': 2139652}, {'source_id': 'GDCA-F-b3b621acbb258fdfd9d5', 'source_path': 'sources/gdca-recon/2026-11-14/files/b3b621acbb258fdfd9d5_1258.pdf', 'source_sha256': 'f3061e69a3b83c3609118d2b9201b0bf3e30ffd31baef74cfdbb5af77ee6480b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1258.pdf', 'bytes': 1907440}, {'source_id': 'GDCA-F-938add67c38af2ee8d50', 'source_path': 'sources/gdca-recon/2026-11-14/files/938add67c38af2ee8d50_23.pdf', 'source_sha256': '5269fe438009ef50002cb3f0abed61875ccf8bf03d66628d5789301cff4c7c7e', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/23.pdf', 'bytes': 120508}, {'source_id': 'GDCA-F-b08ab6add4cd8f5756fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/b08ab6add4cd8f5756fd_312.pdf', 'source_sha256': '81756dfe8ae5041a165ed83b81b492b4676ed43e25b44b933d79042cd0b3a819', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/312.pdf', 'bytes': 2054650}, {'source_id': 'GDCA-F-735b2ea51f65c457b331', 'source_path': 'sources/gdca-recon/2026-11-14/files/735b2ea51f65c457b331_26.pdf', 'source_sha256': 'db098eab49250a45fce5813f4701986ee5211683fdc9fcc169544de0eb08d714', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/26.pdf', 'bytes': 87658}, {'source_id': 'GDCA-F-db896b26577221b01fdc', 'source_path': 'sources/gdca-recon/2026-11-14/files/db896b26577221b01fdc_300.pdf', 'source_sha256': 'f1889e89b24fe33db0c57e85636f532e91081728bb8bd1b94f1b788deb775943', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/300.pdf', 'bytes': 2042414}, {'source_id': 'GDCA-F-2c2d99202c431d1fb55c', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c2d99202c431d1fb55c_754.pdf', 'source_sha256': 'a08eafeb36cd34d3c6a71afd4be59f6a082fac1cfe481691036be53b9d596bfa', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/754.pdf', 'bytes': 1723553}, {'source_id': 'GDCA-F-bfa1bf9679b924c7f9ba', 'source_path': 'sources/gdca-recon/2026-11-14/files/bfa1bf9679b924c7f9ba_578.pdf', 'source_sha256': 'a32c6e367eb1611f551edb13d40c2e6cb6424481c30825e2885e37b5ba0f425a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/578.pdf', 'bytes': 1701981}, {'source_id': 'GDCA-F-7bf3c40ba2c5af11778b', 'source_path': 'sources/gdca-recon/2026-11-14/files/7bf3c40ba2c5af11778b_485.pdf', 'source_sha256': '544c73bdadcaf82ce3c333dcc326c253bca10ff6b014349f29c0fd709e863a34', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/485.pdf', 'bytes': 1736455}, {'source_id': 'GDCA-F-899d94df6e47620c0043', 'source_path': 'sources/gdca-recon/2026-11-14/files/899d94df6e47620c0043_451.pdf', 'source_sha256': 'bff72449dbfb6e066377609de00fc319c685d9072da31cedf91dcfe2d99a1d1a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/451.pdf', 'bytes': 2208973}, {'source_id': 'GDCA-F-f8528108252e080ff34d', 'source_path': 'sources/gdca-recon/2026-11-14/files/f8528108252e080ff34d_395.pdf', 'source_sha256': '935cad4d3e9dc889a2bd8daf4161da286e8f1310b37acf5a47e0cfc46d0b4b15', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/395.pdf', 'bytes': 2203177}, {'source_id': 'GDCA-F-eb0a230278316cbc239f', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb0a230278316cbc239f_289.pdf', 'source_sha256': '448f61903fcd20ce3a0ec859bb4961c3a4a271369707d9d50bbf68b1681a9051', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/289.pdf', 'bytes': 1864584}, {'source_id': 'GDCA-F-bdf656a98029b7b4bc33', 'source_path': 'sources/gdca-recon/2026-11-14/files/bdf656a98029b7b4bc33_416.pdf', 'source_sha256': '5685cf0db3b506bf570231dbe630f30f5e48a2187eeed4724a17fe63c9bce4b1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/416.pdf', 'bytes': 2109305}, {'source_id': 'GDCA-F-1a7ba0419fc195c4c56c', 'source_path': 'sources/gdca-recon/2026-11-14/files/1a7ba0419fc195c4c56c_1238.pdf', 'source_sha256': '485a98b21b6b76afee4ee93ff8075dc5a8564f9f2030d521b0800d6407ad344c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1238.pdf', 'bytes': 2073749}, {'source_id': 'GDCA-F-462e9e551a118c253b14', 'source_path': 'sources/gdca-recon/2026-11-14/files/462e9e551a118c253b14_1016.pdf', 'source_sha256': '950dd8ea57107da6ab3c6f4992140c0bcf501700cbe8b37481b3feada686a7e4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1016.pdf', 'bytes': 1833753}, {'source_id': 'GDCA-F-2acdaf16b2c9876fcbc6', 'source_path': 'sources/gdca-recon/2026-11-14/files/2acdaf16b2c9876fcbc6_48.pdf', 'source_sha256': '4bb6c002222b6f6ce449253ddde915e21d8fe818cd95da315220c5af46061fd4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/48.pdf', 'bytes': 98904}, {'source_id': 'GDCA-F-9c2938caf7c62209c573', 'source_path': 'sources/gdca-recon/2026-11-14/files/9c2938caf7c62209c573_746.pdf', 'source_sha256': 'da7c4724242220747b095b4e5a151dc58f4ef74dfdfc1322e1c8daf8b4f48cbf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/746.pdf', 'bytes': 1867848}, {'source_id': 'GDCA-F-22bb26091847f6b925e9', 'source_path': 'sources/gdca-recon/2026-11-14/files/22bb26091847f6b925e9_786.pdf', 'source_sha256': '4a4efe69e578f58646b148ea3d72be70fbd4eac53541a0556c72d323a8a9cf31', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/786.pdf', 'bytes': 1909245}, {'source_id': 'GDCA-F-065266b40a76df70a337', 'source_path': 'sources/gdca-recon/2026-11-14/files/065266b40a76df70a337_1089.pdf', 'source_sha256': '7d7e5914e91d58071774227b3897843d17917d3f48a828ce116752ef1dff1c6c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1089.pdf', 'bytes': 1822386}, {'source_id': 'GDCA-F-d6bc4ec5a9661d0a80a4', 'source_path': 'sources/gdca-recon/2026-11-14/files/d6bc4ec5a9661d0a80a4_1317.pdf', 'source_sha256': 'b4a8fd45e0c52631230b1d4fc67370152f46ae1f312ef73eb7d25e16d8d2b266', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1317.pdf', 'bytes': 1803838}, {'source_id': 'GDCA-F-9cfa3ad6da76e179d1c6', 'source_path': 'sources/gdca-recon/2026-11-14/files/9cfa3ad6da76e179d1c6_25.pdf', 'source_sha256': 'e072bd828c977019a591561713715f3a1047426e85bc1b77a866a469ca277163', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/25.pdf', 'bytes': 129029}, {'source_id': 'GDCA-F-a16c2baa6b260e25f858', 'source_path': 'sources/gdca-recon/2026-11-14/files/a16c2baa6b260e25f858_566.pdf', 'source_sha256': '35dd25bc34114ec0696ef904025ac995fc70aa9c12eeb11369a63a0e32f1a43d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/566.pdf', 'bytes': 2292482}, {'source_id': 'GDCA-F-9ba23cbb5718576cc0c0', 'source_path': 'sources/gdca-recon/2026-11-14/files/9ba23cbb5718576cc0c0_584.pdf', 'source_sha256': '83cb4709b9b5cc1bcb36c61fc121a1326789f8551ae437084dd864a8c7d0f824', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/584.pdf', 'bytes': 1755799}, {'source_id': 'GDCA-F-856ce5fab025b8cf6809', 'source_path': 'sources/gdca-recon/2026-11-14/files/856ce5fab025b8cf6809_441.pdf', 'source_sha256': 'abf529bdcd2661ece7d5f7982f5dbf196625371d0488b9a06f7cc12f6074ef2b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/441.pdf', 'bytes': 2067301}, {'source_id': 'GDCA-F-85bb0fb183d057a03033', 'source_path': 'sources/gdca-recon/2026-11-14/files/85bb0fb183d057a03033_105.pdf', 'source_sha256': '429adb6f6269438ccc5605abdc31bdf230522e610b40f1f7a10f9e40127fcc25', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/105.pdf', 'bytes': 2927768}, {'source_id': 'GDCA-F-5811dcae9de7e3d621c5', 'source_path': 'sources/gdca-recon/2026-11-14/files/5811dcae9de7e3d621c5_1093.pdf', 'source_sha256': 'cb3550159fda3863c674f88f035c26ffb462410725577ea489535f02c89bca2b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1093.pdf', 'bytes': 1899692}, {'source_id': 'GDCA-F-8a29f03e15f4354d4e57', 'source_path': 'sources/gdca-recon/2026-11-14/files/8a29f03e15f4354d4e57_639.pdf', 'source_sha256': '02471e6681920da2780c9c4da2e42700b42e8d7d2d01db0b0c2c2534ee72079e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/639.pdf', 'bytes': 1882811}, {'source_id': 'GDCA-F-30026af03ad1d376102e', 'source_path': 'sources/gdca-recon/2026-11-14/files/30026af03ad1d376102e_993.pdf', 'source_sha256': 'a4e341ccd05209ad76841a2ca6da72708e00af0fb230be42219b9f82bc9a7082', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/993.pdf', 'bytes': 2458677}, {'source_id': 'GDCA-F-15446e906ec26a5bf97a', 'source_path': 'sources/gdca-recon/2026-11-14/files/15446e906ec26a5bf97a_748.pdf', 'source_sha256': '67b4c47ac1268bd9a631e6da2955d5afd00954c7138b6bb6ce7def24d6dadad9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/748.pdf', 'bytes': 1980051}, {'source_id': 'GDCA-F-9770ed503b092b637b90', 'source_path': 'sources/gdca-recon/2026-11-14/files/9770ed503b092b637b90_423.pdf', 'source_sha256': '5eb96d01867d08376b9bb4cfa58ef022b74b16b80fe0ee07acafb20b8d4095de', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/423.pdf', 'bytes': 5708722}, {'source_id': 'GDCA-F-3963bd621139a2ed7fbf', 'source_path': 'sources/gdca-recon/2026-11-14/files/3963bd621139a2ed7fbf_458.pdf', 'source_sha256': '197bcc19beae694677237323004ab21b6c3c5882f85440a080e845f5a288dd01', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/458.pdf', 'bytes': 1692490}, {'source_id': 'GDCA-F-13ff59597c39622e8671', 'source_path': 'sources/gdca-recon/2026-11-14/files/13ff59597c39622e8671_1138.pdf', 'source_sha256': '9ea077972527837e625d3ccc8451715ed7eafac08f3d2865abc76f93a11d5405', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1138.pdf', 'bytes': 2682291}, {'source_id': 'GDCA-F-2c73ab3b4b213bf86caf', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c73ab3b4b213bf86caf_1220.pdf', 'source_sha256': 'cc5f220acd4723a5b098cd5d277e0fbbf6938089ccf54b14d8b2fe38d3d50ca8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1220.pdf', 'bytes': 1878822}, {'source_id': 'GDCA-F-2f2d828c3602666f57fa', 'source_path': 'sources/gdca-recon/2026-11-14/files/2f2d828c3602666f57fa_437.pdf', 'source_sha256': 'b34735561559aeb441288814f8e48cdba321a4dfbf99a3b0cf4ecb85bd81dfb7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/437.pdf', 'bytes': 2027501}, {'source_id': 'GDCA-F-59beff7fffc5d0f35c43', 'source_path': 'sources/gdca-recon/2026-11-14/files/59beff7fffc5d0f35c43_1452.pdf', 'source_sha256': '1662e5bcf1a2fe3b631fbf93e438ab827c96b806d629b0b4632445ad617ae5fd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1452.pdf', 'bytes': 2431952}, {'source_id': 'GDCA-F-0fd4c7cda132be6e414c', 'source_path': 'sources/gdca-recon/2026-11-14/files/0fd4c7cda132be6e414c_704.pdf', 'source_sha256': '06164ffdf932ab3a3845793023a946985609b5ae1d0b48c8611534a252965ccc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/704.pdf', 'bytes': 2447199}, {'source_id': 'GDCA-F-94ec3d52bcd17e945782', 'source_path': 'sources/gdca-recon/2026-11-14/files/94ec3d52bcd17e945782_764.pdf', 'source_sha256': '5060b622da59b50ee5af8708c7fbe07cece348006fb6f5da7a6782c09866b09c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/764.pdf', 'bytes': 2027569}, {'source_id': 'GDCA-F-4f441262111db7c675ca', 'source_path': 'sources/gdca-recon/2026-11-14/files/4f441262111db7c675ca_1414.pdf', 'source_sha256': '2e73980b4a507169419255b35009460916fefb7122008b4d4cc804c6a2a6b13e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1414.pdf', 'bytes': 1880726}, {'source_id': 'GDCA-F-11ec2b7bfb23f663fe7c', 'source_path': 'sources/gdca-recon/2026-11-14/files/11ec2b7bfb23f663fe7c_519.pdf', 'source_sha256': 'e228ab992bb7c78a8af0eb0167537b365223365823364fc2ef809cc6e4e9337d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/519.pdf', 'bytes': 1630220}, {'source_id': 'GDCA-F-56f19dd0a5f9b7120bbd', 'source_path': 'sources/gdca-recon/2026-11-14/files/56f19dd0a5f9b7120bbd_885.pdf', 'source_sha256': '1cffc7ec178f37390bcdfdd802b072b1347b357d40a51d6ea5137180388c4918', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/885.pdf', 'bytes': 2106803}, {'source_id': 'GDCA-F-d0bc80a91ce0eef98079', 'source_path': 'sources/gdca-recon/2026-11-14/files/d0bc80a91ce0eef98079_711.pdf', 'source_sha256': 'b99a2d5890a5ce113ddf1cf2f481fca34be0ff535d65d7c6ba42af260932efd8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/711.pdf', 'bytes': 1807648}, {'source_id': 'GDCA-F-00013bbc9ff8ef8cd333', 'source_path': 'sources/gdca-recon/2026-11-14/files/00013bbc9ff8ef8cd333_164.pdf', 'source_sha256': '296aa397f74acc31b111bb258a5471aba93f766e70db145acc54d103a3a73912', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/164.pdf', 'bytes': 1942765}, {'source_id': 'GDCA-F-2d96df898bb0a115ff31', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d96df898bb0a115ff31_1206.pdf', 'source_sha256': 'c35eca546b7737551608bc5eaa90e1c6f071e97e812ec6569888ffc6c16170d2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1206.pdf', 'bytes': 1915069}, {'source_id': 'GDCA-F-2db4b3b2fb4a8113aa12', 'source_path': 'sources/gdca-recon/2026-11-14/files/2db4b3b2fb4a8113aa12_503.pdf', 'source_sha256': '5369b74a43b4ec7626ea00dfb78dd6fae5a29a04111abdb70689bfdac6f27e7f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/503.pdf', 'bytes': 1766070}, {'source_id': 'GDCA-F-5a99c30d98184516885b', 'source_path': 'sources/gdca-recon/2026-11-14/files/5a99c30d98184516885b_661.pdf', 'source_sha256': '0013447c1f9450cf02f5e8e163dff08bf1c0413858378591a59832f45d8357d7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/661.pdf', 'bytes': 1971001}, {'source_id': 'GDCA-F-b7bd9f90cdfe9b8002a7', 'source_path': 'sources/gdca-recon/2026-11-14/files/b7bd9f90cdfe9b8002a7_726.pdf', 'source_sha256': '8dcb38e948c9e88ed138a3deab3b1f8f062971f3c29ed70b868588ed3624bb40', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/726.pdf', 'bytes': 2057543}, {'source_id': 'GDCA-F-dcf190cd916ae944b753', 'source_path': 'sources/gdca-recon/2026-11-14/files/dcf190cd916ae944b753_1132.pdf', 'source_sha256': '2e0baadc69c5fa56a624bd15d3666993da28e900a33a54a3739104e05d3dc9d6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1132.pdf', 'bytes': 1834836}, {'source_id': 'GDCA-F-6eb15f686a85182de9fe', 'source_path': 'sources/gdca-recon/2026-11-14/files/6eb15f686a85182de9fe_921.pdf', 'source_sha256': '2f0a9f376180d55e6b6851bccc597bd92d76d75fe0a942d3e1be504c0c8069d4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/921.pdf', 'bytes': 1748664}, {'source_id': 'GDCA-F-0164579c6e089d678c21', 'source_path': 'sources/gdca-recon/2026-11-14/files/0164579c6e089d678c21_37.pdf', 'source_sha256': '7f3c2bd4c47c07e50c81dc71686fb53e5bd1ed2bec60ea6d8802648dcd14ff06', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/37.pdf', 'bytes': 112794}, {'source_id': 'GDCA-F-2250f8259bb8dca1e800', 'source_path': 'sources/gdca-recon/2026-11-14/files/2250f8259bb8dca1e800_759.pdf', 'source_sha256': '1622020bcb378b55c04c5a27e55c6df01c50e5f68762167e2177cc0c4bc2276c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/759.pdf', 'bytes': 1915013}, {'source_id': 'GDCA-F-a3d41a8d7e4f6322ec46', 'source_path': 'sources/gdca-recon/2026-11-14/files/a3d41a8d7e4f6322ec46_1324.pdf', 'source_sha256': '6f7b3c8d8df859d5b7eb5b38d19dfe10ac3b0f0f4dabc186adf8e19f18aefa80', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1324.pdf', 'bytes': 2407437}, {'source_id': 'GDCA-F-8c6dc44d8511274edf09', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c6dc44d8511274edf09_414.pdf', 'source_sha256': 'b6db87c5e7a4426d74e24b80cb2f0b6d1b4e9d43f2ab4f8c608fbe55117f42b6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/414.pdf', 'bytes': 2184960}, {'source_id': 'GDCA-F-c9bd58e6c0c65df59d96', 'source_path': 'sources/gdca-recon/2026-11-14/files/c9bd58e6c0c65df59d96_928.pdf', 'source_sha256': '72fce55ba67a3298d33779c642fee1d4a67b9486320733803203d443a3bef7cb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/928.pdf', 'bytes': 1959709}, {'source_id': 'GDCA-F-25a3c98c418f1673beed', 'source_path': 'sources/gdca-recon/2026-11-14/files/25a3c98c418f1673beed_902.pdf', 'source_sha256': '8b6af561907c882ffc0a662ddf77c3ccc2fbccd06277695de2f3de8f450aa1bc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/902.pdf', 'bytes': 1895252}, {'source_id': 'GDCA-F-98d09fcc3d931627d4c8', 'source_path': 'sources/gdca-recon/2026-11-14/files/98d09fcc3d931627d4c8_509.pdf', 'source_sha256': '58675aacb50131d3d6b63c174199af130aa0aff81eeb3640e111fffcb3381c03', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/509.pdf', 'bytes': 2017509}, {'source_id': 'GDCA-F-438ca172c3ab047dd3a0', 'source_path': 'sources/gdca-recon/2026-11-14/files/438ca172c3ab047dd3a0_1477.pdf', 'source_sha256': '9dc374194c7df2cba15352a2a7145ec59572ff8b531b2275f2cdd68ca33fb58c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1477.pdf', 'bytes': 2368517}, {'source_id': 'GDCA-F-e11ea15c90b28ff55acd', 'source_path': 'sources/gdca-recon/2026-11-14/files/e11ea15c90b28ff55acd_290.pdf', 'source_sha256': '1a4f15f498faa3979d29fcc136309f7ff1a943a257b01412cd3b50c61b84a04b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/290.pdf', 'bytes': 1965848}, {'source_id': 'GDCA-F-1584458c464fa43e7415', 'source_path': 'sources/gdca-recon/2026-11-14/files/1584458c464fa43e7415_396.pdf', 'source_sha256': 'cf42f773052b47b4c64620bb1b6e4ef31b800a57419b9ad568e3f7c332806b5a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/396.pdf', 'bytes': 1855062}, {'source_id': 'GDCA-F-5eb9e26f37ca1d69302a', 'source_path': 'sources/gdca-recon/2026-11-14/files/5eb9e26f37ca1d69302a_983.pdf', 'source_sha256': 'fded2bbd6d3770f8885a3c625064ffd27f94bdb8fa1502ebcafc3fee8f11e680', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/983.pdf', 'bytes': 2129143}, {'source_id': 'GDCA-F-41dd2b4fe5f3e0e69f39', 'source_path': 'sources/gdca-recon/2026-11-14/files/41dd2b4fe5f3e0e69f39_262.pdf', 'source_sha256': '3d8b4006a1f3fe69db3fbddd452101a4cb422013f5ec4a94bfeb6938e2e9e632', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/262.pdf', 'bytes': 1825402}, {'source_id': 'GDCA-F-ce19568885d18eafddcd', 'source_path': 'sources/gdca-recon/2026-11-14/files/ce19568885d18eafddcd_1072.pdf', 'source_sha256': '3c219b056c13938734ddceabff7bb1304d6fe88f550d88aa2dab86e6a94cec7d', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1072.pdf', 'bytes': 906994}, {'source_id': 'GDCA-F-eba74a6e876980c62d0f', 'source_path': 'sources/gdca-recon/2026-11-14/files/eba74a6e876980c62d0f_511.pdf', 'source_sha256': 'b8e46814d41e6e1502bb99700c6b7ab835500cbac5cb075d121005b904f20c4f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/511.pdf', 'bytes': 2035688}, {'source_id': 'GDCA-F-565085a0a0ac93c7c7f7', 'source_path': 'sources/gdca-recon/2026-11-14/files/565085a0a0ac93c7c7f7_1086.pdf', 'source_sha256': 'f9e18827147a0eb7bc1e105c7a3d69f6823bbbe557b873529091c7f5faf01c25', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1086.pdf', 'bytes': 2031841}, {'source_id': 'GDCA-F-fa1f03254584e4a7dd17', 'source_path': 'sources/gdca-recon/2026-11-14/files/fa1f03254584e4a7dd17_78.pdf', 'source_sha256': 'a957a81f0d903d722a7bc21980a9a6d2f609dde31e4e603c5219c577c42822d6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/78.pdf', 'bytes': 2095012}, {'source_id': 'GDCA-F-d99623232b2b7120db37', 'source_path': 'sources/gdca-recon/2026-11-14/files/d99623232b2b7120db37_1308.pdf', 'source_sha256': '66c2786e0a76b295cc68c2c9ac512e61aa98c3045dbd9707bee1c646a98bb733', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1308.pdf', 'bytes': 1896392}, {'source_id': 'GDCA-F-cb69658275f363cbeb4b', 'source_path': 'sources/gdca-recon/2026-11-14/files/cb69658275f363cbeb4b_294.pdf', 'source_sha256': '50a80f94d1f84815eeaf822bd5abcd4912b765cc63ea24623691249c07fb38e8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/294.pdf', 'bytes': 2036470}, {'source_id': 'GDCA-F-37c277d3bcfd188d0851', 'source_path': 'sources/gdca-recon/2026-11-14/files/37c277d3bcfd188d0851_1144.pdf', 'source_sha256': 'f85416c85d551c82a15c71062b65b00f716e7ccfce23ceae37120212639ed823', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1144.pdf', 'bytes': 1911357}, {'source_id': 'GDCA-F-7f4aa547d9088ae4f7c1', 'source_path': 'sources/gdca-recon/2026-11-14/files/7f4aa547d9088ae4f7c1_1175.pdf', 'source_sha256': '8c634804b77c8a4975fbd23342c20b6478159369f920775c80aa5590f459ea07', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1175.pdf', 'bytes': 2054747}, {'source_id': 'GDCA-F-423ebbbc11f7af7f2e30', 'source_path': 'sources/gdca-recon/2026-11-14/files/423ebbbc11f7af7f2e30_880.pdf', 'source_sha256': 'ee98d1b56ecccb7e11dcee94cab9ad74fed4a48f270a2efbaf9fbbdaebbf6441', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/880.pdf', 'bytes': 1740178}, {'source_id': 'GDCA-F-1d5af738bbabbda787ac', 'source_path': 'sources/gdca-recon/2026-11-14/files/1d5af738bbabbda787ac_460.pdf', 'source_sha256': '150f6bb2fc2efd5e60a7388cb33f8f47fffe99397de94ba11bf591b0ae4cd28f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/460.pdf', 'bytes': 2150689}, {'source_id': 'GDCA-F-b03ef2e17d0463080d77', 'source_path': 'sources/gdca-recon/2026-11-14/files/b03ef2e17d0463080d77_349.pdf', 'source_sha256': 'c3901b91d5a7b936d6c00c6edd40e72b5facaae9f5c8209020516c96b2a87aeb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/349.pdf', 'bytes': 1926866}, {'source_id': 'GDCA-F-d2bc95813dbbca6bf787', 'source_path': 'sources/gdca-recon/2026-11-14/files/d2bc95813dbbca6bf787_595.pdf', 'source_sha256': 'b0fe54b8b55339957ad8ff5f81b5813759d102e3ba39b9c0bb788e61a8171df8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/595.pdf', 'bytes': 1876242}, {'source_id': 'GDCA-F-32a0a524eef47613841c', 'source_path': 'sources/gdca-recon/2026-11-14/files/32a0a524eef47613841c_1313.pdf', 'source_sha256': 'e4770fbb76577b93326f1e36d9e8a7710ab658b64a2d850d68df78d40f87880d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1313.pdf', 'bytes': 2048964}, {'source_id': 'GDCA-F-9d383201ef8823cbcd57', 'source_path': 'sources/gdca-recon/2026-11-14/files/9d383201ef8823cbcd57_1494.pdf', 'source_sha256': '8b35664129fe8f70dbb29f4976895c876b4066ebfcee9bccee132eeaf7ba109b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1494.pdf', 'bytes': 1991191}, {'source_id': 'GDCA-F-2d5a83c56289a4beeeb7', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d5a83c56289a4beeeb7_180.pdf', 'source_sha256': 'f2e4b240e8866461cf9f042b346c0e26216d0264bc9e0161e8339b36315d4517', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/180.pdf', 'bytes': 1938144}, {'source_id': 'GDCA-F-263fcfde9427849b1e9a', 'source_path': 'sources/gdca-recon/2026-11-14/files/263fcfde9427849b1e9a_156.pdf', 'source_sha256': '7578e396db38ae541eedd34c1793fa027acda7c1ad39552e004748d08e629c19', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/156.pdf', 'bytes': 1951285}, {'source_id': 'GDCA-F-b431c238ccb9ecf81f9a', 'source_path': 'sources/gdca-recon/2026-11-14/files/b431c238ccb9ecf81f9a_1087.pdf', 'source_sha256': '86ae50ff59c952ee1bd0055689da4ba0773d2599529a83b01c825fd80ba26821', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1087.pdf', 'bytes': 1934036}, {'source_id': 'GDCA-F-708775709135471a7acf', 'source_path': 'sources/gdca-recon/2026-11-14/files/708775709135471a7acf_531.pdf', 'source_sha256': 'daaace5b5f45eb68ebb2c1ba5acfa4d6d9e1dffc4f42cef78bf4033a0f559e7d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/531.pdf', 'bytes': 2155202}, {'source_id': 'GDCA-F-fed1b5ec5361f41ff9e4', 'source_path': 'sources/gdca-recon/2026-11-14/files/fed1b5ec5361f41ff9e4_91.pdf', 'source_sha256': 'e7c38efdfa6303ef1cf0568716a337b8a117286fb405ae9e5dcc60e53b7c8486', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/91.pdf', 'bytes': 1918713}, {'source_id': 'GDCA-F-7c8d80c4b86b657d8981', 'source_path': 'sources/gdca-recon/2026-11-14/files/7c8d80c4b86b657d8981_967.pdf', 'source_sha256': '3148ce4b5ab78065ad88af4220ac114ff17c8cff016f2f7cf00e0d098ae796f1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/967.pdf', 'bytes': 1924079}, {'source_id': 'GDCA-F-d1344c8e85acd389bbd8', 'source_path': 'sources/gdca-recon/2026-11-14/files/d1344c8e85acd389bbd8_188.pdf', 'source_sha256': 'e9175d987a63f2e3e673508ffaed13a57d664c5367314ddc97eb54881e725d96', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/188.pdf', 'bytes': 2053598}, {'source_id': 'GDCA-F-03dcdde0d48373fae1ad', 'source_path': 'sources/gdca-recon/2026-11-14/files/03dcdde0d48373fae1ad_1249.pdf', 'source_sha256': '5940b8b27655bb807705ef8e16fe72fe94688f0f3aca481963ccad5733163aa5', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1249.pdf', 'bytes': 1090702}, {'source_id': 'GDCA-F-5dcd3a044a6c5fb76875', 'source_path': 'sources/gdca-recon/2026-11-14/files/5dcd3a044a6c5fb76875_866.pdf', 'source_sha256': 'c236ed8e77aadf4b55de03b3961c49ddd9049f54920279680ec4ae94320572be', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/866.pdf', 'bytes': 2205805}, {'source_id': 'GDCA-F-1dfd32b03fb0b08fff8d', 'source_path': 'sources/gdca-recon/2026-11-14/files/1dfd32b03fb0b08fff8d_352.pdf', 'source_sha256': '4cad5c88c778894e1d74c8bdfedfaaacac5f7e6be1f2757ec5da5a814f19e842', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/352.pdf', 'bytes': 2100063}, {'source_id': 'GDCA-F-8c43af05eb24ddcbcd6e', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c43af05eb24ddcbcd6e_718.pdf', 'source_sha256': '4c0d76421d0a2c91405fdbf7faa9b7560bbf7e95889f24543ccbd7d41a42f674', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/718.pdf', 'bytes': 2189507}, {'source_id': 'GDCA-F-9258ba9b51f8489492de', 'source_path': 'sources/gdca-recon/2026-11-14/files/9258ba9b51f8489492de_675.pdf', 'source_sha256': 'a085b112bbfb27a09898efc5f107cd0439be95e060f6c9619808003d6e05e76f', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/675.pdf', 'bytes': 2854216}, {'source_id': 'GDCA-F-184de255a57f225824ed', 'source_path': 'sources/gdca-recon/2026-11-14/files/184de255a57f225824ed_942.pdf', 'source_sha256': '553fc45c819d9c7652416e30761c97f8a5504940af73df488ae7fdcb17175e33', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/942.pdf', 'bytes': 1831790}, {'source_id': 'GDCA-F-99c716035c2a4faf4ff8', 'source_path': 'sources/gdca-recon/2026-11-14/files/99c716035c2a4faf4ff8_914.pdf', 'source_sha256': '7d5857af793442d958d0f5f1483e76e6331fdb15649bd9a96259b2a185d83f23', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/914.pdf', 'bytes': 1953144}, {'source_id': 'GDCA-F-9c81b3ae9e25c2481d37', 'source_path': 'sources/gdca-recon/2026-11-14/files/9c81b3ae9e25c2481d37_131.pdf', 'source_sha256': 'e975b8a3ba621fc37fd7182224f98141845b0d9d559adaf7c3a33158edbbef4b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/131.pdf', 'bytes': 1948406}, {'source_id': 'GDCA-F-bb8fc5440e93030df5ed', 'source_path': 'sources/gdca-recon/2026-11-14/files/bb8fc5440e93030df5ed_354.pdf', 'source_sha256': '5a7941e686ab31ea69ef3a82e97cd8057d308db328aa7f568cc9562523f8fb0d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/354.pdf', 'bytes': 1965149}, {'source_id': 'GDCA-F-a792eb754ef5947cd7cd', 'source_path': 'sources/gdca-recon/2026-11-14/files/a792eb754ef5947cd7cd_1312.pdf', 'source_sha256': 'f5331adcdb1e8a85406964f320f1230738ae3706cef2ee7e49954ba18c809e8d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1312.pdf', 'bytes': 1782858}, {'source_id': 'GDCA-F-7b4f7a578f37a11307b0', 'source_path': 'sources/gdca-recon/2026-11-14/files/7b4f7a578f37a11307b0_1475.pdf', 'source_sha256': '01994b8092875389e10263430ba0b2dc767c1cfb97a78f4d59ebaeddcb9d93e9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1475.pdf', 'bytes': 2637289}, {'source_id': 'GDCA-F-3afd144d3fef2a24a27a', 'source_path': 'sources/gdca-recon/2026-11-14/files/3afd144d3fef2a24a27a_1176.pdf', 'source_sha256': '7e9693ee24c8e880cf290d4d6e050299f6428682fd1fe0dae22286a450002caf', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1176.pdf', 'bytes': 2053846}, {'source_id': 'GDCA-F-fdb0d4f3a804693224e8', 'source_path': 'sources/gdca-recon/2026-11-14/files/fdb0d4f3a804693224e8_1302.pdf', 'source_sha256': '0ea66153f925722a244a9cc8434033a7088ba352613e5fb11edd7475973e86e9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1302.pdf', 'bytes': 2090415}, {'source_id': 'GDCA-F-01f24d275324e8228f10', 'source_path': 'sources/gdca-recon/2026-11-14/files/01f24d275324e8228f10_1027.pdf', 'source_sha256': 'b27522dd52b9323a28a357e96e771b69eeb8046d21d1e1145a2a717cd3d78452', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1027.pdf', 'bytes': 2169063}, {'source_id': 'GDCA-F-27ddc1dea9ba04900cc5', 'source_path': 'sources/gdca-recon/2026-11-14/files/27ddc1dea9ba04900cc5_1104.pdf', 'source_sha256': '40f8fa66c901c9ed79b72d1fb589e32102ea977e57b3a60d5ec64edd22493c62', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1104.pdf', 'bytes': 2057737}, {'source_id': 'GDCA-F-b2900157cc7bac0a6020', 'source_path': 'sources/gdca-recon/2026-11-14/files/b2900157cc7bac0a6020_438.pdf', 'source_sha256': '9b8c28ae14eab7311dfb7f7b86050c0ba16a0cf223718843e49f150c9ebb5b83', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/438.pdf', 'bytes': 1978276}, {'source_id': 'GDCA-F-c7ac28cf8659c4093dac', 'source_path': 'sources/gdca-recon/2026-11-14/files/c7ac28cf8659c4093dac_574.pdf', 'source_sha256': '174635513e6417e8b7c0024d04afefc9152eb2164d6f36ab087517a9535b9f66', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/574.pdf', 'bytes': 1932158}, {'source_id': 'GDCA-F-c4f4fa114c7d4bd0b1ad', 'source_path': 'sources/gdca-recon/2026-11-14/files/c4f4fa114c7d4bd0b1ad_121.pdf', 'source_sha256': 'bb3bddecf464a5fdfe8206be7b1b496c58fea65457252086bd36273c8641f623', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/121.pdf', 'bytes': 2508023}, {'source_id': 'GDCA-F-a74e24c9d2b190e5fca7', 'source_path': 'sources/gdca-recon/2026-11-14/files/a74e24c9d2b190e5fca7_564.pdf', 'source_sha256': 'adf3c1e6d9a6e66ca36bcdbfd286e8422bd3f794e719f9f906a0bfa2178d46ad', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/564.pdf', 'bytes': 923096}, {'source_id': 'GDCA-F-8dd3b545315c7c0cc82a', 'source_path': 'sources/gdca-recon/2026-11-14/files/8dd3b545315c7c0cc82a_837.pdf', 'source_sha256': 'd5e404929b4ff0cac411dfb03c202f60cef72e75f0518be9757b8ecc9b717c12', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/837.pdf', 'bytes': 1627510}, {'source_id': 'GDCA-F-d035675163646b64a444', 'source_path': 'sources/gdca-recon/2026-11-14/files/d035675163646b64a444_575.pdf', 'source_sha256': '99526665256bc8dd83c18246dd825d4b06a7623f59498b5d201b9fe69d13db46', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/575.pdf', 'bytes': 2033042}, {'source_id': 'GDCA-F-d38aab61922bd600c551', 'source_path': 'sources/gdca-recon/2026-11-14/files/d38aab61922bd600c551_1048.pdf', 'source_sha256': '93297b8fba1513837d11495043503fb1e3e7dd4710348ce4b18992ec9b164b7f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1048.pdf', 'bytes': 1916785}, {'source_id': 'GDCA-F-6c28b2c09d772603c198', 'source_path': 'sources/gdca-recon/2026-11-14/files/6c28b2c09d772603c198_1091.pdf', 'source_sha256': '6dc1c576dab4f864d478d8f697ddeb79ec9c4ae99a37a272802b0402cc1c3a9a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1091.pdf', 'bytes': 1996316}, {'source_id': 'GDCA-F-fc2207a2bd05d6dd1b3c', 'source_path': 'sources/gdca-recon/2026-11-14/files/fc2207a2bd05d6dd1b3c_218.pdf', 'source_sha256': 'fe63c96963d6e983cf1643908a83b127266972b2d4dfbccb93294f5e26157d58', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/218.pdf', 'bytes': 2109429}, {'source_id': 'GDCA-F-2c2343cfec9a496a4a7c', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c2343cfec9a496a4a7c_762.pdf', 'source_sha256': '9eed04b6beae27875d0950c9d662d080695e4f1af05e1d9e5a280001e4f98d73', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/762.pdf', 'bytes': 2111405}, {'source_id': 'GDCA-F-c27b635be9fc4fd48174', 'source_path': 'sources/gdca-recon/2026-11-14/files/c27b635be9fc4fd48174_647.pdf', 'source_sha256': '4fd2c66c7a29ff8c7df06ee7590d9ab91b12d69191ffd4dec8f61baedcacc284', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/647.pdf', 'bytes': 1761773}, {'source_id': 'GDCA-F-21704acbbd84af74d34b', 'source_path': 'sources/gdca-recon/2026-11-14/files/21704acbbd84af74d34b_1499.pdf', 'source_sha256': '2df13743ae058bc5111e2224617fcb48f29037d6324b4c24c2d4c2134fdfffa3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1499.pdf', 'bytes': 1909042}, {'source_id': 'GDCA-F-d1aae9fa9bc1e332eeab', 'source_path': 'sources/gdca-recon/2026-11-14/files/d1aae9fa9bc1e332eeab_900.pdf', 'source_sha256': 'e7dee84de81dfd6d167677c17f6e84a122bd5436235ea8d10b54a4a365fc6d2c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/900.pdf', 'bytes': 1761986}, {'source_id': 'GDCA-F-dfe13fd0c6d30a32149e', 'source_path': 'sources/gdca-recon/2026-11-14/files/dfe13fd0c6d30a32149e_379.pdf', 'source_sha256': 'a0c90d3c4fce45abed4b76f2f552fee13e4549b44ebe1f9017581160b37b7dc8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/379.pdf', 'bytes': 2158954}, {'source_id': 'GDCA-F-9fe3ed46d6e930a99070', 'source_path': 'sources/gdca-recon/2026-11-14/files/9fe3ed46d6e930a99070_1455.pdf', 'source_sha256': 'c0283bfd26e12d6d864f21e76357d377bdd360c30bf48982e07b0531f829004a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1455.pdf', 'bytes': 2166780}, {'source_id': 'GDCA-F-ec88f282f4fa89dd2719', 'source_path': 'sources/gdca-recon/2026-11-14/files/ec88f282f4fa89dd2719_489.pdf', 'source_sha256': 'fe25116b86e25e9eb6dfd7109f4789502d9864ac3969699ef0ad6a93f0262918', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/489.pdf', 'bytes': 2143041}, {'source_id': 'GDCA-F-894f11f3ce048e7007e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/894f11f3ce048e7007e1_685.pdf', 'source_sha256': '9df43a9f08ffddbb0f790284bccc6de708a7309b55b1ccbb9737c7229b020915', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/685.pdf', 'bytes': 2027100}, {'source_id': 'GDCA-F-3e9673f87c525e1fe55f', 'source_path': 'sources/gdca-recon/2026-11-14/files/3e9673f87c525e1fe55f_1053.pdf', 'source_sha256': '1fb5e7eeeabb1113da39443fc7da5bd7d35aa1fef56662e4a95f305e4a391fb0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1053.pdf', 'bytes': 2056689}, {'source_id': 'GDCA-F-b221d749209fad4e06d4', 'source_path': 'sources/gdca-recon/2026-11-14/files/b221d749209fad4e06d4_706.pdf', 'source_sha256': '3f92a9c0cfb3aa4aa867fefea7695e371e228d21be7b304de2e89e7df0cf1e69', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/706.pdf', 'bytes': 1696353}, {'source_id': 'GDCA-F-f355f85d6132e2270e87', 'source_path': 'sources/gdca-recon/2026-11-14/files/f355f85d6132e2270e87_81.pdf', 'source_sha256': 'bf77b3c34bd322260892cf7deb4dbce0d05912c5c7f0dd54d692ba5dcb066e52', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/81.pdf', 'bytes': 1726155}, {'source_id': 'GDCA-F-6fbfdc4a19947242cc09', 'source_path': 'sources/gdca-recon/2026-11-14/files/6fbfdc4a19947242cc09_888.pdf', 'source_sha256': '71dec5fad131b6957b6e5ab009c83ee9bc4735f20a8b6e6ca909fccec42958d7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/888.pdf', 'bytes': 1994193}, {'source_id': 'GDCA-F-44c0ffb7768aae65b877', 'source_path': 'sources/gdca-recon/2026-11-14/files/44c0ffb7768aae65b877_745.pdf', 'source_sha256': '446a6c52cc281af1bcf319d8cf8e9ea872440bfbdd9973c6c73c90405a396c5f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/745.pdf', 'bytes': 1841129}, {'source_id': 'GDCA-F-be683a6ecea6a2dc1561', 'source_path': 'sources/gdca-recon/2026-11-14/files/be683a6ecea6a2dc1561_724.pdf', 'source_sha256': 'fe5af7cb91f88cc3870ef0aa3f315f1c6140c578ec4eab1b49b1d57510a08e0c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/724.pdf', 'bytes': 1821259}, {'source_id': 'GDCA-F-3b0230ff408daa052fb8', 'source_path': 'sources/gdca-recon/2026-11-14/files/3b0230ff408daa052fb8_67.pdf', 'source_sha256': 'a9cf3ad9dd7ea479454addefd68ec6a62b63004dfaffc641fea74566e2784c58', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/67.pdf', 'bytes': 2074916}, {'source_id': 'GDCA-F-0e3942729445240fe2b8', 'source_path': 'sources/gdca-recon/2026-11-14/files/0e3942729445240fe2b8_1381.pdf', 'source_sha256': '6fd99fd66bea8c37f29517aec6c4c7664d5cc4b71247b37d87d44945a8e58027', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1381.pdf', 'bytes': 2467489}, {'source_id': 'GDCA-F-7fb1d5b6c6a22f1720df', 'source_path': 'sources/gdca-recon/2026-11-14/files/7fb1d5b6c6a22f1720df_1128.pdf', 'source_sha256': '651848f03f31ea64f9196ee11e75d2976123c55534e19850efbfacb8ad449a57', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1128.pdf', 'bytes': 2040090}, {'source_id': 'GDCA-F-54fc209e863d4944ec55', 'source_path': 'sources/gdca-recon/2026-11-14/files/54fc209e863d4944ec55_586.pdf', 'source_sha256': '9ca77130aa3f05943124c4d266654532fc924ab31d77890156330cd421c4a3ce', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/586.pdf', 'bytes': 1952142}, {'source_id': 'GDCA-F-30dcb452ac0145ac8ca5', 'source_path': 'sources/gdca-recon/2026-11-14/files/30dcb452ac0145ac8ca5_415.pdf', 'source_sha256': 'e964463be84fc02e8906017f92be508335a3714de9774093e6a67a75b5f1fd51', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/415.pdf', 'bytes': 2163207}, {'source_id': 'GDCA-F-e2d5411492ffe776be34', 'source_path': 'sources/gdca-recon/2026-11-14/files/e2d5411492ffe776be34_101.pdf', 'source_sha256': 'f48e80eccc303d22cbb9a2d98cb7f50be7987c4056e90cd4ce53354ea3daedd3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/101.pdf', 'bytes': 2487172}, {'source_id': 'GDCA-F-17a4d1df5fa84b869e0f', 'source_path': 'sources/gdca-recon/2026-11-14/files/17a4d1df5fa84b869e0f_1228.pdf', 'source_sha256': '7410fb479930e0afa7e38d86658282c0b290339ce28790d37ad2d5fb0654784f', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1228.pdf', 'bytes': 2382582}, {'source_id': 'GDCA-F-0a9a6c249fb1746ebc09', 'source_path': 'sources/gdca-recon/2026-11-14/files/0a9a6c249fb1746ebc09_106.pdf', 'source_sha256': '5b6cadb52a6b72ef7d4c983e8cc68d1edf4854afc4b16e9f8a54ebce812e6afa', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/106.pdf', 'bytes': 2018838}, {'source_id': 'GDCA-F-64fc5dc0b3cc4b7f6728', 'source_path': 'sources/gdca-recon/2026-11-14/files/64fc5dc0b3cc4b7f6728_149.pdf', 'source_sha256': '2f4e6c7529720566235b3c14e19ae71642bd01a373d6036385d1d1b638ae7a6e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/149.pdf', 'bytes': 1868353}, {'source_id': 'GDCA-F-93043d2f4868cba5a8e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/93043d2f4868cba5a8e3_286.pdf', 'source_sha256': 'b9dee65d205b5ea0c7b095a11a6f82c7bf9a8c30fffbc35f46998deb7fc8f57b', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/286.pdf', 'bytes': 1633712}, {'source_id': 'GDCA-F-7ee04dad5857c25468b9', 'source_path': 'sources/gdca-recon/2026-11-14/files/7ee04dad5857c25468b9_901.pdf', 'source_sha256': '7e701e02829aa787ef8e31ea68cd89be4628f41ae34a67a11d5520b7bdbb1b83', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/901.pdf', 'bytes': 1271429}, {'source_id': 'GDCA-F-7626fd24c02c30d91609', 'source_path': 'sources/gdca-recon/2026-11-14/files/7626fd24c02c30d91609_141.pdf', 'source_sha256': 'd501f1e636b4f97893e2ac9bf6db2c9139a091a82cd1a18582c7ac19fb1d6aea', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/141.pdf', 'bytes': 2131171}, {'source_id': 'GDCA-F-fa752d3b2be95c02478a', 'source_path': 'sources/gdca-recon/2026-11-14/files/fa752d3b2be95c02478a_539.pdf', 'source_sha256': 'cce93295b1aa3a26333bd688bf96610609ec91091e87d99ed29dcd44b136a492', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/539.pdf', 'bytes': 1856860}, {'source_id': 'GDCA-F-712d27dc938ef7f2256f', 'source_path': 'sources/gdca-recon/2026-11-14/files/712d27dc938ef7f2256f_148.pdf', 'source_sha256': 'e490baaa7e02a698fe5a1cb341cbb58ea5fde2023a95d18850d16169eef5f824', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/148.pdf', 'bytes': 2271479}, {'source_id': 'GDCA-F-72d9304f21fafa6611ca', 'source_path': 'sources/gdca-recon/2026-11-14/files/72d9304f21fafa6611ca_459.pdf', 'source_sha256': '7c84d3b481bcc3a14f3aae760f1c017a98cc7e428f25f51ebbdada9a2c2c7751', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/459.pdf', 'bytes': 2136929}, {'source_id': 'GDCA-F-50ac5f111d1d128264e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/50ac5f111d1d128264e3_443.pdf', 'source_sha256': '86bd2958fed76afe42d89cd5146b0da05cc999cba34124462ddb9521713dd256', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/443.pdf', 'bytes': 1891401}, {'source_id': 'GDCA-F-a7a8738a918720ed99e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/a7a8738a918720ed99e3_44.pdf', 'source_sha256': 'cdbc182316b90b54ca36c95446e1e7140d8c2cef8c88ee873026f36698a773b4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/44.pdf', 'bytes': 85738}, {'source_id': 'GDCA-F-4d395f7bcc2f80123e31', 'source_path': 'sources/gdca-recon/2026-11-14/files/4d395f7bcc2f80123e31_973.pdf', 'source_sha256': 'a979598a94856e0de52ebf13b1d634889149203b82e6f2975f21ac7cbd5fc029', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/973.pdf', 'bytes': 2230831}, {'source_id': 'GDCA-F-d4e84ed327ce5b7b09fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/d4e84ed327ce5b7b09fd_1149.pdf', 'source_sha256': 'ee74bfe4e0a69cb39a3a2b097b2f254731f6656580699e2378656bef898660f5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1149.pdf', 'bytes': 1844059}, {'source_id': 'GDCA-F-a74de7bab22dbeaf5b7d', 'source_path': 'sources/gdca-recon/2026-11-14/files/a74de7bab22dbeaf5b7d_974.pdf', 'source_sha256': '3def32566af5b7c6a90c5dea008aac4f52f3da268288e823a44709b33589d1ad', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/974.pdf', 'bytes': 2090378}, {'source_id': 'GDCA-F-80da08d5a563ce74db03', 'source_path': 'sources/gdca-recon/2026-11-14/files/80da08d5a563ce74db03_725.pdf', 'source_sha256': '61606a6bee5f59bd99c1d97a2deb5b1af1439a1e7da47dd275674ba4faaccbd2', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/725.pdf', 'bytes': 2366396}, {'source_id': 'GDCA-F-6d473afb1791268d33a7', 'source_path': 'sources/gdca-recon/2026-11-14/files/6d473afb1791268d33a7_547.pdf', 'source_sha256': '880f61f5758f84b5eabc740bffb244d3fba6a31b22ee81e0db714a706ba1330a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/547.pdf', 'bytes': 1994284}, {'source_id': 'GDCA-F-d8214d72d38941899c16', 'source_path': 'sources/gdca-recon/2026-11-14/files/d8214d72d38941899c16_992.pdf', 'source_sha256': 'f97c68dafc2bcdcc87601c9b0dc9b3778cb29677011993eb85b3fcbfafb9b9c1', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/992.pdf', 'bytes': 2571405}, {'source_id': 'GDCA-F-f545c98c22d94e9fbd79', 'source_path': 'sources/gdca-recon/2026-11-14/files/f545c98c22d94e9fbd79_1441.pdf', 'source_sha256': 'c3d6ab66890b574faff7a76b2bb409e102706bb179de45c1c9724ae87bc74f12', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1441.pdf', 'bytes': 2299360}, {'source_id': 'GDCA-F-a50da517110e8fb9840b', 'source_path': 'sources/gdca-recon/2026-11-14/files/a50da517110e8fb9840b_505.pdf', 'source_sha256': 'b21dc88a437d6354064dcce769f6cffb15450b62da8df0dcfd63ffd558bb0f53', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/505.pdf', 'bytes': 2167088}, {'source_id': 'GDCA-F-0669bbf79eb1e5717702', 'source_path': 'sources/gdca-recon/2026-11-14/files/0669bbf79eb1e5717702_976.pdf', 'source_sha256': '21a5d8df1de96b5a62ff21c3f41bcaa740d5d3d91225e7762399751620f47e4a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/976.pdf', 'bytes': 1981971}, {'source_id': 'GDCA-F-48c83980f3769b8c5950', 'source_path': 'sources/gdca-recon/2026-11-14/files/48c83980f3769b8c5950_99.pdf', 'source_sha256': '0e59ee08dbbf5869545b0959f30399dbd6bc3bb893aa2c519b46ec66bfe435f9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/99.pdf', 'bytes': 2177957}, {'source_id': 'GDCA-F-323e95c438458bef2610', 'source_path': 'sources/gdca-recon/2026-11-14/files/323e95c438458bef2610_399.pdf', 'source_sha256': '292a011144615f614b0b2521ed3fe21ef635083dee4c1f33c2e922accf703504', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/399.pdf', 'bytes': 1835920}, {'source_id': 'GDCA-F-03f3ad0eccfb9709635e', 'source_path': 'sources/gdca-recon/2026-11-14/files/03f3ad0eccfb9709635e_52.pdf', 'source_sha256': 'cec175802000c1d695956fb4fc62b3878370532fed1569da59a964366af2c95e', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/52.pdf', 'bytes': 2269734}, {'source_id': 'GDCA-F-8b5539feb6dc76e55f15', 'source_path': 'sources/gdca-recon/2026-11-14/files/8b5539feb6dc76e55f15_774.pdf', 'source_sha256': '46470f597c81227bbe7dba90fc6e6a2790019a6cdd84928ecc734fa486f39c0f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/774.pdf', 'bytes': 1747527}, {'source_id': 'GDCA-F-f124954ba76481d3998f', 'source_path': 'sources/gdca-recon/2026-11-14/files/f124954ba76481d3998f_329.pdf', 'source_sha256': 'bfb7eb7a8297c46bb3674055467a556128d88fd64bd8856075e86719e05d2915', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/329.pdf', 'bytes': 2276771}, {'source_id': 'GDCA-F-6528a20ee88375701695', 'source_path': 'sources/gdca-recon/2026-11-14/files/6528a20ee88375701695_1378.pdf', 'source_sha256': '7904c50497531d79386de9b9f1475547b61c81b68d51f12a9f48c663b3b88676', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1378.pdf', 'bytes': 2608010}, {'source_id': 'GDCA-F-35ad087a5df775da9c9d', 'source_path': 'sources/gdca-recon/2026-11-14/files/35ad087a5df775da9c9d_1189.pdf', 'source_sha256': 'e27c8c1c0c1c679c7b54d436addf82163dbe9e076205ef460068816667f10e1f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1189.pdf', 'bytes': 1706974}, {'source_id': 'GDCA-F-9a20844e7a15888c9d3a', 'source_path': 'sources/gdca-recon/2026-11-14/files/9a20844e7a15888c9d3a_1377.pdf', 'source_sha256': 'bd6235d8432a4af84d8fc6afd5478ce918b79935068f175b46366620fcda7f92', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1377.pdf', 'bytes': 2808491}, {'source_id': 'GDCA-F-d8c25858fed44ff8bc8e', 'source_path': 'sources/gdca-recon/2026-11-14/files/d8c25858fed44ff8bc8e_1318.pdf', 'source_sha256': '913119629df51c82d638ef8ceaa841fe5049c9f7e2bbf6c534c4db6d4aaf3646', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1318.pdf', 'bytes': 2109685}, {'source_id': 'GDCA-F-a74b8ee17582785c4879', 'source_path': 'sources/gdca-recon/2026-11-14/files/a74b8ee17582785c4879_380.pdf', 'source_sha256': 'e14301fc52464a15010863999a1c4fcccdfd54241b5f9104752d000e46fe22df', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/380.pdf', 'bytes': 1976041}, {'source_id': 'GDCA-F-7e83aaaa2a58fc99e3d7', 'source_path': 'sources/gdca-recon/2026-11-14/files/7e83aaaa2a58fc99e3d7_10.pdf', 'source_sha256': '263dd3a7846d2bff0d01ff0b25d727180b463e4d9a3cc99c4c2374c9cefa2d2c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/10.pdf', 'bytes': 2531416}, {'source_id': 'GDCA-F-2a677d8fe9e041a4995d', 'source_path': 'sources/gdca-recon/2026-11-14/files/2a677d8fe9e041a4995d_996.pdf', 'source_sha256': '702bb50e3460fc079812f907a07d8474805b9b792cc3386f31eb694a724c806e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/996.pdf', 'bytes': 2046461}, {'source_id': 'GDCA-F-aab8ec9b3e8257b116b0', 'source_path': 'sources/gdca-recon/2026-11-14/files/aab8ec9b3e8257b116b0_190.pdf', 'source_sha256': '8e14d5b38ea365a32a46fc5ca17059641641c2a7ffa53b30ed01be0f89b9b901', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/190.pdf', 'bytes': 1963897}, {'source_id': 'GDCA-F-cd55029f1b7cca1a111f', 'source_path': 'sources/gdca-recon/2026-11-14/files/cd55029f1b7cca1a111f_1579.pdf', 'source_sha256': 'f6bb880488d40605d413d045190c35517986ed292b7ec1e9412eba359a85ea19', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1579.pdf', 'bytes': 2414612}, {'source_id': 'GDCA-F-1d9fa2af07179ff25811', 'source_path': 'sources/gdca-recon/2026-11-14/files/1d9fa2af07179ff25811_1406.pdf', 'source_sha256': 'e7c5c097e87fd60d582619ae56d2eb3b1d69fe9a881831dac7fb7bb5a51aa903', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1406.pdf', 'bytes': 2541843}, {'source_id': 'GDCA-F-2bbca38fdf6ac497b28a', 'source_path': 'sources/gdca-recon/2026-11-14/files/2bbca38fdf6ac497b28a_278.pdf', 'source_sha256': '8833a86d459f2e451bebe5fe15d218b507b666d089a86e6261b90204a6792d49', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/278.pdf', 'bytes': 2230555}, {'source_id': 'GDCA-F-25b5f7ad1bf43287c261', 'source_path': 'sources/gdca-recon/2026-11-14/files/25b5f7ad1bf43287c261_1035.pdf', 'source_sha256': '73bd326e14cab374d30f7f8fcef05c40fb79fe942f2ab2f48a76ab5bf1418fb5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1035.pdf', 'bytes': 1882572}, {'source_id': 'GDCA-F-a58f4da42eda44b3b012', 'source_path': 'sources/gdca-recon/2026-11-14/files/a58f4da42eda44b3b012_979.pdf', 'source_sha256': '5fd2ffb20b9c48fc81d969a7ff208a0a90694dcc5be41d95274638749e5aa4c5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/979.pdf', 'bytes': 1907217}, {'source_id': 'GDCA-F-3751f071a7ffbc2b9617', 'source_path': 'sources/gdca-recon/2026-11-14/files/3751f071a7ffbc2b9617_499.pdf', 'source_sha256': '081bd414e961242ee976a172410d1a01bda0a2b3a95bebdbf945bd1755ee7a5c', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/499.pdf', 'bytes': 2628707}, {'source_id': 'GDCA-F-3c241cbc26579e589fa2', 'source_path': 'sources/gdca-recon/2026-11-14/files/3c241cbc26579e589fa2_222.pdf', 'source_sha256': '5d2b240b81007f2b150b1731f310885ccf6675fe5791a580cbc291af762b337a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/222.pdf', 'bytes': 1857110}, {'source_id': 'GDCA-F-64ab4ecaf52b863f4246', 'source_path': 'sources/gdca-recon/2026-11-14/files/64ab4ecaf52b863f4246_1309.pdf', 'source_sha256': 'f83d9bd3934c067dfe9d32dbfe9fe72f7f48121c1f7658ef525df64a372fdd7f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1309.pdf', 'bytes': 1922023}, {'source_id': 'GDCA-F-1cf02c1fcd12443da7cb', 'source_path': 'sources/gdca-recon/2026-11-14/files/1cf02c1fcd12443da7cb_1076.pdf', 'source_sha256': '7927735d7876f1e4987fce06e84f45fbd6a57494f765b5d0c6455fc296c3a6e6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1076.pdf', 'bytes': 2112452}, {'source_id': 'GDCA-F-8e30e3e7a83e0b5946f9', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e30e3e7a83e0b5946f9_927.pdf', 'source_sha256': '60a43395708eac4a9d824d4a80ba5a8cacff0a76f4aeb0802c42fac619b57bfd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/927.pdf', 'bytes': 1976905}, {'source_id': 'GDCA-F-10ac6f34a9facb8a2adc', 'source_path': 'sources/gdca-recon/2026-11-14/files/10ac6f34a9facb8a2adc_641.pdf', 'source_sha256': '3a71171712400c041f735e4c6d0c534af2f643fa2b1a8ad65cabbe3226090452', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/641.pdf', 'bytes': 2021948}, {'source_id': 'GDCA-F-da68a50a18d54362aa4b', 'source_path': 'sources/gdca-recon/2026-11-14/files/da68a50a18d54362aa4b_664.pdf', 'source_sha256': '5d1987669a442ea2a914e71cc121c3da1fa7e152c41179b31cd8f8bac1f45930', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/664.pdf', 'bytes': 2710204}, {'source_id': 'GDCA-F-6eaae25fb05a632213e6', 'source_path': 'sources/gdca-recon/2026-11-14/files/6eaae25fb05a632213e6_829.pdf', 'source_sha256': '1d82aa806100f1428166768f463f4138377212c5b60d96888b8b9871d98f4c30', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/829.pdf', 'bytes': 1731410}, {'source_id': 'GDCA-F-7a6529e6cab330b0e635', 'source_path': 'sources/gdca-recon/2026-11-14/files/7a6529e6cab330b0e635_571.pdf', 'source_sha256': '67593b9aae3a3dc714db9e6132b0575b5a84c75a10da85a546d08d2f72747ddb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/571.pdf', 'bytes': 2122987}, {'source_id': 'GDCA-F-59ba1e398954f3aa63a2', 'source_path': 'sources/gdca-recon/2026-11-14/files/59ba1e398954f3aa63a2_694.pdf', 'source_sha256': 'e1f0cb755f046776d60ef870ad924f6f9c80fc5ddc40e3a8a3a1ec130b72490b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/694.pdf', 'bytes': 2107354}, {'source_id': 'GDCA-F-30b5bb76c0d3c5019ed2', 'source_path': 'sources/gdca-recon/2026-11-14/files/30b5bb76c0d3c5019ed2_948.pdf', 'source_sha256': '1c943ac7199cfacbd1e0b101ba66f193fbe44ba883002c0c7331c516dcc4a41d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/948.pdf', 'bytes': 2076272}, {'source_id': 'GDCA-F-45f30c053f01df22f151', 'source_path': 'sources/gdca-recon/2026-11-14/files/45f30c053f01df22f151_1289.pdf', 'source_sha256': '5aada74f605f033b1a05374290b7dbd2014cb15b16d2574cdcb2c099795ae0bf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1289.pdf', 'bytes': 2004990}, {'source_id': 'GDCA-F-d13cca5898866e51769a', 'source_path': 'sources/gdca-recon/2026-11-14/files/d13cca5898866e51769a_833.pdf', 'source_sha256': '1541ce678356e003f052b662db9e597b47e514c9cc8b3f9f34ebcac76196c129', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/833.pdf', 'bytes': 2132952}, {'source_id': 'GDCA-F-c4953ee861890d90f42c', 'source_path': 'sources/gdca-recon/2026-11-14/files/c4953ee861890d90f42c_147.pdf', 'source_sha256': '3a05457b6b38e1c13019839bbd1fef61de572ae8430379f804e736095ae619bd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/147.pdf', 'bytes': 2253514}, {'source_id': 'GDCA-F-eb2c768726980e70fd3e', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb2c768726980e70fd3e_520.pdf', 'source_sha256': 'ddf0d0e5a266aa4ac67573950fe571dd9432579aadd199599c204072230cf922', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/520.pdf', 'bytes': 2009452}, {'source_id': 'GDCA-F-5a8f16440c17621a1bf5', 'source_path': 'sources/gdca-recon/2026-11-14/files/5a8f16440c17621a1bf5_1079.pdf', 'source_sha256': '057902e6278e406dc55dbb89a3a7bfa6034454bd8073db8082adb52b4a235160', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1079.pdf', 'bytes': 2194780}, {'source_id': 'GDCA-F-3d79125e6fa97543e89a', 'source_path': 'sources/gdca-recon/2026-11-14/files/3d79125e6fa97543e89a_1400.pdf', 'source_sha256': '0edd1a0dca444f2fcc4c61c2716a060c9ecaf2980268cb828ec5ca66e91d0fc6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1400.pdf', 'bytes': 2314568}, {'source_id': 'GDCA-F-6e0331245f8b8608f2cc', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e0331245f8b8608f2cc_145.pdf', 'source_sha256': 'd703da28555df64bc301982fabee80d17011c52f60e83383858d38b7807a7e80', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/145.pdf', 'bytes': 1894959}, {'source_id': 'GDCA-F-e99ba35892a5caff37b6', 'source_path': 'sources/gdca-recon/2026-11-14/files/e99ba35892a5caff37b6_1092.pdf', 'source_sha256': 'ab0bd2a017fcfa12e0be8176fc8f6ec4a1b07e023637f4f2dfcda7cc3a1240c9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1092.pdf', 'bytes': 1426326}, {'source_id': 'GDCA-F-098e83dea383396999de', 'source_path': 'sources/gdca-recon/2026-11-14/files/098e83dea383396999de_1482.pdf', 'source_sha256': 'ac5c57f42a308bbf90bd252194932ca77b3d1fcafee5a815117a525160a880b3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1482.pdf', 'bytes': 2271651}, {'source_id': 'GDCA-F-cab997f5fdfa618ea004', 'source_path': 'sources/gdca-recon/2026-11-14/files/cab997f5fdfa618ea004_701.pdf', 'source_sha256': 'b9039661a734f8b81b4a7200b09fcf24d8a99a0fbfc2c812e7263ba9438f83d0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/701.pdf', 'bytes': 1898550}, {'source_id': 'GDCA-F-4393456c0e5bd2287ce1', 'source_path': 'sources/gdca-recon/2026-11-14/files/4393456c0e5bd2287ce1_960.pdf', 'source_sha256': 'be5d303036b149796f71c25459aed5e7131f7ef7393df321528f6a3d688d9b6f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/960.pdf', 'bytes': 2019935}, {'source_id': 'GDCA-F-e229b9e0832f6cdc0abc', 'source_path': 'sources/gdca-recon/2026-11-14/files/e229b9e0832f6cdc0abc_735.pdf', 'source_sha256': '2020015ea4bfc158c1fc8638892e4edc74f6e96275e4fe0ed70b3d227b371618', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/735.pdf', 'bytes': 2076575}, {'source_id': 'GDCA-F-449d7d4d457d9337598b', 'source_path': 'sources/gdca-recon/2026-11-14/files/449d7d4d457d9337598b_171.pdf', 'source_sha256': '1fdda456c0ba82ace0068fd9cbdff9655392aed3bba016f85789c9e2ed86ea64', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/171.pdf', 'bytes': 1604511}, {'source_id': 'GDCA-F-ee92fcaab5396cc1c3b5', 'source_path': 'sources/gdca-recon/2026-11-14/files/ee92fcaab5396cc1c3b5_1286.pdf', 'source_sha256': 'c523b4755c5eb10341b362bb6e81a4956e376a35ba05fbc9031498b00656ffd4', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1286.pdf', 'bytes': 1410700}, {'source_id': 'GDCA-F-565cca52a47cdf009d02', 'source_path': 'sources/gdca-recon/2026-11-14/files/565cca52a47cdf009d02_1560.pdf', 'source_sha256': 'c22fa6c6fc5e9cf129f367cf4bccf9e81586193b6f5f7e4f5ce5e5c6dee108fc', 'document_pages': 11, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1560.pdf', 'bytes': 4241404}, {'source_id': 'GDCA-F-e08bf43e1ac0e5e1f1b5', 'source_path': 'sources/gdca-recon/2026-11-14/files/e08bf43e1ac0e5e1f1b5_431.pdf', 'source_sha256': 'ea0090f435c62e1f59882279c7833186a340a48970695b34fc5b7e0a53c9ea96', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/431.pdf', 'bytes': 1932668}, {'source_id': 'GDCA-F-dbcf2ec8fa27be29bf9c', 'source_path': 'sources/gdca-recon/2026-11-14/files/dbcf2ec8fa27be29bf9c_903.pdf', 'source_sha256': '6b039100638d7a7b67c172871270cc71fc208350630bbc0a25cb3f8aefb52f51', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/903.pdf', 'bytes': 2091243}, {'source_id': 'GDCA-F-9c64ee4d21140d9dd704', 'source_path': 'sources/gdca-recon/2026-11-14/files/9c64ee4d21140d9dd704_203.pdf', 'source_sha256': 'b822271a1453b06fe4f3851ceeb2e48403bfd91db65c1c2f715a80b717af9f49', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/203.pdf', 'bytes': 1950310}, {'source_id': 'GDCA-F-88548382d0ec0fd20a72', 'source_path': 'sources/gdca-recon/2026-11-14/files/88548382d0ec0fd20a72_877.pdf', 'source_sha256': '599a9be82189e206c8445a1d03cff7c331a8483fe7b02c4957aec8b8bb9c0ec5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/877.pdf', 'bytes': 1751874}, {'source_id': 'GDCA-F-cb4734bee41c44ef6ebb', 'source_path': 'sources/gdca-recon/2026-11-14/files/cb4734bee41c44ef6ebb_929.pdf', 'source_sha256': '84cdbb7998fc5d98241701aba3732f0c41c1349fad4c73534e9ae08b04325a97', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/929.pdf', 'bytes': 1821861}, {'source_id': 'GDCA-F-00d38832a8315b1b8b1f', 'source_path': 'sources/gdca-recon/2026-11-14/files/00d38832a8315b1b8b1f_1213.pdf', 'source_sha256': 'f1b29ef08f25cc9c484dc0b6a7eadfdb96617194486b3fd0e4cf79e778d55a06', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1213.pdf', 'bytes': 1863477}, {'source_id': 'GDCA-F-3edeffe54640f61b8967', 'source_path': 'sources/gdca-recon/2026-11-14/files/3edeffe54640f61b8967_1230.pdf', 'source_sha256': '79f8f11ee35c4a12a823dea25de3323dc4c708be0a984cc82b2041dad39ac8f4', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1230.pdf', 'bytes': 918701}, {'source_id': 'GDCA-F-0e2e253bc34fba8babb1', 'source_path': 'sources/gdca-recon/2026-11-14/files/0e2e253bc34fba8babb1_840.pdf', 'source_sha256': 'f7c8d2cd3503e66707cd6d4e5f98d5ac373610e16f52e32d0bcd7c43de001ccc', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/840.pdf', 'bytes': 850463}, {'source_id': 'GDCA-F-6039f1aa3f3d2fa4a452', 'source_path': 'sources/gdca-recon/2026-11-14/files/6039f1aa3f3d2fa4a452_61.pdf', 'source_sha256': 'c3cdd9ee416afcaf515be351914e49365e1c75d58d2a069683bacb0bd20f92a0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/61.pdf', 'bytes': 2121802}, {'source_id': 'GDCA-F-48f5cb1a82265e9baae9', 'source_path': 'sources/gdca-recon/2026-11-14/files/48f5cb1a82265e9baae9_533.pdf', 'source_sha256': '1cd21ec61d75034f82f0f78519e877a8eee75f21df771886c66b6228c59e611d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/533.pdf', 'bytes': 2131993}, {'source_id': 'GDCA-F-b25e4699adf167f7466d', 'source_path': 'sources/gdca-recon/2026-11-14/files/b25e4699adf167f7466d_207.pdf', 'source_sha256': '716d9cadaed512d158bf538a0121a6785cc2a3e0de67a38c164996a9c1db6f4e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/207.pdf', 'bytes': 1886604}, {'source_id': 'GDCA-F-f5246c37bec4d437ed0f', 'source_path': 'sources/gdca-recon/2026-11-14/files/f5246c37bec4d437ed0f_898.pdf', 'source_sha256': '783b385ce719b6f6a074df3ae221b170b61f8b98fa0510c746dfbd4d91bb5099', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/898.pdf', 'bytes': 1804149}, {'source_id': 'GDCA-F-88831178e0ea75425c77', 'source_path': 'sources/gdca-recon/2026-11-14/files/88831178e0ea75425c77_279.pdf', 'source_sha256': '5512e44f24b4387c5d656d80ff160671d30703b1d06f74e19b66dfae2849373e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/279.pdf', 'bytes': 2427093}, {'source_id': 'GDCA-F-89ec28f30825287f952b', 'source_path': 'sources/gdca-recon/2026-11-14/files/89ec28f30825287f952b_185.pdf', 'source_sha256': '76a34138ee501891b0d35af2afdc22a0136735b1b1d9fa58655a3ed768187056', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/185.pdf', 'bytes': 1688998}, {'source_id': 'GDCA-F-38b18436cf4f1a4253c0', 'source_path': 'sources/gdca-recon/2026-11-14/files/38b18436cf4f1a4253c0_260.pdf', 'source_sha256': '7841904418c79fd2fd348c71adc7f4ca489ea06cf5a512398ee52f20c695afa6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/260.pdf', 'bytes': 1801885}, {'source_id': 'GDCA-F-102dee7cd35f727cc905', 'source_path': 'sources/gdca-recon/2026-11-14/files/102dee7cd35f727cc905_953.pdf', 'source_sha256': '43a06d4ec780a51d1c341601019a3b340fdadd8e48161e2064f9ccbed6382599', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/953.pdf', 'bytes': 2238696}, {'source_id': 'GDCA-F-443c16cda7680b10b19a', 'source_path': 'sources/gdca-recon/2026-11-14/files/443c16cda7680b10b19a_1280.pdf', 'source_sha256': 'b717fe9653e6e7c4e5a439cfce2d931f228250bf92b2c5815ab0fe87b25a0837', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1280.pdf', 'bytes': 1992611}, {'source_id': 'GDCA-F-4fcf5b215f9bdd2755bd', 'source_path': 'sources/gdca-recon/2026-11-14/files/4fcf5b215f9bdd2755bd_1145.pdf', 'source_sha256': '732292fd6e6f54ab045ff98cd7d90e897c6f4c0c1abc7707bacd03bb2978683b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1145.pdf', 'bytes': 1852759}, {'source_id': 'GDCA-F-0c354543cda1d27ffc67', 'source_path': 'sources/gdca-recon/2026-11-14/files/0c354543cda1d27ffc67_1364.pdf', 'source_sha256': '53b6e45f414c5a7ee8222c185c6d8769bec1ab4096b6f989bf48d6c966896f19', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1364.pdf', 'bytes': 1929151}, {'source_id': 'GDCA-F-a98ff4aea3ebd6e0b235', 'source_path': 'sources/gdca-recon/2026-11-14/files/a98ff4aea3ebd6e0b235_1463.pdf', 'source_sha256': 'ae7dc41da1054aa41e090b0fe15f58dc53ef2fc77ad08e18a9b934a7dc5599e3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1463.pdf', 'bytes': 2274290}, {'source_id': 'GDCA-F-b76dc7e19c52a50310d5', 'source_path': 'sources/gdca-recon/2026-11-14/files/b76dc7e19c52a50310d5_1056.pdf', 'source_sha256': '7e5e083fda07d50e62d14227f1bd4f53a804311ec89030265fe30120dd633b43', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1056.pdf', 'bytes': 1872498}, {'source_id': 'GDCA-F-ca2173b32ad544f019d3', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca2173b32ad544f019d3_235.pdf', 'source_sha256': '171bbc07efae413ec11a8ae346a37bdeff6ae61e57522c9ccce2238a4b01dfd1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/235.pdf', 'bytes': 1971659}, {'source_id': 'GDCA-F-27270f5d7ac4ce65ec7b', 'source_path': 'sources/gdca-recon/2026-11-14/files/27270f5d7ac4ce65ec7b_534.pdf', 'source_sha256': '59d5b53b2f716bfa4df73d40e56f492e5300b332a67fa6531a4afa55e739216e', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/534.pdf', 'bytes': 1427239}, {'source_id': 'GDCA-F-a6ceace30deb74fa5875', 'source_path': 'sources/gdca-recon/2026-11-14/files/a6ceace30deb74fa5875_455.pdf', 'source_sha256': 'dfdf2df8474ca0cf00c4606fc822838badd30cb628408434207728746e23aba1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/455.pdf', 'bytes': 1937317}, {'source_id': 'GDCA-F-915cef34bc44b469b201', 'source_path': 'sources/gdca-recon/2026-11-14/files/915cef34bc44b469b201_323.pdf', 'source_sha256': 'a5ef38a08f7c980adfcf49945c52f5f587311b9be649814442885e530e459a3b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/323.pdf', 'bytes': 1773267}, {'source_id': 'GDCA-F-f8d6baccb0fe3f246c05', 'source_path': 'sources/gdca-recon/2026-11-14/files/f8d6baccb0fe3f246c05_1583.pdf', 'source_sha256': '9f7546eaf895e4fb097887e2d64b15aafa4abe7813d7e095dc0394c6b66f4364', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1583.pdf', 'bytes': 2381981}, {'source_id': 'GDCA-F-37bec178797bf2e22252', 'source_path': 'sources/gdca-recon/2026-11-14/files/37bec178797bf2e22252_1199.pdf', 'source_sha256': '0f2538bb3c6305af4fc50b57f2e90d1fb242ac95d62bf1c89985f4dddf50ddd1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1199.pdf', 'bytes': 2137005}, {'source_id': 'GDCA-F-25c5ca4fb8e854b4cbd7', 'source_path': 'sources/gdca-recon/2026-11-14/files/25c5ca4fb8e854b4cbd7_1100.pdf', 'source_sha256': 'b22c20bf07ceae75c3967b812f38abbe0750108617dd189a339e10c1359fa5b9', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1100.pdf', 'bytes': 2253442}, {'source_id': 'GDCA-F-00146879b36e926eecb0', 'source_path': 'sources/gdca-recon/2026-11-14/files/00146879b36e926eecb0_727.pdf', 'source_sha256': 'cb238d8934f6d3cec243379a589713494d2dec1f2ad9a42a4f697b7ebb30911d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/727.pdf', 'bytes': 2113828}, {'source_id': 'GDCA-F-5230ff673ad3c3774a20', 'source_path': 'sources/gdca-recon/2026-11-14/files/5230ff673ad3c3774a20_189.pdf', 'source_sha256': '6f546cc7fe8d6343c93f59f4c08459bdbb77c71a54eeca15ea44571bdcf6a5f9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/189.pdf', 'bytes': 1959692}, {'source_id': 'GDCA-F-50553561ce7d233e06fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/50553561ce7d233e06fd_177.pdf', 'source_sha256': '04a765e1aef26a46f4d79aeade2624c94c113edd72bbb760b21c28bca6f36bad', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/177.pdf', 'bytes': 1925560}, {'source_id': 'GDCA-F-7d637dfe75b5c7755ec9', 'source_path': 'sources/gdca-recon/2026-11-14/files/7d637dfe75b5c7755ec9_63.pdf', 'source_sha256': '38955bba9657d20ced93b901c838b2ccf2a11a1e043ff9a039ba018d537cd0b6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/63.pdf', 'bytes': 2189280}, {'source_id': 'GDCA-F-52c19ca0e659d66599c0', 'source_path': 'sources/gdca-recon/2026-11-14/files/52c19ca0e659d66599c0_320.pdf', 'source_sha256': '146e3e0343ed750af3b2bb65912229c7b73e128ccae776b4a5366cfd7db8f684', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/320.pdf', 'bytes': 1913167}, {'source_id': 'GDCA-F-f807c40e72275d75142d', 'source_path': 'sources/gdca-recon/2026-11-14/files/f807c40e72275d75142d_1178.pdf', 'source_sha256': 'f921fefc447fe3212e041c43086119f8a8b5e43aab272db6029d7923b55ceb15', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1178.pdf', 'bytes': 1797296}, {'source_id': 'GDCA-F-bc596462282d443bcce6', 'source_path': 'sources/gdca-recon/2026-11-14/files/bc596462282d443bcce6_966.pdf', 'source_sha256': '37667191199554c8e748cca47a64307714e78dfdfb03d8491d2e2f3cc8ebdb89', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/966.pdf', 'bytes': 2325643}, {'source_id': 'GDCA-F-df0353e7b3bd4de28d02', 'source_path': 'sources/gdca-recon/2026-11-14/files/df0353e7b3bd4de28d02_1281.pdf', 'source_sha256': '5791d34d70cc3ecc3cc90414931fd5b20729f6c518f514ce7622ef6a3677373a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1281.pdf', 'bytes': 1984466}, {'source_id': 'GDCA-F-e2cd52dc3180889972ae', 'source_path': 'sources/gdca-recon/2026-11-14/files/e2cd52dc3180889972ae_1099.pdf', 'source_sha256': '38c64e5e8cd2adcb99413c9311af8f72d0c657486100b2fd68c424ccc5a79f43', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1099.pdf', 'bytes': 1877560}, {'source_id': 'GDCA-F-69cd14765081220ba725', 'source_path': 'sources/gdca-recon/2026-11-14/files/69cd14765081220ba725_215.pdf', 'source_sha256': 'dafedc2bc04b815c666b4f317e211e4a1b33ac0dd46ac83b7b0350aa2bdf3e03', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/215.pdf', 'bytes': 1630782}, {'source_id': 'GDCA-F-5e418feb9a67ba439ac3', 'source_path': 'sources/gdca-recon/2026-11-14/files/5e418feb9a67ba439ac3_1049.pdf', 'source_sha256': 'ada675602fa9206d3310438b5e472663bda21c620521ebdcb533ca508192a1e3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1049.pdf', 'bytes': 2057669}, {'source_id': 'GDCA-F-53d38082f161c503fed9', 'source_path': 'sources/gdca-recon/2026-11-14/files/53d38082f161c503fed9_871.pdf', 'source_sha256': 'ad55790f5164a0856adc3989bca2dd639f55736dbca43061d690816d53041b9a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/871.pdf', 'bytes': 1802058}, {'source_id': 'GDCA-F-9d03ecb03396207bb690', 'source_path': 'sources/gdca-recon/2026-11-14/files/9d03ecb03396207bb690_1250.pdf', 'source_sha256': '932441d689e15a4f7a74df0682112c6ed903e7c17feeadc2830fe9b150faa890', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1250.pdf', 'bytes': 1897656}, {'source_id': 'GDCA-F-342776e862e7588e221a', 'source_path': 'sources/gdca-recon/2026-11-14/files/342776e862e7588e221a_56.pdf', 'source_sha256': 'cb92ad8b2bf53564bc069f5b8ff7317ebbd0e0820d3f4fe0bc1ff87387524bfc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/56.pdf', 'bytes': 1923613}, {'source_id': 'GDCA-F-3f435d09fca5ce86a4a2', 'source_path': 'sources/gdca-recon/2026-11-14/files/3f435d09fca5ce86a4a2_555.pdf', 'source_sha256': '1b38bc504de50bc06617200d2a363e86837d44c5eb555a1775ce481e6f802834', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/555.pdf', 'bytes': 1731836}, {'source_id': 'GDCA-F-ea45df870dee15733f24', 'source_path': 'sources/gdca-recon/2026-11-14/files/ea45df870dee15733f24_1215.pdf', 'source_sha256': '305b536588239dfda0fe90df56878ad9051172655e7a416c86b9a4b1eb7eecbf', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1215.pdf', 'bytes': 2378805}, {'source_id': 'GDCA-F-9c9ecabdcad990a31d6f', 'source_path': 'sources/gdca-recon/2026-11-14/files/9c9ecabdcad990a31d6f_115.pdf', 'source_sha256': '53ec2012d20fee6b5bf9d257e6db90e911345cd34197f36f62b706205b727149', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/115.pdf', 'bytes': 2149595}, {'source_id': 'GDCA-F-4371d8e389dbbc7c0094', 'source_path': 'sources/gdca-recon/2026-11-14/files/4371d8e389dbbc7c0094_602.pdf', 'source_sha256': '7278008c8d3a0ec577f811bcfbe39394ffd9c84f08c48ad9c5772aa3014d9552', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/602.pdf', 'bytes': 1795614}, {'source_id': 'GDCA-F-83f853b9dda7ebcd4735', 'source_path': 'sources/gdca-recon/2026-11-14/files/83f853b9dda7ebcd4735_1074.pdf', 'source_sha256': 'f65f422c12c87d186007ceed359dd401d8acd8837f05ee872e1145e593e581d5', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1074.pdf', 'bytes': 2118403}, {'source_id': 'GDCA-F-53348155544eb2cd8a04', 'source_path': 'sources/gdca-recon/2026-11-14/files/53348155544eb2cd8a04_265.pdf', 'source_sha256': 'f077208f5d8902afbff0c4dd4530409ca3ae73de0d5ffe9b6d2b99705dd6f4c9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/265.pdf', 'bytes': 1950231}, {'source_id': 'GDCA-F-fbeb5379e464a26a2fd6', 'source_path': 'sources/gdca-recon/2026-11-14/files/fbeb5379e464a26a2fd6_673.pdf', 'source_sha256': '7d713d8e06047cf0b16c9c2f154bca384aa73400089c299863b8f9735f0cd371', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/673.pdf', 'bytes': 1897821}, {'source_id': 'GDCA-F-e0123b51c3ca0a404769', 'source_path': 'sources/gdca-recon/2026-11-14/files/e0123b51c3ca0a404769_1043.pdf', 'source_sha256': '49701ecd483368e03020983a638ad32a1a4dbb7604485bdd13450fac8950d96f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1043.pdf', 'bytes': 2103363}, {'source_id': 'GDCA-F-3314dcaf3eae8d6995d7', 'source_path': 'sources/gdca-recon/2026-11-14/files/3314dcaf3eae8d6995d7_1055.pdf', 'source_sha256': '51475278bf6e6b909968576779357637fbc22495866baf9c83e6e64531487d42', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1055.pdf', 'bytes': 2001063}, {'source_id': 'GDCA-F-c8407179e9130c58b184', 'source_path': 'sources/gdca-recon/2026-11-14/files/c8407179e9130c58b184_341.pdf', 'source_sha256': 'a5ca82203a5fee4f6ccfe477e07e62933f624cd054439477bbdaa80d3973d8f9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/341.pdf', 'bytes': 1983456}, {'source_id': 'GDCA-F-f5d3b0e1262cc5ad4d9a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f5d3b0e1262cc5ad4d9a_299.pdf', 'source_sha256': 'f371bfbe074f6b09b4c45cc88aa7b15cffbde2b857c88e369d6f4bbe85baf964', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/299.pdf', 'bytes': 1889172}, {'source_id': 'GDCA-F-a643e75a8ab6a44588e8', 'source_path': 'sources/gdca-recon/2026-11-14/files/a643e75a8ab6a44588e8_783.pdf', 'source_sha256': 'f0767e87be70462579e4ddde8c12ce161acec4fdcdc765fbeb6f89524badc4d4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/783.pdf', 'bytes': 1819351}, {'source_id': 'GDCA-F-e433850c2d7f83fa70af', 'source_path': 'sources/gdca-recon/2026-11-14/files/e433850c2d7f83fa70af_676.pdf', 'source_sha256': 'f3348483cb3209afed27abf3e528bdb41c68326cdd72af458f0af06f28e9a0d9', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/676.pdf', 'bytes': 2297292}, {'source_id': 'GDCA-F-b9ebab0b9d9567fd6718', 'source_path': 'sources/gdca-recon/2026-11-14/files/b9ebab0b9d9567fd6718_1263.pdf', 'source_sha256': 'afdcd1dcfe37477a48a2e41ed6a2174b7c8eda2b25732a380edc56f84cd2dc43', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1263.pdf', 'bytes': 2236113}, {'source_id': 'GDCA-F-eadbe1afa7f81133c67d', 'source_path': 'sources/gdca-recon/2026-11-14/files/eadbe1afa7f81133c67d_462.pdf', 'source_sha256': '1293208d76fa8b2ae72d0e35d211d95716958791880d8e03c682fdde861f9ab6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/462.pdf', 'bytes': 1915190}, {'source_id': 'GDCA-F-dbcf2ec402b607971918', 'source_path': 'sources/gdca-recon/2026-11-14/files/dbcf2ec402b607971918_64.pdf', 'source_sha256': 'f26c5ae150a05be0758006de83e4790744a6b9f572b68ea7f5e3efdd77cba2e0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/64.pdf', 'bytes': 1917828}, {'source_id': 'GDCA-F-be4ec4a5c92f34906a9e', 'source_path': 'sources/gdca-recon/2026-11-14/files/be4ec4a5c92f34906a9e_608.pdf', 'source_sha256': '68c7e65be1173da379a662100e8c5d90dcd9338ffac393868a6286d50d62d094', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/608.pdf', 'bytes': 2739574}, {'source_id': 'GDCA-F-8cb490da4ca114cd1e81', 'source_path': 'sources/gdca-recon/2026-11-14/files/8cb490da4ca114cd1e81_168.pdf', 'source_sha256': 'f626833da2e089d0f6f44bb7c4c9a1069b70b51be96cb5865b74234887dc7d69', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/168.pdf', 'bytes': 1877694}, {'source_id': 'GDCA-F-31c0773689eeca0f3d24', 'source_path': 'sources/gdca-recon/2026-11-14/files/31c0773689eeca0f3d24_404.pdf', 'source_sha256': '839ca52a2fac1b4e47b9c654a172a6c78b929019f1bc3c06482f475db162241c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/404.pdf', 'bytes': 2258077}, {'source_id': 'GDCA-F-e4e302f31dc9ffbff7b4', 'source_path': 'sources/gdca-recon/2026-11-14/files/e4e302f31dc9ffbff7b4_163.pdf', 'source_sha256': '9d07b03d1e97545951deb78457a10b9b795a810f0a1379cb70fef46f903d7ca2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/163.pdf', 'bytes': 2110504}, {'source_id': 'GDCA-F-aea94e5b01fd5c0dc251', 'source_path': 'sources/gdca-recon/2026-11-14/files/aea94e5b01fd5c0dc251_984.pdf', 'source_sha256': '3a99cbd73576c806e9507728b1968d3b64a353fdea1e747745f6172605b8c130', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/984.pdf', 'bytes': 1579600}, {'source_id': 'GDCA-F-3ff2a9f3869d61fb288f', 'source_path': 'sources/gdca-recon/2026-11-14/files/3ff2a9f3869d61fb288f_154.pdf', 'source_sha256': '946e7fcdfa848f1538ff05da346cf7be71bae54985fff66f680581c3e4b356f7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/154.pdf', 'bytes': 1816246}, {'source_id': 'GDCA-F-d2ffe956c8d0c36f824c', 'source_path': 'sources/gdca-recon/2026-11-14/files/d2ffe956c8d0c36f824c_436.pdf', 'source_sha256': '077381d9615d5b70e04d12f25d980893dd456e7cd919fe68eea3816da4a8c140', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/436.pdf', 'bytes': 1753276}, {'source_id': 'GDCA-F-999078966f8b4adf8d48', 'source_path': 'sources/gdca-recon/2026-11-14/files/999078966f8b4adf8d48_1007.pdf', 'source_sha256': '3637ae6998b7d12998a6d7f8b7e614270ff66b51dc50472b23569ba9faab8049', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1007.pdf', 'bytes': 2143037}, {'source_id': 'GDCA-F-060755549123435901b9', 'source_path': 'sources/gdca-recon/2026-11-14/files/060755549123435901b9_1323.pdf', 'source_sha256': 'ac7771d9e26cafc95c6efd159d127747a0d6817e42c4ea922d2b509c55c0ac08', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1323.pdf', 'bytes': 2159073}, {'source_id': 'GDCA-F-febec6b9f251138d7d55', 'source_path': 'sources/gdca-recon/2026-11-14/files/febec6b9f251138d7d55_824.pdf', 'source_sha256': 'bfd0b580006ba5eef968e85dd67b8067e5f78549f27203031830007c12ca5cd3', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/824.pdf', 'bytes': 2945887}, {'source_id': 'GDCA-F-aa197e237717ba8bcbaf', 'source_path': 'sources/gdca-recon/2026-11-14/files/aa197e237717ba8bcbaf_556.pdf', 'source_sha256': '6e48d5ea181c545da5aef46d267d629ff905ea26f55cefe9314000840d0a0fe9', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/556.pdf', 'bytes': 2432822}, {'source_id': 'GDCA-F-ea94ce826b52fbb2c84f', 'source_path': 'sources/gdca-recon/2026-11-14/files/ea94ce826b52fbb2c84f_689.pdf', 'source_sha256': '37654231f8840d1ab78b77f705283221aa7cc1e170c22a649f9b0dacc7b07423', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/689.pdf', 'bytes': 2054007}, {'source_id': 'GDCA-F-3b290a28e20cb9898b4f', 'source_path': 'sources/gdca-recon/2026-11-14/files/3b290a28e20cb9898b4f_1458.pdf', 'source_sha256': '369b8095b22dbd906da5deba72c5afa63269cb401b2a1126ad5925c1b6cf6c32', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1458.pdf', 'bytes': 3214719}, {'source_id': 'GDCA-F-4dc81cb705df6337e434', 'source_path': 'sources/gdca-recon/2026-11-14/files/4dc81cb705df6337e434_383.pdf', 'source_sha256': 'd68c0a009671af8f752fcc4e597efa6b65e7c9de6b8af0fafb43884e8256451b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/383.pdf', 'bytes': 1776283}, {'source_id': 'GDCA-F-95304552f4b5ed6336bc', 'source_path': 'sources/gdca-recon/2026-11-14/files/95304552f4b5ed6336bc_1229.pdf', 'source_sha256': '6817c12868dd639c4d36912731234ca4e44eef10eb78b20f3b17ffd312222cf9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1229.pdf', 'bytes': 1879399}, {'source_id': 'GDCA-F-27bac3285d1cdbc4ad03', 'source_path': 'sources/gdca-recon/2026-11-14/files/27bac3285d1cdbc4ad03_678.pdf', 'source_sha256': 'bc73334ec9a47e7823380d71610d17c79e5b3af1c7e420be15e0a0b290e8726d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/678.pdf', 'bytes': 2161534}, {'source_id': 'GDCA-F-06784f1f9bbe7fed6974', 'source_path': 'sources/gdca-recon/2026-11-14/files/06784f1f9bbe7fed6974_1094.pdf', 'source_sha256': 'e2ecac8f7cf75442546575758a8a5a41f074e24a8ede1a21f89556e672685906', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1094.pdf', 'bytes': 2009723}, {'source_id': 'GDCA-F-e5f3b7ca01eb575e196f', 'source_path': 'sources/gdca-recon/2026-11-14/files/e5f3b7ca01eb575e196f_1131.pdf', 'source_sha256': '747f9c014c6757e90d19a392cb8eea2f2148417f484d2e7aa7e8392f1188043d', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1131.pdf', 'bytes': 1321485}, {'source_id': 'GDCA-F-bd518c0733a75d7c2221', 'source_path': 'sources/gdca-recon/2026-11-14/files/bd518c0733a75d7c2221_1177.pdf', 'source_sha256': '66f52b12352e558ccd882003a0a84f11f4450ace1e016a08c3924a0c7a9d4313', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1177.pdf', 'bytes': 2141800}, {'source_id': 'GDCA-F-cff5ffceca0a67dd6a0d', 'source_path': 'sources/gdca-recon/2026-11-14/files/cff5ffceca0a67dd6a0d_1409.pdf', 'source_sha256': '4f04afb1be6d108198b0f1d8ec2dfef157302c8abd68d75f5df5ce4ae363f070', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1409.pdf', 'bytes': 2124380}, {'source_id': 'GDCA-F-656ea5007d25d33b726f', 'source_path': 'sources/gdca-recon/2026-11-14/files/656ea5007d25d33b726f_488.pdf', 'source_sha256': 'fbec53c94ca130d1193f83bc4a0fec23f6b7facba55626cab0d77769d725e19d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/488.pdf', 'bytes': 2106852}, {'source_id': 'GDCA-F-b2be56e6778daaceb535', 'source_path': 'sources/gdca-recon/2026-11-14/files/b2be56e6778daaceb535_393.pdf', 'source_sha256': '49424b5cfffbb17ebffce0ef73362a24858df9e990643f3fa193b326e8692286', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/393.pdf', 'bytes': 1617475}, {'source_id': 'GDCA-F-385d351c0aa4565e7423', 'source_path': 'sources/gdca-recon/2026-11-14/files/385d351c0aa4565e7423_6.pdf', 'source_sha256': 'fc92e32576870f484c3765f8e0b7ed3a3e83a59e3faf14ac64d022e981e8a77d', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/6.pdf', 'bytes': 1734429}, {'source_id': 'GDCA-F-aa4584351d3e8d38eadf', 'source_path': 'sources/gdca-recon/2026-11-14/files/aa4584351d3e8d38eadf_1236.pdf', 'source_sha256': 'f8b1a4f7f27249ccabef25f4783a596f5255944d078add56f6a97619fb7ab600', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1236.pdf', 'bytes': 1732055}, {'source_id': 'GDCA-F-62114b76fb8a842b9ca4', 'source_path': 'sources/gdca-recon/2026-11-14/files/62114b76fb8a842b9ca4_757.pdf', 'source_sha256': '1573282ba43de403c2989767d8a3c218ae3641a74fddb39b91ec9db346fdffd8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/757.pdf', 'bytes': 1786631}, {'source_id': 'GDCA-F-f1ec6d65f74237bf4506', 'source_path': 'sources/gdca-recon/2026-11-14/files/f1ec6d65f74237bf4506_166.pdf', 'source_sha256': '64c84bfe1d01d05b48f0005824f00600ae773813a0610533ee3440684c1f6a05', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/166.pdf', 'bytes': 2164549}, {'source_id': 'GDCA-F-61891cf6c7589ba12aa7', 'source_path': 'sources/gdca-recon/2026-11-14/files/61891cf6c7589ba12aa7_1432.pdf', 'source_sha256': '8e8dc422124c36cbcaad5f71e1f2b677a828d02dd92c8121d6735b608ff128d1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1432.pdf', 'bytes': 2834539}, {'source_id': 'GDCA-F-c4f89aa4875d8a5aca9e', 'source_path': 'sources/gdca-recon/2026-11-14/files/c4f89aa4875d8a5aca9e_100.pdf', 'source_sha256': 'd8540a7f78f17916bd3bc0eeed9058849adacac911ecc7ec50e8038adcb63c5f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/100.pdf', 'bytes': 2142543}, {'source_id': 'GDCA-F-4a0c3ccbda998f7feeb4', 'source_path': 'sources/gdca-recon/2026-11-14/files/4a0c3ccbda998f7feeb4_364.pdf', 'source_sha256': 'f7b81c6f06a8ac7a0bb53ca6380e6a4ffddff2b58e6cd07610ce38a85ac3acbc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/364.pdf', 'bytes': 2192796}, {'source_id': 'GDCA-F-8c2d76e6e499d3abb806', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c2d76e6e499d3abb806_1429.pdf', 'source_sha256': '366a2e9b579e74bf7875aaaf2d61956530d51f655038fb268bcad10690220349', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1429.pdf', 'bytes': 2921975}, {'source_id': 'GDCA-F-834d3d2e253e8f69c84a', 'source_path': 'sources/gdca-recon/2026-11-14/files/834d3d2e253e8f69c84a_756.pdf', 'source_sha256': 'e0f328544bf3bba075d6c44b863db9ecfaa549de649eb50be94099099502060a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/756.pdf', 'bytes': 2168532}, {'source_id': 'GDCA-F-620157a34f0f1e5e4baa', 'source_path': 'sources/gdca-recon/2026-11-14/files/620157a34f0f1e5e4baa_738.pdf', 'source_sha256': '4341bc26a37ccce14c14cb7f6f2caafc40119a5ff7c7af4b0abe5eceea4fbbd2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/738.pdf', 'bytes': 2056729}, {'source_id': 'GDCA-F-2efedc33b4e946c7ef67', 'source_path': 'sources/gdca-recon/2026-11-14/files/2efedc33b4e946c7ef67_982.pdf', 'source_sha256': '2698aa240145041c4dec35a9cc412a3fdbcc0237957b101d7ee5f8eff4e2e1c3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/982.pdf', 'bytes': 2165952}, {'source_id': 'GDCA-F-53e7d84ef907cd357814', 'source_path': 'sources/gdca-recon/2026-11-14/files/53e7d84ef907cd357814_512.pdf', 'source_sha256': 'a4499d972ff5c235a17cb7da53ffb070bd122e02b9091d5bb8e0c8d75cd50260', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/512.pdf', 'bytes': 1804940}, {'source_id': 'GDCA-F-f2d63753a9beb9d2e617', 'source_path': 'sources/gdca-recon/2026-11-14/files/f2d63753a9beb9d2e617_1465.pdf', 'source_sha256': '4f48b4dbe43c9ec576f881d1e405c66cc21347ce946f3f4d7b567a37d8dcb308', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1465.pdf', 'bytes': 2303858}, {'source_id': 'GDCA-F-a2359761d5bf7af1a4c0', 'source_path': 'sources/gdca-recon/2026-11-14/files/a2359761d5bf7af1a4c0_621.pdf', 'source_sha256': '776085a3ca7d5a2b8ba2d80539790ff74b2d285556577e105d712537f85213af', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/621.pdf', 'bytes': 1804302}, {'source_id': 'GDCA-F-7842f04f2cbb243a5d88', 'source_path': 'sources/gdca-recon/2026-11-14/files/7842f04f2cbb243a5d88_428.pdf', 'source_sha256': '56f14c566c7681d993eae711e4c6ee8ae80f44a5f1e28c727469f1f3c7c9c88d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/428.pdf', 'bytes': 2276665}, {'source_id': 'GDCA-F-da010e5b117b5a4aad9c', 'source_path': 'sources/gdca-recon/2026-11-14/files/da010e5b117b5a4aad9c_1098.pdf', 'source_sha256': '8c660e233549d7b367a4bbd5d03ffb16d85966b9b52433fcd953d89c9e665c43', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1098.pdf', 'bytes': 1702451}, {'source_id': 'GDCA-F-f6cc4549578dfdbecdc2', 'source_path': 'sources/gdca-recon/2026-11-14/files/f6cc4549578dfdbecdc2_1217.pdf', 'source_sha256': 'edc9f3217609351a60f348e78d0f00b5278313ab0dd15732d6052c0e654ca172', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1217.pdf', 'bytes': 2408369}, {'source_id': 'GDCA-F-19f24ba0b64c033d4a68', 'source_path': 'sources/gdca-recon/2026-11-14/files/19f24ba0b64c033d4a68_366.pdf', 'source_sha256': '21dc02748e6659f278c68711417b6c9dc5c35923020ef9dcf67051e2c4f76678', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/366.pdf', 'bytes': 2166108}, {'source_id': 'GDCA-F-2cb8b45cd81332158c79', 'source_path': 'sources/gdca-recon/2026-11-14/files/2cb8b45cd81332158c79_524.pdf', 'source_sha256': '81c7405f052e88dfc78233c52457cc516c3967e7ba02f3253317057dd8751395', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/524.pdf', 'bytes': 1630092}, {'source_id': 'GDCA-F-a36a02e5e9b2d06e77ab', 'source_path': 'sources/gdca-recon/2026-11-14/files/a36a02e5e9b2d06e77ab_1108.pdf', 'source_sha256': '51e0f21337adb4451596332a18592d8c227f346a6348249a3fbb49513b03efa7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1108.pdf', 'bytes': 2115114}, {'source_id': 'GDCA-F-7448b20c938bbc35152a', 'source_path': 'sources/gdca-recon/2026-11-14/files/7448b20c938bbc35152a_378.pdf', 'source_sha256': '81e7b2c24bc6b95ef123fd9b783bd8832bef6c4a1c5f4124344dfb1da6c8efab', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/378.pdf', 'bytes': 2927351}, {'source_id': 'GDCA-F-c8b6504a5bdfe83440ee', 'source_path': 'sources/gdca-recon/2026-11-14/files/c8b6504a5bdfe83440ee_239.pdf', 'source_sha256': 'f1e970d7f350fdea1c2e0ae38d02e045cd051fdd240143ac79417f29c41b18b1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/239.pdf', 'bytes': 1989616}, {'source_id': 'GDCA-F-8e56983c62134690c111', 'source_path': 'sources/gdca-recon/2026-11-14/files/8e56983c62134690c111_433.pdf', 'source_sha256': '431af3191be1ae532ef848fd8050c90531d065f70611839cceb7b63d8a96fd69', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/433.pdf', 'bytes': 2158186}, {'source_id': 'GDCA-F-6e051b3041a57a8ab7ef', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e051b3041a57a8ab7ef_693.pdf', 'source_sha256': '61b420d7742baad17665e5718e448cb83baac3aa37b1e89cf16f3754d3765d64', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/693.pdf', 'bytes': 1721551}, {'source_id': 'GDCA-F-c38dc103c6601bf33858', 'source_path': 'sources/gdca-recon/2026-11-14/files/c38dc103c6601bf33858_334.pdf', 'source_sha256': 'cf1b7d80ec182a4a979770c556d34f2bc55c0ae743c653da2982decf44629efb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/334.pdf', 'bytes': 2192913}, {'source_id': 'GDCA-F-8623ccc5b991406e697c', 'source_path': 'sources/gdca-recon/2026-11-14/files/8623ccc5b991406e697c_691.pdf', 'source_sha256': '04e0cf920180d3b613e39c575cff1120091122b6e891dd4b64e8a8b7ab785254', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/691.pdf', 'bytes': 2002175}, {'source_id': 'GDCA-F-50a622639598156c70be', 'source_path': 'sources/gdca-recon/2026-11-14/files/50a622639598156c70be_834.pdf', 'source_sha256': '47291a16a22ca96b1a6d82166e41d7f3ba3c32b21bed89a09957c5a935c74b45', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/834.pdf', 'bytes': 1899301}, {'source_id': 'GDCA-F-a314e5b4170e5687a6b7', 'source_path': 'sources/gdca-recon/2026-11-14/files/a314e5b4170e5687a6b7_93.pdf', 'source_sha256': 'da92d83075abdfd16bedb45a210401a87c60a61c82f27a0222891df75015153d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/93.pdf', 'bytes': 1793960}, {'source_id': 'GDCA-F-da47ffeddaf2640eb769', 'source_path': 'sources/gdca-recon/2026-11-14/files/da47ffeddaf2640eb769_930.pdf', 'source_sha256': '5f5e51971b06c94c1ca6fd9a7a2bdb7a2a5874576acb1fb2d84de486ff0e7385', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/930.pdf', 'bytes': 1843728}, {'source_id': 'GDCA-F-4efee661897e7d376381', 'source_path': 'sources/gdca-recon/2026-11-14/files/4efee661897e7d376381_84.pdf', 'source_sha256': '80c4ee52c8a2988a744bf2cec2d71ba960aba21a7832a3a3bd928e75835f34d4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/84.pdf', 'bytes': 2042543}, {'source_id': 'GDCA-F-d2f2cf8c8c801240add5', 'source_path': 'sources/gdca-recon/2026-11-14/files/d2f2cf8c8c801240add5_611.pdf', 'source_sha256': '46efba20b0ce75a3068cbdd467a7577dc22e5feaa33c95615302fcf08f4d22fc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/611.pdf', 'bytes': 1888130}, {'source_id': 'GDCA-F-29ade8be9c257edd5f67', 'source_path': 'sources/gdca-recon/2026-11-14/files/29ade8be9c257edd5f67_807.pdf', 'source_sha256': 'e3de30ba15b1c1554b52fa708a513e284c4d7d7c9b7aae6d3ede3d0e579980a1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/807.pdf', 'bytes': 1761937}, {'source_id': 'GDCA-F-f20fc66ddf53c576ef91', 'source_path': 'sources/gdca-recon/2026-11-14/files/f20fc66ddf53c576ef91_591.pdf', 'source_sha256': '65e0af64f7ae289df9cf62a55dd9b5a0a1b5d1762148383b29f3e54ab3225fba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/591.pdf', 'bytes': 1889446}, {'source_id': 'GDCA-F-fca432b61fa7df2819a4', 'source_path': 'sources/gdca-recon/2026-11-14/files/fca432b61fa7df2819a4_114.pdf', 'source_sha256': 'c9b7a19494f439b21655711b925d027098686e8d4c7fc3a01386daf6ac7aff5d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/114.pdf', 'bytes': 1993276}, {'source_id': 'GDCA-F-1220a6f0c2227c9bd391', 'source_path': 'sources/gdca-recon/2026-11-14/files/1220a6f0c2227c9bd391_515.pdf', 'source_sha256': '5ac632b19adca64239bbcff4925fd2903bfcb502172d3611af11820a8962b8c7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/515.pdf', 'bytes': 1742737}, {'source_id': 'GDCA-F-6e012275b9e7059aa567', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e012275b9e7059aa567_82.pdf', 'source_sha256': 'ee414f59cc91db8ee05e4bbe4b92b0af0ec501e558ccf261ae84c0f17ff479c6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/82.pdf', 'bytes': 1825169}, {'source_id': 'GDCA-F-1170afb67e75ea87db1b', 'source_path': 'sources/gdca-recon/2026-11-14/files/1170afb67e75ea87db1b_1026.pdf', 'source_sha256': '89ff72c0677efa964021eb8be39100922407dea3edfd4739df66dae08dc6910b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1026.pdf', 'bytes': 1922711}, {'source_id': 'GDCA-F-8f5b1290f7314d7caaf2', 'source_path': 'sources/gdca-recon/2026-11-14/files/8f5b1290f7314d7caaf2_184.pdf', 'source_sha256': '35bcb8a19d2bb798488ccdb3d5354c87673411c7645660e063e45bc2f92c666b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/184.pdf', 'bytes': 2104856}, {'source_id': 'GDCA-F-d35f26c491403ccceada', 'source_path': 'sources/gdca-recon/2026-11-14/files/d35f26c491403ccceada_1075.pdf', 'source_sha256': 'c48ba366cc0380dbfdd98a96a0d23e73b8990b694a39ba75f18d3fd3a30c6336', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1075.pdf', 'bytes': 1994446}, {'source_id': 'GDCA-F-d7155d03e34853557f1b', 'source_path': 'sources/gdca-recon/2026-11-14/files/d7155d03e34853557f1b_872.pdf', 'source_sha256': 'c58612fe7fad8ba00593d8e56f9493d322ae95aee2aa99382422371daa9d362f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/872.pdf', 'bytes': 2003588}, {'source_id': 'GDCA-F-7e0d5c12f0116470a0d8', 'source_path': 'sources/gdca-recon/2026-11-14/files/7e0d5c12f0116470a0d8_353.pdf', 'source_sha256': '03a718e2c6e34fd62298620457427b51c4ebeab6e9511647fa56a039f1f2c326', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/353.pdf', 'bytes': 1790164}, {'source_id': 'GDCA-F-f33e26111d7efebb56da', 'source_path': 'sources/gdca-recon/2026-11-14/files/f33e26111d7efebb56da_882.pdf', 'source_sha256': '36257518bb5140c397856cf49c678708848d2579e914195e66b8da77211107cd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/882.pdf', 'bytes': 2060802}, {'source_id': 'GDCA-F-fc0086cb1e5be312d82c', 'source_path': 'sources/gdca-recon/2026-11-14/files/fc0086cb1e5be312d82c_1567.pdf', 'source_sha256': '88947c491a04051c621c6a65cebc725dd7fdb926a23737153d0296ffc3ddc807', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1567.pdf', 'bytes': 3341824}, {'source_id': 'GDCA-F-6e634802e6a41bd93f3a', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e634802e6a41bd93f3a_296.pdf', 'source_sha256': 'cd462fc8806c91f91e3cec4b3477674731ffd5752fc850ed9943e6fc09420b20', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/296.pdf', 'bytes': 1838857}, {'source_id': 'GDCA-F-5dc8db7b70dda2c5ed82', 'source_path': 'sources/gdca-recon/2026-11-14/files/5dc8db7b70dda2c5ed82_580.pdf', 'source_sha256': '2bce95dc70f9d8b5f537230d7094906a922b297e2ef30044d4992768a880e697', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/580.pdf', 'bytes': 2299921}, {'source_id': 'GDCA-F-011407a8fd08cf7a5be6', 'source_path': 'sources/gdca-recon/2026-11-14/files/011407a8fd08cf7a5be6_802.pdf', 'source_sha256': 'c2f6aef217b76a8c5b23373019c9153131ed85f2243bbee7dd0676c1aef65c35', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/802.pdf', 'bytes': 2403198}, {'source_id': 'GDCA-F-e3432cb9989cd8b835f3', 'source_path': 'sources/gdca-recon/2026-11-14/files/e3432cb9989cd8b835f3_980.pdf', 'source_sha256': 'd90a7bdb1588c785786827af7d0a1141293b49be6b1ec1f395a38b6685bfdb88', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/980.pdf', 'bytes': 1962919}, {'source_id': 'GDCA-F-4fd5be6d82f7a43f2b47', 'source_path': 'sources/gdca-recon/2026-11-14/files/4fd5be6d82f7a43f2b47_652.pdf', 'source_sha256': '88e63dea9916c7224c2ddda3375683c6fee45cadb31ad2a557163c6d05cc693d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/652.pdf', 'bytes': 1810586}, {'source_id': 'GDCA-F-27f51b65bbf747b238bc', 'source_path': 'sources/gdca-recon/2026-11-14/files/27f51b65bbf747b238bc_654.pdf', 'source_sha256': 'eafb38ba6eac2b6bc5c9f26c04574d20a251d4e5e71d360c05b7c41ce549780d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/654.pdf', 'bytes': 1726605}, {'source_id': 'GDCA-F-a3a023851e5082a3da0c', 'source_path': 'sources/gdca-recon/2026-11-14/files/a3a023851e5082a3da0c_891.pdf', 'source_sha256': '020234576c618ad2d807d56670bac9dc160afe0e2eb450cc0acde8bef108ef05', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/891.pdf', 'bytes': 946357}, {'source_id': 'GDCA-F-7db042e5a4abea71ceb1', 'source_path': 'sources/gdca-recon/2026-11-14/files/7db042e5a4abea71ceb1_412.pdf', 'source_sha256': '98f73f503631f9aaa1a5212fbb277bf054982bf7d23e482d14440b0b20f21175', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/412.pdf', 'bytes': 1878254}, {'source_id': 'GDCA-F-9f9d0a0ec87ea9950c4c', 'source_path': 'sources/gdca-recon/2026-11-14/files/9f9d0a0ec87ea9950c4c_387.pdf', 'source_sha256': '59d3a2ac2b64624ec125716a59405405cdaac5878b7a4fc86a678227ec921d08', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/387.pdf', 'bytes': 2766814}, {'source_id': 'GDCA-F-258bd98c95a3188d53d8', 'source_path': 'sources/gdca-recon/2026-11-14/files/258bd98c95a3188d53d8_463.pdf', 'source_sha256': 'cf6f13a076f7b32cf2b8f124096ac415fe79915e2ba0e0f5e53b69566ad76868', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/463.pdf', 'bytes': 1710812}, {'source_id': 'GDCA-F-54d747a127cd995fc50b', 'source_path': 'sources/gdca-recon/2026-11-14/files/54d747a127cd995fc50b_1082.pdf', 'source_sha256': '9b94823d7fc4f56273360a243c6e6fec98a0cb5c2dc8ccfdb7fe016ae14bae3b', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1082.pdf', 'bytes': 1255613}, {'source_id': 'GDCA-F-067592704f4e4bbd5c8e', 'source_path': 'sources/gdca-recon/2026-11-14/files/067592704f4e4bbd5c8e_1407.pdf', 'source_sha256': '0e5a00ec517e6aedbc6fbebad4e115280ae82fb7fd93a396416e273557a6821c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1407.pdf', 'bytes': 2120701}, {'source_id': 'GDCA-F-78eaac0a85cc4c40bc3c', 'source_path': 'sources/gdca-recon/2026-11-14/files/78eaac0a85cc4c40bc3c_264.pdf', 'source_sha256': 'f26ecbcab01c0334436eedcabdaef4ddaba00d91c81f1e78115f445f8039f1e0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/264.pdf', 'bytes': 1906119}, {'source_id': 'GDCA-F-20b8231fb22a6ec4abb9', 'source_path': 'sources/gdca-recon/2026-11-14/files/20b8231fb22a6ec4abb9_361.pdf', 'source_sha256': 'f516316c22fbe95717a4a6e0c576256d376f55ed035eb757b8fe7fd1635871ae', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/361.pdf', 'bytes': 1855261}, {'source_id': 'GDCA-F-a9ebd796b1a5f0054a07', 'source_path': 'sources/gdca-recon/2026-11-14/files/a9ebd796b1a5f0054a07_1451.pdf', 'source_sha256': '4e17870936c9b1982ed9a132ddba3c1a03e15e47df532739f3b10b690e1948d9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1451.pdf', 'bytes': 2616758}, {'source_id': 'GDCA-F-5af0ea59abc9a920504b', 'source_path': 'sources/gdca-recon/2026-11-14/files/5af0ea59abc9a920504b_232.pdf', 'source_sha256': '8d7c955e73fa3ad2d799119c2dd5d9507f84aee8ac2e1195b23d6dd9e4cb65ba', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/232.pdf', 'bytes': 2002621}, {'source_id': 'GDCA-F-d16a36fd651dfdbfa3ac', 'source_path': 'sources/gdca-recon/2026-11-14/files/d16a36fd651dfdbfa3ac_1415.pdf', 'source_sha256': 'cac1422b60641d2aac60cbd7b5c63b878307991df3adf1287013152b606ab2d7', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1415.pdf', 'bytes': 980091}, {'source_id': 'GDCA-F-f8bb4439deb0d46fe64a', 'source_path': 'sources/gdca-recon/2026-11-14/files/f8bb4439deb0d46fe64a_213.pdf', 'source_sha256': '3b77c4750680ab3aeac28166f430c721038d2bd1cb287c7b6bb67621bd3d0670', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/213.pdf', 'bytes': 3670349}, {'source_id': 'GDCA-F-444a234e45d2d0905ef3', 'source_path': 'sources/gdca-recon/2026-11-14/files/444a234e45d2d0905ef3_1397.pdf', 'source_sha256': '1abd7f901f559753de917a77664c50753eb1f0919f817a87a867335f3199d7e7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1397.pdf', 'bytes': 2371032}, {'source_id': 'GDCA-F-a58858f4c58ce0edc99c', 'source_path': 'sources/gdca-recon/2026-11-14/files/a58858f4c58ce0edc99c_1065.pdf', 'source_sha256': '635d1f43999e19710bcf65edfc1d02a7b2e5dad20ef02d9d5916e3ca25be5b33', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1065.pdf', 'bytes': 2169430}, {'source_id': 'GDCA-F-de94f7425cdb76c43ae4', 'source_path': 'sources/gdca-recon/2026-11-14/files/de94f7425cdb76c43ae4_1227.pdf', 'source_sha256': '51efcc779b6735d97179ccebd4948b8dbd32ecfba68a06f303582ec9917367ba', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1227.pdf', 'bytes': 2013519}, {'source_id': 'GDCA-F-0eb2945f6cb73a11a54a', 'source_path': 'sources/gdca-recon/2026-11-14/files/0eb2945f6cb73a11a54a_552.pdf', 'source_sha256': '6085909252451a2eef16ee1964394cf894aee54aeb4ed772db6631dce949882c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/552.pdf', 'bytes': 1839630}, {'source_id': 'GDCA-F-dd66f1831b4754b32b23', 'source_path': 'sources/gdca-recon/2026-11-14/files/dd66f1831b4754b32b23_1448.pdf', 'source_sha256': 'ff3db0d5a3753c2c3e4a248b8b4784ff7cd1e8dd0f39d29e0e73f4dae06422a4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1448.pdf', 'bytes': 2264411}, {'source_id': 'GDCA-F-78299fb489e42cfd5c4e', 'source_path': 'sources/gdca-recon/2026-11-14/files/78299fb489e42cfd5c4e_175.pdf', 'source_sha256': '9f1c0b200cd96ceb4483eb929cca7ca1429485016d1f3a67c29dc34c09d61617', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/175.pdf', 'bytes': 2090060}, {'source_id': 'GDCA-F-669edfcabd10c47a15a7', 'source_path': 'sources/gdca-recon/2026-11-14/files/669edfcabd10c47a15a7_319.pdf', 'source_sha256': '8c3a655b0804aaee59bc24df154152ee9abfe8c6c602513d42c9e43a59be6178', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/319.pdf', 'bytes': 1987112}, {'source_id': 'GDCA-F-2bbf974e03ea6b311b57', 'source_path': 'sources/gdca-recon/2026-11-14/files/2bbf974e03ea6b311b57_118.pdf', 'source_sha256': '7b156321bed1f52704a02e8ec64d33e627b28e320ed76a16668770fdf51ebe0d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/118.pdf', 'bytes': 2034284}, {'source_id': 'GDCA-F-3e3f1a37346a67bddf44', 'source_path': 'sources/gdca-recon/2026-11-14/files/3e3f1a37346a67bddf44_1172.pdf', 'source_sha256': '4efd08702cacd86d321b8b585b2cc8f55fcdd9cc2660f759244a09a51c2b2509', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1172.pdf', 'bytes': 1927454}, {'source_id': 'GDCA-F-46fd44550a01b39f9a87', 'source_path': 'sources/gdca-recon/2026-11-14/files/46fd44550a01b39f9a87_986.pdf', 'source_sha256': 'e504eb5a5d9fbf68aa93db86754dc065ca5c1e79f8a680b77d677c0c5f9c520f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/986.pdf', 'bytes': 1821878}, {'source_id': 'GDCA-F-5f15a8d4955061a66ecd', 'source_path': 'sources/gdca-recon/2026-11-14/files/5f15a8d4955061a66ecd_999.pdf', 'source_sha256': '056b1cda5844ec607a98a3531386c82d124788281295fe3b12e96990d31cd5e1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/999.pdf', 'bytes': 1797793}, {'source_id': 'GDCA-F-3e133e791b319522ea48', 'source_path': 'sources/gdca-recon/2026-11-14/files/3e133e791b319522ea48_817.pdf', 'source_sha256': '4809c8592240d3c0facb023d20722e6e7fe35dd6952bd3b5a813a9e3608c38dc', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/817.pdf', 'bytes': 1806256}, {'source_id': 'GDCA-F-4064f170ed2217575d46', 'source_path': 'sources/gdca-recon/2026-11-14/files/4064f170ed2217575d46_1562.pdf', 'source_sha256': '87831a4d1503b4172246f90a7919c169a5c8b5147f3b1a22be0f1d80817da5ff', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1562.pdf', 'bytes': 3853046}, {'source_id': 'GDCA-F-f35473848d8dbbe80cb6', 'source_path': 'sources/gdca-recon/2026-11-14/files/f35473848d8dbbe80cb6_426.pdf', 'source_sha256': '56b7941924fd335f45039930c7cc12085d7330f61e42aa5c7ec318f99d2cb03a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/426.pdf', 'bytes': 2288433}, {'source_id': 'GDCA-F-0a8f61ba66ef9b50a8d3', 'source_path': 'sources/gdca-recon/2026-11-14/files/0a8f61ba66ef9b50a8d3_1288.pdf', 'source_sha256': '015968ebfa334f4e9d6c418b746c47b91fa9d6772b98632840f6710e68332df4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1288.pdf', 'bytes': 2162951}, {'source_id': 'GDCA-F-ab6e33244ca2aeaa754e', 'source_path': 'sources/gdca-recon/2026-11-14/files/ab6e33244ca2aeaa754e_69.pdf', 'source_sha256': '96076047936b0d359a8faa70b559f4fa5b198eda362d2defb699bcc3ada63a3e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/69.pdf', 'bytes': 1934776}, {'source_id': 'GDCA-F-f43b0fe3f0cadf5e5b08', 'source_path': 'sources/gdca-recon/2026-11-14/files/f43b0fe3f0cadf5e5b08_543.pdf', 'source_sha256': '9e1e42c8dc1cd8e5cb5552d8f3f82d2d37ed1f67c45c8865f49c1c988c1da1b6', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/543.pdf', 'bytes': 1906678}, {'source_id': 'GDCA-F-a12974bd9603cea934ef', 'source_path': 'sources/gdca-recon/2026-11-14/files/a12974bd9603cea934ef_372.pdf', 'source_sha256': 'f39c64f074c438cbd17fd773e51e7298610b031ef8f0cdf9ddc45c9263c1693f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/372.pdf', 'bytes': 1924993}, {'source_id': 'GDCA-F-7f68d1b6b3d9b2945641', 'source_path': 'sources/gdca-recon/2026-11-14/files/7f68d1b6b3d9b2945641_28.pdf', 'source_sha256': '1b8264ce896c7060346e708da4343fb86d4f2df2d131ef859721c1f3981d34d9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/28.pdf', 'bytes': 91742}, {'source_id': 'GDCA-F-8de851065b564fd8639f', 'source_path': 'sources/gdca-recon/2026-11-14/files/8de851065b564fd8639f_14.pdf', 'source_sha256': '79d7242e7a98e512095327900b9d377531ac35767683559c738ce9cc16b2b9de', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/14.pdf', 'bytes': 2991698}, {'source_id': 'GDCA-F-7fae246a685d2d50d07e', 'source_path': 'sources/gdca-recon/2026-11-14/files/7fae246a685d2d50d07e_87.pdf', 'source_sha256': '53a178b1eeab434af50efcaca9f657e40710274b5c4f13e91afcb0221913f4af', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/87.pdf', 'bytes': 2193082}, {'source_id': 'GDCA-F-6c65a67ecfac41e95fdf', 'source_path': 'sources/gdca-recon/2026-11-14/files/6c65a67ecfac41e95fdf_192.pdf', 'source_sha256': '2de6f27ad88bd72232331259ca618602a14eaf2dbddd4d90eb70e4f609617eb0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/192.pdf', 'bytes': 1945921}, {'source_id': 'GDCA-F-55a8fa08ebd734462e39', 'source_path': 'sources/gdca-recon/2026-11-14/files/55a8fa08ebd734462e39_198.pdf', 'source_sha256': '575d89f01be820e89c882f9a1872a3ef33d3a9e62ff7079c0784980f8c798f86', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/198.pdf', 'bytes': 1792817}, {'source_id': 'GDCA-F-08b3310547bbc6888dec', 'source_path': 'sources/gdca-recon/2026-11-14/files/08b3310547bbc6888dec_583.pdf', 'source_sha256': '85ae5ad83ba9e6e002d32022bdd39da216ee0b34d8a1ee4ec58715849239a8d0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/583.pdf', 'bytes': 2193468}, {'source_id': 'GDCA-F-1961927e60c4ed0e753f', 'source_path': 'sources/gdca-recon/2026-11-14/files/1961927e60c4ed0e753f_831.pdf', 'source_sha256': '87335195cda9eb7936e756ce35efd7d0832ecd02b4f2b904afe89129ac0dec8e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/831.pdf', 'bytes': 1979575}, {'source_id': 'GDCA-F-76bacaf2175c0f4c39a3', 'source_path': 'sources/gdca-recon/2026-11-14/files/76bacaf2175c0f4c39a3_723.pdf', 'source_sha256': 'fac11eabdd0d1f9a6dfec47b20dbfc6b75a746cc88d657c8d7b63f61a2ecf631', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/723.pdf', 'bytes': 1790314}, {'source_id': 'GDCA-F-bdbba4406ab301eb1a21', 'source_path': 'sources/gdca-recon/2026-11-14/files/bdbba4406ab301eb1a21_636.pdf', 'source_sha256': '11c94bc72f94b0b9ddf42f1bcc7920e4d2731c37ea73e45dc32cde810f8b830a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/636.pdf', 'bytes': 1924776}, {'source_id': 'GDCA-F-1863e27d2b3f44b70310', 'source_path': 'sources/gdca-recon/2026-11-14/files/1863e27d2b3f44b70310_89.pdf', 'source_sha256': '303fcde64436020e9410576ca239522138f5990b1396cee19f9ce47474333aa6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/89.pdf', 'bytes': 1827081}, {'source_id': 'GDCA-F-75edd2cea91558ec50b7', 'source_path': 'sources/gdca-recon/2026-11-14/files/75edd2cea91558ec50b7_183.pdf', 'source_sha256': '5d35583c33c4cf9e3af4577d8edf35c28508e581e573274f6ad785e8abdac265', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/183.pdf', 'bytes': 2329460}, {'source_id': 'GDCA-F-b9c1d8cf213becf0623e', 'source_path': 'sources/gdca-recon/2026-11-14/files/b9c1d8cf213becf0623e_295.pdf', 'source_sha256': 'e4add9e5a79f8add1e4c7ff0908e2baf7909a581bfc6aadc0b29e8b8997632bf', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/295.pdf', 'bytes': 2291254}, {'source_id': 'GDCA-F-e75511fdd6431bbd3a1a', 'source_path': 'sources/gdca-recon/2026-11-14/files/e75511fdd6431bbd3a1a_922.pdf', 'source_sha256': '3832cb1d03a06b6d1fb2d30aa46d79d635dea663ade15826727c6928a06c4340', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/922.pdf', 'bytes': 1855829}, {'source_id': 'GDCA-F-8be3bde995ba22d88944', 'source_path': 'sources/gdca-recon/2026-11-14/files/8be3bde995ba22d88944_1393.pdf', 'source_sha256': '46ea0c84a54f152aa5de0af10c22187d72b9cf553f6a2a692d7bdbc9b173508d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1393.pdf', 'bytes': 2019811}, {'source_id': 'GDCA-F-497e9e5d7e1b93910b8b', 'source_path': 'sources/gdca-recon/2026-11-14/files/497e9e5d7e1b93910b8b_1427.pdf', 'source_sha256': '9767afb796315bc8180231379bdf09abab67348fec07fd51e11cb92fbbc51fff', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1427.pdf', 'bytes': 2242810}, {'source_id': 'GDCA-F-e35a69b9ea822c2a7b65', 'source_path': 'sources/gdca-recon/2026-11-14/files/e35a69b9ea822c2a7b65_429.pdf', 'source_sha256': '096e81f8fcf057aad0dd829d88ec4fc00459d800f70c2a3ca1ea662c340f9cf9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/429.pdf', 'bytes': 2154727}, {'source_id': 'GDCA-F-752d8071cb4747a93163', 'source_path': 'sources/gdca-recon/2026-11-14/files/752d8071cb4747a93163_1061.pdf', 'source_sha256': 'ed5f35d497a800eda2b9eb78e353704c5527fe470bc7fb21e5160e994bc1f861', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1061.pdf', 'bytes': 1811771}, {'source_id': 'GDCA-F-de232b70b4e38c928010', 'source_path': 'sources/gdca-recon/2026-11-14/files/de232b70b4e38c928010_1106.pdf', 'source_sha256': '96af99f0bd480edf2bd56f19dcc8d6f873d748d4af5a625d9e3686fb704aa435', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1106.pdf', 'bytes': 1977386}, {'source_id': 'GDCA-F-082984b9c565a4fa6b82', 'source_path': 'sources/gdca-recon/2026-11-14/files/082984b9c565a4fa6b82_1162.pdf', 'source_sha256': '472cc2630d29c2dcbfb1c904a4a3703d050c37cac1b82f3b9112c8a2762b3e46', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1162.pdf', 'bytes': 5403760}, {'source_id': 'GDCA-F-1ca647427d749e1ad072', 'source_path': 'sources/gdca-recon/2026-11-14/files/1ca647427d749e1ad072_592.pdf', 'source_sha256': 'b43d38e134f9a7f658c2a597d0d260d672b1738a83a650398590442bc4ca4ff1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/592.pdf', 'bytes': 1789386}, {'source_id': 'GDCA-F-33afa7a726215d3e68be', 'source_path': 'sources/gdca-recon/2026-11-14/files/33afa7a726215d3e68be_5.pdf', 'source_sha256': 'bc370688c641b879fa26a9322ddfa450767962acca1093f216079f70cb48ceee', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/5.pdf', 'bytes': 2649287}, {'source_id': 'GDCA-F-eb84ecbfbf3863e6f690', 'source_path': 'sources/gdca-recon/2026-11-14/files/eb84ecbfbf3863e6f690_957.pdf', 'source_sha256': 'c3636483ca94f671c61f3a75bf13bf6bb34d8cc22ff389288389ec31d11cbe8b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/957.pdf', 'bytes': 2011019}, {'source_id': 'GDCA-F-ca3ea02c640cdb861beb', 'source_path': 'sources/gdca-recon/2026-11-14/files/ca3ea02c640cdb861beb_1471.pdf', 'source_sha256': 'ae412dd63d0b10b7531571afbc7a69f9e6f815183f8522e1d6664217a84b4669', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1471.pdf', 'bytes': 2901316}, {'source_id': 'GDCA-F-a5563225fc0738c90798', 'source_path': 'sources/gdca-recon/2026-11-14/files/a5563225fc0738c90798_731.pdf', 'source_sha256': '14ead75ef1757fa42fa29e1b8c869070b3616a44aedb30e88912da5127c78f1d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/731.pdf', 'bytes': 1966345}, {'source_id': 'GDCA-F-032d13e22a02e41e2d6a', 'source_path': 'sources/gdca-recon/2026-11-14/files/032d13e22a02e41e2d6a_1241.pdf', 'source_sha256': '44c47c010244c028bcbc852cf0f13a545d027b35490f4ea45263504a116870b0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1241.pdf', 'bytes': 1613835}, {'source_id': 'GDCA-F-2c640fc911a1759fb4f2', 'source_path': 'sources/gdca-recon/2026-11-14/files/2c640fc911a1759fb4f2_1222.pdf', 'source_sha256': '362cf5acfac74b27e4a488e6e380fe3ff45c3ca699aa7357e7750091f2aa2ab6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1222.pdf', 'bytes': 1807545}, {'source_id': 'GDCA-F-e525e669d2481e2a807d', 'source_path': 'sources/gdca-recon/2026-11-14/files/e525e669d2481e2a807d_193.pdf', 'source_sha256': '832203d26ba3ebdf99fb5469ffc85ef8c734463aa80968cc1223849687d5e7cc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/193.pdf', 'bytes': 1971800}, {'source_id': 'GDCA-F-1b3dc7efcad25a0d9f10', 'source_path': 'sources/gdca-recon/2026-11-14/files/1b3dc7efcad25a0d9f10_1293.pdf', 'source_sha256': 'f51abf07279eb831a491cb9336b2dced88fbfede85e4cbe3651313cf3d99ba19', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1293.pdf', 'bytes': 2273643}, {'source_id': 'GDCA-F-21383636bb071260e33d', 'source_path': 'sources/gdca-recon/2026-11-14/files/21383636bb071260e33d_98.pdf', 'source_sha256': '55f23aeee0ee0fdca8fbe01f0dd40c47556d7b782c0c32325d65c1efbf9ab1a9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/98.pdf', 'bytes': 1932599}, {'source_id': 'GDCA-F-6e6a337debdbdb5caead', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e6a337debdbdb5caead_1301.pdf', 'source_sha256': '0948b896486762e31d1e8ed7d2e6c9392934753b2c5c9abe0fa6d73b6c7a02dd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1301.pdf', 'bytes': 1832239}, {'source_id': 'GDCA-F-1a9b75ae5b1a865dd739', 'source_path': 'sources/gdca-recon/2026-11-14/files/1a9b75ae5b1a865dd739_1239.pdf', 'source_sha256': '3c08670f2175889551e14768e001884da07382e67fcd29beca69ccff1314929d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1239.pdf', 'bytes': 1734392}, {'source_id': 'GDCA-F-02bde54d8a07cb91bc16', 'source_path': 'sources/gdca-recon/2026-11-14/files/02bde54d8a07cb91bc16_1186.pdf', 'source_sha256': '5de491ae5b64f1f592e6605653854ddfbfd42f9c173fa54c343a933aa9e557c1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1186.pdf', 'bytes': 1900468}, {'source_id': 'GDCA-F-5710d67c299ad2f693c2', 'source_path': 'sources/gdca-recon/2026-11-14/files/5710d67c299ad2f693c2_1014.pdf', 'source_sha256': '4285700e3feda7e8c8b0c2670c81b26cfdec5e7d653521ca04c9486e0dc6fc7e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1014.pdf', 'bytes': 1562919}, {'source_id': 'GDCA-F-86154a1836d099f1b841', 'source_path': 'sources/gdca-recon/2026-11-14/files/86154a1836d099f1b841_1164.pdf', 'source_sha256': '0640adb9f9e1274e9fe3ef9bfe0a4c265a24e4385f74d2e383bfa50fc2f1c2a5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1164.pdf', 'bytes': 1770122}, {'source_id': 'GDCA-F-ee5256d698180dbb9747', 'source_path': 'sources/gdca-recon/2026-11-14/files/ee5256d698180dbb9747_1073.pdf', 'source_sha256': '212228611bdd6c510862d6b20bb43727d70905df64b02279c3557eb231d2f36b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1073.pdf', 'bytes': 1786100}, {'source_id': 'GDCA-F-4b0ce3df4030bedd16a2', 'source_path': 'sources/gdca-recon/2026-11-14/files/4b0ce3df4030bedd16a2_1571.pdf', 'source_sha256': '2cd82e27f058c2989da791d6b68bb37d0c9bce546db3bf05760a4cc80ddf3e49', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1571.pdf', 'bytes': 2622770}, {'source_id': 'GDCA-F-6f1f5df07d0c6efb6ccf', 'source_path': 'sources/gdca-recon/2026-11-14/files/6f1f5df07d0c6efb6ccf_15.pdf', 'source_sha256': '00645bc209b0f56b2dcd7f3381460a27df69c3e00b446b3ba67e8d58d086e502', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/15.pdf', 'bytes': 2705435}, {'source_id': 'GDCA-F-89e759c54c7c1ff142be', 'source_path': 'sources/gdca-recon/2026-11-14/files/89e759c54c7c1ff142be_1223.pdf', 'source_sha256': 'f65738edcf2c1f3b531a231958c82730cc35788ee73fec24f73c623d918e0b64', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1223.pdf', 'bytes': 1876537}, {'source_id': 'GDCA-F-3f267ec1c550bfcae881', 'source_path': 'sources/gdca-recon/2026-11-14/files/3f267ec1c550bfcae881_1202.pdf', 'source_sha256': '2576b9eb3da30670df8f6f6675c7130647ccd4ab50c0fa1281c115c0c2ac4dff', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1202.pdf', 'bytes': 2063251}, {'source_id': 'GDCA-F-d70e84d3d9993f906822', 'source_path': 'sources/gdca-recon/2026-11-14/files/d70e84d3d9993f906822_567.pdf', 'source_sha256': '94d18025a8a6c7ab035294b10aa6ce88114e23d9231c8dba0f781764539d855c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/567.pdf', 'bytes': 1773755}, {'source_id': 'GDCA-F-3db4311a4279b22e728c', 'source_path': 'sources/gdca-recon/2026-11-14/files/3db4311a4279b22e728c_288.pdf', 'source_sha256': 'd152776f52016ed9120bab6e6bbed5705a12dcbd1adf3b6a2e21dbaf2e04a3e3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/288.pdf', 'bytes': 1778137}, {'source_id': 'GDCA-F-21fa001a27ddb7ec9248', 'source_path': 'sources/gdca-recon/2026-11-14/files/21fa001a27ddb7ec9248_1445.pdf', 'source_sha256': '2e0ff08daa4b729dca28bda0605498c53db4d2fe49f1efcb233b337f1c6785d1', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1445.pdf', 'bytes': 2663135}, {'source_id': 'GDCA-F-67f45b725886b703a505', 'source_path': 'sources/gdca-recon/2026-11-14/files/67f45b725886b703a505_528.pdf', 'source_sha256': '908ee1525131036c20593bca52d136ce644344302ae986d254e224f8732b38de', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/528.pdf', 'bytes': 2054050}, {'source_id': 'GDCA-F-9961bd9781d892e1953e', 'source_path': 'sources/gdca-recon/2026-11-14/files/9961bd9781d892e1953e_1210.pdf', 'source_sha256': 'a6536a056818242141b264eeb85fab25e2bd50b9e6ab865e5285e2fd1b45e6e1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1210.pdf', 'bytes': 1924736}, {'source_id': 'GDCA-F-e5b2e189b0c854aa4841', 'source_path': 'sources/gdca-recon/2026-11-14/files/e5b2e189b0c854aa4841_108.pdf', 'source_sha256': 'f7850c0bc9f5d8784108370cedb1bbedfa4735fcfd822891a9b136ff1bac30a3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/108.pdf', 'bytes': 1988495}, {'source_id': 'GDCA-F-19dd263fa87d178d3549', 'source_path': 'sources/gdca-recon/2026-11-14/files/19dd263fa87d178d3549_140.pdf', 'source_sha256': 'd65eff14d5d6d4ab30a3a71616b9162f4899d82c65b458eacf94b64a209de89e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/140.pdf', 'bytes': 2271224}, {'source_id': 'GDCA-F-5a4b435087791adfe6df', 'source_path': 'sources/gdca-recon/2026-11-14/files/5a4b435087791adfe6df_1282.pdf', 'source_sha256': 'dd77f33541b4d47f6cfd12bf4df12f9055e3a7d471a02cdda1d09cc899908f5a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1282.pdf', 'bytes': 1957274}, {'source_id': 'GDCA-F-f6e2e7d47719afa1d8d5', 'source_path': 'sources/gdca-recon/2026-11-14/files/f6e2e7d47719afa1d8d5_599.pdf', 'source_sha256': 'e3f05301c9af5a5172710d90af02307f489fefa695ef8f2b216acd7bc116f1a8', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/599.pdf', 'bytes': 1839303}, {'source_id': 'GDCA-F-b19123a380a7aa52d41b', 'source_path': 'sources/gdca-recon/2026-11-14/files/b19123a380a7aa52d41b_705.pdf', 'source_sha256': '084d6e48cdd28ed78fae2e845628555def10739e5f7278bb0a49d9a3884eeccf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/705.pdf', 'bytes': 1788955}, {'source_id': 'GDCA-F-831419374ad57811389c', 'source_path': 'sources/gdca-recon/2026-11-14/files/831419374ad57811389c_1283.pdf', 'source_sha256': 'b568065bb82f4149be99963796ca366204c2d3c966ecc3e0191038632ffcaeab', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1283.pdf', 'bytes': 2005325}, {'source_id': 'GDCA-F-404213335dc559029be9', 'source_path': 'sources/gdca-recon/2026-11-14/files/404213335dc559029be9_605.pdf', 'source_sha256': '428cbcc73a01d3df5024f0295314bffdf6c0badedb2da891d38337920caaa160', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/605.pdf', 'bytes': 2014389}, {'source_id': 'GDCA-F-3a2ad17c235cd31c40cb', 'source_path': 'sources/gdca-recon/2026-11-14/files/3a2ad17c235cd31c40cb_330.pdf', 'source_sha256': '0debd51418c146375eed2894329e1c05b5bcffee2464cf6b05457710802f74fb', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/330.pdf', 'bytes': 1995119}, {'source_id': 'GDCA-F-dcc29795e230b0d9b153', 'source_path': 'sources/gdca-recon/2026-11-14/files/dcc29795e230b0d9b153_475.pdf', 'source_sha256': 'b2af047ba89fe3f5150a5122d93d29cd64d6759d74830b55a1d2ac94f5d565f4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/475.pdf', 'bytes': 1829959}, {'source_id': 'GDCA-F-3e89ef863c69398caba8', 'source_path': 'sources/gdca-recon/2026-11-14/files/3e89ef863c69398caba8_1058.pdf', 'source_sha256': 'e37b10914e8c08f32edddb9e7f60b50e95a82b45101eb026b711c63bcd4d9ee7', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1058.pdf', 'bytes': 1983050}, {'source_id': 'GDCA-F-a0bcd7ede2b8a5c8f7bf', 'source_path': 'sources/gdca-recon/2026-11-14/files/a0bcd7ede2b8a5c8f7bf_1276.pdf', 'source_sha256': '12b08c29bde40e67e40b8254595998a49e1a24f277b98587715dc8777ec7ce6f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1276.pdf', 'bytes': 1826856}, {'source_id': 'GDCA-F-b31e7a054df2f1aecd22', 'source_path': 'sources/gdca-recon/2026-11-14/files/b31e7a054df2f1aecd22_92.pdf', 'source_sha256': 'c24797e0edd319bb9f59c9b58035cc9b26274509edf444d9fe433f880a507545', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/92.pdf', 'bytes': 1932120}, {'source_id': 'GDCA-F-5ca3517628504d7c89dd', 'source_path': 'sources/gdca-recon/2026-11-14/files/5ca3517628504d7c89dd_915.pdf', 'source_sha256': '767735255f389f832c4e2c30a4a7bbe9ee17295b1a6a14f05ed922848596222c', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/915.pdf', 'bytes': 1348357}, {'source_id': 'GDCA-F-49bf113d54410e794257', 'source_path': 'sources/gdca-recon/2026-11-14/files/49bf113d54410e794257_1124.pdf', 'source_sha256': '748935496307d85eea47525b82340ba9154cb28f79d47d0061bbf23afa18893d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1124.pdf', 'bytes': 2400320}, {'source_id': 'GDCA-F-cd466657fd877a8d3089', 'source_path': 'sources/gdca-recon/2026-11-14/files/cd466657fd877a8d3089_469.pdf', 'source_sha256': 'c6b15c3b2c2254f12fe266ee9e17fbe26b90ba9574de73046117f84099debc0d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/469.pdf', 'bytes': 1773147}, {'source_id': 'GDCA-F-e5b4c84aa37b40bf0364', 'source_path': 'sources/gdca-recon/2026-11-14/files/e5b4c84aa37b40bf0364_596.pdf', 'source_sha256': 'd0e81d45523bad2db0119c39297aac4e4f5590ab6ca395c626f9dacc3c01f3be', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/596.pdf', 'bytes': 1746082}, {'source_id': 'GDCA-F-4334e3c751eef8596ebe', 'source_path': 'sources/gdca-recon/2026-11-14/files/4334e3c751eef8596ebe_987.pdf', 'source_sha256': '1c9d399c2e59e269959bd73888a0c2d60b0323ace62e38bc9d45db89bd57aab2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/987.pdf', 'bytes': 1932578}, {'source_id': 'GDCA-F-8ffdce30bfdc8a6fd43f', 'source_path': 'sources/gdca-recon/2026-11-14/files/8ffdce30bfdc8a6fd43f_1551.pdf', 'source_sha256': '8a24b754d27a3d4bdf8e31b6c26ae5bff2faf7ebb751822e41fc2d90876ae49c', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1551.pdf', 'bytes': 3697399}, {'source_id': 'GDCA-F-b20d53c87b0e8ac128a9', 'source_path': 'sources/gdca-recon/2026-11-14/files/b20d53c87b0e8ac128a9_945.pdf', 'source_sha256': '535d6a40bb41be62e1258917dc784e7439ae8ace7f4500f89dcdaadf704020b5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/945.pdf', 'bytes': 1562200}, {'source_id': 'GDCA-F-f019d70d2b912aa25a56', 'source_path': 'sources/gdca-recon/2026-11-14/files/f019d70d2b912aa25a56_1411.pdf', 'source_sha256': 'b42ee112e6e40f10909af49a5c46a90e90fe63198b6be2804a447ea08f5634a1', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1411.pdf', 'bytes': 3151548}, {'source_id': 'GDCA-F-9e5793c5f0b8529cd41c', 'source_path': 'sources/gdca-recon/2026-11-14/files/9e5793c5f0b8529cd41c_342.pdf', 'source_sha256': '5440d0147131fd9e5569cc4565031b1ffa041020d9d7343e0ae7eaee9f9139ba', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/342.pdf', 'bytes': 2234512}, {'source_id': 'GDCA-F-80d14fbc5a635f8875c6', 'source_path': 'sources/gdca-recon/2026-11-14/files/80d14fbc5a635f8875c6_229.pdf', 'source_sha256': 'bbdc1318a99c0a55b569d301cc1f6b960b84ca543c71727b7a7d90a1d66fdcd5', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/229.pdf', 'bytes': 2199812}, {'source_id': 'GDCA-F-8cf3da0c7255276aa972', 'source_path': 'sources/gdca-recon/2026-11-14/files/8cf3da0c7255276aa972_250.pdf', 'source_sha256': '43bad91409cf3e17d8ada6125733d0df861c6335578759d12074064b94bd828c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/250.pdf', 'bytes': 2452781}, {'source_id': 'GDCA-F-a282008a0b42da4c15ec', 'source_path': 'sources/gdca-recon/2026-11-14/files/a282008a0b42da4c15ec_1212.pdf', 'source_sha256': '22ecb746be8d9d7073f1e582af76c2a6ed9b89c30d9339d1b4a0fffd5d26dae9', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1212.pdf', 'bytes': 1994374}, {'source_id': 'GDCA-F-c3964b8e59b2f3cdb5ef', 'source_path': 'sources/gdca-recon/2026-11-14/files/c3964b8e59b2f3cdb5ef_1417.pdf', 'source_sha256': '589d39c8e6bfa46366851d93262905adfdc92555dd9c6a514e78134a547c3cea', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1417.pdf', 'bytes': 2100197}, {'source_id': 'GDCA-F-b3c8b5f43971e9fbcc5d', 'source_path': 'sources/gdca-recon/2026-11-14/files/b3c8b5f43971e9fbcc5d_862.pdf', 'source_sha256': 'f333175fe78372f8e38182a82ae257513882a3ff36ad16b146d520f09c85fe66', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/862.pdf', 'bytes': 2759188}, {'source_id': 'GDCA-F-0388dd9b6b90d7c980e2', 'source_path': 'sources/gdca-recon/2026-11-14/files/0388dd9b6b90d7c980e2_550.pdf', 'source_sha256': '9cfb677e29ce9b290d540a772fe6b83b39191ff99d6e507528027b619f623c0b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/550.pdf', 'bytes': 1851999}, {'source_id': 'GDCA-F-29019f821cdf85717830', 'source_path': 'sources/gdca-recon/2026-11-14/files/29019f821cdf85717830_446.pdf', 'source_sha256': '42680e18d4368a9aae444abcb9fbf93054315dc4b8ba6af40ac42b6e16bc6d4d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/446.pdf', 'bytes': 1951647}, {'source_id': 'GDCA-F-049706217377a617f2f1', 'source_path': 'sources/gdca-recon/2026-11-14/files/049706217377a617f2f1_761.pdf', 'source_sha256': 'e1ce72f52dda8221b85eb7f692fda3c6535050f064c43444ad2b914b85bc997a', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/761.pdf', 'bytes': 2065538}, {'source_id': 'GDCA-F-2b1e78a2d0fe6e4bf0e8', 'source_path': 'sources/gdca-recon/2026-11-14/files/2b1e78a2d0fe6e4bf0e8_367.pdf', 'source_sha256': 'a8b8083e1adcb4160a28131d8f2975900cf18f6b1f8aae30ade176d13265f6dd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/367.pdf', 'bytes': 1948699}, {'source_id': 'GDCA-F-5edcd6b7c0d6fa1151c9', 'source_path': 'sources/gdca-recon/2026-11-14/files/5edcd6b7c0d6fa1151c9_345.pdf', 'source_sha256': 'e48fb5145fd3f7f2da06517d074c0906a251c5bd7686d702693d04899f6d4fe8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/345.pdf', 'bytes': 2095955}, {'source_id': 'GDCA-F-9aa6d8306825775d5ae2', 'source_path': 'sources/gdca-recon/2026-11-14/files/9aa6d8306825775d5ae2_1386.pdf', 'source_sha256': '14dce226c90a7aac00d4e3dace5997ac90559aefaa758d124014482b8f8a3091', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1386.pdf', 'bytes': 2211480}, {'source_id': 'GDCA-F-67613336cc78de159a72', 'source_path': 'sources/gdca-recon/2026-11-14/files/67613336cc78de159a72_1428.pdf', 'source_sha256': '8ba6569fec0931adac0f0e1d0b09adc3af50ffafda02bd89d50cb7bd1d193458', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1428.pdf', 'bytes': 2294900}, {'source_id': 'GDCA-F-066c64c2806003547160', 'source_path': 'sources/gdca-recon/2026-11-14/files/066c64c2806003547160_682.pdf', 'source_sha256': '1779127a5a70440e51093075c9f81ee958f252a4d48ae91edefc4819fbfb9af2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/682.pdf', 'bytes': 1721635}, {'source_id': 'GDCA-F-5f55bf73130a98944353', 'source_path': 'sources/gdca-recon/2026-11-14/files/5f55bf73130a98944353_71.pdf', 'source_sha256': 'cdaa259a59532bc189063783799bce5f0f5df5e16dd153887c8d2e1f6fc92c40', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/71.pdf', 'bytes': 1809969}, {'source_id': 'GDCA-F-0bdad7c536ae55ce9f26', 'source_path': 'sources/gdca-recon/2026-11-14/files/0bdad7c536ae55ce9f26_931.pdf', 'source_sha256': '92fcb769348d44bb469ed8601a7201e441b867531ff49f9d7a2308aa14b40ade', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/931.pdf', 'bytes': 2019317}, {'source_id': 'GDCA-F-8c452b28383ca65f192c', 'source_path': 'sources/gdca-recon/2026-11-14/files/8c452b28383ca65f192c_1559.pdf', 'source_sha256': 'e9d5e1e7af35035531c32209b95b21f1c6e7c7db01b34b74c7b91f6cdfc5edf7', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1559.pdf', 'bytes': 3116660}, {'source_id': 'GDCA-F-658be140e5a4e230d348', 'source_path': 'sources/gdca-recon/2026-11-14/files/658be140e5a4e230d348_1297.pdf', 'source_sha256': 'a46523480d54ef4d9c8652d69470e9f7afe4540e1dcee1738383ffd5f43ccdf7', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1297.pdf', 'bytes': 1878767}, {'source_id': 'GDCA-F-47262bb6acf67e13d4e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/47262bb6acf67e13d4e1_656.pdf', 'source_sha256': '0ac73892a22ea4a227421d0d7b2dff73155e95567136a0a2d814f9bad77f48b4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/656.pdf', 'bytes': 1972821}, {'source_id': 'GDCA-F-05ea25b484e93d4be628', 'source_path': 'sources/gdca-recon/2026-11-14/files/05ea25b484e93d4be628_1306.pdf', 'source_sha256': '200088aa1037eae03776000be3367f6fedc4e9c4831034f2800d20f675853597', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1306.pdf', 'bytes': 1904202}, {'source_id': 'GDCA-F-b7783daa4dfc812c94c2', 'source_path': 'sources/gdca-recon/2026-11-14/files/b7783daa4dfc812c94c2_47.pdf', 'source_sha256': 'e5ee8f2b5061dd1987f380a00080a50074278aaa779d7edd58e5b4081967535e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/47.pdf', 'bytes': 98674}, {'source_id': 'GDCA-F-b510825165b766bb8338', 'source_path': 'sources/gdca-recon/2026-11-14/files/b510825165b766bb8338_1129.pdf', 'source_sha256': '1155286c6b8d01a033b544f0fdf40e379c4d506687a96f5a5c3b7e267c6986e4', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1129.pdf', 'bytes': 2034497}, {'source_id': 'GDCA-F-38d4796b79502e06f423', 'source_path': 'sources/gdca-recon/2026-11-14/files/38d4796b79502e06f423_1500.pdf', 'source_sha256': '6491a6c692ca82b62b7d9e9be09746213a4f62fd2f8b91f17c6d141cff44e943', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1500.pdf', 'bytes': 1279946}, {'source_id': 'GDCA-F-812a196b2fb16702a3be', 'source_path': 'sources/gdca-recon/2026-11-14/files/812a196b2fb16702a3be_1137.pdf', 'source_sha256': '9063f1e21818202d2723ef8cc962113b683bc61b5a6542c486356ff8e7bc8f23', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1137.pdf', 'bytes': 2130906}, {'source_id': 'GDCA-F-48706ded13a7c717e845', 'source_path': 'sources/gdca-recon/2026-11-14/files/48706ded13a7c717e845_765.pdf', 'source_sha256': 'ca3d0e1ae66cbd871604e95e4c77683411988f8cca346ac9c4a8eccb87cefc03', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/765.pdf', 'bytes': 2064706}, {'source_id': 'GDCA-F-a72a595753910bbca466', 'source_path': 'sources/gdca-recon/2026-11-14/files/a72a595753910bbca466_107.pdf', 'source_sha256': 'a8fd25affc97562cf5ecf0e83084d2d92e73bc18129e1e5dc363d75c83ec10fd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/107.pdf', 'bytes': 2237007}, {'source_id': 'GDCA-F-051ba338ef78c9634f1d', 'source_path': 'sources/gdca-recon/2026-11-14/files/051ba338ef78c9634f1d_484.pdf', 'source_sha256': '43df35a5d419947a1570f1772df35a45f535817d3ddb9f56b065a455f9672d82', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/484.pdf', 'bytes': 1771429}, {'source_id': 'GDCA-F-942d41b765850c5a03fb', 'source_path': 'sources/gdca-recon/2026-11-14/files/942d41b765850c5a03fb_165.pdf', 'source_sha256': '25d6c0a5d399051bc676e3938977a99a48c96f53c372952fbd0bc85a0fd26240', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/165.pdf', 'bytes': 1808267}, {'source_id': 'GDCA-F-6524ed731a48cddeb143', 'source_path': 'sources/gdca-recon/2026-11-14/files/6524ed731a48cddeb143_863.pdf', 'source_sha256': '3eb60c4ac2e4a58ceb48eb4e2d11ce7628f5038b3d845040993cc17aa954ce1f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/863.pdf', 'bytes': 2135347}, {'source_id': 'GDCA-F-c68601037038a09ea70e', 'source_path': 'sources/gdca-recon/2026-11-14/files/c68601037038a09ea70e_38.pdf', 'source_sha256': 'fc190bde7f3afd326ffd9920256e8d955c91efd0d50cdcc8082e019dfd722136', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/38.pdf', 'bytes': 94992}, {'source_id': 'GDCA-F-ea470d350f9cdc0a4537', 'source_path': 'sources/gdca-recon/2026-11-14/files/ea470d350f9cdc0a4537_1195.pdf', 'source_sha256': '7f0003ad3a72fbc1bca4319901c30bf574b174b94d93ef03ff8727b0be877749', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1195.pdf', 'bytes': 1856700}, {'source_id': 'GDCA-F-8d0ae4950364b5b3f51a', 'source_path': 'sources/gdca-recon/2026-11-14/files/8d0ae4950364b5b3f51a_1040.pdf', 'source_sha256': '98514fee519a3a7437a21abfeb2fa086476e530e17727f19aa7cad04b84218cb', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1040.pdf', 'bytes': 2084212}, {'source_id': 'GDCA-F-c7287fd0b8f1e7c1bf09', 'source_path': 'sources/gdca-recon/2026-11-14/files/c7287fd0b8f1e7c1bf09_1209.pdf', 'source_sha256': '98946d18095760f739f19c4a300eb31cad7b235e7fdefe706540bd2afba9a436', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1209.pdf', 'bytes': 1741013}, {'source_id': 'GDCA-F-e6dd77851b17a33bfbb2', 'source_path': 'sources/gdca-recon/2026-11-14/files/e6dd77851b17a33bfbb2_338.pdf', 'source_sha256': '33dc64048c953c6082c3393c71ad87adc5afa8ff4245f0768f66eb1d262dd328', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/338.pdf', 'bytes': 2491961}, {'source_id': 'GDCA-F-f3d02f4e635212825204', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3d02f4e635212825204_538.pdf', 'source_sha256': 'a3236b4f4f4a8b6c77b3ad2785cd2426653dbeed720e2b0372e5720685188449', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/538.pdf', 'bytes': 1788984}, {'source_id': 'GDCA-F-f0c4fe44c2584561d146', 'source_path': 'sources/gdca-recon/2026-11-14/files/f0c4fe44c2584561d146_1383.pdf', 'source_sha256': '9e8c88e3d23c6cfd7ea08b670eb0545cb31ec179871519dbc68bbdfc332ac534', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1383.pdf', 'bytes': 2247645}, {'source_id': 'GDCA-F-01d4998b0c3af3686f41', 'source_path': 'sources/gdca-recon/2026-11-14/files/01d4998b0c3af3686f41_266.pdf', 'source_sha256': 'e81084efb0458d887ad212d60b7abe050f6746e4446e13e030c6e58bcbdf559f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/266.pdf', 'bytes': 1921908}, {'source_id': 'GDCA-F-380e9f0058e08204e9f5', 'source_path': 'sources/gdca-recon/2026-11-14/files/380e9f0058e08204e9f5_660.pdf', 'source_sha256': '17f600a9887bf2c9b1fe079b607ed5ff4e627991b2d87301c4ca9831562ef5c2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/660.pdf', 'bytes': 1905861}, {'source_id': 'GDCA-F-1b22c36a262492a5fa3b', 'source_path': 'sources/gdca-recon/2026-11-14/files/1b22c36a262492a5fa3b_684.pdf', 'source_sha256': '8b08dde41c56c9f754009bccd547ec69557d4e14d12b40e7c4c45068c1a76889', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/684.pdf', 'bytes': 2061339}, {'source_id': 'GDCA-F-9d58f5a901fa6b1d685e', 'source_path': 'sources/gdca-recon/2026-11-14/files/9d58f5a901fa6b1d685e_340.pdf', 'source_sha256': '07fdd7bdcf4b0fa0f43248fd2b80448b39f5c5e245808cf212d51dfc30731526', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/340.pdf', 'bytes': 2334400}, {'source_id': 'GDCA-F-f7f4a6d74b69dbde6584', 'source_path': 'sources/gdca-recon/2026-11-14/files/f7f4a6d74b69dbde6584_1133.pdf', 'source_sha256': 'c6faaa98e4f4cab66889f81eab66d4683a0458287140013cfbaf35c9636420b2', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1133.pdf', 'bytes': 2020642}, {'source_id': 'GDCA-F-f4c881234893d6f0d336', 'source_path': 'sources/gdca-recon/2026-11-14/files/f4c881234893d6f0d336_251.pdf', 'source_sha256': 'dbe926ad9fba648883d3ad05c4f429bbad9bfe90f69338377b496b555c0b9855', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/251.pdf', 'bytes': 1435679}, {'source_id': 'GDCA-F-5c17e95513936d98f699', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c17e95513936d98f699_445.pdf', 'source_sha256': '1098def502a3aa87a3773c11ff28da6d6f181719bc2d97e9330ad1141fc098dd', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/445.pdf', 'bytes': 2555118}, {'source_id': 'GDCA-F-da179b2a5b06b762f2cf', 'source_path': 'sources/gdca-recon/2026-11-14/files/da179b2a5b06b762f2cf_700.pdf', 'source_sha256': '90d5bc20e11859e7981b0654e3b256f694022a08743ca6f0a90503d8f03abb2c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/700.pdf', 'bytes': 2040792}, {'source_id': 'GDCA-F-66dd6a8742b4ae70675b', 'source_path': 'sources/gdca-recon/2026-11-14/files/66dd6a8742b4ae70675b_408.pdf', 'source_sha256': '685129428c64c05bc0755cad62398e0b2bbefe0c73d540ec4108bac385383db1', 'document_pages': 10, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/408.pdf', 'bytes': 4200703}, {'source_id': 'GDCA-F-dc74685f5d69c1cb6a7f', 'source_path': 'sources/gdca-recon/2026-11-14/files/dc74685f5d69c1cb6a7f_1234.pdf', 'source_sha256': '6c377330c5ab80fa0070f22ed6089b84c90a34d4b9e7c9311d175426b77d8e21', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1234.pdf', 'bytes': 1912377}, {'source_id': 'GDCA-F-d8b56dfa7fdcadcfe3c4', 'source_path': 'sources/gdca-recon/2026-11-14/files/d8b56dfa7fdcadcfe3c4_1232.pdf', 'source_sha256': '543983e02271f80fad3d04fb70104c5703437d343e935003bd5b43d434dd7895', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1232.pdf', 'bytes': 1986609}, {'source_id': 'GDCA-F-ebb2617e35a8308da48b', 'source_path': 'sources/gdca-recon/2026-11-14/files/ebb2617e35a8308da48b_1158.pdf', 'source_sha256': '6ce5bd6e5c24c0c288d7c74bdb77b5b6e2188bad2c6c03cff4033f7e7523077c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1158.pdf', 'bytes': 2043909}, {'source_id': 'GDCA-F-2989c20c673d0d399b53', 'source_path': 'sources/gdca-recon/2026-11-14/files/2989c20c673d0d399b53_55.pdf', 'source_sha256': 'c98f7fe42ef2fc7f4f5751bdeb434fcd69cbbcd92e36fc8ccdb8078dead160c4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/55.pdf', 'bytes': 1891136}, {'source_id': 'GDCA-F-a8a2d1bb397107afe9d9', 'source_path': 'sources/gdca-recon/2026-11-14/files/a8a2d1bb397107afe9d9_614.pdf', 'source_sha256': 'eeb0ed56b2705b523ef990ec46a3710514fc7e97a4b2f49ea4fa7ed6c67dc3d9', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/614.pdf', 'bytes': 2356436}, {'source_id': 'GDCA-F-7dc9b41ef1a2a085f7ba', 'source_path': 'sources/gdca-recon/2026-11-14/files/7dc9b41ef1a2a085f7ba_305.pdf', 'source_sha256': 'dcd789fcc29786fc7bf162fd4d15cb51534a9d2b8a824859ac20b8a1ef72d5df', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/305.pdf', 'bytes': 2721078}, {'source_id': 'GDCA-F-72becf4324b533b117a3', 'source_path': 'sources/gdca-recon/2026-11-14/files/72becf4324b533b117a3_1300.pdf', 'source_sha256': '2c9419d0a05b5986875d3a407e8cd5d9c164d7b7c70e71958c89ff85bb00b289', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1300.pdf', 'bytes': 2208488}, {'source_id': 'GDCA-F-1a7847acd322f65338cc', 'source_path': 'sources/gdca-recon/2026-11-14/files/1a7847acd322f65338cc_917.pdf', 'source_sha256': 'b061a99dce4279c500f673302652616b1837011810bdc409a24365748e9c64c1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/917.pdf', 'bytes': 1755517}, {'source_id': 'GDCA-F-50bd2ecc383b41547c17', 'source_path': 'sources/gdca-recon/2026-11-14/files/50bd2ecc383b41547c17_1028.pdf', 'source_sha256': 'bbf060da17984075430deab675076ce3c1a3de839835e77ac29d499dae050e0b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1028.pdf', 'bytes': 1785181}, {'source_id': 'GDCA-F-663cc0735124e971781b', 'source_path': 'sources/gdca-recon/2026-11-14/files/663cc0735124e971781b_623.pdf', 'source_sha256': 'fad4cb7d27eda6c14648e38393ff437a8d8ad0ecc93917eb73d841f08f542816', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/623.pdf', 'bytes': 1876703}, {'source_id': 'GDCA-F-56e374c25ed2ef61fd0c', 'source_path': 'sources/gdca-recon/2026-11-14/files/56e374c25ed2ef61fd0c_1320.pdf', 'source_sha256': 'e45fdbfecb4b25d32808d8a16cce1a379950d191690b37c321f5bbc41afd1c5d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1320.pdf', 'bytes': 2005922}, {'source_id': 'GDCA-F-09b26a7979fc73766037', 'source_path': 'sources/gdca-recon/2026-11-14/files/09b26a7979fc73766037_1012.pdf', 'source_sha256': '7334336e9f20d4f010c42e8b538cf29d0979642f9e650bbe4a42a864047be46b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1012.pdf', 'bytes': 2171149}, {'source_id': 'GDCA-F-86b9e7cac096d88d8f83', 'source_path': 'sources/gdca-recon/2026-11-14/files/86b9e7cac096d88d8f83_128.pdf', 'source_sha256': '11618497149c08c225a87a0f2c911869d5c269faa51acdd977871249f4529e69', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/128.pdf', 'bytes': 2248069}, {'source_id': 'GDCA-F-5086edc9884503ce63be', 'source_path': 'sources/gdca-recon/2026-11-14/files/5086edc9884503ce63be_216.pdf', 'source_sha256': '5c15a71fb56a626db6588b0538d55d671c19b96ec6b902a4b828352866c28a44', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/216.pdf', 'bytes': 1790283}, {'source_id': 'GDCA-F-5bb4c0d8557dc49ca47d', 'source_path': 'sources/gdca-recon/2026-11-14/files/5bb4c0d8557dc49ca47d_1021.pdf', 'source_sha256': '3e23b95a09e0886f8f2a5ec697dbd25b69012638c7e04cc76c35d69bf91751fd', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1021.pdf', 'bytes': 2039826}, {'source_id': 'GDCA-F-5debcf2be86c75efdaec', 'source_path': 'sources/gdca-recon/2026-11-14/files/5debcf2be86c75efdaec_4.pdf', 'source_sha256': 'b7dbb51dc4b066b704d4aca1970faa3f7be9891acd57afe46e9fafb21d6a976c', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/4.pdf', 'bytes': 3034840}, {'source_id': 'GDCA-F-a878ecf6fa78dff50b69', 'source_path': 'sources/gdca-recon/2026-11-14/files/a878ecf6fa78dff50b69_958.pdf', 'source_sha256': '47f2601c8a8ac2b8cb5be26adcf23f49056f96517801a99b415166677068abce', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/958.pdf', 'bytes': 2545514}, {'source_id': 'GDCA-F-4f051fa6b6be00690cf6', 'source_path': 'sources/gdca-recon/2026-11-14/files/4f051fa6b6be00690cf6_103.pdf', 'source_sha256': 'ba4943854ba96cdd329f51f1c9c98b8049f4e0042ff1ff1cdeec0a657e69b422', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/103.pdf', 'bytes': 2882369}, {'source_id': 'GDCA-F-d1f5f60dba41cfca48fd', 'source_path': 'sources/gdca-recon/2026-11-14/files/d1f5f60dba41cfca48fd_752.pdf', 'source_sha256': '0aeb0e17e2b079516ecf6edbbda3e131b9c000d7e60d8b72dbad1091286728a9', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/752.pdf', 'bytes': 1239270}, {'source_id': 'GDCA-F-024788700449ca6a92e0', 'source_path': 'sources/gdca-recon/2026-11-14/files/024788700449ca6a92e0_1485.pdf', 'source_sha256': 'c7e5c8cc628850571824732227fce84f6c222cc9d8e9ca36ed81b81905cd4638', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1485.pdf', 'bytes': 2227872}, {'source_id': 'GDCA-F-a7c2688ea5ed7b86af2d', 'source_path': 'sources/gdca-recon/2026-11-14/files/a7c2688ea5ed7b86af2d_1233.pdf', 'source_sha256': 'd82a02a41550bc433baf15f03e4dd460f03805524c930c3efded32b605528b7c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1233.pdf', 'bytes': 1759673}, {'source_id': 'GDCA-F-0dab2b324d1306db0091', 'source_path': 'sources/gdca-recon/2026-11-14/files/0dab2b324d1306db0091_211.pdf', 'source_sha256': '85fe83ed4fb499197818be5bf82e856b94b6c3a1a2c4d46bc9d5df7b6079a6e8', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/211.pdf', 'bytes': 2011149}, {'source_id': 'GDCA-F-e64fccfb1657ff83a62c', 'source_path': 'sources/gdca-recon/2026-11-14/files/e64fccfb1657ff83a62c_1114.pdf', 'source_sha256': '8157ef5fe2a394bc3bce8e6d934b5fa79f26df1667998f2ddda40aee9fe63c5c', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1114.pdf', 'bytes': 1987552}, {'source_id': 'GDCA-F-d5f5f7b4f6ecd1fe0f64', 'source_path': 'sources/gdca-recon/2026-11-14/files/d5f5f7b4f6ecd1fe0f64_1182.pdf', 'source_sha256': '95890a31648a9543eda87765a43234179c29002b250745dcb7e40fb7c5d763da', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1182.pdf', 'bytes': 2332261}, {'source_id': 'GDCA-F-9ab54b290589d2a0a686', 'source_path': 'sources/gdca-recon/2026-11-14/files/9ab54b290589d2a0a686_311.pdf', 'source_sha256': '5d78c5cc6623b1e107168a40dd1399bb34dc963993b69ee880c0d265ff95710f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/311.pdf', 'bytes': 2141961}, {'source_id': 'GDCA-F-7f387ec0d4774fbc4281', 'source_path': 'sources/gdca-recon/2026-11-14/files/7f387ec0d4774fbc4281_778.pdf', 'source_sha256': '64414724a59132bb86dac02823988035c5785a285037cf1789ad515d97caf421', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/778.pdf', 'bytes': 2042460}, {'source_id': 'GDCA-F-3ce8a0e8d16b74c0c6e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/3ce8a0e8d16b74c0c6e5_1088.pdf', 'source_sha256': '280fb796897c7d7548193aa3273e3abd99b033943739590302be810c78237e8f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1088.pdf', 'bytes': 1797898}, {'source_id': 'GDCA-F-7dc53afb70c52f891122', 'source_path': 'sources/gdca-recon/2026-11-14/files/7dc53afb70c52f891122_1294.pdf', 'source_sha256': '5d20a482d137ee32300399e4f71f479afdc13e88566b43dec0d906a047729b2c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1294.pdf', 'bytes': 2260505}, {'source_id': 'GDCA-F-8ba025361e9e2a1f2f6e', 'source_path': 'sources/gdca-recon/2026-11-14/files/8ba025361e9e2a1f2f6e_1299.pdf', 'source_sha256': '5dfb9ad98ea283dee67c16b5008f2b5096ac73aece1c45d86d48acbde23c7f3b', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1299.pdf', 'bytes': 1902498}, {'source_id': 'GDCA-F-d0f83569ef3bdfc5d89e', 'source_path': 'sources/gdca-recon/2026-11-14/files/d0f83569ef3bdfc5d89e_405.pdf', 'source_sha256': '13ca0e324bbe871dae3eb6d09b80a887e2b67e83861c24ad67bba30110a52fcf', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/405.pdf', 'bytes': 2160927}, {'source_id': 'GDCA-F-669f37d546098e3c039a', 'source_path': 'sources/gdca-recon/2026-11-14/files/669f37d546098e3c039a_911.pdf', 'source_sha256': '0308fa2d5cfa4aa8dfab146e31515af98d48e0885ef89c2c7bf7a23cb20cb81f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/911.pdf', 'bytes': 2058720}, {'source_id': 'GDCA-F-9ac5b5b0b934a9a126f5', 'source_path': 'sources/gdca-recon/2026-11-14/files/9ac5b5b0b934a9a126f5_1025.pdf', 'source_sha256': 'e9e91c83ed8e039c675867eafa7b06bbe8e9600468cdc5665c2ae0bf747b9cd1', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1025.pdf', 'bytes': 1924941}, {'source_id': 'GDCA-F-89f8df0cb205ca18c3cd', 'source_path': 'sources/gdca-recon/2026-11-14/files/89f8df0cb205ca18c3cd_506.pdf', 'source_sha256': '81fb8802339ebf8c1db78e17dbafa9fa3ae31bade4540a987489add5e849f6b5', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/506.pdf', 'bytes': 2011649}, {'source_id': 'GDCA-F-69ba2fa74cc2fd7be944', 'source_path': 'sources/gdca-recon/2026-11-14/files/69ba2fa74cc2fd7be944_179.pdf', 'source_sha256': '70e5a0044b10e4e00c32cc4ac56a1c48adbb305e37ac8ca701453f8b30755577', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/179.pdf', 'bytes': 2036262}, {'source_id': 'GDCA-F-65fa7553e3d1e495669f', 'source_path': 'sources/gdca-recon/2026-11-14/files/65fa7553e3d1e495669f_253.pdf', 'source_sha256': '64f65075f60507c98dd6c412a924a50908d8315a3790244a7371ac1777d881be', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/253.pdf', 'bytes': 2126343}, {'source_id': 'GDCA-F-e893e38b47222409a67d', 'source_path': 'sources/gdca-recon/2026-11-14/files/e893e38b47222409a67d_865.pdf', 'source_sha256': '50a7a2960174e4cf5401c4485923b7fc479c1bfe07609a93ba4f9b7caa002cf0', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/865.pdf', 'bytes': 1582058}, {'source_id': 'GDCA-F-f618bc1aa66c919ef4f1', 'source_path': 'sources/gdca-recon/2026-11-14/files/f618bc1aa66c919ef4f1_1174.pdf', 'source_sha256': 'd4a5b7462b0f4a7f1f932325a1b7dc03df37940bb2c464c1f15335278283d352', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1174.pdf', 'bytes': 6739781}, {'source_id': 'GDCA-F-015a1df20114a9aa316c', 'source_path': 'sources/gdca-recon/2026-11-14/files/015a1df20114a9aa316c_1561.pdf', 'source_sha256': '3cf90960fb5bcef560e205355ff2ca96a27cca4ea8fdf8f1848488846221b5e5', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1561.pdf', 'bytes': 3418653}, {'source_id': 'GDCA-F-9fbd46dafd081a118baf', 'source_path': 'sources/gdca-recon/2026-11-14/files/9fbd46dafd081a118baf_826.pdf', 'source_sha256': 'b72a46dbc1f6012fbb1cef0a2e444e5ccf16bb447b19dbbd5d3745527673a6c7', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/826.pdf', 'bytes': 2816898}, {'source_id': 'GDCA-F-bfa59463e3750fc63f49', 'source_path': 'sources/gdca-recon/2026-11-14/files/bfa59463e3750fc63f49_666.pdf', 'source_sha256': 'b9ebded4749bb82906d5a5734de85ff4cee154273d91726cfde2c30bac79ecef', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/666.pdf', 'bytes': 2216277}, {'source_id': 'GDCA-F-3709966e853bbe4b67fc', 'source_path': 'sources/gdca-recon/2026-11-14/files/3709966e853bbe4b67fc_811.pdf', 'source_sha256': 'c0577c1f12597d02cdf29e7167642b23f6ed43c131e2679fc6f1aea89eacaab3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/811.pdf', 'bytes': 2063325}, {'source_id': 'GDCA-F-94a4d6477760597edfbf', 'source_path': 'sources/gdca-recon/2026-11-14/files/94a4d6477760597edfbf_1041.pdf', 'source_sha256': 'a1a149802a4b12ea6eaa3e4fd89e843eea64eee1a649d4bfb8d235a4924713ea', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1041.pdf', 'bytes': 2169086}, {'source_id': 'GDCA-F-d7b75b513df62b496d22', 'source_path': 'sources/gdca-recon/2026-11-14/files/d7b75b513df62b496d22_186.pdf', 'source_sha256': '32f8b1bcbbb45ca325c2087229bb3d34b7263f8dd90a11057b1dc999e4dd9f8e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/186.pdf', 'bytes': 1875710}, {'source_id': 'GDCA-F-34eff81c9b0f5b94d036', 'source_path': 'sources/gdca-recon/2026-11-14/files/34eff81c9b0f5b94d036_763.pdf', 'source_sha256': '8cb0f5e7daa67e753ea32f02418672353aaf14d643241bd723ee7aca4a30fe90', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/763.pdf', 'bytes': 2106679}, {'source_id': 'GDCA-F-29890ee52ef84d598900', 'source_path': 'sources/gdca-recon/2026-11-14/files/29890ee52ef84d598900_417.pdf', 'source_sha256': 'f3501ca4fe56c93ae7c6700879b8af851967d761f57a03df36c888c54aeff18a', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/417.pdf', 'bytes': 2442505}, {'source_id': 'GDCA-F-29616d49e85723ee7967', 'source_path': 'sources/gdca-recon/2026-11-14/files/29616d49e85723ee7967_1442.pdf', 'source_sha256': '03bc15360118f34ff1b1fc12dd828c96f343b2578aa342743e463c090523e836', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1442.pdf', 'bytes': 2304807}, {'source_id': 'GDCA-F-b43296d56a4785766c59', 'source_path': 'sources/gdca-recon/2026-11-14/files/b43296d56a4785766c59_769.pdf', 'source_sha256': 'd2266fb4801b69050fc666019b32385a3c280567dd9926b0836d4f2dfa707c8d', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/769.pdf', 'bytes': 1713236}, {'source_id': 'GDCA-F-473064bad0a3ecb59352', 'source_path': 'sources/gdca-recon/2026-11-14/files/473064bad0a3ecb59352_13.pdf', 'source_sha256': '76ddb9666115f0b5c2485b3ace56f27dba639f8b8cfd34ff73aab2df509719b4', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/13.pdf', 'bytes': 2757934}, {'source_id': 'GDCA-F-6415f7ae88ca30a5ef08', 'source_path': 'sources/gdca-recon/2026-11-14/files/6415f7ae88ca30a5ef08_466.pdf', 'source_sha256': '28f4b2014fb40867d8613da0f9c5bb765111c9268be2e1439ed918d2f8cdc25b', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/466.pdf', 'bytes': 1223018}, {'source_id': 'GDCA-F-9066215a3414a8177ce6', 'source_path': 'sources/gdca-recon/2026-11-14/files/9066215a3414a8177ce6_988.pdf', 'source_sha256': '27d9a5cb98b40dfb5fbc285ee269e639040759bf7eeb64a63881e701cb2d6210', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/988.pdf', 'bytes': 1949083}, {'source_id': 'GDCA-F-690d376c6101c4cfe474', 'source_path': 'sources/gdca-recon/2026-11-14/files/690d376c6101c4cfe474_468.pdf', 'source_sha256': 'f3d92fbce5cbf9903b256d6f3a01608d283960dd4536f50e191cac0d2d28048d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/468.pdf', 'bytes': 1723986}, {'source_id': 'GDCA-F-0d9d68141cf90bfa39b4', 'source_path': 'sources/gdca-recon/2026-11-14/files/0d9d68141cf90bfa39b4_1379.pdf', 'source_sha256': 'd1ff954cfeba5faf4e5b12a00c7d210f529ab267da609cd745d8f1aff60b8f57', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1379.pdf', 'bytes': 2589446}, {'source_id': 'GDCA-F-92cdb008c842d16b0fc0', 'source_path': 'sources/gdca-recon/2026-11-14/files/92cdb008c842d16b0fc0_130.pdf', 'source_sha256': '7d7df8131943a5b8b383c6f18e10fa578be20431f42a6b70b29063c60809df8f', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/130.pdf', 'bytes': 2005028}, {'source_id': 'GDCA-F-7bfaa27f6dd8a920cdc2', 'source_path': 'sources/gdca-recon/2026-11-14/files/7bfaa27f6dd8a920cdc2_133.pdf', 'source_sha256': 'ce00db7db245eb334934097c8614447ef0c61cd30b3b35923ec0eada28e76e6d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/133.pdf', 'bytes': 2340528}, {'source_id': 'GDCA-F-6e5fc37db0a2b08e8fa6', 'source_path': 'sources/gdca-recon/2026-11-14/files/6e5fc37db0a2b08e8fa6_245.pdf', 'source_sha256': 'a98d40afd68d1e8bf006760ef79e5a22687db1c246287690d996865bd51d12f2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/245.pdf', 'bytes': 1840547}, {'source_id': 'GDCA-F-08919ab8bb87d8b1561e', 'source_path': 'sources/gdca-recon/2026-11-14/files/08919ab8bb87d8b1561e_744.pdf', 'source_sha256': 'd4ca26e8024f677474c0935c13795824fc33753bd4422b6478f672cc4932a312', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/744.pdf', 'bytes': 1842692}, {'source_id': 'GDCA-F-f99619c6b10442635985', 'source_path': 'sources/gdca-recon/2026-11-14/files/f99619c6b10442635985_1275.pdf', 'source_sha256': '2b96afb4f087dc174fac4f239d394c10a1d7879744ba2cd557f7c69510266ace', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1275.pdf', 'bytes': 1837699}, {'source_id': 'GDCA-F-8ac01d20b8e739f42cc3', 'source_path': 'sources/gdca-recon/2026-11-14/files/8ac01d20b8e739f42cc3_1314.pdf', 'source_sha256': 'b11bb5c3035b3d69a8073c4948cee57a1248bfab9146f4e040f09b981fcf6824', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1314.pdf', 'bytes': 2017643}, {'source_id': 'GDCA-F-9a59a50f880abff7dff1', 'source_path': 'sources/gdca-recon/2026-11-14/files/9a59a50f880abff7dff1_1038.pdf', 'source_sha256': 'c44de15c99df8a3d9c76e839e9898ddebab4b73f7b5fb9cd72671495bb3dab4c', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1038.pdf', 'bytes': 2145893}, {'source_id': 'GDCA-F-c4ba0e4f975634457508', 'source_path': 'sources/gdca-recon/2026-11-14/files/c4ba0e4f975634457508_375.pdf', 'source_sha256': '917055ed9faac66925f02c38925b1c62aa95884da14dd75154afe93331b74e3f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/375.pdf', 'bytes': 2136746}, {'source_id': 'GDCA-F-2874abeff6044b2b9c8c', 'source_path': 'sources/gdca-recon/2026-11-14/files/2874abeff6044b2b9c8c_1375.pdf', 'source_sha256': '6f065e346243f4c5b534e6171287f0c744da64e88166ed03663819b9f2acf8ce', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1375.pdf', 'bytes': 2598346}, {'source_id': 'GDCA-F-6a60b50d28329c639c36', 'source_path': 'sources/gdca-recon/2026-11-14/files/6a60b50d28329c639c36_842.pdf', 'source_sha256': '7856434dded5503f7125b39c75a8e80ae55700d6f06f641dd98992ddc0eaaf23', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/842.pdf', 'bytes': 2056112}, {'source_id': 'GDCA-F-18380aa82014f7c10d62', 'source_path': 'sources/gdca-recon/2026-11-14/files/18380aa82014f7c10d62_841.pdf', 'source_sha256': '6e6c09022d7ae96e322c2e74edef3715c51a27ba59dde1a85e95a0bb8c0e62fe', 'document_pages': 8, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/841.pdf', 'bytes': 3566613}, {'source_id': 'GDCA-F-d2190d49920b40e35883', 'source_path': 'sources/gdca-recon/2026-11-14/files/d2190d49920b40e35883_843.pdf', 'source_sha256': 'be35df52fa5ee1ad4084fe6cdf9e2c0eb23fe7f4308a9ac887f88b82f8c54702', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/843.pdf', 'bytes': 1973377}, {'source_id': 'GDCA-F-0ee8df6223fd94c2c962', 'source_path': 'sources/gdca-recon/2026-11-14/files/0ee8df6223fd94c2c962_844.pdf', 'source_sha256': '6c08af1a36acec39a0d28829d9045db267321413085f2cb501b7579bfee958bc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/844.pdf', 'bytes': 1913873}, {'source_id': 'GDCA-F-83c2208b189fb317f2e6', 'source_path': 'sources/gdca-recon/2026-11-14/files/83c2208b189fb317f2e6_846.pdf', 'source_sha256': 'd653cf406752d5e51d905d3789f920e06ef9d2d8a34fe4671920867f9a8c13d4', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/846.pdf', 'bytes': 1347203}, {'source_id': 'GDCA-F-9de1a8d4ff15dc90d464', 'source_path': 'sources/gdca-recon/2026-11-14/files/9de1a8d4ff15dc90d464_845.pdf', 'source_sha256': 'a7c58118e9576c99988ef55faa73d1634745415bbe83a569eb689b43c7de00c3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/845.pdf', 'bytes': 1897880}, {'source_id': 'GDCA-F-4788f9f8c0da8401fa1f', 'source_path': 'sources/gdca-recon/2026-11-14/files/4788f9f8c0da8401fa1f_847.pdf', 'source_sha256': '4f16abd131132e8b261d27ef248622b58c3640af01842015c4475e9ee10c5754', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/847.pdf', 'bytes': 1965043}, {'source_id': 'GDCA-F-6dac322c1b6250b36585', 'source_path': 'sources/gdca-recon/2026-11-14/files/6dac322c1b6250b36585_848.pdf', 'source_sha256': '91e8d6d358bf853c2c624060759349ab45988be72a8c4e02c440cf6231c50954', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/848.pdf', 'bytes': 2069901}, {'source_id': 'GDCA-F-ae1fb6916274ddc15937', 'source_path': 'sources/gdca-recon/2026-11-14/files/ae1fb6916274ddc15937_849.pdf', 'source_sha256': 'daa7b7a88f3a61672ae746bbd1d4377863994886c5eaff4da92f4e45d85cf1bc', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/849.pdf', 'bytes': 1842027}, {'source_id': 'GDCA-F-5eea01772577e236715f', 'source_path': 'sources/gdca-recon/2026-11-14/files/5eea01772577e236715f_850.pdf', 'source_sha256': '6133954cceff09712ba8490042c3994d96830ce4bfeb33d9500ee01ca8d554b9', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/850.pdf', 'bytes': 1757286}, {'source_id': 'GDCA-F-2f9d28141d4be72d77e5', 'source_path': 'sources/gdca-recon/2026-11-14/files/2f9d28141d4be72d77e5_851.pdf', 'source_sha256': 'f7eec8b5fb241dc5f173283db73b176b7bd1263de7219a806956f64cbf61a751', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/851.pdf', 'bytes': 1164373}, {'source_id': 'GDCA-F-d053dc2cf8d52d544166', 'source_path': 'sources/gdca-recon/2026-11-14/files/d053dc2cf8d52d544166_853.pdf', 'source_sha256': 'd0f21e78f244cb87f0190fdb1b8759e8dbab72ea3b93b9141ac8b30e0d617d4e', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/853.pdf', 'bytes': 2082339}, {'source_id': 'GDCA-F-3949a4956824a8bfd79c', 'source_path': 'sources/gdca-recon/2026-11-14/files/3949a4956824a8bfd79c_852.pdf', 'source_sha256': '1aa69f58378c3457580e48c3eac0707c209bbb5b2dff5c28ba6262e28a22f9be', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/852.pdf', 'bytes': 2542298}, {'source_id': 'GDCA-F-bf3589c49b4b179c7f1b', 'source_path': 'sources/gdca-recon/2026-11-14/files/bf3589c49b4b179c7f1b_1326.pdf', 'source_sha256': '2fd141776ae576545adb2f93164345caff8ecdf0827e0a7a09a20167f760b287', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1326.pdf', 'bytes': 1796854}, {'source_id': 'GDCA-F-94d9c1b400be4faf630e', 'source_path': 'sources/gdca-recon/2026-11-14/files/94d9c1b400be4faf630e_1327.pdf', 'source_sha256': '11e303bcec519ccb3956a3ab9b4d1ba777c07cc69c22aad0f718ae71867aa202', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1327.pdf', 'bytes': 1907344}, {'source_id': 'GDCA-F-1d53af2d3ca972e16013', 'source_path': 'sources/gdca-recon/2026-11-14/files/1d53af2d3ca972e16013_1328.pdf', 'source_sha256': 'c9556027580e63d56363103ee934e66370abcc8cc375b833fa442a3380852806', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1328.pdf', 'bytes': 1845831}, {'source_id': 'GDCA-F-73c23d545294a6145084', 'source_path': 'sources/gdca-recon/2026-11-14/files/73c23d545294a6145084_1329.pdf', 'source_sha256': '421f30dd1608af1c0290c46304c3ef9e9ecdfc07ae6b0f671c0ed998fb94e465', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1329.pdf', 'bytes': 1922953}, {'source_id': 'GDCA-F-4dea4b9e2862e1720945', 'source_path': 'sources/gdca-recon/2026-11-14/files/4dea4b9e2862e1720945_1330.pdf', 'source_sha256': 'f5553aa4dd8e8745161afffc6e2354999898b85fd538c87e6dd88c2006e81c26', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1330.pdf', 'bytes': 2027358}, {'source_id': 'GDCA-F-7948463195d7462b7c6d', 'source_path': 'sources/gdca-recon/2026-11-14/files/7948463195d7462b7c6d_1331.pdf', 'source_sha256': '8ac8d357689fbed21afe73ec823c04d8fb184424a1f62b20d0ed356c6c7dfb38', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1331.pdf', 'bytes': 2035248}, {'source_id': 'GDCA-F-f5a0459955c97ab1b265', 'source_path': 'sources/gdca-recon/2026-11-14/files/f5a0459955c97ab1b265_1333.pdf', 'source_sha256': '49421d75691779d1b4ebaa551f90095db9d48a28aa33c2342771a5c31ec7e99b', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1333.pdf', 'bytes': 2620299}, {'source_id': 'GDCA-F-e40c762f3901d755985d', 'source_path': 'sources/gdca-recon/2026-11-14/files/e40c762f3901d755985d_1332.pdf', 'source_sha256': '9ac79db2205b6562a27fd33e6e59651c143ddbc2debd7ca9c9ecaf4d77b16868', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1332.pdf', 'bytes': 2124510}, {'source_id': 'GDCA-F-40f489d257c1026d1d1c', 'source_path': 'sources/gdca-recon/2026-11-14/files/40f489d257c1026d1d1c_1334.pdf', 'source_sha256': 'c9abd6c7f351180d8fb3828cc061496bd53abb5959790aeac4a881e6ee841812', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1334.pdf', 'bytes': 1923282}, {'source_id': 'GDCA-F-d18c664811ac3f20377a', 'source_path': 'sources/gdca-recon/2026-11-14/files/d18c664811ac3f20377a_1336.pdf', 'source_sha256': 'a2047fad4b865f211caf15707d48ee5b7056842766f13af40cc0f04a7e269857', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1336.pdf', 'bytes': 1862744}, {'source_id': 'GDCA-F-dd6fb75d975ea86379e6', 'source_path': 'sources/gdca-recon/2026-11-14/files/dd6fb75d975ea86379e6_1335.pdf', 'source_sha256': '0039ab5d6d4dba9e5e34578b82a382ba7a67d46a45a9860a4e69b8652ce88432', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1335.pdf', 'bytes': 1941327}, {'source_id': 'GDCA-F-cd1f363ddf7e8f85270c', 'source_path': 'sources/gdca-recon/2026-11-14/files/cd1f363ddf7e8f85270c_1337.pdf', 'source_sha256': 'fc90506e16f144f111fdad81beeb5f8f52d237751780e58e90b3337090a642d2', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1337.pdf', 'bytes': 1933647}, {'source_id': 'GDCA-F-063c0e107dc70df577c5', 'source_path': 'sources/gdca-recon/2026-11-14/files/063c0e107dc70df577c5_1338.pdf', 'source_sha256': '25582c53c364b11c077ee34c716575317b190aafc8cbd53cab06d0c8f31bb2b4', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1338.pdf', 'bytes': 1831867}, {'source_id': 'GDCA-F-e445378b1c191b08c9ec', 'source_path': 'sources/gdca-recon/2026-11-14/files/e445378b1c191b08c9ec_1339.pdf', 'source_sha256': '21908e3eabfa5369951ab98a2397bc9e8900307afd8a0aa5c4ed89c4399a69a3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1339.pdf', 'bytes': 1901368}, {'source_id': 'GDCA-F-349a9b108fc38be335ad', 'source_path': 'sources/gdca-recon/2026-11-14/files/349a9b108fc38be335ad_1341.pdf', 'source_sha256': 'bfc1fc916cb58b905458a2315ee07b1aaa72cc8ad89baddb0d367a1bddd23bcd', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1341.pdf', 'bytes': 1760820}, {'source_id': 'GDCA-F-5fc3003da65858a9aa71', 'source_path': 'sources/gdca-recon/2026-11-14/files/5fc3003da65858a9aa71_1340.pdf', 'source_sha256': 'cf1367a74d7931e7652213d010c54fa3f183bde3f1aa5933b34145868248df94', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1340.pdf', 'bytes': 1885911}, {'source_id': 'GDCA-F-8d152720b231d03f23e3', 'source_path': 'sources/gdca-recon/2026-11-14/files/8d152720b231d03f23e3_1342.pdf', 'source_sha256': '43eff42e3113bca214ac5efabeef787defef2d22bf72e69f54fc77d5ceaa5d1f', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1342.pdf', 'bytes': 2046415}, {'source_id': 'GDCA-F-19889f96c8a9a28f7e9d', 'source_path': 'sources/gdca-recon/2026-11-14/files/19889f96c8a9a28f7e9d_1343.pdf', 'source_sha256': 'f3e33ecc1764b4864d2001fde4bb3558e40250e4105fc019c1f4527a111623f3', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1343.pdf', 'bytes': 1317195}, {'source_id': 'GDCA-F-a4a6deebd1927daa98ae', 'source_path': 'sources/gdca-recon/2026-11-14/files/a4a6deebd1927daa98ae_1344.pdf', 'source_sha256': '255b0173421ed01a013f2a2a046b1763defca767de37eeb4bca725be45931eb3', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1344.pdf', 'bytes': 1708683}, {'source_id': 'GDCA-F-24a6dd6d44c6cec7669e', 'source_path': 'sources/gdca-recon/2026-11-14/files/24a6dd6d44c6cec7669e_1345.pdf', 'source_sha256': '40529d20444ecad755a301d27b1f4c28055b18fc2ddee4828922ad7f1bda76a3', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1345.pdf', 'bytes': 2121137}, {'source_id': 'GDCA-F-263b7ca94553d9fa239c', 'source_path': 'sources/gdca-recon/2026-11-14/files/263b7ca94553d9fa239c_1346.pdf', 'source_sha256': 'e71d56ce4d6bad71474dd3c7be94ffd7533df10e96ca5a02e91f8624c1edb8f0', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1346.pdf', 'bytes': 2345541}, {'source_id': 'GDCA-F-5391963dc07ac365ea7f', 'source_path': 'sources/gdca-recon/2026-11-14/files/5391963dc07ac365ea7f_1348.pdf', 'source_sha256': 'd23859d17fdf124d6050054596cb5016f5e6beb0417c22ab8ae8e990a8a565ea', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1348.pdf', 'bytes': 1837312}, {'source_id': 'GDCA-F-bee3e7113470f978b036', 'source_path': 'sources/gdca-recon/2026-11-14/files/bee3e7113470f978b036_1347.pdf', 'source_sha256': '65d70c450a9da6225b23b7d6bed00ff85882202fef449c5c926762e6d03ec5b4', 'document_pages': 7, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1347.pdf', 'bytes': 2940658}, {'source_id': 'GDCA-F-bafea78e704cbe99794f', 'source_path': 'sources/gdca-recon/2026-11-14/files/bafea78e704cbe99794f_1349.pdf', 'source_sha256': '3bd84b834540d7abef01bd47948583de5162212c1dad5612460c7b7d9731fd0a', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1349.pdf', 'bytes': 1864145}, {'source_id': 'GDCA-F-a97c8b1db14bb0f45b76', 'source_path': 'sources/gdca-recon/2026-11-14/files/a97c8b1db14bb0f45b76_1351.pdf', 'source_sha256': 'db4e1f2757cb42fea7c2f96a88f8fa0141f92af15f2c58706c08e99bc1548da6', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1351.pdf', 'bytes': 1707356}, {'source_id': 'GDCA-F-e9014ad5a959ef4f0745', 'source_path': 'sources/gdca-recon/2026-11-14/files/e9014ad5a959ef4f0745_1350.pdf', 'source_sha256': '98b6fd8c05d8b3f1fe1b9d8a9056d82948b0309755894dc7b0f45cd272ee185d', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1350.pdf', 'bytes': 1751944}, {'source_id': 'GDCA-F-59e88f9791d4bc7d3f4d', 'source_path': 'sources/gdca-recon/2026-11-14/files/59e88f9791d4bc7d3f4d_1352.pdf', 'source_sha256': '63cdea9ce108d71b6e11b4e780efe34cf73c8128738ae526b0b7964221653663', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1352.pdf', 'bytes': 2119741}, {'source_id': 'GDCA-F-1e71a2b19d8728b2883f', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e71a2b19d8728b2883f_1353.pdf', 'source_sha256': '17e4e9c2c450c0be8dfd80a4d5355843d800f09e80b6f9708628618ba8233242', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1353.pdf', 'bytes': 1909741}, {'source_id': 'GDCA-F-2d5fec3bfed173411fc7', 'source_path': 'sources/gdca-recon/2026-11-14/files/2d5fec3bfed173411fc7_1354.pdf', 'source_sha256': 'b8a2fa0007d9697bdd92feb16555cb9458b3c1b3e208997fa11f0afe8a6cd739', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1354.pdf', 'bytes': 1688739}, {'source_id': 'GDCA-F-bacb89a77be3d90d2cb7', 'source_path': 'sources/gdca-recon/2026-11-14/files/bacb89a77be3d90d2cb7_1357.pdf', 'source_sha256': '933214f9e95ea9658045a8be77cc66d971b1872e9bf286b4e46f82157a8735e2', 'document_pages': 2, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1357.pdf', 'bytes': 935204}, {'source_id': 'GDCA-F-bfce43b6ba7d3e433f7a', 'source_path': 'sources/gdca-recon/2026-11-14/files/bfce43b6ba7d3e433f7a_1355.pdf', 'source_sha256': 'd19c05d19b356586db5b7f0e3469d6c7780825eef27f2929eae4ab0c33bd14ec', 'document_pages': 3, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1355.pdf', 'bytes': 1394180}, {'source_id': 'GDCA-F-9441c330820393894c0b', 'source_path': 'sources/gdca-recon/2026-11-14/files/9441c330820393894c0b_1356.pdf', 'source_sha256': 'a2d9ca1331a06947eda2dbd06ededdc82336a75dd62769f6feae022b749aba0e', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1356.pdf', 'bytes': 1685692}, {'source_id': 'GDCA-F-1bac28dbcf342060baf2', 'source_path': 'sources/gdca-recon/2026-11-14/files/1bac28dbcf342060baf2_1358.pdf', 'source_sha256': '6d885cbcf8d6a60358dc4d75e68726802b37da8797a4e18e095c227a43808653', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1358.pdf', 'bytes': 2573646}, {'source_id': 'GDCA-F-0d6681b9d39f9c241057', 'source_path': 'sources/gdca-recon/2026-11-14/files/0d6681b9d39f9c241057_1359.pdf', 'source_sha256': '59483a667d9f9232bb723c97c1226297dd498b3f9b8afbaaada83ee6b6e30697', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1359.pdf', 'bytes': 1839504}, {'source_id': 'GDCA-F-13a4736ba8e15f26441d', 'source_path': 'sources/gdca-recon/2026-11-14/files/13a4736ba8e15f26441d_1360.pdf', 'source_sha256': 'cb6817e338f66239b09484d7be9df0369f0d37c281b68a9f3f08a5b8c8bbb9be', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1360.pdf', 'bytes': 1881841}, {'source_id': 'GDCA-F-56c9effb563d6d811208', 'source_path': 'sources/gdca-recon/2026-11-14/files/56c9effb563d6d811208_1362.pdf', 'source_sha256': '05dbb6f886840aad073590015aaa165b86dc0cf7b169d5c080beca45a04352ec', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1362.pdf', 'bytes': 2114354}, {'source_id': 'GDCA-F-bd280f260e495612e986', 'source_path': 'sources/gdca-recon/2026-11-14/files/bd280f260e495612e986_1361.pdf', 'source_sha256': '3d87ae286cf48bc3ebf7557a1d007d9d50169966fd6c33abdd182d48943c355b', 'document_pages': 5, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1361.pdf', 'bytes': 2017785}, {'source_id': 'GDCA-F-f7dbe05436f5f1366831', 'source_path': 'sources/gdca-recon/2026-11-14/files/f7dbe05436f5f1366831_1363.pdf', 'source_sha256': 'a321a52330a4e88035c2ef08a0b91c7ed85fcce9d709155e58c2915ff3783c20', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1363.pdf', 'bytes': 2239058}, {'source_id': 'GDCA-F-1578e0d110331576ca13', 'source_path': 'sources/gdca-recon/2026-11-14/files/1578e0d110331576ca13_1366.pdf', 'source_sha256': '2dbf4d3e405c1a61e0d54a530a2d51084eba9b9ee0dc9c758e34afb1f84ad51f', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1366.pdf', 'bytes': 2214746}, {'source_id': 'GDCA-F-1e48a98887f392c064bb', 'source_path': 'sources/gdca-recon/2026-11-14/files/1e48a98887f392c064bb_1367.pdf', 'source_sha256': 'a3e055a30c78a484607d5e67a0f84826ad46102733c92888550a2590a7bd0480', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1367.pdf', 'bytes': 1394598}, {'source_id': 'GDCA-F-aaf095395abf2c51ab81', 'source_path': 'sources/gdca-recon/2026-11-14/files/aaf095395abf2c51ab81_1365.pdf', 'source_sha256': 'ac026b210fcac08fa1c3afb2d78b103618d397cf5c10add496cabbfef517917d', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1365.pdf', 'bytes': 2400884}, {'source_id': 'GDCA-F-a220ec59d1d0923c20c5', 'source_path': 'sources/gdca-recon/2026-11-14/files/a220ec59d1d0923c20c5_1368.pdf', 'source_sha256': '7308263f7f901fe6097d9edeb530de71aeba2c0b1016bac007841e9c1be43c76', 'document_pages': 6, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1368.pdf', 'bytes': 2541434}, {'source_id': 'GDCA-F-a149652b0cb5de579b0e', 'source_path': 'sources/gdca-recon/2026-11-14/files/a149652b0cb5de579b0e_1392.pdf', 'source_sha256': '50f43fcb145ec5e210e53bf34fad66ece58bb6e5598224d8596b0dca5ad60639', 'document_pages': 4, 'url': 'https://recon.gdca.gov.sy/upload/pdfs/1392.pdf', 'bytes': 1915039}]
PILOT = [{'source_id': 'GDCA-F-21f013fb092490ebed6a', 'source_path': 'sources/gdca-recon/2026-11-14/files/21f013fb092490ebed6a_39.pdf', 'source_sha256': 'd113eff7e5a2c5887fd85ba5d112f28685bbf1669a337f4dda1ffc81ce7a72b7', 'page': 1, 'document_pages': 5}, {'source_id': 'GDCA-F-21f013fb092490ebed6a', 'source_path': 'sources/gdca-recon/2026-11-14/files/21f013fb092490ebed6a_39.pdf', 'source_sha256': 'd113eff7e5a2c5887fd85ba5d112f28685bbf1669a337f4dda1ffc81ce7a72b7', 'page': 3, 'document_pages': 5}, {'source_id': 'GDCA-F-21f013fb092490ebed6a', 'source_path': 'sources/gdca-recon/2026-11-14/files/21f013fb092490ebed6a_39.pdf', 'source_sha256': 'd113eff7e5a2c5887fd85ba5d112f28685bbf1669a337f4dda1ffc81ce7a72b7', 'page': 5, 'document_pages': 5}, {'source_id': 'GDCA-F-5eea01772577e236715f', 'source_path': 'sources/gdca-recon/2026-11-14/files/5eea01772577e236715f_850.pdf', 'source_sha256': '6133954cceff09712ba8490042c3994d96830ce4bfeb33d9500ee01ca8d554b9', 'page': 1, 'document_pages': 4}, {'source_id': 'GDCA-F-5eea01772577e236715f', 'source_path': 'sources/gdca-recon/2026-11-14/files/5eea01772577e236715f_850.pdf', 'source_sha256': '6133954cceff09712ba8490042c3994d96830ce4bfeb33d9500ee01ca8d554b9', 'page': 2, 'document_pages': 4}, {'source_id': 'GDCA-F-5eea01772577e236715f', 'source_path': 'sources/gdca-recon/2026-11-14/files/5eea01772577e236715f_850.pdf', 'source_sha256': '6133954cceff09712ba8490042c3994d96830ce4bfeb33d9500ee01ca8d554b9', 'page': 4, 'document_pages': 4}, {'source_id': 'GDCA-F-5c967522121de0cfdbfb', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c967522121de0cfdbfb_270.pdf', 'source_sha256': 'd420c2f014a4fcc287e59e5c8c0825c20ecf1bb7155117412d157e9cc145aab2', 'page': 1, 'document_pages': 4}, {'source_id': 'GDCA-F-5c967522121de0cfdbfb', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c967522121de0cfdbfb_270.pdf', 'source_sha256': 'd420c2f014a4fcc287e59e5c8c0825c20ecf1bb7155117412d157e9cc145aab2', 'page': 2, 'document_pages': 4}, {'source_id': 'GDCA-F-5c967522121de0cfdbfb', 'source_path': 'sources/gdca-recon/2026-11-14/files/5c967522121de0cfdbfb_270.pdf', 'source_sha256': 'd420c2f014a4fcc287e59e5c8c0825c20ecf1bb7155117412d157e9cc145aab2', 'page': 4, 'document_pages': 4}, {'source_id': 'GDCA-F-f9394133fb5f2ed02665', 'source_path': 'sources/gdca-recon/2026-11-14/files/f9394133fb5f2ed02665_944.pdf', 'source_sha256': '85b521262f83ff044b902b0642d514583280c104101dcebd0d4f48ae1de005b7', 'page': 1, 'document_pages': 5}, {'source_id': 'GDCA-F-f9394133fb5f2ed02665', 'source_path': 'sources/gdca-recon/2026-11-14/files/f9394133fb5f2ed02665_944.pdf', 'source_sha256': '85b521262f83ff044b902b0642d514583280c104101dcebd0d4f48ae1de005b7', 'page': 3, 'document_pages': 5}, {'source_id': 'GDCA-F-f9394133fb5f2ed02665', 'source_path': 'sources/gdca-recon/2026-11-14/files/f9394133fb5f2ed02665_944.pdf', 'source_sha256': '85b521262f83ff044b902b0642d514583280c104101dcebd0d4f48ae1de005b7', 'page': 5, 'document_pages': 5}, {'source_id': 'GDCA-F-51f8f6d532398d9c79e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/51f8f6d532398d9c79e1_893.pdf', 'source_sha256': '274fe2c5b164d72821f53b7a0be0721740fb0a0592891ddb1793eae71f8c07e0', 'page': 1, 'document_pages': 5}, {'source_id': 'GDCA-F-51f8f6d532398d9c79e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/51f8f6d532398d9c79e1_893.pdf', 'source_sha256': '274fe2c5b164d72821f53b7a0be0721740fb0a0592891ddb1793eae71f8c07e0', 'page': 3, 'document_pages': 5}, {'source_id': 'GDCA-F-51f8f6d532398d9c79e1', 'source_path': 'sources/gdca-recon/2026-11-14/files/51f8f6d532398d9c79e1_893.pdf', 'source_sha256': '274fe2c5b164d72821f53b7a0be0721740fb0a0592891ddb1793eae71f8c07e0', 'page': 5, 'document_pages': 5}, {'source_id': 'GDCA-F-19f24ba0b64c033d4a68', 'source_path': 'sources/gdca-recon/2026-11-14/files/19f24ba0b64c033d4a68_366.pdf', 'source_sha256': '21dc02748e6659f278c68711417b6c9dc5c35923020ef9dcf67051e2c4f76678', 'page': 1, 'document_pages': 5}, {'source_id': 'GDCA-F-19f24ba0b64c033d4a68', 'source_path': 'sources/gdca-recon/2026-11-14/files/19f24ba0b64c033d4a68_366.pdf', 'source_sha256': '21dc02748e6659f278c68711417b6c9dc5c35923020ef9dcf67051e2c4f76678', 'page': 3, 'document_pages': 5}, {'source_id': 'GDCA-F-19f24ba0b64c033d4a68', 'source_path': 'sources/gdca-recon/2026-11-14/files/19f24ba0b64c033d4a68_366.pdf', 'source_sha256': '21dc02748e6659f278c68711417b6c9dc5c35923020ef9dcf67051e2c4f76678', 'page': 5, 'document_pages': 5}, {'source_id': 'GDCA-F-86f70605b3b56c3f7de3', 'source_path': 'sources/gdca-recon/2026-11-14/files/86f70605b3b56c3f7de3_435.pdf', 'source_sha256': '3fadd658847b7326d1bbaac2bdf08aacbe996696f60f83864fec4fb28b499935', 'page': 1, 'document_pages': 5}, {'source_id': 'GDCA-F-86f70605b3b56c3f7de3', 'source_path': 'sources/gdca-recon/2026-11-14/files/86f70605b3b56c3f7de3_435.pdf', 'source_sha256': '3fadd658847b7326d1bbaac2bdf08aacbe996696f60f83864fec4fb28b499935', 'page': 3, 'document_pages': 5}, {'source_id': 'GDCA-F-86f70605b3b56c3f7de3', 'source_path': 'sources/gdca-recon/2026-11-14/files/86f70605b3b56c3f7de3_435.pdf', 'source_sha256': '3fadd658847b7326d1bbaac2bdf08aacbe996696f60f83864fec4fb28b499935', 'page': 5, 'document_pages': 5}, {'source_id': 'GDCA-F-f3031731ed58f986f672', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3031731ed58f986f672_425.pdf', 'source_sha256': '6a2120a049b25bc744bc1e82f644c91feced6cd2dbf47bae1c8dc08b1912da89', 'page': 1, 'document_pages': 6}, {'source_id': 'GDCA-F-f3031731ed58f986f672', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3031731ed58f986f672_425.pdf', 'source_sha256': '6a2120a049b25bc744bc1e82f644c91feced6cd2dbf47bae1c8dc08b1912da89', 'page': 3, 'document_pages': 6}, {'source_id': 'GDCA-F-f3031731ed58f986f672', 'source_path': 'sources/gdca-recon/2026-11-14/files/f3031731ed58f986f672_425.pdf', 'source_sha256': '6a2120a049b25bc744bc1e82f644c91feced6cd2dbf47bae1c8dc08b1912da89', 'page': 6, 'document_pages': 6}, {'source_id': 'GDCA-F-909bb9805469d7713707', 'source_path': 'sources/gdca-recon/2026-11-14/files/909bb9805469d7713707_1457.pdf', 'source_sha256': '211e74d53bb72a5f8387ab304b31ab1e5b328dc9b14b89be50118e2826e5739c', 'page': 1, 'document_pages': 7}, {'source_id': 'GDCA-F-909bb9805469d7713707', 'source_path': 'sources/gdca-recon/2026-11-14/files/909bb9805469d7713707_1457.pdf', 'source_sha256': '211e74d53bb72a5f8387ab304b31ab1e5b328dc9b14b89be50118e2826e5739c', 'page': 4, 'document_pages': 7}, {'source_id': 'GDCA-F-909bb9805469d7713707', 'source_path': 'sources/gdca-recon/2026-11-14/files/909bb9805469d7713707_1457.pdf', 'source_sha256': '211e74d53bb72a5f8387ab304b31ab1e5b328dc9b14b89be50118e2826e5739c', 'page': 7, 'document_pages': 7}]
BASE_CONFIG = {'schema_version': 1, 'surya_version': '0.24.0', 'model_repo': 'datalab-to/surya-ocr-2.5-gguf', 'model_revision': 'eb78671ae9af9bdddfac2e25507c55337b6098e2', 'model_file': 'surya-2.5-Q8_0.gguf', 'projector_file': 'surya-2.5-mmproj.gguf', 'llama_release': 'b12403', 'llama_archive_url': 'https://github.com/ggml-org/llama.cpp/releases/download/b12403/llama-b12403-bin-win-cpu-x64.zip', 'llama_archive_sha256': '8ac66c6b8707d104f8564673d8e12d0f3c5564784f3494cb1fb3063e8eec6070', 'dpi': 192, 'threads': 4, 'parallel_pages': 1, 'context_tokens': 24576, 'max_output_tokens': 12288, 'page_timeout_seconds': 900, 'archive': 'sources/gdca-recon/2026-11-14', 'output': 'evidence/extracts/gdca-surya'}
MODELS = [{'path': 'ocr/runtime/model/surya-2.5-Q8_0.gguf', 'sha256': '5ab34c76845f66d2882ec5c89735894b700e0a7decd08898ab962b2afb9aca57', 'bytes': 628680096, 'url': 'https://huggingface.co/datalab-to/surya-ocr-2.5-gguf/resolve/eb78671ae9af9bdddfac2e25507c55337b6098e2/surya-2.5-Q8_0.gguf'}, {'path': 'ocr/runtime/model/surya-2.5-mmproj.gguf', 'sha256': 'f245dbf784cf73683a273a37fefae0d0d7983b083a5d2fecf9309476f861f659', 'bytes': 207345920, 'url': 'https://huggingface.co/datalab-to/surya-ocr-2.5-gguf/resolve/eb78671ae9af9bdddfac2e25507c55337b6098e2/surya-2.5-mmproj.gguf'}]
WORKER_SOURCE = 'from pathlib import Path\nimport json, hashlib, os\nimport time\nfrom datetime import datetime, timezone\nROOT=Path(os.environ["GDCA_OCR_ROOT"]).resolve()\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\n\ndef digest(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    os.replace(temporary, path)\n\ndef safe_path(value):\n    path = (ROOT / value).resolve()\n    if not path.is_relative_to(ROOT):\n        raise ValueError(\'Path outside project\')\n    return path\n\ndef worker(job_path, port):\n    import pypdfium2 as pdfium\n    from bs4 import BeautifulSoup\n    from surya.settings import settings\n    from surya import SuryaInferenceManager, RecognitionPredictor\n    job = read(job_path)\n    config = job[\'config\']\n    settings.SURYA_INFERENCE_BACKEND = \'llamacpp\'\n    settings.SURYA_INFERENCE_URL = f\'http://127.0.0.1:{port}/v1\'\n    settings.SURYA_INFERENCE_HOST = \'127.0.0.1\'\n    settings.SURYA_INFERENCE_AUTOSTART = False\n    settings.SURYA_INFERENCE_PARALLEL = 1\n    settings.SURYA_INFERENCE_API_KEY = \'local-only\'\n    settings.SURYA_MODEL_CHECKPOINT = \'surya-local\'\n    settings.SURYA_INFERENCE_TIMEOUT_SECONDS = config[\'page_timeout_seconds\'] - 30\n    settings.SURYA_MAX_TOKENS_FULL_PAGE = config[\'max_output_tokens\']\n    settings.IMAGE_DPI_HIGHRES = config[\'dpi\']\n    settings.DISABLE_TQDM = True\n    source = safe_path(job[\'source_path\'])\n    if digest(source) != job[\'source_sha256\']:\n        raise ValueError(\'Original checksum changed\')\n    started = time.monotonic()\n    with pdfium.PdfDocument(source) as pdf:\n        page = pdf[job[\'page\'] - 1]\n        bitmap = page.render(scale=config[\'dpi\'] / 72)\n        image = bitmap.to_pil().convert(\'RGB\')\n        bitmap.close()\n        page.close()\n    preview = safe_path(job[\'preview\'])\n    preview.parent.mkdir(parents=True, exist_ok=True)\n    image.save(preview)\n    manager = SuryaInferenceManager(method=\'llamacpp\')\n    try:\n        prediction = RecognitionPredictor(manager)([image], full_page=True)[0].model_dump(mode=\'json\')\n    finally:\n        manager.stop()\n        image.close()\n    text = \'\\n\\n\'.join((BeautifulSoup(b.get(\'html\', \'\'), \'html.parser\').get_text(\' \', strip=True) for b in prediction[\'blocks\']))\n    errors = any((b.get(\'error\') for b in prediction[\'blocks\']))\n    result = {**{k: job[k] for k in (\'source_id\', \'source_path\', \'source_sha256\', \'page\', \'document_pages\', \'profile\')}, \'created_at\': now(), \'elapsed_seconds\': round(time.monotonic() - started, 3), \'review_status\': \'unreviewed\', \'classification\': \'unknown\', \'extraction_method\': \'machine_ocr\', \'status\': \'needs_review\' if errors or not text.strip() else \'extracted\', \'uncertainty\': \'All OCR is unverified; names, numbers, shares, stamps and handwriting require comparison with the original.\', \'text\': text, \'ocr\': prediction, \'image_path\': job[\'preview\'], \'config\': config}\n    if digest(source) != job[\'source_sha256\']:\n        raise ValueError(\'Original checksum changed during OCR\')\n    target = safe_path(job[\'result\'])\n    if target.exists():\n        raise FileExistsError(\'Refusing to overwrite completed extraction\')\n    write(target, result)\n    target.with_suffix(\'.txt\').write_text(text, encoding=\'utf-8\')\nif __name__ == "__main__":\n    import sys\n    worker(sys.argv[1],int(sys.argv[2]))\n'
WORKER=WORK/'page_worker.py'
WORKER.write_text(WORKER_SOURCE,encoding='utf-8')
BY_ID={d['source_id']:d for d in DOCUMENTS}
assert len(DOCUMENTS)==1522 and sum(d['document_pages'] for d in DOCUMENTS)==7072
print('الفهرس العام جاهز:',len(DOCUMENTS),'ملفاً')


الفهرس العام جاهز: 1522 ملفاً


## 3 — الحفظ وتنزيل الأوزان
عند تفعيل Drive، يكتب الدفتر إلى مجلد النتائج المسمى أعلاه فقط ولا يقرأ ملفات Drive الأخرى. إذا عطّلت Drive فالنتائج مؤقتة، ويجب تنزيلها قبل انتهاء الجلسة. الملفات الأصلية تُجلب عند الحاجة، ولا تُحمّل جميعها مسبقاً.

In [ ]:
#@title 4. Models and Drive cache | الأوزان والحفظ
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8');os.replace(temp,path)
def fetch(url,path,expected):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        if sha(path)==expected:return path
        raise ValueError('ملف موجود ببصمة مختلفة؛ لم يتم استبداله: '+str(path))
    part=path.with_suffix(path.suffix+'.part')
    print('تنزيل وتحقق:',path.name,flush=True)
    for attempt in range(4):
        offset=part.stat().st_size if part.exists() else 0
        req=urllib.request.Request(url,headers={'Range':f'bytes={offset}-'} if offset else {})
        try:
            with urllib.request.urlopen(req,timeout=60) as response:
                append=bool(offset and response.status==206)
                if append and not response.headers.get('Content-Range','').startswith(f'bytes {offset}-'):raise ValueError('Invalid resume range')
                if not append:offset=0
                expected_size=offset+int(response.headers.get('Content-Length',0))
                last_report=time.monotonic()
                with part.open('ab' if append else 'wb') as f:
                    while chunk:=response.read(1024*1024):
                        f.write(chunk);offset+=len(chunk)
                        if time.monotonic()-last_report>=10:
                            print(f'{path.name}: {offset/1048576:.1f} / {expected_size/1048576:.1f} MiB',flush=True)
                            last_report=time.monotonic()
            if expected_size and part.stat().st_size!=expected_size:raise OSError('Incomplete response')
            if sha(part)!=expected:raise ValueError('بصمة المصدر مختلفة عن الأرشيف المحلي؛ لم يُستخدم الملف')
            os.replace(part,path);return path
        except OSError:
            if attempt==3:raise
            print('انقطع الاتصال؛ إعادة المحاولة',attempt+2,'من 4',flush=True)
            time.sleep(2)
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE=Path('/content/drive/MyDrive')/DRIVE_FOLDER
else:SAVE=WORK/'session-checkpoints'
SAVE.mkdir(parents=True,exist_ok=True)
for asset in MODELS:
    filename=Path(asset['path']).name
    path=WORK/'models'/filename
    # Keep verified weights in Drive too, so a replacement runtime can reuse them.
    cache=SAVE/'models'/filename
    if cache.exists():
        if sha(cache)!=asset['sha256']:raise ValueError('Cached model checksum mismatch')
        if not path.exists():path.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(cache,path)
    fetch(asset['url'],path,asset['sha256'])
    if not cache.exists():cache.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(path,cache)
print('الأوزان متحققة؛ الحفظ في:',SAVE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
الأوزان متحققة؛ الحفظ في: /content/drive/MyDrive/GDCA-OCR-public-archive


## 4 — تشغيل الدفعة
الافتراضي ثلاث صفحات. لزيادة التغطية اختر `MAX_PAGES=27` مع `RUN_FULL=False`. لتشغيل دفعة من الأرشيف بعد مراجعة القياس، غيّر `RUN_FULL=True` وعدّل حجم الدفعة ثم أعد تشغيل خلية الإعدادات وهذه الخلية. يُحفظ كل ناتج قبل الانتقال للصفحة التالية، وتُتجاوز النتائج المكتملة والمتطابقة عند الاستئناف. تنتهي الدفعة عند أول فشل. المقاطعة تحفظ الصفحات السابقة وتغلق محرك OCR؛ لكنها لا تنهي احتساب جلسة Colab نفسها.

In [19]:
#@title 5. Run OCR batch | تشغيل الدفعة
CONFIG={**BASE_CONFIG,'dpi':DPI,'page_timeout_seconds':PAGE_TIMEOUT_SECONDS,'backend':'llamacpp-cuda','threads':4}
PROFILE_INFO={'config':CONFIG,'gpu':GPU_INFO,'llama_commit':LLAMA_COMMIT,'models':MODELS,
              'worker_sha256':hashlib.sha256(WORKER_SOURCE.encode()).hexdigest(),
              'python':sys.version,'packages':PACKAGES,'binary_sha256':sha(BINARY)}
PROFILE=hashlib.sha256(json.dumps(PROFILE_INFO,sort_keys=True).encode()).hexdigest()[:16]
OUT=WORK/'evidence/extracts/gdca-surya'/PROFILE
CHECKPOINT=SAVE/'results'/PROFILE
OUT.mkdir(parents=True,exist_ok=True);CHECKPOINT.mkdir(parents=True,exist_ok=True)
atomic_json(OUT/'profile.json',PROFILE_INFO);atomic_json(CHECKPOINT/'profile.json',PROFILE_INFO)
plan=([{'source_id':d['source_id'],'source_path':d['source_path'],'source_sha256':d['source_sha256'],
        'page':p,'document_pages':d['document_pages']} for d in DOCUMENTS for p in range(1,d['document_pages']+1)] if RUN_FULL else PILOT)
pending=[]
for page in plan:
    key=page['source_id']+f"/page-{page['page']:04}"
    marker=CHECKPOINT/(key+'.complete.json')
    if marker.exists():
        record=json.loads(marker.read_text())
        assert record['source_sha256']==page['source_sha256'] and record['profile']==PROFILE
        for name,expected in record['files'].items():
            candidate=CHECKPOINT/name
            assert candidate.resolve().is_relative_to(CHECKPOINT.resolve())
            assert sha(candidate)==expected,'Checkpoint integrity failure'
        continue
    pending.append((page,key))
print('المنجز سابقاً:',len(plan)-len(pending),'المتبقي:',len(pending),'حجم الدفعة:',min(MAX_PAGES,len(pending)))
server=None
run_started=time.monotonic();measurements=[];failures=[]
env=os.environ.copy()
for key in list(env):
    if key.startswith(('SURYA_','LLAMA_CPP_','VLLM_')) or key.upper().endswith('_PROXY'):env.pop(key,None)
env.update(GDCA_OCR_ROOT=str(WORK),HF_HUB_OFFLINE='1',HF_HUB_DISABLE_TELEMETRY='1',NO_PROXY='127.0.0.1,localhost',PYTHONUTF8='1')
try:
    if pending:
        with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
        cmd=[str(BINARY),'-m',str(WORK/'models'/CONFIG['model_file']),'--mmproj',str(WORK/'models'/CONFIG['projector_file']),
             '--host','127.0.0.1','--port',str(port),'--parallel','1','-ngl','99','--ctx-size',str(CONFIG['context_tokens']),
             '--threads','4','--alias','surya-local','--jinja','--special']
        with (OUT/'server.log').open('ab') as log:server=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env)
        opener=urllib.request.build_opener(urllib.request.ProxyHandler({}));deadline=time.monotonic()+180
        while True:
            if server.poll() is not None:raise RuntimeError('محرك GPU توقف؛ راجع server.log')
            try:
                with opener.open(f'http://127.0.0.1:{port}/health',timeout=2) as response:
                    if response.status==200:break
            except OSError:pass
            if time.monotonic()>deadline:raise TimeoutError('Model startup timeout')
            time.sleep(1)
        gpu_used=subprocess.check_output(['nvidia-smi','--query-compute-apps=pid,used_memory','--format=csv,noheader'],text=True)
        print('استهلاك GPU بعد تحميل النموذج:',gpu_used)
        if str(server.pid) not in gpu_used:raise RuntimeError('لم يتم التأكد من استخدام GPU؛ توقف دون الرجوع إلى CPU')
    for page,key in pending[:MAX_PAGES]:
        if time.monotonic()-run_started>=MAX_RUN_MINUTES*60:break
        if (SAVE/'STOP').exists():print('إيقاف مطلوب بعد الصفحة السابقة');break
        document=BY_ID[page['source_id']]
        assert urllib.parse.urlsplit(document['url']).hostname=='recon.gdca.gov.sy'
        target=OUT/(key+'.json')
        print('معالجة',page['source_id'],'الصفحة',page['page'],flush=True)
        task_start=time.monotonic()
        try:
            source=fetch(document['url'],WORK/page['source_path'],page['source_sha256'])
            job={**page,'profile':PROFILE,'config':CONFIG,'result':str(target.relative_to(WORK)),
                 'preview':str(target.with_suffix('.png').relative_to(WORK))}
            jobfile=WORK/'job.json';atomic_json(jobfile,job)
            # A result without a checkpoint marker may be from an interrupted copy.
            # Validate and checkpoint it; never overwrite it blindly.
            if not target.exists():
                with (OUT/'worker.log').open('ab') as log:
                    subprocess.run([sys.executable,str(WORKER),str(jobfile),str(port)],check=True,
                                   timeout=PAGE_TIMEOUT_SECONDS,stdout=log,stderr=subprocess.STDOUT,env=env)
            result=json.loads(target.read_text())
            assert result['source_sha256']==page['source_sha256'] and result['profile']==PROFILE
            assert sha(source)==page['source_sha256']
            assets={}
            for suffix in ['.json','.txt']+(['.png'] if SAVE_PREVIEWS else []):
                origin=target.with_suffix(suffix)
                if suffix=='.txt' and not origin.exists():origin.write_text(result['text'],encoding='utf-8')
                dest=CHECKPOINT/(key+suffix);dest.parent.mkdir(parents=True,exist_ok=True)
                temp=dest.with_suffix(dest.suffix+'.tmp');shutil.copyfile(origin,temp);os.replace(temp,dest)
                assets[key+suffix]=sha(dest)
            record={**page,'profile':PROFILE,'status':result['status'],'review_status':'unreviewed','files':assets,
                    'elapsed_seconds':result['elapsed_seconds'],'wall_seconds':time.monotonic()-task_start}
            atomic_json(CHECKPOINT/(key+'.complete.json'),record)
            measurements.append(record)
            atomic_json(CHECKPOINT/'progress.json',{'planned':len(plan),'completed_total':len(plan)-len(pending)+len(measurements),
                        'latest':record,'review_status':'unreviewed'})
            print('حُفظت الصفحة؛ زمن OCR:',round(result['elapsed_seconds'],1),'ثانية',flush=True)
            if result['status']!='extracted':raise RuntimeError('الصفحة تحتاج مراجعة تقنية قبل المتابعة')
        except Exception as exc:
            failures.append({**page,'error':str(exc)})
            print('توقفت الدفعة:',str(exc));break
finally:
    if server is not None:
        server.terminate()
        try:server.wait(timeout=15)
        except subprocess.TimeoutExpired:server.kill();server.wait()
    SUMMARY={'profile':PROFILE,'mode':'full-batch' if RUN_FULL else 'pilot','gpu':GPU_INFO,'planned':len(plan),
             'new_pages':len(measurements),'results':measurements,'failures':failures,
             'wall_seconds':time.monotonic()-run_started,'review_status':'unreviewed'}
    atomic_json(CHECKPOINT/'last-run.json',SUMMARY)
    for name in ['server.log','worker.log']:
        if (OUT/name).exists():shutil.copyfile(OUT/name,CHECKPOINT/name)
print('انتهت الدفعة. الخادم توقف؛ افصل جلسة Colab عند انتهاء العمل لتجنب استمرار استهلاك الرصيد.')


المنجز سابقاً: 6 المتبقي: 7066 حجم الدفعة: 7066
استهلاك GPU بعد تحميل النموذج: 30796, 1444 MiB

معالجة GDCA-F-bd52dab60901e8f15f72 الصفحة 4
حُفظت الصفحة؛ زمن OCR: 21.1 ثانية
معالجة GDCA-F-f2de3362e20d8837cc50 الصفحة 1
تنزيل وتحقق: f2de3362e20d8837cc50_665.pdf
حُفظت الصفحة؛ زمن OCR: 18.7 ثانية
معالجة GDCA-F-f2de3362e20d8837cc50 الصفحة 2
حُفظت الصفحة؛ زمن OCR: 14.7 ثانية
معالجة GDCA-F-f2de3362e20d8837cc50 الصفحة 3
حُفظت الصفحة؛ زمن OCR: 17.9 ثانية
معالجة GDCA-F-f2de3362e20d8837cc50 الصفحة 4
حُفظت الصفحة؛ زمن OCR: 13.6 ثانية
معالجة GDCA-F-f2de3362e20d8837cc50 الصفحة 5
حُفظت الصفحة؛ زمن OCR: 24.2 ثانية
معالجة GDCA-F-f2de3362e20d8837cc50 الصفحة 6
حُفظت الصفحة؛ زمن OCR: 30.5 ثانية
معالجة GDCA-F-f92a4b22a10eea59b585 الصفحة 1
تنزيل وتحقق: f92a4b22a10eea59b585_638.pdf
حُفظت الصفحة؛ زمن OCR: 23.4 ثانية
معالجة GDCA-F-f92a4b22a10eea59b585 الصفحة 2
حُفظت الصفحة؛ زمن OCR: 18.5 ثانية
معالجة GDCA-F-f92a4b22a10eea59b585 الصفحة 3
حُفظت الصفحة؛ زمن OCR: 18.2 ثانية
معالجة GDCA-F-f92a4b22a10eea59b585 الصفح

## 5 — قياس الوقت وتنزيل النتائج
التقدير حسابي من الصفحات المنجزة فقط؛ لا يشمل تكلفة بناء المحرك أو كل التنزيلات والانقطاعات أو المراجعة البشرية. لا يحسب الدفتر رصيد حسابك أو سعر الـGPU. راجع ذلك من واجهة Colab.

In [20]:
#@title 6. Benchmark and preview | القياس والمعاينة
times=[r['elapsed_seconds'] for r in SUMMARY['results'] if r['status']=='extracted']
if times:
    mean=statistics.mean(times)
    print('عدد الصفحات المقاسة:',len(times),'المتوسط:',round(mean,2),'ثانية/صفحة')
    print('إسقاط حسابي لـ7072 صفحة:',round(mean*7072/3600,2),'ساعة OCR، وليس وعداً بالمدة')
    first=SUMMARY['results'][0]
    result=json.loads((CHECKPOINT/(first['source_id']+f"/page-{first['page']:04}.json")).read_text())
    print('معاينة نص غير مراجَع:\n',result['text'][:1500])
else:print('لا توجد صفحات ناجحة تكفي لحساب تقدير. راجع last-run.json وworker.log')


عدد الصفحات المقاسة: 1921 المتوسط: 20.94 ثانية/صفحة
إسقاط حسابي لـ7072 صفحة: 41.13 ساعة OCR، وليس وعداً بالمدة
معاينة نص غير مراجَع:
 رقم الصفحة : ٩ مكرر رقم العقار : ١٥/٥٥٩ النوع الشرعي : أميري المساحة : رقم عقد التملك اسماء المالكين الحصة السهمية سبب التملك بموجب العقد رقم ٥٦٨١ تاريخ ١٩٩٧/١٢/٢٩ ٣٩٥٥ عصام بن خليل دياب الكامل افرازاً و اختصاصاً صححت الأوصاف و أصبحت كما يلي : ١٩٩٠/١٢/١٠ ١٥ / دكان في الطابق الأرضي بناؤها من حجر و اسمنت مسلح تحتوي على مرحاض و مغسلة و سقيفة و ملجأ في الطلبق تحت الأرضي مشترك بين جميع المقاسم الإشارات : حقوق الارتفاق النسخ المسلمة ( السندات ) ٥٨٨ /٣/١٩٩٨ دعوى على صحيفة العقار لمصلحة رجب ديوب حقوق الارتفاق الطبيعية ١٣٧٨ /٤/١٩٩٣ أعطي على الصفحة الأصلية سند عصام دياب

أساس التسجيل : سجل هذا العقار افرازاً طابقياً بالعقد ٣٩٥٥ تاريخ ١٩٩٠/١٢/١٠

رقم الصفحة : ١٠ رقم العقار : ١٦/٥٥٩ النوع الشرعي : أميري المساحة : رقم عقد التملك اسماء المالكين الحصة السهمية سبب التملك بموجب العقد رقم ٥٦٨١ تاريخ ١٩٩٧/١٢/٢٩ ٤٩٧٧ محمد هشام بن محمود سلام الكامل هبة صححت الأوصاف و أصبحت ك

In [21]:
#@title 7. Export results | تنزيل النتائج
# تنزيل ملف النتائج إلى اللابتوب. لا يحوي نماذج أو أصول PDF.
from google.colab import files
archive=shutil.make_archive(str(WORK/('gdca-surya-'+PROFILE)),'zip',root_dir=CHECKPOINT)
files.download(archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## بعد التجربة

- افتح **Runtime → Disconnect and delete runtime** عندما تنتهي؛ إغلاق محرك OCR وحده لا ينهي جلسة GPU المدفوعة.
- احتفظ بملف ZIP وأعده إلى المشروع للمراجعة واستيراد النتائج تحت `evidence/extracts`، دون استبدال الوثائق الأصلية.
- للاستئناف على جلسة جديدة، شغّل الخلايا بنفس الإعدادات ونوع GPU؛ اختلاف المحرك أو الحزم أو الجهاز ينشئ profile جديداً حتى لا تختلط التجارب.
- إذا تعذر جلب PDF أو اختلفت بصمته، تتوقف الدفعة. لا يُستبدل تلقائياً بنسخة أحدث؛ يمكن تجهيز حزمة من الأصول المحلية المطابقة لاحقاً.
- أدوات الإيقاف: مقاطعة خلية التشغيل، أو إنشاء ملف `STOP` في مجلد الحفظ لتوقف الدفعة بين الصفحات. أعد تسميته قبل الاستئناف.

المصادر التقنية: [Surya](https://github.com/datalab-to/surya)، [llama.cpp CUDA](https://github.com/ggml-org/llama.cpp/blob/b12403/docs/build.md#cuda)، [حدود Colab](https://research.google.com/colaboratory/faq.html#resource-limits).


## 6 — تشخيص اختياري عند توقف خطوة
يعرض حالة GPU والمساحة وآخر سجل للمحرك. لا يعيد تشغيل التثبيت أو المعالجة.

In [22]:
#@title 8. Diagnostics | تشخيص اختياري
import subprocess, shutil
from pathlib import Path
print('Free local disk GiB:',round(shutil.disk_usage('/content').free/2**30,1))
subprocess.run(['nvidia-smi'],check=False,timeout=30)
for name in ['server.log','worker.log']:
    root=globals().get('OUT')
    path=Path(root)/name if root else None
    if path and path.exists():
        print('---',name,'---')
        print('\n'.join(path.read_text(errors='replace').splitlines()[-30:]))


Free local disk GiB: 220.6
--- server.log ---
719.21.102.882 I slot print_timing: id  0 | task 4065181 |    graphs reused =    4040806
719.21.102.883 I slot print_timing: id  0 | task 4065181 |   graph reallocs =          0    /     0        (same size) / (changed size)
719.21.103.792 I slot      release: id  0 | task 4065181 | stop processing: n_tokens = 6302, truncated = 0
719.26.089.978 I slot get_availabl: id  0 | task -1 | selected slot by LRU, n_tokens = 6302, t_last = 48933777468
719.26.090.460 W srv         alloc:  - making room for prompt cache entry, removing oldest entry (size = 118.211 MiB)
719.26.141.918 I slot launch_slot_: id  0 | task 4067423 | processing task, is_child = 0
719.31.836.751 I slot print_timing: id  0 | task 4067423 | n_gen =    450, tg = 149.44 t/s, tg_3s = 149.71 t/s
719.34.842.512 I slot print_timing: id  0 | task 4067423 | n_gen =    918, tg = 152.56 t/s, tg_3s = 155.70 t/s
719.37.845.113 I slot print_timing: id  0 | task 4067423 | n_gen =   1389, tg =